# Autonomous MV/LV voltage-management notebook
## Single-file BFM–SOCP identification, local inverter synthesis and autonomous-SVR evaluation

This is the self-contained version of V4.9. It does not require
`opf_mvlv_core_v4_9.py` or any manually uploaded Python source file. Network
data and meteorological profiles are downloaded automatically, while the model
implementation is embedded below in collapsible cells.

For a fresh experiment, edit only the short `CFG` block and execute the cells
in order. After a Colab interruption, keep the same `campaign_id`, rerun the
setup and embedded-engine cells, and continue from Section 14; completed
scenario/day checkpoints are not repeated.


### Scenarios evaluated under the same autonomous phase-wise LDC

| Scenario | Inverter operation | Purpose |
|---|---|---|
| **S1** | all inverters at unity PF | physical autonomous-LDC baseline |
| **S2** | all inverters at capacitive PF = 0.96 | uniform-support counterfactual |
| **S3-RL** | sparse A/B/C settings selected through contextual Q-learning | proposed implementation |
| **S3-direct** | the same rewards, but $a_g=\arg\max_a R_g(a)$ | classifier ablation |
| **S4** | individualized OPF-derived Volt--VAR curve at every inverter | published-style non-sparse baseline |

S1--S4 use the same network, voltage bounds, inverter capability, LDC logic,
initial state and exact fixed-tap SVR relation.  Therefore, differences are due
to inverter settings and their spatial selection, rather than to an imposed tap
trajectory.


### Workflow and stage contract

| Stage | Essential input | Essential output |
|---|---|---|
| Data and profiles | MV/LV spreadsheets; 10-min irradiance and ambient temperature | validated radial network and 144-sample profiles |
| E1 identification | BFM--SOCP with relaxed tap integrality and relaxed policy linkage for $Q_{\rm pv}$ | continuous tap trajectory and $Q^\star(t)$ |
| E2 identification | causal integer projection of E1 taps | discrete identification candidate |
| Physical $Q^\star$ reconstruction | fixed E2 taps and $Q^\star$ | physically closed $(V,Q^\star)$ points |
| Device synthesis | hourly medians of 10-min points | A/B/C settings and all-inverter S4 curves |
| Closed-loop evaluation | fixed local settings and native LDC | exact-SVR S1--S4 metrics |
| Multiday transfer | frozen selected settings and consecutive days | weekly taps, DRP, DRC and $\mathrm{FD}_{95}$ |

`Q*` is used only as a training/identification target.  It is not dispatched in
the final scenarios.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import copy, gc, hashlib, importlib.util, json, math, os, subprocess, sys, time

# Change only this block for a new experiment.
CFG = SimpleNamespace(
    source_date_in_json="15/02/2019",
    physical_year=2019,
    output_root="/content/drive/MyDrive/ACOPF/public_v4_9",
    campaign_id="public_v4_9_summer_source_2019_02_15",
    cplex_executable=(
        "/opt/ibm/ILOG/CPLEX_Studio2212/cplex/bin/x86-64_linux/cplex"),
    cplex_time_limit_e1_s=600,
    cplex_time_limit_e2_s=1200,
    cplex_threads=4,
    run_heavy_identification=True,
    run_sparse_refinement=True,
    run_multiday=True,
    multiday_scenarios=("S1", "S3_RL"),
    colab_session_budget_s=8*3600 + 30*60,
    kmeans_seed=42,
)

OUTPUT = Path(CFG.output_root)
RUN_ID = CFG.campaign_id
RUN_DIR = OUTPUT / RUN_ID
print(json.dumps(vars(CFG), indent=2, ensure_ascii=False, default=str))


## 1. Explicit environment preparation

Importing the model does **not** mount Google Drive, install software or accept
a solver license.  Those actions are explicit below.  IBM CPLEX must be used
under a valid license.  Set `INSTALL_CPLEX_FROM_DRIVE=True` only when the
installer belongs to the user and its terms have already been accepted.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
INSTALL_CPLEX_FROM_DRIVE = False

required = ["pyomo", "pandas", "numpy", "openpyxl", "matplotlib", "psutil"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        check=True,
    )

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive", force_remount=False)

OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0], "| Colab:", IN_COLAB)


In [ ]:
WORK = Path("/content/acopf_public_v4_9") if IN_COLAB else Path.cwd()/"_public_run"
INPUT_REPO = WORK/"ACOPF_input"
WORK.mkdir(parents=True, exist_ok=True)

if not (INPUT_REPO/"data"/"network"/"buses_1.xlsx").is_file():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/DarleneJD/ACOPF.git", str(INPUT_REPO),
    ], check=True)

DATA_DIR = INPUT_REPO/"data"
NETWORK_DIR = DATA_DIR/"network"
METEOROLOGY_DIR = DATA_DIR/"meteorology"
PROFILES_DIR = METEOROLOGY_DIR/"profiles_10min"
representative_json = METEOROLOGY_DIR/"dicionario_8_dias_representativos.json"
if not representative_json.is_file():
    raise FileNotFoundError(f"JSON representativo não encontrado: {representative_json}")

raw_days = json.loads(representative_json.read_text(encoding="utf-8"))
flat_dates = [date for season in raw_days.values() for date in season]

def representative_day_index(requested_date, available_dates):
    from datetime import datetime
    if requested_date in available_dates:
        return available_dates.index(requested_date)
    requested = datetime.strptime(requested_date, "%d/%m/%Y")
    matches = []
    for index, value in enumerate(available_dates):
        parsed = datetime.strptime(value, "%d/%m/%Y")
        if (parsed.day, parsed.month) == (requested.day, requested.month):
            matches.append(index)
    if len(matches) != 1:
        raise ValueError(f"Não foi possível resolver {requested_date!r}: {available_dates}")
    return matches[0]

source_index = representative_day_index(CFG.source_date_in_json, flat_dates)
RESOLVED_SOURCE_DATE = flat_dates[source_index]
os.environ.update({
    "OPF_REPRESENTATIVE_DAYS_JSON": str(representative_json),
    "OPF_REPRESENTATIVE_DAY_INDEX": str(source_index),
    "OPF_AUTO_SETUP_ENVIRONMENT": "0",
    "OPF_RUN_8_DAY_CAMPAIGN": "0",
})

BUSES_FILE = NETWORK_DIR/"buses_1.xlsx"
BRANCHES_FILE = NETWORK_DIR/"branches_1.xlsx"
SOURCE_COMMIT = subprocess.run(
    ["git", "-C", str(INPUT_REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("Inputs prepared at", INPUT_REPO)
print("Representative day resolved as", RESOLVED_SOURCE_DATE)


## Embedded model implementation

The following cells contain the complete implementation required by this
notebook. They are split only between top-level definitions and may be kept
collapsed during normal use. The mathematical sections that follow map the
published equations to the corresponding Pyomo constraints.


In [ ]:
#@title Internal model implementation 1/12 { display-mode: "form" }
"""intrahorario2.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1tx-9gVOCC_hhiyfUnZupKF8LoYEPtYN2
"""

# -*- coding: utf-8 -*-
"""CODEX-OPF_SOC_CPLEX.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/18y08IXF2DcU2hNzWqHZYmLqcj2CbcmDu

# Ambiente
"""

import os
import shutil
import subprocess
import sys
import importlib.util
from pathlib import Path


VERSION = 'V4.9-PUBLIC-NOTEBOOK-S1S2S3S4-EXACT-SVR-ABLATION-MULTIDAY'
RUNTIME_SIGNATURE = 'V4.9_EXACT_FIXED_TAP_SVR_S4_ABLATION_STATEFUL_MULTIDAY_20260918'

DRIVE_ROOT = Path('/content/drive')
CPLEX_INSTALLER_DRIVE = (
    DRIVE_ROOT / 'MyDrive' / 'Solvers' /
    'cplex_studio2212.linux_x86_64.bin')
CPLEX_INSTALLER_LOCAL = Path('/content/cplex_studio2212.linux_x86_64.bin')
CPLEX_INSTALL_DIR = Path('/opt/ibm/ILOG/CPLEX_Studio2212')
CPLEX_EXE = str(
    CPLEX_INSTALL_DIR / 'cplex' / 'bin' / 'x86-64_linux' / 'cplex')


def ensure_drive_and_cplex():
    """Monta o Drive e instala o CPLEX silenciosamente somente se necessário.

    A primeira autorização do Google Drive ainda depende da confirmação de
    segurança da conta Google. Depois de autorizada, as reexecuções ficam
    automáticas e a instalação é ignorada quando CPLEX_EXE já existe.
    """
    try:
        from google.colab import drive
    except ImportError as exc:
        if not Path(CPLEX_EXE).is_file():
            raise RuntimeError(
                'Execução fora do Colab e CPLEX não encontrado em '
                f'{CPLEX_EXE}.') from exc
    else:
        if not (DRIVE_ROOT / 'MyDrive').is_dir():
            print('[ambiente] Montando Google Drive...')
            drive.mount(str(DRIVE_ROOT), force_remount=False)
        else:
            print('[ambiente] Google Drive já está montado.')

    cplex_exe = Path(CPLEX_EXE)
    if cplex_exe.is_file() and os.access(cplex_exe, os.X_OK):
        print(f'[ambiente] CPLEX já instalado: {cplex_exe}')
    else:
        if not CPLEX_INSTALLER_DRIVE.is_file():
            raise FileNotFoundError(
                'Instalador do CPLEX não encontrado no Drive: '
                f'{CPLEX_INSTALLER_DRIVE}')

        print('[ambiente] Copiando instalador do CPLEX do Drive...')
        shutil.copy2(CPLEX_INSTALLER_DRIVE, CPLEX_INSTALLER_LOCAL)
        CPLEX_INSTALLER_LOCAL.chmod(0o755)

        # Propriedades oficiais do InstallAnywhere para instalação silenciosa.
        # LICENSE_ACCEPTED=true pressupõe que o usuário possui licença válida e
        # aceita os termos IBM ao executar este notebook.
        response_file = Path('/content/cplex_silent.properties')
        response_file.write_text(
            '\n'.join([
                'INSTALLER_UI=silent',
                'LICENSE_ACCEPTED=true',
                'USER_INPUT_SEGMENT_TRUE=0',
                'USER_INPUT_SEGMENT_FALSE=1',
                f'USER_INSTALL_DIR={CPLEX_INSTALL_DIR}',
                'INSTALLER_LOCALE=en',
                'CPLEX_STUDIO_README=0',
                'CPLEX_STUDIO_IDE=0',
                '',
            ]),
            encoding='utf-8')

        print('[ambiente] Instalando CPLEX 22.1.2 em modo silencioso...')
        subprocess.run(
            [str(CPLEX_INSTALLER_LOCAL), '-f', str(response_file)],
            check=True)
        if not cplex_exe.is_file():
            raise RuntimeError(
                'O instalador terminou, mas o executável do CPLEX não foi '
                f'encontrado em {cplex_exe}.')
        cplex_exe.chmod(cplex_exe.stat().st_mode | 0o111)
        response_file.unlink(missing_ok=True)
        CPLEX_INSTALLER_LOCAL.unlink(missing_ok=True)
        print(f'[ambiente] CPLEX instalado com sucesso: {cplex_exe}')

    cplex_bin = str(cplex_exe.parent)
    path_entries = os.environ.get('PATH', '').split(os.pathsep)
    if cplex_bin not in path_entries:
        os.environ['PATH'] = cplex_bin + os.pathsep + os.environ.get('PATH', '')
    return str(cplex_exe)


AUTO_SETUP_ENVIRONMENT = os.environ.get('OPF_AUTO_SETUP_ENVIRONMENT', '0') == '1'
if AUTO_SETUP_ENVIRONMENT:
    CPLEX_EXE = ensure_drive_and_cplex()
else:
    # O notebook público possui uma célula explícita de preparação. Importar o
    # módulo não deve montar Drive, instalar software nem aceitar licença.
    CPLEX_EXE = os.environ.get('CPLEX_EXE', CPLEX_EXE)


def ensure_python_dependencies():
    """Instala silenciosamente apenas os pacotes Python ausentes no Colab."""
    required = {
        'pyomo': 'pyomo',
        'pandas': 'pandas',
        'numpy': 'numpy',
        'openpyxl': 'openpyxl',
        'matplotlib': 'matplotlib',
    }
    missing = [package for module, package in required.items()
               if importlib.util.find_spec(module) is None]
    if not missing:
        print('[ambiente] Dependências Python já instaladas.')
        return
    print('[ambiente] Instalando dependências Python: ' + ', '.join(missing))
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *missing],
        check=True)


if AUTO_SETUP_ENVIRONMENT:
    ensure_python_dependencies()

# -*- coding: utf-8 -*-
"""
================================================================================
  OPF BFM-SOCP V3.9.3-ESTÁTICO-MULTIMODO — controles diários implementáveis
  Sistema com topologia análoga a alimentador radial | 144 × 10 min
================================================================================
  Formulação:
    - Branch Flow Model (Baran & Wu 1989; Farivar & Low 2013)
    - Aproximação cônica sucessiva: l_sq * V̄_i² >= P_ij² + Q_ij²
    - Refinamento até V̄_i≈V_i e auditoria ex-post de l_sq·V_i²=P_ij²+Q_ij²
    - Perdas Joule e de núcleo no balanço; núcleo é uma das quatro metas
    - Tap SVR discreto (MISOCP): tap_pos∈{0..32}, tap_pu∈[0.9000,1.1000] p.u.
    - Cada atuação percorre um degrau; a F.O. penaliza Σ|tap(t)-tap(t-1)|
    - Classificador por aprendizagem por reforço contextual: cada equipamento
      físico recebe FP fixo, Q constante, FP=1,00 ou Volt-VAR estático
    - FP é escolhido em {0,85; 0,88; 0,90; 0,92; 0,95; 0,96; 0,97; 0,98; 0,99}
      e a direção capacitiva/indutiva é inferida do despacho ótimo Q*
    - Q constante é escolhido em degraus de 0,05*Snom até ±0,44*Snom
    - O modo selecionado é único nas fases de um mesmo inversor trifásico
    - Filtro físico/elétrico obrigatório antes da seleção operacional
    - F.O. normalizada: 0,55 taps + 0,20 desvio + 0,10 desequilíbrio
      + 0,15 perdas no núcleo; Joule, margem, Q e corte são salvaguardas
    - Mantém a busca V3.7.4 de curvas Volt-VAR estáticas e trajetórias
      vizinhas de 31 taps da baseline LDC física
    - Diferença da V3.7.5: pré-triagem sem solver, catálogo sem modelos/Q(t),
      coleta imediata de clones e reexecução física apenas do vencedor
    - Mantém a correção V3.9.1: a mistura amortecida também é projetada no
      limite de rampa, e os relatórios separam FP fixo de Q constante
    - Mantém a V3.9.2: se o Grupo B inicial estiver vazio, a triagem usa A/C
      provisoriamente e testa até duas curvas em rollouts individuais/cumulativos
      A/C→B; a classificação só muda após replay físico e dominância ex-post
    - Diferença da V3.9.2: todos os equipamentos continuam testando A/B/C sem
      cotas, mas a classificação e a triagem incorporam distância elétrica
      Σ|Z| e sensibilidade Σ|X|; a correlação distância×|Q| é auditada ex-post
    - Arquivo único para Colab, sem memória de otimização entre períodos;
      preserva o OPF diário da V3.7.6 e amplia o Grupo A para controles estáticos
      não-Volt-VAR (família de FP fixo e Q constante)
    - Dominância ex-post: o estado aplicado deve reduzir simultaneamente as
      quatro metas vs. baseline LDC, inclusive depois da validação híbrida;
      caso contrário, o arquivo operacional é bloqueado como recomendação
    - Reparo E2: se a projeção inteira primária for inviável, testa trajetórias
      causais vizinhas; SCA utilizável segue para replay físico obrigatório
    - Relaxação SOCP com folga nunca é chamada de fluxo: Q* e taps são apenas
      candidatos até o replay fechar l·V²=P²+Q²
    - Regra de materialidade e não ação: ganhos marginais não mudam settings
    - Fronteira de Pareto: taps + desvio + desequilíbrio + núcleo
    - Salvaguarda: perdas técnicas totais não podem piorar materialmente
    - Limitação de rampa do comando Q para evitar saltos impraticáveis
    - Solver: CPLEX em subproblemas QCP convexos sucessivos

  Solver:
    opt = SolverFactory('cplex',
              executable='/opt/ibm/ILOG/CPLEX_Studio2212/cplex/bin/x86-64_linux/cplex')
    results = opt.solve(model, tee=True).write()
================================================================================
"""

import gc, math, sys, time
import pyomo.environ as pyo

try:
    import pandas as pd
    import numpy as np
except ImportError:
    sys.exit("pip install pandas openpyxl numpy")

# ══════════════════════════════════════════════════════════════════════════════
# PARÂMETROS GLOBAIS
# ══════════════════════════════════════════════════════════════════════════════

DT_MIN     = 10                      # minutos por período
N_PERIODS  = 144                     # 24h × 60min / 10min
HOURS      = list(range(N_PERIODS))

T_VV_START = 42                      # 07:00
T_VV_END   = 108                     # 18:00

# ═══════════════════════════════════════════════════════════════════════════
# PERFIS TEMPORAIS — 144 pontos diretos (Δt = 10 min, 24h)
# ═══════════════════════════════════════════════════════════════════════════
# Todos os perfis vêm de medições reais com resolução de 10 min, dispensando
# a etapa de interpolação. Os valores são fatores adimensionais em [0,1] que
# multiplicam os valores nominais declarados na planilha (kW, kVAr).
#
# Fontes:
#   TECHNICH BRASIL — curva de carga industrial/comercial MT característica
#                      (baseload elevado, range ≈ 0,71–1,00, pico às 16-17h)
#   curvaR          — curva de carga residencial BT (pico vespertino,
#                      range ≈ 0,19–1,00, máximo às 18-19h)
#   temp            — temperatura ambiente [°C] (perfil diário medido)
#   irrad           — irradiância global normalizada [W/m²/STC]
# ═══════════════════════════════════════════════════════════════════════════

# Curva TECHNICH BRASIL — cargas MT (perfil industrial/comercial real)
TECHNICHDOBRASIL = [
    0.909, 0.909, 0.909, 0.909, 0.909, 0.909,  # 00:00–01:00
    0.721, 0.721, 0.721, 0.721, 0.721, 0.721,  # 01:00–02:00
    0.714, 0.714, 0.714, 0.714, 0.714, 0.714,  # 02:00–03:00
    0.721, 0.721, 0.721, 0.721, 0.721, 0.721,  # 03:00–04:00
    0.718, 0.718, 0.718, 0.718, 0.718, 0.718,  # 04:00–05:00
    0.755, 0.755, 0.755, 0.755, 0.755, 0.755,  # 05:00–06:00
    0.761, 0.761, 0.761, 0.761, 0.761, 0.761,  # 06:00–07:00
    0.934, 0.934, 0.934, 0.934, 0.934, 0.934,  # 07:00–08:00
    1.000, 1.000, 1.000, 1.000, 1.000, 1.000,  # 08:00–09:00
    0.986, 0.986, 0.986, 0.986, 0.986, 0.986,  # 09:00–10:00
    0.992, 0.992, 0.992, 0.992, 0.992, 0.992,  # 10:00–11:00
    0.883, 0.883, 0.883, 0.883, 0.883, 0.883,  # 11:00–12:00
    0.783, 0.783, 0.783, 0.783, 0.783, 0.783,  # 12:00–13:00
    0.780, 0.780, 0.780, 0.780, 0.780, 0.780,  # 13:00–14:00
    0.934, 0.934, 0.934, 0.934, 0.934, 0.934,  # 14:00–15:00
    0.919, 0.919, 0.919, 0.919, 0.919, 0.919,  # 15:00–16:00
    0.738, 0.738, 0.738, 0.738, 0.738, 0.738,  # 16:00–17:00
    0.893, 0.893, 0.893, 0.893, 0.893, 0.893,  # 17:00–18:00
    0.889, 0.889, 0.889, 0.889, 0.889, 0.889,  # 18:00–19:00
    0.898, 0.898, 0.898, 0.898, 0.898, 0.898,  # 19:00–20:00
    0.885, 0.885, 0.885, 0.885, 0.885, 0.885,  # 20:00–21:00
    0.792, 0.792, 0.792, 0.792, 0.792, 0.792,  # 21:00–22:00
    0.817, 0.817, 0.817, 0.817, 0.817, 0.817,  # 22:00–23:00
    0.926, 0.926, 0.926, 0.926, 0.926, 0.926,  # 23:00–24:00
]
assert len(TECHNICHDOBRASIL) == N_PERIODS, "TECHNICHDOBRASIL deve ter 144 pontos"

# Curva R — cargas BT residenciais (pico vespertino)
CURVA_R = [
    0.289, 0.307, 0.319, 0.330, 0.342, 0.362,  # 00:00–01:00
    0.206, 0.223, 0.235, 0.246, 0.258, 0.278,  # 01:00–02:00
    0.190, 0.207, 0.219, 0.230, 0.242, 0.262,  # 02:00–03:00
    0.185, 0.202, 0.214, 0.225, 0.237, 0.257,  # 03:00–04:00
    0.190, 0.207, 0.219, 0.230, 0.242, 0.262,  # 04:00–05:00
    0.378, 0.396, 0.408, 0.419, 0.431, 0.451,  # 05:00–06:00
    0.488, 0.506, 0.518, 0.529, 0.541, 0.560,  # 06:00–07:00
    0.498, 0.516, 0.528, 0.539, 0.552, 0.571,  # 07:00–08:00
    0.493, 0.511, 0.523, 0.534, 0.546, 0.566,  # 08:00–09:00
    0.383, 0.401, 0.413, 0.424, 0.436, 0.456,  # 09:00–10:00
    0.415, 0.433, 0.444, 0.455, 0.468, 0.487,  # 10:00–11:00
    0.504, 0.522, 0.534, 0.545, 0.557, 0.576,  # 11:00–12:00
    0.472, 0.490, 0.502, 0.513, 0.525, 0.545,  # 12:00–13:00
    0.378, 0.396, 0.408, 0.419, 0.431, 0.451,  # 13:00–14:00
    0.415, 0.433, 0.444, 0.455, 0.468, 0.487,  # 14:00–15:00
    0.389, 0.406, 0.418, 0.429, 0.442, 0.461,  # 15:00–16:00
    0.425, 0.443, 0.455, 0.466, 0.478, 0.498,  # 16:00–17:00
    0.713, 0.731, 0.743, 0.754, 0.766, 0.785,  # 17:00–18:00
    0.896, 0.914, 0.926, 0.937, 0.950, 0.969,  # 18:00–19:00 PICO
    0.880, 0.899, 0.911, 0.922, 0.934, 0.953,  # 19:00–20:00
    0.928, 0.946, 0.958, 0.969, 0.981, 1.000,  # 20:00–21:00 PICO
    0.786, 0.804, 0.816, 0.827, 0.840, 0.859,  # 21:00–22:00
    0.744, 0.762, 0.775, 0.786, 0.798, 0.817,  # 22:00–23:00
    0.462, 0.480, 0.492, 0.503, 0.515, 0.534,  # 23:00–24:00
]
assert len(CURVA_R) == N_PERIODS, "CURVA_R deve ter 144 pontos"

# Perfis medidos fornecidos em temperatura_2880.txt e irradiancia_2880.txt.
# Irradiância: 2.880 pontos = 30 dias × 96 intervalos de 15 min.
# Temperatura: os primeiros 2.880 dos 23.040 pontos correspondem aos mesmos
# 30 dias (correlação amostra a amostra T×G=0,9713). Sem uma data específica,
# adota-se o perfil médio diário dos 30 dias alinhados.
#
# Os arquivos repetem cada valor horário em quatro intervalos de 15 min. Para
# converter médias de intervalo de 15 para 10 min sem alterar energia/média:
# cada par A,B (30 min) vira A, (A+B)/2, B. No arquivo atual isso equivale a
# repetir cada média horária seis vezes no grid de 10 min.

REPRESENTATIVE_DAYS_URL = (
    'https://raw.githubusercontent.com/DarleneJD/ACOPF/main/data/meteorology/'
    'dicionario_8_dias_representativos.json')
REPRESENTATIVE_DAYS_LOCAL = os.environ.get(
    'OPF_REPRESENTATIVE_DAYS_JSON',
    '/content/dicionario_8_dias_representativos.json')

def _load_representative_day_list(local_path=REPRESENTATIVE_DAYS_LOCAL):
    import json
    from urllib.request import urlopen
    path = Path(local_path)
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        with urlopen(REPRESENTATIVE_DAYS_URL, timeout=60) as response:
            path.write_bytes(response.read())
    raw = json.loads(path.read_text(encoding='utf-8'))
    days = []
    for season, season_days in raw.items():
        for date, samples in season_days.items():
            if len(samples) != N_PERIODS:
                raise ValueError(
                    f'{season}/{date}: esperado {N_PERIODS} pontos; '
                    f'recebido {len(samples)}')
            expected = [f'{10*k//60:02d}:{10*k%60:02d}'
                        for k in range(N_PERIODS)]
            received = [str(row['hora']) for row in samples]
            if received != expected:
                raise ValueError(f'{season}/{date}: grade horária inválida')
            days.append({
                'season': season,
                'date': date,
                'slug': date.replace('/', '-') + '_' + season.lower(),
                'samples': samples,
            })
    if len(days) != 8:
        raise ValueError(
            f'Esperados 8 dias representativos; recebidos {len(days)}')
    return days

REPRESENTATIVE_DAYS = _load_representative_day_list()
SELECTED_DAY_INDEX = int(os.environ.get('OPF_REPRESENTATIVE_DAY_INDEX', '0'))
if not 0 <= SELECTED_DAY_INDEX < len(REPRESENTATIVE_DAYS):
    raise IndexError('OPF_REPRESENTATIVE_DAY_INDEX fora de 0..7')
SELECTED_DAY = REPRESENTATIVE_DAYS[SELECTED_DAY_INDEX]
PROFILE_SOURCE = (
    f"dicionario_8_dias_representativos:{SELECTED_DAY['season']}:"
    f"{SELECTED_DAY['date']}")
PROFILE_DAYS_AVERAGED = 1
TEMP_AMB = [float(row['ambient_temperature_c'])
            for row in SELECTED_DAY['samples']]
PV_TEMP_INPUT_KIND = 'ambient'
assert len(TEMP_AMB) == N_PERIODS, 'TEMP_AMB deve ter 144 pontos'
if PV_TEMP_INPUT_KIND not in ('ambient', 'cell'):
    raise ValueError("PV_TEMP_INPUT_KIND deve ser 'ambient' ou 'cell'")
_temp_finite = all(math.isfinite(_temp) for _temp in TEMP_AMB)
TEMP_AMB_DATA_OK = (_temp_finite and min(TEMP_AMB) >= -20.0 and
                    max(TEMP_AMB) <= 60.0)
PV_TEMP_DATA_OK = (
    TEMP_AMB_DATA_OK if PV_TEMP_INPUT_KIND == 'ambient' else
    (_temp_finite and min(TEMP_AMB) >= -40.0 and max(TEMP_AMB) <= 100.0))
IRRAD = [float(row['irradiance_pu']) for row in SELECTED_DAY['samples']]
assert len(IRRAD) == N_PERIODS, 'IRRAD deve ter 144 pontos'


# ═══════════════════════════════════════════════════════════════════════════
# MODELO PV COM TEMPERATURA DA CÉLULA — NREL/SAM (Sandia)
# ═══════════════════════════════════════════════════════════════════════════
# A potência efetiva do painel PV depende da irradiância E da temperatura:
#
#   T_cell = T_amb + (G_t / G_STC) · (NOCT - 20)         [°C]
#   P_eff  = P_STC · (G_t/G_STC) · [1 + γ·(T_cell - 25)] [kW]
#
# Onde:
#   NOCT = 45°C   (Nominal Operating Cell Temperature, valor típico)
#   γ    = -0.0040 /°C (coef. térmico de potência, típico de Si cristalino)
#   G_STC= 1000 W/m² (irradiância de referência)
#
# A temperatura da célula reduz a potência ativa: para T_cell=60°C,
# a perda é (60-25)·0.004 = 14% em relação ao STC.
# Fonte: Skoplaki & Palyvos (2009); IEC 61853-2
NOCT          = 45.0     # °C — temperatura nominal de operação da célula
GAMMA_PV      = -0.0040  # /°C — coeficiente térmico (Si cristalino típico)
T_STC         = 25.0     # °C — temperatura de referência STC
G_STC_NORM    = 1.0      # irradiância STC normalizada (G_t já vem em p.u.)

def pv_temperature_derate(irr_norm, temp_input_c):
    """Calcula T_cell e fator de derate térmico do painel PV.

    Args:
        irr_norm: irradiância normalizada (G_t/G_STC), em [0,1]
        temp_input_c: temperatura ambiente ou da célula, conforme
                      PV_TEMP_INPUT_KIND [°C]
    Returns:
        (t_cell, derate): T_cell em °C e fator multiplicativo de P_avail
    """
    if PV_TEMP_INPUT_KIND == 'ambient':
        t_cell = temp_input_c + irr_norm * (NOCT - 20.0)
    else:
        t_cell = temp_input_c
    derate = 1.0 + GAMMA_PV * (t_cell - T_STC)
    return t_cell, max(derate, 0.5)    # piso 50% para evitar valores absurdos

# Vetor Pavail_factor por período: P_eff/P_nom = irrad·derate(irrad, T_amb)
PV_EFF_FACTOR = []
PV_T_CELL     = []
for _k in range(N_PERIODS):
    _t_cell, _derate = pv_temperature_derate(IRRAD[_k], TEMP_AMB[_k])
    PV_T_CELL.append(_t_cell)
    PV_EFF_FACTOR.append(IRRAD[_k] * _derate)

# ═══════════════════════════════════════════════════════════════════════════
# MAPEAMENTO DE PERFIS DE CARGA POR BARRA
# ═══════════════════════════════════════════════════════════════════════════
# MT (Load_MT do xlsx) → curva TECHNICH BRASIL
# BT (laterais BT)     → curva R (residencial)
# Pode-se sobrepor por barra específica via _MT_LOAD_PROFILE_MAP se necessário
IRR_PROFILE      = IRRAD              # alias (compatibilidade)
LOAD_PROFILE     = CURVA_R            # default BT — residencial
LOAD_PROFILE_MT  = TECHNICHDOBRASIL   # default MT — technichdobrasil
LOAD_INDUSTRIAL  = TECHNICHDOBRASIL
LOAD_COMERCIAL   = TECHNICHDOBRASIL
LOAD_RESIDENCIAL = CURVA_R

# Mapeamento opcional barra → tipo (vazio = usa default MT/BT por level)
_MT_LOAD_PROFILE_MAP = {}   # vazio: todas as barras MT usam TECHNICHDOBRASIL
_LOAD_BY_TYPE = {
    'industrial':  TECHNICHDOBRASIL,
    'comercial':   TECHNICHDOBRASIL,
    'residencial': CURVA_R,
    'mt_default':  TECHNICHDOBRASIL,
    'bt_default':  CURVA_R,
}

# ═══════════════════════════════════════════════════════════════════════════
# FUNÇÃO OBJETIVO V3.7.6 — quatro metas normalizadas
# ═══════════════════════════════════════════════════════════════════════════
# Os quatro termos primários são divididos por escalas físicas de referência.
# Sem essa normalização, um coeficiente numericamente maior não representa
# necessariamente maior prioridade. O peso de taps é maior que a soma dos
# outros três; a seleção ex-post ainda aplica dominância e prioridade
# lexicográfica, pois uma soma ponderada isolada não garante redução de todos.
OBJ_WEIGHT_TAP = 0.55
OBJ_WEIGHT_VOLTAGE_DEVIATION = 0.20
OBJ_WEIGHT_UNBALANCE = 0.10
OBJ_WEIGHT_CORE = 0.15
if abs(OBJ_WEIGHT_TAP + OBJ_WEIGHT_VOLTAGE_DEVIATION +
       OBJ_WEIGHT_UNBALANCE + OBJ_WEIGHT_CORE - 1.0) > 1.0e-12:
    raise ValueError("Os quatro pesos primários da F.O. devem somar 1")
if OBJ_WEIGHT_TAP <= (OBJ_WEIGHT_VOLTAGE_DEVIATION +
                      OBJ_WEIGHT_UNBALANCE + OBJ_WEIGHT_CORE):
    raise ValueError("O peso de taps deve ser estritamente dominante")

# Salvaguardas operacionais normalizadas. Não substituem as quatro metas.
OBJ_WEIGHT_VOLTAGE_MARGIN = 0.02
OBJ_WEIGHT_JOULE_SAFEGUARD = 0.02
OBJ_WEIGHT_Q_USE_SAFEGUARD = 0.01
OBJ_WEIGHT_Q_RAMP_SAFEGUARD = 0.005
W_RAMP_DIA  = 0.10  # suavização ΔQpv (fora pico solar)
W_RAMP_PICO = 0.01  # suavização ΔQpv (pico solar)
IRR_PICO   = 0.50   # threshold pico solar
OBJ_WEIGHT_CURTAILMENT = 50.0  # último recurso, normalizado por energia disponível

# Referências nominais usadas na F.O. Todas são adimensionais em p.u.
# As barras MT com carga de potência constante (modelo 1) usam 0,95 p.u.;
# as demais barras MT e todas as barras BT usam a referência nominal 1,00 p.u.
MV_VREF_CONST_P_PU = 0.95
MV_VREF_OTHER_PU = 1.00
LV_VREF_PU = 1.00

# Certificação física posterior à classificação. P/Q e taps do ponto de
# equilíbrio são congelados, mas TODAS as tensões não-slack permanecem livres.
# Isto evita sobre-determinar o BFM ao fixar simultaneamente P, Q e V nos
# terminais Volt-VAR. A consistência do Grupo B é auditada pela deriva de
# tensão em relação ao ponto fixo que originou o comando Q(V).
ENABLE_HYBRID_POST_CLASSIFICATION_VALIDATION = True
HYBRID_FIXED_INJECTION_TOL_PU = 1.0e-9
HYBRID_B_EQUILIBRIUM_V_DRIFT_TOL_PU = 5.0e-5

# Baseline S1: resolve-se o BFM físico e reconstrói-se causalmente o LDC até
# que a própria trajetória de taps seja um ponto fixo do fluxo + relé.
BASELINE_LDC_CLOSED_LOOP_MAX_ITER = 12

# Controle
FP_MIN_VV  = 0.85   # fator de potência mínimo (IEEE 1547 Cat B)
QPV_MAX_FRAC_S = 0.44  # envelope Volt-VAR anunciado: |Q|≤0,44·S_nom
# Na descoberta, Q*(t) é limitado apenas pela capacidade aparente. O piso de
# FP volta a ser obrigatório quando a política A/B/C é construída e validada.
DISCOVERY_RELAX_FP_FLOOR = True
VV_FIT_AGGREGATION = 'hourly_median'
VV_PERIODS_PER_HOUR = 6
VV_MIN_POINTS_PER_HOUR = 2
ENABLE_3PH_INVERTER_COUPLING = True
ENABLE_ZERO_CURTAILMENT_TEST = True
FORCE_ZERO_CURTAILMENT_IN_OPERATIONAL_REPLAY = True
ENABLE_RECOMMENDED_SETTINGS_VALIDATION = True
# Validação principal: curva IEEE fixa com trajetórias programadas de tap.
# produzida pelo OPF. O modo 'autonomous_ldc' é um cenário distinto, útil como
# diagnóstico, mas não deve substituir silenciosamente a trajetória candidata.
SETTINGS_TAP_MODE = 'autonomous_ldc'  # S3 final: taps emergem da lógica LDC; OPF é só referência de identificação
SETTINGS_MAX_ITER = 30
SETTINGS_V_TOL = 1.0e-5       # p.u.
SETTINGS_Q_TOL = 1.0e-5       # p.u.
SVR_EXACT_RESIDUAL_TOL = 1.0e-7  # p.u.; somente para replays com tap fixo
SETTINGS_DAMPING = 0.50       # amortecimento do ponto fixo Q(V)
# A busca usa tolerância suficiente para ordenar candidatos, mas tem limite
# próprio. O vencedor volta a ser resolvido com os valores rigorosos acima.
SEARCH_CANDIDATE_MAX_ITER = 16
SEARCH_CANDIDATE_V_TOL = 5.0e-5
SEARCH_CANDIDATE_Q_TOL = 5.0e-5
VALIDATION_V_MIN = 0.80       # bounds numéricos do fluxo ex-post, não PRODIST

# A campanha de dias representativos possui 144 amostras de 10 min. DRP/DRC
# e FD95 calculados nesse horizonte são usados apenas como triagem comparativa
# S1--S2--S3. A avaliação regulatória formal é reservada à sequência semanal
# de 1008 amostras e não é inferida a partir de um único dia representativo.
FORMAL_REGULATORY_WINDOW_PERIODS = 1008
DAILY_SCREEN_DRP_MAX_PCT = 3.0
DAILY_SCREEN_DRC_MAX_PCT = 0.5
DAILY_SCREEN_METRIC_TOL_PCT = 1.0e-3

VALIDATION_V_MAX = 1.10
# O produto l·V² é tratado por aproximação cônica sucessiva (SCA), pois o
# writer LP/QCP recebe V̄² como parâmetro. Estes limites controlam a consistência
# entre o parâmetro e a tensão resolvida e a exatidão física do BFM.
SCA_MAX_ITER = 5
SCA_V_TOL = 1.0e-4
SCA_DAMPING = 1.0
BFM_MAX_AGG_GAP_PCT = 0.50
BFM_MAX_PHYSICAL_VIOL_PU = 1.0e-3
BFM_MAX_VDROP_RESID_PU2 = 1.0e-3
# Reparo automático da trajetória discreta antes de abandonar o Estágio 2.
# As estratégias são estáticas e auditáveis; nenhuma relaxa limites elétricos.
ENABLE_E2_TAP_REPAIR = True
E2_TAP_REPAIR_MAX_SOLVES = 6       # inclui a projeção primária
E2_UNKNOWN_RETRY_VIOLATION_CUTOFF = 1.0e-3
E2_TAP_REPAIR_STRATEGIES = (
    ('ceil', 0), ('round', 1), ('ceil', 1),
    ('round', 2), ('floor', 0), ('round', -1),
)
ENFORCE_CYCLIC_TAP = False    # só usar se os perfis t=0 e t=T forem cíclicos
PENALIZE_TERMINAL_TAP = True  # contabiliza o retorno futuro sem forçá-lo em t=T
# A margem é uma meta preventiva, não um novo limite físico/regulatório. Ela
# deve ser soft para não tornar o OPF inviável durante o atraso inicial do SVR.
V_SECURITY_MARGIN = 0.005     # margem interna desejada [p.u.], penalizada na F.O.
ENABLE_NIGHT_VAR = False      # True somente se o inversor suportar VAR noturno
QPV_MIN_SOLAR_FRAC = 0.05     # abaixo de 5% de P_STC, Qpv=0 se night-VAR=False

# Cenário V3.7.6-GLOBAL-IMPLEMENTABLE — busca e validação física.
# Pontos reproduzidos em NREL 84633, Tabela I (IEEE 1547 Volt-VAR curve).
# Convenção: Q>0 injeta reativo (capacitivo) e Q<0 absorve (indutivo).
# Flag legado. False preserva a classificação heterogênea aprendida; True
# reproduz o experimento anterior com todos os inversores em Volt-VAR.
IEEE1547_VV_ALL_PV = False
IEEE1547_V1 = 0.92
IEEE1547_V2 = 0.98
IEEE1547_V3 = 1.02
IEEE1547_V4 = 1.08
IEEE1547_Q_FRAC = 0.44
if abs(IEEE1547_Q_FRAC-QPV_MAX_FRAC_S) > 1.0e-12:
    raise ValueError("Envelope Qpv e curva IEEE devem usar a mesma fração de Snom")

# Experimento: Qpv*(t) livre no OPF seguido de uma curva Volt-VAR estatica,
# individual e ajustada sobre toda a janela temporal de cada inversor.
USE_INDIVIDUAL_FITTED_VV_CURVES = True

# IMPORTABILIDADE PARA O OPERADOR
# --------------------------------
# Cada item abaixo é um único conjunto de parâmetros aplicado durante TODO o
# horizonte. A busca escolhe um perfil antes da operação; V1..V4 e Qmax NÃO são
# reprogramados a cada t. Assim, a variação de Q(t) é somente a resposta local
# e autônoma da curva à tensão medida, como ocorreria no controlador real.
# Apenas o primeiro perfil é a curva IEEE 1547 Cat.B padrão. Os demais são
# alternativas operacionais dentro do mesmo envelope de capacidade e são
# identificados sem o rótulo de "curva IEEE padrão".
OPERATIONAL_VV_PROFILES = (
    {'name': 'opf_individual_fit',
     'V1': 0.92, 'V2': 0.98, 'V3': 1.02, 'V4': 1.08, 'q_frac': 0.44},
    {'name': 'ieee_catb_default',
     'V1': 0.92, 'V2': 0.98, 'V3': 1.02, 'V4': 1.08, 'q_frac': 0.44},
    # Família de busca V3.7.4. Todos os pontos são constantes nas 24 h. A
    # gradação de Qmax evita que suporte de tensão compre melhoria de V com
    # aumento excessivo de corrente e perdas Joule.
    {'name': 'vv_loss_aware_q10',
     'V1': 0.90, 'V2': 0.99, 'V3': 1.01, 'V4': 1.10, 'q_frac': 0.10},
    {'name': 'vv_gentle_q15',
     'V1': 0.90, 'V2': 0.995, 'V3': 1.020, 'V4': 1.10, 'q_frac': 0.15},
    {'name': 'vv_balanced_q20',
     'V1': 0.90, 'V2': 1.000, 'V3': 1.020, 'V4': 1.10, 'q_frac': 0.20},
    {'name': 'vv_narrow_q20',
     'V1': 0.92, 'V2': 0.995, 'V3': 1.005, 'V4': 1.08, 'q_frac': 0.20},
    {'name': 'vv_support_q30',
     'V1': 0.90, 'V2': 1.005, 'V3': 1.025, 'V4': 1.10, 'q_frac': 0.30},
    {'name': 'vv_narrow_q30',
     'V1': 0.92, 'V2': 1.000, 'V3': 1.010, 'V4': 1.08, 'q_frac': 0.30},
    {'name': 'vv_support_q44',
     'V1': 0.92, 'V2': 1.010, 'V3': 1.025, 'V4': 1.08, 'q_frac': 0.44},
    {'name': 'vv_wide_deadband_q20',
     'V1': 0.90, 'V2': 0.97, 'V3': 1.03, 'V4': 1.10, 'q_frac': 0.20},
)
for _vv_profile in OPERATIONAL_VV_PROFILES:
    if not (_vv_profile['V1'] < _vv_profile['V2'] <= _vv_profile['V3'] <
            _vv_profile['V4']):
        raise ValueError(f"Breakpoints Volt-VAR inválidos: {_vv_profile}")
    if not (0.0 < _vv_profile['q_frac'] <= QPV_MAX_FRAC_S):
        raise ValueError(f"Qmax Volt-VAR fora do envelope: {_vv_profile}")
INVERTER_PARAMETERS_STATIC_OVER_HORIZON = True

# A classificação inicial A/B/C permanece congelada durante o horizonte. A
# busca de curva usa B quando ele existe; se estiver vazio, A/C são apenas uma
# população provisória para a triagem e só migram a B após replay físico.
# Toda curva escolhida é comum, estática e programada uma única vez.
RL_KEEP_GROUP_B_IEEE = False
ACTIVE_OPERATIONAL_VV_PROFILE_NAMES = (
    ('opf_individual_fit',) if USE_INDIVIDUAL_FITTED_VV_CURVES else
    ('ieee_catb_default',) if RL_KEEP_GROUP_B_IEEE else
    tuple(p['name'] for p in OPERATIONAL_VV_PROFILES))

# Além de parâmetros estáticos, limita-se a variação do comando resultante.
# 0,10·Snom por intervalo de 10 min leva pelo menos 50 min para ir de 0 a
# 0,44·Snom. Desligamento por ausência de irradiância permanece imediato.
ENABLE_Q_COMMAND_SLEW_LIMIT = True
Q_COMMAND_MAX_STEP_FRAC_S = 0.10
Q_COMMAND_SLEW_TOL = 1.0e-8

# Seleção operacional V3.7.6. O escore usa os quatro termos da F.O.; taps
# têm prioridade lexicográfica entre candidatos que reduzem simultaneamente
# todas as metas. Joule permanece apenas como salvaguarda de perdas totais.
OP_WEIGHT_TAP = 0.55
OP_WEIGHT_VOLTAGE_DEVIATION = 0.20
OP_WEIGHT_UNBALANCE = 0.10
OP_WEIGHT_CORE = 0.15
if abs(OP_WEIGHT_TAP + OP_WEIGHT_VOLTAGE_DEVIATION +
       OP_WEIGHT_UNBALANCE + OP_WEIGHT_CORE - 1.0) > 1.0e-12:
    raise ValueError("Pesos operacionais devem somar 1")
if OP_WEIGHT_TAP <= (OP_WEIGHT_VOLTAGE_DEVIATION +
                     OP_WEIGHT_UNBALANCE + OP_WEIGHT_CORE):
    raise ValueError("Taps devem ter peso operacional dominante")
OP_REQUIRE_TAP_OPS_NO_GREATER_THAN_BASE = True

# Uma recomendação diferente do tap-base só é emitida se TODOS os critérios
# materiais abaixo forem atendidos. Sem candidato simultâneo, aplica-se a regra
# de não ação e mantém-se a classificação RL A/B/C com a agenda tap-base.
OP_MIN_TAP_SAVING_STEPS = 1
OP_MIN_CORE_SAVING_KWH = 0.25
OP_MIN_VOLTAGE_DEVIATION_SAVING_FRAC = 1.0e-4
OP_MIN_UNBALANCE_SAVING_FRAC = 1.0e-4
OP_DOMINANCE_ABS_TOL_PU = 1.0e-8
OP_MIN_REL_SCORE_IMPROVEMENT = 0.001       # 0,1% no escore normalizado
OP_MAX_TOTAL_LOSS_INCREASE_FRAC = 0.001    # no máximo +0,1% vs. baseline física

# Uma triagem multiobjetivo preserva no máximo dois perfis. Com B existente,
# eles são avaliados na mesma trajetória tap-base; sem B, seguem diretamente
# para os rollouts A/C→B, evitando replays eletricamente idênticos.
VV_SCREEN_MAX_PROFILES = 2
# Na V3.7.5 a triagem inicial deixa de resolver todas as curvas. Uma métrica
# substituta, calculada sobre V do tap-base e Q* do OPF, escolhe no máximo duas
# curvas; somente elas podem passar pelo fechamento físico BFM. Isso limita
# tempo e memória sem transformar a triagem heurística em certificação elétrica.
VV_SURROGATE_VOLT_GAIN_PU_PER_QFRAC = 0.04
VV_SURROGATE_WEIGHT_VOLTAGE = 0.50
VV_SURROGATE_WEIGHT_Q_TRACKING = 0.35
VV_SURROGATE_WEIGHT_Q_USAGE = 0.15
# Curvas quase inativas não ocupam uma vaga apenas por minimizarem Q. A IEEE
# continua disponível como referência física já resolvida no tap-base.
VV_SURROGATE_MIN_ACTIVE_Q_RMS_FRAC_S = 0.005
VV_SURROGATE_INACTIVE_TRACKING_NRMSE = 0.95
# As reduções adicionais sobre a trajetória OPF de 20 taps ficaram fora da
# faixa adequada na execução V3.7.3. A V3.7.4 concentra o custo computacional
# nas curvas e nos vizinhos de 31 taps da baseline LDC.
OPERATIONAL_GLOBAL_TAP_REDUCTIONS = ()
TAP_REDUCTION_VV_PROFILE_NAMES = tuple(
    p['name'] for p in OPERATIONAL_VV_PROFILES)
OPERATIONAL_TAP_REDUCTION_PHASE_SETS = ((1,), (2,), (3,), (1,2,3))

# A V3.7.3 testava apenas trajetórias derivadas do OPF de 20 taps. A V3.7.4
# também parte dos 32 taps do LDC e suprime uma única atuação. Esses vizinhos
# de 31 taps são a mudança mínima capaz de reduzir desgaste sem abandonar a
# qualidade de tensão da baseline. Na V3.7.6, seis cronogramas direcionais
# cobrem as três fases e priorizam menor exposição acumulada a tap².
ENABLE_LDC_NEAR_TAP_SEARCH = False
LDC_NEAR_MAX_SCHEDULES = 6

# Refinamento da política RL por rollouts físicos globais. A classificação
# local fornece candidatos; equipamentos completos dos Grupos A ou C são movidos
# temporariamente para B e avaliados no BFM. A mudança é estática nas 24 h.
# A classificação desta versão já usa as curvas individuais ajustadas. O antigo
# refinamento A/C->B tentava materializar curvas para registros sem V1..V4 e
# também confundia a atribuição causal; fica desativado neste experimento.
ENABLE_GLOBAL_POLICY_REFINEMENT = False
GLOBAL_POLICY_MAX_DEVICE_ROLLOUTS = 4
# Quando o RL não forma Grupo B, a triagem Volt-VAR usa provisoriamente os
# membros A/C e os replays globais testam duas curvas. Para manter o orçamento
# de memória da V3.9.1, cada curva recebe somente um rollout individual e um
# rollout cumulativo com os equipamentos mais promissores.
GLOBAL_POLICY_MAX_VV_PROFILES = 2
GLOBAL_POLICY_INDIVIDUALS_PER_PROFILE = 1
GLOBAL_POLICY_CUMULATIVE_DEVICE_COUNT = 4
# Rollouts continuam sendo decididos pelo BFM. A sensibilidade acumulada X do
# caminho serve somente como prior de triagem: o mesmo ΔQ tende a produzir
# maior efeito de tensão em barras eletricamente remotas de uma rede radial.
GLOBAL_POLICY_DISTANCE_RANK_WEIGHT = 1.00
GLOBAL_POLICY_MAX_TAP_ROLLOUTS = 3
GLOBAL_POLICY_MIN_REWARD_GAIN = 1.0e-4
GLOBAL_POLICY_TOTAL_LOSS_PENALTY = 0.25

# Busca lexicográfica de trajetórias de tap. O candidato base é preservado; os
# demais reduzem temporariamente um degrau somente na janela solar, por fase ou
# nas três fases. O tap é restaurado antes do período noturno. A escolha por
# perdas de núcleo ocorre somente entre candidatos fisicamente fechados,
# mecanicamente válidos e integralmente dentro da faixa adequada.
ENABLE_CORE_LOSS_CANDIDATE_SEARCH = False  # refinamento/esparsificação fica para segunda etapa; não programa taps
# As janelas solares geravam doze fechamentos físicos adicionais e repetiam
# uma família que as versões anteriores já mostraram pouco promissora. Elas
# permanecem documentadas abaixo, mas ficam fora da busca padrão V3.7.6.
ENABLE_SOLAR_WINDOW_SEARCH = False
CORE_LOSS_TAP_REDUCTIONS = (0,)  # reduções globais desativadas na V3.2
CORE_LOSS_SELECTIVE_WINDOWS = (
    # nome, fases, início, fim exclusivo, redução [degraus]
    ('solar_08_16_ph1', (1,), int(8*60/DT_MIN), int(16*60/DT_MIN), 1),
    ('solar_08_16_ph2', (2,), int(8*60/DT_MIN), int(16*60/DT_MIN), 1),
    ('solar_08_16_ph3', (3,), int(8*60/DT_MIN), int(16*60/DT_MIN), 1),
    ('solar_08_16_all', (1,2,3), int(8*60/DT_MIN), int(16*60/DT_MIN), 1),
    ('solar_10_16_all', (1,2,3), int(10*60/DT_MIN), int(16*60/DT_MIN), 1),
    ('solar_10_14_all', (1,2,3), int(10*60/DT_MIN), int(14*60/DT_MIN), 1),
)
# A faixa "adequada" amostra a amostra é mantida como métrica, mas a aceitação
# segue DRP/DRC e FD95 como screening do cenário-base. Isso evita rejeitar um
# replay por diferenças numéricas mínimas (por exemplo V=0,9297 p.u.) quando o
# próprio baseline físico contém ocorrências precárias no horizonte diário.
CORE_CANDIDATE_REQUIRE_ALL_ADEQUATE = False

# Métricas legadas preservadas como referências e features da política RL.
FIT_C_MAX_NRMSE = 0.08        # Q≈0: erro máximo relativo à capacidade observada
FIT_A_MAX_NRMSE = 0.15        # Q=kP: erro máximo para aceitar FP fixo
FIT_PAR_SIMPLICITY = 1.10     # aceita A se até 10% pior que Volt-VAR

# ── Classificação por aprendizagem por reforço contextual ───────────────────
# Problema de decisão estática: um contextual bandit é mais apropriado que um
# MDP temporal, pois a ação é escolhida uma vez antes da operação e permanece
# fixa nas 24 h. Não há solução sequencial, warm-start entre períodos ou mudança
# de parâmetros a cada 10 min. Permanece apenas o limitador operacional de rampa
# do comando Q já usado na V3.7.6. O Q-learning usa gamma=0.
ENABLE_RL_INVERTER_CLASSIFICATION = True
RL_ACTIONS = ('A', 'B', 'C')  # A=estático não-VV, B=Volt-VAR, C=FP1.00

# Biblioteca implementável do Grupo A. Cada equipamento recebe UM único modo
# e UM único conjunto de parâmetros para todo o horizonte de 24 h. O sinal de
# Q é parte do parâmetro estático; não é reconfigurado entre períodos.
PREFERRED_FIXED_POWER_FACTOR = 0.96
# 0.85 permanece disponível como limite operacional, mas deixa de ser a única
# alternativa. A ordem da tupla não decide o resultado: o desempate explícito
# abaixo favorece 0.96 apenas quando as métricas anteriores são equivalentes.
FIXED_POWER_FACTOR_CANDIDATES = (
    0.85, 0.88, 0.90, 0.92, 0.95, 0.96, 0.97, 0.98, 0.99)
FP_EQUIVALENCE_RMSE_TOL_FRAC_S = 0.005
CONSTANT_Q_FRAC_S_CANDIDATES = ()
# Desempate pequeno: FP fixo é preferido quando Q constante quase não melhora
# a aderência. A penalidade equivale a 0,5% de Snom no RMSE físico.
CONSTANT_Q_SELECTION_PENALTY_FRAC_S = 0.005
RL_RANDOM_SEED = 20260720
RL_TRAINING_EPISODES = 12000
RL_LEARNING_RATE = 0.12
RL_EPSILON_START = 0.35
RL_EPSILON_END = 0.02
RL_POLICY_LOCAL_REWARD_BLEND = 0.35
RL_NRMSE_REWARD_CAP = 3.0

# Penalidades da recompensa, todas adimensionais na mesma escala do NRMSE.
# Favorecem o controle mais simples quando a aderência ao Q* é equivalente,
# sem impedir IEEE quando há excursões de tensão ou troca de sinal de Q.
RL_COMPLEXITY_PENALTY = {'A': 0.020, 'B': 0.045, 'C': 0.000}
# C (FP=1) prevalece quando o suporte ótimo é pequeno ou quando A/B não melhora
# materialmente a aderência. Evita ativar reativo em todos os equipamentos.
C_MAX_NRMSE_REGRET = 0.08
C_MIN_Q_RMS_FRAC_S_FOR_REACTIVE = 0.04
C_MIN_REACTIVE_SUPPORT_PRIORITY = 0.035
UNITY_COUNTERFACTUAL_MAX_SCHEDULES = 3
PARETO_MAX_VOLTAGE_DEVIATION_INCREASE_FRAC = 0.01
PARETO_MAX_TOTAL_LOSS_INCREASE_FRAC = 0.001
SPARSE_REACTIVE_FRACTIONS = (0.25, 0.50, 0.75, 1.00)
# V3: busca esparsa operacional com LDC autônomo. Os prefixos são definidos
# sobre equipamentos físicos (não terminais por fase) e avaliados por replay
# completo. O limite pode ser alterado via OPF_SPARSE_MAX_REPLAYS.
ENABLE_AUTONOMOUS_SPARSE_POLICY_SEARCH = True
AUTONOMOUS_SPARSE_PREFIX_FRACTIONS = (0.10, 0.20, 0.30, 0.40, 0.50, 0.65, 0.80, 1.00)
AUTONOMOUS_SPARSE_MAX_REPLAYS = int(os.environ.get('OPF_SPARSE_MAX_REPLAYS', '8'))
SPARSE_MIN_DV_IMPROVEMENT_FRAC = 0.001
SPARSE_MIN_LOSS_IMPROVEMENT_FRAC = 0.001
SPARSE_PREANCHOR_TAP_OPS = (22, 24, 26)
SPARSE_PREANCHOR_MAX_VOLTAGE_DEVIATION_INCREASE_FRAC = 0.02
SPARSE_PREANCHOR_MAX_LOSS_INCREASE_FRAC = 0.02
SPARSE_PREANCHORS_PER_TAP_BUDGET = 1
# Busca coordenada das três fases. O orçamento é total (não por fase) e a
# trajetória é construída causalmente; assim não se misturam planos diários
# inteiros independentes, o que anteriormente criava desequilíbrio artificial.
JOINT_TAP_BUDGETS = (20, 22, 24, 26, 28, 30, 31)
JOINT_TAP_BLEND_FACTORS = (0.25, 0.50, 0.75)
JOINT_TAP_MAX_PHASE_SEPARATION = 2
JOINT_TAP_MAX_CANDIDATES_PER_BUDGET = 3
RL_STATIC_BIDIRECTIONAL_PENALTY = 0.20
RL_STATIC_CLIPPING_PENALTY = 0.15
RL_UNITY_SUPPORT_PENALTY = 0.12
# A distância não determina o grupo. Ela apenas aumenta o custo de escolher
# FP=1 quando um ponto eletricamente sensível e remoto também apresenta demanda
# efetiva de suporte em Q*/tensão. Sem demanda local, o termo vale zero.
RL_DISTANCE_SUPPORT_PENALTY = 0.10
RL_DISTANCE_STATE_CUTS = (0.33, 0.66)

# Cobertura não é imposta a qualquer custo. Se um modo não aparecer, ele só é
# atribuído ao dispositivo de menor arrependimento quando a perda de retorno é
# inferior ao limite abaixo; caso contrário, a ausência é reportada.
# Não força a presença de A/B/C na recomendação. Um modo ausente é um resultado
# válido; alternativas Volt-VAR são testadas depois por rollouts físicos globais.
RL_ENABLE_SAFE_MODE_COVERAGE = False
RL_MAX_COVERAGE_REGRET = 0.08
# Operações de tap por fase: limite físico razoável.
# O número emerge da hierarquia W_dv > W_tap_ops, mas com um teto sensato:
# 12 ops/fase = 1 operação a cada 2 horas em média, compatível com a vida
# útil mecânica do comutador IEEE Cooper típico em operação normal.
# Acima disso, a baseline OpenDSS (33-39 ops/dia totais, ≈12/fase) já
# indica que o sistema está em estresse.
N_TAP_MAX  = 12     # degraus máximos por fase no horizonte de 24h

# ═══════════════════════════════════════════════════════════════════════════
# BASES DO SISTEMA — por nível de tensão (linha-neutro)
# ═══════════════════════════════════════════════════════════════════════════
# Conforme Kersting (Distribution System Modeling and Analysis, cap. 6) e
# Bergen & Vittal (Power Systems Analysis), em sistemas multinível com
# transformadores, adota-se:
#   - S_base ÚNICA para todo o sistema (kVA/fase)
#   - V_base por nível de tensão (lado do transformador), linha-neutro
#   - Z_base = V_base² / S_base — única por nível de tensão
# A configuração do circuito (trifásico, bifásico, monofásico) NÃO altera a
# Z_base; afeta apenas o número de fases ativas e o conjunto de equações.
SBASE = 1000.0 / 3.0                  # kVA/fase
VMT   = 4.16  / math.sqrt(3)          # kV (linha-neutro, MT)
VBT   = 0.480 / math.sqrt(3)          # kV (linha-neutro, BT)
ZMT   = VMT**2 / SBASE * 1000         # Ω (única para todos os ramos MT)


In [ ]:
#@title Internal model implementation 2/12 { display-mode: "form" }
ZBT   = VBT**2 / SBASE * 1000         # Ω (única para todos os ramos BT)
IMT   = SBASE / VMT                   # kA
IBT   = SBASE / VBT                   # kA
# Aliases retrocompatíveis (manter código existente funcionando)
VBT3 = VBT; ZBT3 = ZBT; IBT3 = IBT
VBT1 = VBT; ZBT1 = ZBT; IBT1 = IBT   # mesma base — sem distinção por nº de fases

# Limites PRODIST
FD_MT_MAX    = 2.0   # %
FD_BT_MAX    = 3.0   # %
K_FD_VUF     = math.sqrt(3) / 6
VUF_MAX_MT   = FD_MT_MAX / (K_FD_VUF * 100)   # ≈ 0.0693 p.u.
VUF_MAX_BT   = FD_BT_MAX / (K_FD_VUF * 100)   # ≈ 0.1039 p.u.
FD_ANEEL_MAX = FD_MT_MAX                        # alias — limite mais restritivo

TR_MT_NORMAL = 1.00
TR_MT_M25    = 0.95
V_ADEQ_MT_LO = 0.93 * TR_MT_NORMAL
V_ADEQ_MT_HI = 1.05 * TR_MT_NORMAL
V_PREC_MT_LO = 0.90 * TR_MT_NORMAL
V_ADEQ_BT_LO = 0.93
V_ADEQ_BT_HI = 1.05
V_PREC_BT_LO = 0.90

# ══════════════════════════════════════════════════════════════════════════════
# UTILITÁRIOS
# ══════════════════════════════════════════════════════════════════════════════

def _norm(val):
    s = str(val).strip()
    return s[:-2] if s.endswith('.0') else s

def _f(val, default=0.0):
    try:
        v = float(val)
        return default if math.isnan(v) else v
    except:
        return default

def _phases(val):
    out = []
    for p in str(val).strip().replace('.', ',').split(','):
        try:
            ph = int(float(p.strip()))
            if ph in (1, 2, 3): out.append(ph)
        except:
            pass
    return sorted(set(out))


_USABLE_TERMINATIONS = ('optimal', 'locallyOptimal', 'feasible')


def _max_model_violation(model):
    """Maior violação de bounds/restrições na solução atualmente carregada."""
    worst = 0.0
    try:
        for var in model.component_data_objects(pyo.Var, active=True):
            val = pyo.value(var, exception=False)
            if val is None:
                return float('inf')
            if var.lb is not None:
                worst = max(worst, pyo.value(var.lb) - val)
            if var.ub is not None:
                worst = max(worst, val - pyo.value(var.ub))
        for con in model.component_data_objects(pyo.Constraint, active=True):
            body = pyo.value(con.body, exception=False)
            if body is None:
                return float('inf')
            if con.lower is not None:
                worst = max(worst, pyo.value(con.lower) - body)
            if con.upper is not None:
                worst = max(worst, body - pyo.value(con.upper))
    except Exception:
        return float('inf')
    return max(worst, 0.0)


def _solve_with_status_recovery(opt, model, tee=False, label='solver'):
    """Resolve, carrega a solução e trata `status=ok/termination=unknown`.

    Alguns retornos do executável CPLEX contêm uma solução completa, mas o
    plugin shell do Pyomo não classifica a condição de término. Nessa situação
    a solução é auditada e marcada apenas como `feasible` — nunca como
    `optimal`. Uma segunda leitura só ocorre quando não há solução auditável.
    """
    result = opt.solve(model, tee=tee, load_solutions=False)
    tc_raw = str(result.solver.termination_condition)
    if len(result.solution) > 0:
        model.solutions.load_from(result)

    tc = tc_raw
    if tc in _USABLE_TERMINATIONS and len(result.solution) == 0:
        # Alguns plugins reportam o término mas não carregam os valores quando
        # load_solutions=False; uma chamada silenciosa completa o carregamento.
        retry = opt.solve(model, tee=False)
        retry_tc = str(retry.solver.termination_condition)
        result = retry
        if retry_tc in _USABLE_TERMINATIONS:
            tc = retry_tc
    violation = (0.0 if tc in _USABLE_TERMINATIONS
                 else _max_model_violation(model))
    if (tc not in _USABLE_TERMINATIONS and
            str(result.solver.status).lower() == 'ok' and
            violation <= 1.0e-5):
        print(f"  [{label}] retorno '{tc_raw}', mas solução auditada: "
              f"max violação={violation:.3e}; classificada como feasible "
              "(otimalidade não certificada).")
        tc = 'feasible'
    elif (tc not in _USABLE_TERMINATIONS and
          str(result.solver.status).lower() == 'ok' and
          violation > E2_UNKNOWN_RETRY_VIOLATION_CUTOFF):
        # Uma iterada com violação material, como a saída "Barrier -
        # Infeasible" do CPLEX, não se torna factível por reler o mesmo QCP.
        # O reparo de taps deve testar outra trajetória em vez de duplicar o
        # tempo de solução.
        print(f"  [{label}] retorno '{tc_raw}' com violação material "
              f"({violation:.3e}); releitura omitida.")
    elif tc not in _USABLE_TERMINATIONS and str(result.solver.status).lower() == 'ok':
        print(f"  [{label}] retorno '{tc_raw}' sem solução auditável; repetindo leitura...")
        try:
            retry = opt.solve(model, tee=False, load_solutions=False)
            if len(retry.solution) > 0:
                model.solutions.load_from(retry)
            retry_tc = str(retry.solver.termination_condition)
            if retry_tc in _USABLE_TERMINATIONS:
                result, tc = retry, retry_tc
            elif len(retry.solution) > 0:
                result, tc = retry, retry_tc
        except Exception as exc:
            print(f"  [{label}] segunda leitura indisponível: {exc}")

    violation = (0.0 if tc in _USABLE_TERMINATIONS
                 else _max_model_violation(model))
    if tc not in _USABLE_TERMINATIONS and violation <= 1.0e-5:
        print(f"  [{label}] solução auditada: max violação={violation:.3e}; "
              "classificada como feasible (otimalidade não certificada).")
        tc = 'feasible'
    return result, tc, tc_raw, violation


def _solve_sca_voltage_consistent(opt, model, cph, hours, label,
                                  tee_first=False):
    """Resolve QCPs sucessivos até os pontos V̄ coincidirem com V resolvido."""
    keys = [(i,j,ph,t) for (i,j,ph) in cph for t in hours]
    bus_keys = list(model.V_bar_bus.index_set())
    v_bar_bus = {k: pyo.value(model.V_bar_bus[k]) for k in bus_keys}
    result = None
    tc = tc_raw = 'notSolved'
    violation = float('inf')
    mismatch = float('inf')
    converged = False
    iteration = 0
    for iteration in range(1, SCA_MAX_ITER + 1):
        result, tc, tc_raw, violation = _solve_with_status_recovery(
            opt, model, tee=(tee_first and iteration == 1),
            label=f'{label}-SCA{iteration}')
        if tc not in _USABLE_TERMINATIONS:
            break
        v_new_bus = {k: pyo.value(model.V[k]) for k in bus_keys}
        mismatch = max(abs(v_new_bus[k] - v_bar_bus[k]) for k in bus_keys)
        print(f"  {label} SCA iter={iteration}: "
              f"max|V−V̄_cone|={mismatch:.3e} p.u. | solver={tc}")
        if mismatch <= SCA_V_TOL:
            converged = True
            break
        if iteration < SCA_MAX_ITER:
            for k in bus_keys:
                v_bar_bus[k] = (SCA_DAMPING*v_new_bus[k] +
                                (1.0-SCA_DAMPING)*v_bar_bus[k])
                model.V_bar_bus[k] = max(v_bar_bus[k], 0.80)
            for i,j,ph,t in keys:
                model.V_sq_ws[i,j,ph,t] = max(
                    v_bar_bus[(i,ph,t)]**2, 0.64)
    return result, tc, tc_raw, violation, converged, iteration, mismatch


def _classify_e2_candidate(termination, sca_converged, exactness):
    """Separa solução convexa utilizável de certificação física do BFM.

    A primeira autoriza somente extrair Q*/taps para o replay. A segunda
    autoriza interpretar o próprio estado E2 como fluxo físico. Confundir os
    dois níveis fez a V3.7.1 descartar uma trajetória ótima antes do replay.
    """
    accepted_for_replay = bool(
        termination in _USABLE_TERMINATIONS and sca_converged)
    relaxation_exact = bool(
        accepted_for_replay and exactness is not None and
        exactness.get('exactness_ok', False))
    return accepted_for_replay, relaxation_exact


def summarize_bfm_exactness(model, data, sets_info):
    """Audita o cone de corrente e a queda exata em tensão quadrática."""
    gaps = []
    vdrop_residuals = []
    rhs_total = 0.0
    for (i,j,ph) in sets_info['cph']:
        if data['branches'][(i,j,ph)].get('level') == 'svr':
            continue
        for t in sets_info['hours']:
            v2 = max(pyo.value(model.V[i,ph,t])**2, 0.0)
            lhs = pyo.value(model.l_sq[i,j,ph,t]) * v2
            rhs = (pyo.value(model.P[i,j,ph,t])**2 +
                   pyo.value(model.Q[i,j,ph,t])**2)
            gaps.append(lhs-rhs)
            rhs_total += rhs
            bd = data['branches'][(i,j,ph)]
            r, x = bd['r_pu'], bd['x_pu']
            vj2 = pyo.value(model.V[j,ph,t])**2
            vj2_bfm = (v2 - 2.0*(r*pyo.value(model.P[i,j,ph,t]) +
                                 x*pyo.value(model.Q[i,j,ph,t])) +
                       (r*r+x*x)*pyo.value(model.l_sq[i,j,ph,t]))
            vdrop_residuals.append(vj2-vj2_bfm)
    abs_total = sum(abs(g) for g in gaps)
    positive_total = sum(max(g, 0.0) for g in gaps)
    max_physical_violation = max([max(-g, 0.0) for g in gaps] or [0.0])
    aggregate_gap_pct = 100.0*abs_total/max(rhs_total, 1.0e-12)
    relaxation_slack_pct = 100.0*positive_total/max(rhs_total, 1.0e-12)
    max_vdrop_residual = max([abs(v) for v in vdrop_residuals] or [0.0])
    rms_vdrop_residual = math.sqrt(
        sum(v*v for v in vdrop_residuals)/max(len(vdrop_residuals), 1))
    exactness_ok = (
        aggregate_gap_pct <= BFM_MAX_AGG_GAP_PCT and
        max_physical_violation <= BFM_MAX_PHYSICAL_VIOL_PU and
        max_vdrop_residual <= BFM_MAX_VDROP_RESID_PU2
    )
    return {
        'aggregate_gap_pct': aggregate_gap_pct,
        'relaxation_slack_pct': relaxation_slack_pct,
        'max_physical_violation_pu': max_physical_violation,
        'max_vdrop_residual_pu2': max_vdrop_residual,
        'rms_vdrop_residual_pu2': rms_vdrop_residual,
        'exactness_ok': exactness_ok,
        'samples': len(gaps),
    }

# ══════════════════════════════════════════════════════════════════════════════
# LEITURA DE DADOS
# ══════════════════════════════════════════════════════════════════════════════

def load_data(buses_file, branches_file):
    LOAD_EXP_P = {1: 0.0, 2: 2.0, 3: 0.0, 4: 1.0, 5: 1.0}
    LOAD_EXP_Q = {1: 0.0, 2: 2.0, 3: 2.0, 4: 2.0, 5: 1.0}

    df = pd.read_excel(buses_file, sheet_name='MT')
    df.columns = df.columns.str.strip()
    df = df[~df['name'].astype(str).str.contains('Source', case=False)]

    buses = {}; slack = None
    for _, r in df.iterrows():
        n = _norm(r['name']); tb = int(_f(r.get('tb', 0), 0))
        buses[n] = {'level': 'mv', 'v_nom_kv': VMT,
                    'phases': _phases(r.get('phases', '1,2,3')), 'tb': tb}
        if tb == 3 and slack is None:
            slack = n

    # Cargas MT
    loads_mt = {}
    try:
        df_lmt = pd.read_excel(buses_file, sheet_name='Load_MT')
        df_lmt.columns = df_lmt.columns.str.strip()
        for _, r in df_lmt.iterrows():
            n   = _norm(r['name'])
            phs = _phases(r.get('phases', '1,2,3'))
            nph = max(len(phs), 1)
            P   = _f(r.get('P_D'), 0.)
            Q   = _f(r.get('Q_D'), 0.)
            mod = int(_f(r.get('Model', 1), 1))
            conn = str(r.get('Conn', 'Wye')).strip().lower()
            # Cargas delta -- Simplificacao S5b:
            # ZY=ZD/3 invalida em sistema desequilibrado (assume Vab=Vbc=Vca).
            # Correcao exata nao-linear; S5b usa potencia de linha sem fator /3.
            is_delta = ('delta' in conn or conn.strip().lower() == 'd')
            for ph in phs:
                P_ph = P / nph / SBASE
                Q_ph = Q / nph / SBASE
                loads_mt[(n, ph)] = {
                    'P_pu':     P_ph,
                    'Q_pu':     Q_ph,
                    'model':    mod,
                    'conn':     conn,
                    'is_delta': is_delta,
                    'aP':       LOAD_EXP_P.get(mod, 0.0),
                    'aQ':       LOAD_EXP_Q.get(mod, 0.0),
                }
    except Exception as e:
        loads_mt = {}
        print(f"  Load_MT: {e}")

    # Cargas BT
    df = pd.read_excel(buses_file, sheet_name='BT')
    df.columns = df.columns.str.strip()
    loads = {}
    for _, r in df.iterrows():
        n = _norm(r['name']); phs = _phases(r.get('phases', '1,2,3'))
        nph = max(len(phs), 1); Vbt = VBT3 if nph > 1 else VBT1
        buses[n] = {'level': 'lv', 'v_nom_kv': Vbt, 'phases': phs, 'tb': 0}
        P_tot = _f(r.get('P_D'), 0.); Q_tot = _f(r.get('Q_D'), 0.)
        for ph in phs:
            # Suportar cargas por fase (colunas P_ph1/P_ph2/P_ph3 no xlsx)
            # Se não existirem, dividir uniformemente por nph
            P_ph = _f(r.get(f'P_ph{ph}'), P_tot / nph)
            Q_ph = _f(r.get(f'Q_ph{ph}'), Q_tot / nph)
            loads[(n, ph)] = {'P_pu': P_ph / SBASE,
                              'Q_pu': Q_ph / SBASE,
                              'model': 1, 'aP': 0.0, 'aQ': 0.0}

    for n, d in buses.items():
        if d['level'] == 'mv':
            for ph in d['phases']:
                if (n, ph) not in loads:
                    loads[(n, ph)] = loads_mt.get(
                        (n, ph),
                        {'P_pu': 0., 'Q_pu': 0., 'model': 1, 'aP': 0.0, 'aQ': 0.0})

    # PV — distinguir inversores trifásicos (parâmetros iguais por fase)
    # e monofásicos (parâmetros independentes por fase)
    # Inversor trifásico: usa kva/3 para cada fase (mesmo dispositivo);
    #   a potência nominal é comum → P_rated = S_nom/nph para as 3 fases.
    # Inversor monofásico: nph=1, parâmetros lidos individualmente.
    # A própria coluna 'phases' identifica o tipo: "1.2.3" é trifásico;
    # "1", "2" ou "3" é monofásico. A planilha não possui coluna 'type'.
    df = pd.read_excel(buses_file, sheet_name='PV')
    df.columns = df.columns.str.strip()
    pv = {}; pv_meta = {}   # pv_meta: metadados para pós-processamento
    for _, r in df.iterrows():
        bus  = _norm(r['Bus'])
        if bus not in buses: continue
        phs  = _phases(r.get('phases', '1,2,3'))
        nph  = max(len(phs), 1)
        kva  = _f(r.get('kva'), 0.)
        Sf   = kva / nph            # potência por fase do inversor
        is3ph = (nph == 3)
        if is3ph:
            # Trifásico: um único dispositivo, com despacho comum nas três fases.
            # Usa a média dos valores não nulos para não privilegiar uma fase por
            # pequenas diferenças numéricas oriundas do fluxo de potência fonte.
            _p3 = [_f(r.get(f'p_pv_{ph}'), 0.0) for ph in phs]
            _p3 = [p for p in _p3 if p > 0.0]
            P_rated_com = sum(_p3) / len(_p3) if _p3 else Sf
            S_nom_com   = max(Sf, P_rated_com / FP_MIN_VV) if P_rated_com > 0 else Sf
            for ph in phs:
                pv[(bus, ph)] = {'S_nom_pu':   S_nom_com / SBASE,
                                 'P_rated_pu': P_rated_com / SBASE,
                                 'is_3ph': True, 'nph': nph}
            pv_meta[bus] = {'type': '3ph', 'kva': kva, 'phases': phs}
        else:
            # Monofásico: parâmetros independentes por fase
            for ph in phs:
                P_rated = _f(r.get(f'p_pv_{ph}'), Sf)
                S_nom   = max(Sf, P_rated / FP_MIN_VV) if P_rated > 0 else Sf
                pv[(bus, ph)] = {'S_nom_pu':   S_nom / SBASE,
                                 'P_rated_pu': P_rated / SBASE,
                                 'is_3ph': False, 'nph': 1}
            pv_meta[bus] = {'type': '1ph', 'kva': kva, 'phases': phs}

    # SVR
    df = pd.read_excel(branches_file, sheet_name='Reg')
    df.columns = df.columns.str.strip()
    svr = {}
    for _, r in df.iterrows():
        ph = int(_f(r.get('phases'), 1))
        mv = _norm(r['mv_bus']); lv = _norm(r['lv_bus'])
        R  = _f(r.get('R_ohm'), 0.0)
        X  = _f(r.get('X_ohm'), 0.0)
        I  = _f(r.get('ctprim_A'), 700.)
        n_tap    = int(_f(r.get('TapNum'), 33))
        step     = _f(r.get('Step'), 0.00625)
        # Faixa bidirecional padrão IEEE Cooper: ±10% em 32 passos
        # tap_pu ∈ [0.900, 1.100] com posição neutra em pos=16 (1.000 p.u.)
        # Permite tanto buck (descer V) quanto boost (subir V) — essencial
        # para cenários com PV onde o fluxo pode se inverter.
        tap_min  = _f(r.get('Tap_min'), 0.90)
        tap_max  = tap_min + (n_tap - 1) * step      # com step=0.00625: 1.100
        tap_init = _f(r.get('Tap_init'), 1.0)         # posição neutra padrão
        # A coluna histórica chama-se Delay_s, mas neste conjunto seu valor foi
        # confirmado como MINUTOS. Com Δt=10 min, 15 min exigem ceil(15/10)=2
        # amostras consecutivas fora da banda antes de cada degrau.
        delay_min = _f(r.get('Delay_min'), _f(r.get('Delay_s'), 15.0))
        d_per     = max(1, math.ceil(delay_min / DT_MIN))

        # Line Drop Compensation (LDC) na base do relé, em volts.
        # r_LDC_V e X_LDC_V já são referidos a Vreg; portanto NÃO devem ser
        # convertidos para ohms nem divididos pela impedância-base da rede.
        ptratio   = _f(r.get('ptratio_V'), 20.0)  # relação PT primário/relé
        kv_lv     = _f(r.get('kv_lv'), VMT)
        vbase_relay_V = kv_lv * 1000.0 / max(ptratio, 1.0e-9)
        R_ldc_V   = _f(r.get('r_LDC_V'), _f(r.get('R_ldc'), 3.0))
        X_ldc_V   = _f(r.get('X_LDC_V'), _f(r.get('X_ldc'), 9.0))
        vreg_V    = _f(r.get('vreg_V'), 122.0)
        band_V    = _f(r.get('band_V'), 2.0)
        svr[ph]  = {
            'mv_bus': mv, 'lv_bus': lv,
            'r_pu':   R / ZMT, 'x_pu': X / ZMT,
            'Imax_pu': I / IMT,
            'tap_min': tap_min, 'tap_max': tap_max,
            'tap_step': step,   'tap_init': tap_init,
            'n_tap':   n_tap,   'delay_periods': d_per,
            'delay_min': delay_min,
            # Parâmetros LDC na base física do relé.
            'ctprim_A': I, 'ptratio': ptratio,
            'vbase_relay_V': vbase_relay_V,
            'R_ldc_V': R_ldc_V, 'X_ldc_V': X_ldc_V,
            'vreg_V': vreg_V, 'band_V': band_V,
        }
        for b in (mv, lv):
            if b not in buses:
                buses[b] = {'level': 'mv', 'v_nom_kv': VMT,
                            'phases': [1,2,3], 'tb': 0}

    # Ramos MT
    df = pd.read_excel(branches_file, sheet_name='MT')
    df.columns = df.columns.str.strip()
    branches = {}
    for _, r in df.iterrows():
        fr = _norm(r['l']); to = _norm(r['k'])
        if to == 'RG60': fr, to = to, fr
        if 'Source' in (fr, to): continue
        R = _f(r['R']); X = _f(r['X']); I = _f(r.get('Imax', 999))
        for ph in _phases(r.get('phase', '1,2,3')):
            branches[(fr, to, ph)] = {
                'r_pu': R/ZMT, 'x_pu': X/ZMT,
                'Imax_pu': I/IMT, 'level': 'mv'}

    # Ramos BT
    df = pd.read_excel(branches_file, sheet_name='BT')
    df.columns = df.columns.str.strip()
    for _, r in df.iterrows():
        fr = _norm(r['l']); to = _norm(r['k'])
        phs = _phases(r.get('phase', '1,2,3'))
        nph = len(phs)
        Zb = ZBT3 if nph > 1 else ZBT1
        Ib = IBT3 if nph > 1 else IBT1
        R = _f(r['R']); X = _f(r['X']); I = _f(r.get('Imax', 999))
        for ph in phs:
            branches[(fr, to, ph)] = {
                'r_pu': R/Zb, 'x_pu': X/Zb,
                'Imax_pu': I/Ib, 'level': 'lv'}

    # Trafos — armazenar parâmetros térmicos (IEEE C57.110 / C57.91)
    # G_core = P_NL_nom / V_nom² → perda no núcleo p.u. proporcional a V²
    # P_LL_R = perdas de carga nominais (Joule) = r · I_nom²
    # theta_TO_R = elevação nominal do óleo de topo (padrão: 55°C)
    df = pd.read_excel(branches_file, sheet_name='Trafos')
    df.columns = df.columns.str.strip()
    trafos = {}   # {trafo_id: {...}} — metadados térmicos
    for _, r in df.iterrows():
        if str(r['trafo_id']) == 'Sub': continue
        mv = _norm(r['mv_bus']); lv = _norm(r['lv_bus'])
        tid = str(r['trafo_id']).strip()
        phs = _phases(r.get('phases', '1,2,3'))
        nph = max(len(phs), 1)
        Vbt = VBT3 if nph == 3 else VBT1
        Zb = Vbt**2 / SBASE * 1000; Ib = SBASE / Vbt
        kva = _f(r['kva']); Sn = kva / nph
        R = _f(r['R_ohm']); X = _f(r['X_ohm'])
        # Perdas em vazio (núcleo): lidas do xlsx ou estimadas como 0.3% de S_nom
        P_NL_nom_kw  = _f(r.get('P_NL_kW'),  Sn * 0.003)   # kW/fase
        # Perdas de carga nominais: lidas ou estimadas como R·I²
        P_LL_R_kw    = _f(r.get('P_LL_kW'),  Sn * (R/Zb))  # kW/fase
        # Condutância de núcleo em p.u.: G_core = P_NL / (V_nom² · S_base)
        G_core_pu    = (P_NL_nom_kw / SBASE) / 1.0**2       # V_nom=1.0 p.u.
        # Elevação nominal do óleo de topo (°C) — padrão IEEE C57.91
        theta_TO_R   = _f(r.get('theta_TO_R'), 55.0)        # °C
        theta_g_R    = _f(r.get('theta_g_R'),  25.0)        # gradiente nominal
        if lv in buses: buses[lv]['v_nom_kv'] = Vbt
        for ph in phs:
            branches[(mv, lv, ph)] = {
                'r_pu': R/Zb, 'x_pu': X/Zb,
                'Imax_pu': Sn/Vbt/Ib, 'level': 'trafo',
                'G_core_pu': G_core_pu,        # condutância de ferro [p.u.]
                'P_NL_nom_kw': P_NL_nom_kw,    # perda vazio nominal [kW/fase]
                'P_LL_R_kw': P_LL_R_kw,        # perda carga nominal [kW/fase]
                'theta_TO_R': theta_TO_R,       # elevação óleo nominal [°C]
                'theta_g_R': theta_g_R,         # gradiente nominal [°C]
                'trafo_id': tid, 'mv_bus': mv, 'lv_bus': lv,
                'nph': nph}
            trafos[tid] = {
                'mv_bus': mv, 'lv_bus': lv, 'phases': phs, 'nph': nph,
                'kva_fase': Sn, 'G_core_pu': G_core_pu,
                'P_NL_nom_kw': P_NL_nom_kw, 'P_LL_R_kw': P_LL_R_kw,
                'theta_TO_R': theta_TO_R,    'theta_g_R': theta_g_R}

    # Completar barras ausentes
    for (fr, to, ph) in list(branches.keys()):
        for b in (fr, to):
            if b not in buses:
                buses[b] = {'level': 'mv', 'v_nom_kv': VMT,
                            'phases': [1,2,3], 'tb': 0}
            if (b, ph) not in loads:
                loads[(b, ph)] = {'P_pu': 0., 'Q_pu': 0.}

    print(f"  Barras: {len(buses)}  Ramos: {len(branches)}")
    print(f"  PVs: {len(pv)}  SVRs: {len(svr)}  Slack: {slack}")
    return {'buses': buses, 'loads': loads, 'loads_mt': loads_mt,
            'pv': pv, 'pv_meta': pv_meta,
            'branches': branches, 'svr': svr, 'slack': slack,
            'trafos': trafos}


# ══════════════════════════════════════════════════════════════════════════════
# WARM-START (forward sweep para inicialização)
# ══════════════════════════════════════════════════════════════════════════════

def compute_warm_start(data, hours, irr, load):
    from collections import deque, defaultdict
    buses    = data['buses']
    loads    = data['loads']
    pv       = data['pv']
    branches = data['branches']
    svr      = data['svr']
    slack    = data['slack']

    adj = defaultdict(list)
    for (fr, to, ph) in branches:
        adj[(fr, ph)].append((to, ph, (fr, to, ph)))
    for ph, d in svr.items():
        adj[(d['mv_bus'], ph)].append((d['lv_bus'], ph, ('SVR', ph)))

    def bfs(root, rphs):
        order, visited = [], set()
        q = deque([(root, ph) for ph in rphs])
        for item in q: visited.add(item)
        while q:
            node = q.popleft(); order.append(node)
            for (to, tph, br) in adj.get(node, []):
                if (to, tph) not in visited:
                    visited.add((to, tph)); q.append((to, tph))
        return order

    order  = bfs(slack, buses[slack]['phases'])
    parent = {}
    for (b, ph) in order:
        for (to, tph, br) in adj.get((b, ph), []):
            if (to, tph) not in parent:
                parent[(to, tph)] = ((b, ph), br)

    V_ws  = {}; P_ws  = {}; Q_ws  = {}
    I2_ws = {}; Ppv_ws = {}; tap_ws = {}

    for t in hours:
        alpha = load[t]; irr_t = irr[t]
        tap_t = {ph: svr[ph]['tap_init'] for ph in svr}

        P_net = {}; Q_net = {}
        for (b, ph) in order:
            # Selecionar perfil de carga: MT vs BT (override em _MT_LOAD_PROFILE_MAP)
            if b in _MT_LOAD_PROFILE_MAP:
                _aw = _LOAD_BY_TYPE[_MT_LOAD_PROFILE_MAP[b]][t]
            else:
                level = buses.get(b, {}).get('level', 'mv')
                _aw = LOAD_PROFILE_MT[t] if level == 'mv' else LOAD_PROFILE[t]
            Pd   = loads.get((b, ph), {'P_pu': 0})['P_pu'] * _aw
            Qd   = loads.get((b, ph), {'Q_pu': 0})['Q_pu'] * _aw
            # PV com derate térmico (mesma função usada no modelo)
            Ppv = pv.get((b, ph), {'P_rated_pu': 0})['P_rated_pu'] * PV_EFF_FACTOR[t]
            Ppv_ws[(b, ph, t)] = Ppv
            P_net[(b, ph)] = Pd - Ppv
            Q_net[(b, ph)] = Qd

        P_br = {}; Q_br = {}
        for (b, ph) in reversed(order):
            pf = P_net[(b, ph)]; qf = Q_net[(b, ph)]
            for (to, tph, br) in adj.get((b, ph), []):
                pf += P_br.get((to, ph), 0)
                qf += Q_br.get((to, ph), 0)
            P_br[(b, ph)] = pf; Q_br[(b, ph)] = qf

        V_ws[(slack, 1, t)] = 1.0
        V_ws[(slack, 2, t)] = 1.0
        V_ws[(slack, 3, t)] = 1.0

        for (b, ph) in order:
            if b == slack:
                V_ws[(b, ph, t)] = 1.0; continue
            if (b, ph) not in parent:
                V_ws[(b, ph, t)] = 0.97; continue
            (up, uph), br_key = parent[(b, ph)]
            Vup = V_ws.get((up, uph, t), 1.0)
            if br_key == ('SVR', ph):
                tap_ws[(ph, t)] = tap_t[ph]
                V_ws[(b, ph, t)] = tap_t[ph] * Vup
                P_ws[('SVR_P', ph, t)] = P_br.get((b, ph), 0)
                Q_ws[('SVR_Q', ph, t)] = Q_br.get((b, ph), 0)
            else:
                (fr, to, p) = br_key
                r = branches[br_key]['r_pu']
                x = branches[br_key]['x_pu']
                Pf = P_br.get((b, ph), 0)
                Qf = Q_br.get((b, ph), 0)
                # Forward sweep com aproximação BFM: V_j≈V_i - rP - xQ + 0.5·z²·l_sq
                # l_sq estimado: (P²+Q²)/Vup² para o warm-start
                z2_ws = r**2 + x**2
                l_sq_ws = (Pf**2 + Qf**2) / max(Vup**2, 0.64)
                V_ws[(b, ph, t)] = max(
                    Vup - r*Pf - x*Qf + 0.5*z2_ws*l_sq_ws, 0.80)
                P_ws[(fr, to, p, t)] = Pf
                Q_ws[(fr, to, p, t)] = Qf
                Vi2 = max(Vup**2, 0.64)
                I2_ws[(fr, to, p, t)] = (Pf**2 + Qf**2) / Vi2

        for ph in svr:
            if (ph, t) not in tap_ws:
                tap_ws[(ph, t)] = svr[ph]['tap_init']

    return V_ws, P_ws, Q_ws, I2_ws, Ppv_ws, tap_ws


# ══════════════════════════════════════════════════════════════════════════════
# CONSTRUÇÃO DO MODELO BFM-SOCP
# ══════════════════════════════════════════════════════════════════════════════


In [ ]:
#@title Internal model implementation 3/12 { display-mode: "form" }
def build_opf(data, hours=HOURS):
    buses    = data['buses']
    branches = data['branches']
    loads    = data['loads']
    loads_mt = data.get('loads_mt', {})
    pv       = data['pv']
    svr      = data['svr']
    slack    = data['slack']

    irr  = {t: IRR_PROFILE[t]  for t in hours}
    load = {t: LOAD_PROFILE[t] for t in hours}

    # ── Conjuntos ────────────────────────────────────────────────────────────
    connected_bph = set()
    for (b, ph) in loads:      connected_bph.add((b, ph))
    for (b, ph) in pv:         connected_bph.add((b, ph))
    for (i, j, ph) in branches:
        connected_bph.add((i, ph)); connected_bph.add((j, ph))
    for ph, d in svr.items():
        connected_bph.add((d['mv_bus'], ph))
        connected_bph.add((d['lv_bus'], ph))

    bph_list  = sorted(connected_bph)
    # V_ref por barra — diferenciado pelo modelo ZIP da carga:
    # • PRODIST Módulo 8 estabelece TR = 1.00 p.u. para todas as barras
    #   regulatoriamente; aqui adotamos uma escolha OPERACIONAL para a F.O.
    #   que difere conforme o modelo ZIP da carga MT.
    # • Cargas com característica de POTÊNCIA CONSTANTE (modelo 1) podem
    #   operar em tensão ligeiramente reduzida (≈0,95 p.u.) sem perda de
    #   atendimento, dado que a potência demandada independe da tensão.
    #   Esta redução libera margem operacional para os reguladores de
    #   tensão e reduz perdas Joule a montante.
    # • Cargas com característica de IMPEDÂNCIA CONSTANTE (modelo 2) ou
    #   CORRENTE CONSTANTE (modelo 5), bem como barras sem carga,
    #   permanecem com V_ref = 1,00 p.u. A referência da F.O. não deve ser
    #   confundida com o ajuste em volts do relé LDC do SVR.
    # • Os limites adequado/precário PRODIST seguem sendo verificados no
    #   pós-processamento independentemente do V_ref operacional.
    _m1_bph = {(b, ph) for (b, ph), ld in loads_mt.items()
                if ld.get('model', 1) == 1}
    # Para a função objetivo: V_ref = 0,95 p.u. somente nas barras MT com
    # carga P-const (modelo 1); V_ref = 1,00 p.u. nas demais barras.
    _vref = {
        (b, ph): (
            MV_VREF_CONST_P_PU
            if (b, ph) in _m1_bph and
               buses.get(b, {}).get('level', 'mv') == 'mv'
            else (MV_VREF_OTHER_PU
                  if buses.get(b, {}).get('level', 'mv') == 'mv'
                  else LV_VREF_PU))
        for (b, ph) in bph_list}

    cph_list  = sorted(branches.keys())
    pvph_list = sorted((b, ph) for (b, ph) in pv if (b, ph) in connected_bph)
    svr_phs   = sorted(svr.keys())
    svr_branches = {(svr[ph]['mv_bus'], svr[ph]['lv_bus'], ph) for ph in svr_phs}

    step_    = svr[svr_phs[0]]['tap_step']
    n_pos    = svr[svr_phs[0]]['n_tap']
    tap_min  = svr[svr_phs[0]]['tap_min']
    tap_max  = svr[svr_phs[0]]['tap_max']
    d_delay  = svr[svr_phs[0]]['delay_periods']

    total_P = sum(v['P_pu'] for (b, ph), v in loads.items()) * max(LOAD_PROFILE)
    P_bnd   = max(total_P * 3.0, 2.0)
    Q_bnd   = P_bnd
    _tan_fp = math.sqrt(1.0 - FP_MIN_VV**2) / FP_MIN_VV

    print(f"\n  BFM-SOCP | BPH={len(bph_list)} CPH={len(cph_list)} "
          f"PVPH={len(pvph_list)} SVR={svr_phs} T={len(hours)}")
    print(f"  Tap: {n_pos} pos  step={step_}  N_TAP_MAX={N_TAP_MAX} degraus/fase")
    print(f"  Delay SVR: {svr[svr_phs[0]]['delay_min']:.1f} min "
          f"→ {d_delay} períodos de {DT_MIN} min (discretização conservadora)")
    print(f"  Envelope inversores: |Qpv|≤{QPV_MAX_FRAC_S:.2f}·S_nom "
          "e P²+Q²≤S_nom²")
    print(f"  {VERSION}: taps/desvio/desequilíbrio/core + RL A/B/C="
          f"{ENABLE_CORE_LOSS_CANDIDATE_SEARCH}")
    print(f"  Implementabilidade: parâmetros/t=0 mudanças | "
          f"ΔQ≤{Q_COMMAND_MAX_STEP_FRAC_S:.2f}·Snom/{DT_MIN}min | "
          f"taps≤tap-base={OP_REQUIRE_TAP_OPS_NO_GREATER_THAN_BASE}")
    print(f"  Pesos primários normalizados: tap={OBJ_WEIGHT_TAP:.2f}, "
          f"desvio={OBJ_WEIGHT_VOLTAGE_DEVIATION:.2f}, "
          f"desequilíbrio={OBJ_WEIGHT_UNBALANCE:.2f}, "
          f"núcleo={OBJ_WEIGHT_CORE:.2f}")
    print(f"  Salvaguarda Joule (W={OBJ_WEIGHT_JOULE_SAFEGUARD:g}) | "
          f"tap cíclico hard={ENFORCE_CYCLIC_TAP} | "
          f"custo terminal={PENALIZE_TERMINAL_TAP} | "
          f"margem V soft={V_SECURITY_MARGIN:.3f} p.u. "
          f"(W={OBJ_WEIGHT_VOLTAGE_MARGIN:g}) | "
          f"night-VAR={ENABLE_NIGHT_VAR}")

    # ── Barras trifásicas para VUF ───────────────────────────────────────────
    bph_set = set(bph_list)
    trif_mt = sorted({b for (b, ph) in bph_list
                      if buses.get(b, {}).get('level') == 'mv'
                      and all((b, p) in bph_set for p in [1, 2, 3])})
    trif_bt = sorted({b for (b, ph) in bph_list
                      if buses.get(b, {}).get('level') == 'lv'
                      and all((b, p) in bph_set for p in [1, 2, 3])})
    all_trif   = sorted(set(trif_mt) | set(trif_bt))
    unbal_pairs = [(1,2),(1,3),(2,3)]

    # ── Coeficientes de carga V-dependente ──────────────────────────────────
    _aP = {(b,ph): loads.get((b,ph),{}).get('aP', 0.0) for (b,ph) in bph_list}
    _aQ = {(b,ph): loads.get((b,ph),{}).get('aQ', 0.0) for (b,ph) in bph_list}

    # Perfil de carga por (barra, fase, período):
    #   - Barras MT (level='mv', exceto slack): TECHNICHDOBRASIL
    #   - Barras BT (level='lv'):               CURVA_R (residencial)
    #   - Override por barra: _MT_LOAD_PROFILE_MAP (industrial/comercial/residencial)
    def _bus_profile(b):
        # Override explícito por barra (se mapeado)
        if b in _MT_LOAD_PROFILE_MAP:
            return _LOAD_BY_TYPE[_MT_LOAD_PROFILE_MAP[b]]
        # Default por level: MT → TECHNICH, BT → CURVA_R
        level = buses.get(b, {}).get('level', 'mv')
        if level == 'mv': return LOAD_PROFILE_MT
        return LOAD_PROFILE
    _alpha_dict = {
        (b, ph, t): _bus_profile(b)[t]
        for (b, ph) in bph_list for t in hours}

    # ── Warm-start ───────────────────────────────────────────────────────────
    V_ws, P_ws, Q_ws, I2_ws, Ppv_ws, tap_ws = compute_warm_start(
        data, hours, irr, load)

    # ── Modelo Pyomo ─────────────────────────────────────────────────────────
    m = pyo.ConcreteModel(name='BFM_SOCP_VVC_144x10min')

    # Sets
    m.T      = pyo.Set(initialize=hours, ordered=True)
    m.T_VV   = pyo.Set(initialize=[t for t in hours if T_VV_START <= t <= T_VV_END])
    m.T_NVV  = pyo.Set(initialize=[t for t in hours if t < T_VV_START or t > T_VV_END])
    m.BPH    = pyo.Set(initialize=bph_list,  dimen=2)
    m.CPH    = pyo.Set(initialize=cph_list,  dimen=3)
    m.PVPH   = pyo.Set(initialize=pvph_list, dimen=2)
    m.SVRPH  = pyo.Set(initialize=svr_phs)

    # Parâmetros
    m.r     = pyo.Param(m.CPH, initialize={k: v['r_pu']    for k,v in branches.items()})
    m.x     = pyo.Param(m.CPH, initialize={k: v['x_pu']    for k,v in branches.items()})
    m.Imax  = pyo.Param(m.CPH, initialize={k: v['Imax_pu'] for k,v in branches.items()})
    m.Pd0   = pyo.Param(m.BPH, initialize={(b,ph): loads.get((b,ph),{'P_pu':0})['P_pu']
                                            for (b,ph) in bph_list})
    m.Qd0   = pyo.Param(m.BPH, initialize={(b,ph): loads.get((b,ph),{'Q_pu':0})['Q_pu']
                                            for (b,ph) in bph_list})
    m.alpha = pyo.Param(m.T, initialize=load)   # BT fallback
    m.alpha_bph = pyo.Param(
        m.BPH, m.T,
        initialize={(b,ph,t): _alpha_dict.get((b,ph,t), load[t])
                    for (b,ph) in bph_list for t in hours})
    m.irr   = pyo.Param(m.T, initialize=irr)
    m.Prat  = pyo.Param(m.PVPH, initialize={(b,ph): pv[(b,ph)]['P_rated_pu']
                                             for (b,ph) in pvph_list})
    m.Snom  = pyo.Param(m.PVPH, initialize={(b,ph): pv[(b,ph)]['S_nom_pu']
                                             for (b,ph) in pvph_list})
    # Pavail considerando temperatura da célula:
    # P_avail(t) = P_STC · irrad(t) · [1 + γ·(T_cell(t)-25)]
    # T_cell(t)  = T_amb(t) + irrad(t)·(NOCT-20)
    # O fator PV_EFF_FACTOR[t] = irrad[t] · derate(t) já encapsula esta relação.
    m.Pavail = pyo.Param(m.PVPH, m.T,
                         initialize={(b,ph,t): pv[(b,ph)]['P_rated_pu']
                                                * PV_EFF_FACTOR[t]
                                     for (b,ph) in pvph_list for t in hours})
    m.Vref   = pyo.Param(m.BPH,
                         initialize={(b,ph): _vref[(b,ph)]
                                     for (b,ph) in bph_list})

    # ── VARIÁVEIS ─────────────────────────────────────────────────────────────

    # Tensão V[b,ph,t] em p.u.
    svr_lv = {svr[ph]['lv_bus'] for ph in svr_phs}
    # V_ref operacional é 0,95/1,00 p.u.; limites regulatórios são separados.

    def v_bounds(m, b, ph, t):
        # Bounds conforme PRODIST Módulo 8 — LIMITE ADEQUADO como teto:
        # - Slack: fixo em 1.0 p.u.
        # - Secundário do SVR (RG60): [0.95, 1.05] — o tap pode estar em
        #   qualquer posição física, mas a tensão resultante respeita o
        #   limite adequado regulatório. Isto evita que o solver use
        #   tap_max=1.10 como artifício para acomodar quedas de tensão
        #   nas pontas, gerando sobretensão na cabeça do alimentador.
        # - Barras MT (1 < V < 69 kV): adequado [0.93, 1.05]
        # - Barras BT (V ≤ 1 kV):       adequado [0.92, 1.05]
        # Os limites adequados do PRODIST são as restrições rígidas. A margem
        # preventiva de V_SECURITY_MARGIN é tratada separadamente como meta
        # soft: o atraso de 15 min do SVR impede que o controlador corrija os
        # primeiros intervalos e não deve, por si só, destruir a factibilidade.
        if b == slack: return (1.0, 1.0)
        if b in svr_lv:
            return (0.95, 1.05)
        if buses.get(b,{}).get('level') == 'mv':
            return (0.93, 1.05)
        return (0.92, 1.05)

    def _v_initialize(m, b, ph, t):
        lo, hi = v_bounds(m, b, ph, t)
        return min(max(V_ws.get((b,ph,t), 0.97), lo), hi)

    m.V = pyo.Var(m.BPH, m.T,
                  domain=pyo.NonNegativeReals,
                  bounds=v_bounds,
                  initialize=_v_initialize)
    # Pontos de linearização de V², atualizados pela SCA. No ponto fixo,
    # 2·V̄·V−V̄² = V², recuperando as relações quadráticas exatas.
    m.V_bar_bus = pyo.Param(
        m.BPH, m.T,
        initialize={(b,ph,t): _v_initialize(None,b,ph,t)
                    for (b,ph) in bph_list for t in hours},
        within=pyo.NonNegativeReals, mutable=True)

    # Fluxos P[i,j,ph,t] e Q[i,j,ph,t]
    m.P = pyo.Var(m.CPH, m.T, domain=pyo.Reals,
                  bounds=(-P_bnd, P_bnd),
                  initialize=lambda m,i,j,ph,t: P_ws.get((i,j,ph,t), 0.0))
    m.Q = pyo.Var(m.CPH, m.T, domain=pyo.Reals,
                  bounds=(-Q_bnd, Q_bnd),
                  initialize=lambda m,i,j,ph,t: Q_ws.get((i,j,ph,t), 0.0))

    # ── l_sq[i,j,ph,t] = |I_ij|²  —  VARIÁVEL CENTRAL DO BFM ────────────────
    # A relação física é l_sq · V_i² ≥ P² + Q². Como o writer QCP usado aqui
    # não aceita o produto l_sq·V_i² variável, cada subproblema usa V̄_i² como
    # parâmetro e o atualiza sucessivamente. A exatidão não é presumida: ela é
    # verificada depois com a tensão V_i efetivamente resolvida.
    # Bound generoso: (P_bnd²+Q_bnd²)/V_min²
    # Imax_pu² subdimensionado para o tronco 650→632 no pico PV (~49 vs 27.7)
    # O termo positivo de perdas favorece o fechamento do cone, mas somente a
    # auditoria residual ex-post autoriza afirmar que a relaxação ficou justa.
    _lsq_ub = (P_bnd**2 + Q_bnd**2) / 0.64   # V_min=0.8 p.u.
    m.l_sq = pyo.Var(m.CPH, m.T,
                     domain=pyo.NonNegativeReals,
                     bounds=(0.0, _lsq_ub),
                     initialize=lambda m,i,j,ph,t: I2_ws.get((i,j,ph,t), 0.0))

    # PV: potência reativa Qpv e curtailment Pcurt
    m.Qpv  = pyo.Var(m.PVPH, m.T, domain=pyo.Reals,
                     bounds=lambda m,b,ph,t: (
                         -(1.0 if DISCOVERY_RELAX_FP_FLOOR else
                           QPV_MAX_FRAC_S)*pv[(b,ph)]['S_nom_pu'],
                          (1.0 if DISCOVERY_RELAX_FP_FLOOR else
                           QPV_MAX_FRAC_S)*pv[(b,ph)]['S_nom_pu']),
                     initialize=0.0)
    m.Pcurt = pyo.Var(m.PVPH, m.T, domain=pyo.NonNegativeReals,
                      bounds=lambda m,b,ph,t: (0.0, pv[(b,ph)]['P_rated_pu']),
                      initialize=0.0)

    # Desvio de tensão linearizado |V - V_ref|
    # dv_pos[b,ph,t] >= V[b,ph,t] - Vref[b,ph]
    # dv_neg[b,ph,t] >= Vref[b,ph] - V[b,ph,t]
    _dv_idx = [(b,ph,t) for (b,ph) in bph_list
               for t in hours if b != slack]
    m.dv_pos = pyo.Var(_dv_idx, domain=pyo.NonNegativeReals,
                        bounds=(0, 0.50), initialize=0.)
    m.dv_neg = pyo.Var(_dv_idx, domain=pyo.NonNegativeReals,
                         bounds=(0, 0.50), initialize=0.)

    # Folgas da margem preventiva. Não relaxam os limites PRODIST acima: medem
    # apenas quanto a solução precisou entrar na faixa de segurança de 0,005
    # p.u. junto às fronteiras adequadas.
    m.v_margin_low = pyo.Var(_dv_idx, domain=pyo.NonNegativeReals,
                             bounds=(0, 0.10), initialize=0.0)
    m.v_margin_high = pyo.Var(_dv_idx, domain=pyo.NonNegativeReals,
                              bounds=(0, 0.10), initialize=0.0)

    # SVR: tap discreto, indicador de movimento e número de degraus percorridos
    m.tap_pos = pyo.Var(m.SVRPH, m.T,
                        domain=pyo.NonNegativeIntegers,
                        bounds=(0, n_pos-1),
                        initialize=lambda m,ph,t: int(round(
                            (svr[ph]['tap_init'] - tap_min) / step_)))
    m.u_tap   = pyo.Var(m.SVRPH, m.T, domain=pyo.Binary, initialize=0)
    m.tap_step_abs = pyo.Var(
        m.SVRPH, m.T, domain=pyo.NonNegativeReals,
        bounds=(0.0, 1.0), initialize=0.0)
    m.tap_terminal_abs = pyo.Var(
        m.SVRPH, domain=pyo.NonNegativeReals,
        bounds=(0.0, n_pos-1), initialize=0.0)

    # SVR: fluxo de potência no regulador
    m.P_svr = pyo.Var(m.SVRPH, m.T, domain=pyo.Reals,
                      bounds=(-P_bnd, P_bnd),
                      initialize=lambda m,ph,t: P_ws.get(('SVR_P',ph,t), 0.0))
    m.Q_svr = pyo.Var(m.SVRPH, m.T, domain=pyo.Reals,
                      bounds=(-Q_bnd, Q_bnd),
                      initialize=lambda m,ph,t: Q_ws.get(('SVR_Q',ph,t), 0.0))

    # Desequilíbrio linearizado: |Vi - Vj|
    _ub_vuf = max(VUF_MAX_MT, VUF_MAX_BT) * math.sqrt(3)
    m.unbal = pyo.Var(
        [(b,ph1,ph2,t) for b in all_trif
         for (ph1,ph2) in unbal_pairs for t in hours],
        domain=pyo.NonNegativeReals,
        bounds=(0, _ub_vuf),
        initialize=0.0)

    # Rampa ΔQpv
    _ramp_bnd = max(pv[(b,ph)]['S_nom_pu'] for (b,ph) in pvph_list) if pvph_list else 0.1
    m.ramp_pos = pyo.Var(m.PVPH, m.T_VV, domain=pyo.NonNegativeReals,
                         bounds=(0, _ramp_bnd*2), initialize=0.0)
    m.ramp_neg = pyo.Var(m.PVPH, m.T_VV, domain=pyo.NonNegativeReals,
                         bounds=(0, _ramp_bnd*2), initialize=0.0)

    # Variável auxiliar para a penalização L1 do uso reativo.
    m.q_abs = pyo.Var(m.PVPH, m.T, domain=pyo.NonNegativeReals,
                      bounds=lambda m,b,ph,t: (0.0, pv[(b,ph)]['S_nom_pu']),
                      initialize=0.0)

    # ── RESTRIÇÕES ────────────────────────────────────────────────────────────

    # R1: Tensão fixa no slack
    m.c_slack = pyo.Constraint(
        m.SVRPH, m.T,
        rule=lambda m,ph,t: m.V[slack,ph,t] == 1.0
        if (slack,ph) in connected_bph else pyo.Constraint.Skip)

    # ── R2: Balanço de potência ativa — BFM EXATO ─────────────────────────────
    # P_entrada_líquida = P_ij - r_ij·l_sq_ij  (perdas descontadas no ramo)
    # Referência: Farivar & Low (2013), eq.(2); Baran & Wu (1989)
    def pred(b, ph): return [(i,j,p) for (i,j,p) in branches if j==b and p==ph]
    def succ(b, ph): return [(i,j,p) for (i,j,p) in branches if i==b and p==ph]
    _core_at_bph = {}
    for (i, j, ph), bd in branches.items():
        g_core = bd.get('G_core_pu', 0.0)
        if g_core > 0.0:
            _core_at_bph[(j, ph)] = _core_at_bph.get((j, ph), 0.0) + g_core

    def bal_P_rule(m, b, ph, t):
        if b == slack: return pyo.Constraint.Skip
        ps = pred(b, ph); ss = succ(b, ph)
        svr_in  = [p for p,d in svr.items() if d['lv_bus']==b and p==ph]
        svr_out = [p for p,d in svr.items() if d['mv_bus']==b and p==ph]
        if not ps and not ss and not svr_in and not svr_out and (b,ph) not in pvph_list:
            return pyo.Constraint.Skip
        # BFM: entrada líquida = P_ij - r_ij·l_sq_ij  (perdas no ramo)
        in_P  = sum(m.P[i,j,p,t] - m.r[i,j,p]*m.l_sq[i,j,p,t] for (i,j,p) in ps)
        out_P = sum(m.P[i,j,p,t] for (i,j,p) in ss)
        svr_i = sum(m.P_svr[p,t] for p in svr_in)
        svr_o = sum(m.P_svr[p,t] for p in svr_out)
        Ppv   = (m.Pavail[b,ph,t] - m.Pcurt[b,ph,t]) if (b,ph) in pvph_list else 0.0
        aP_v  = _aP.get((b,ph), 0.0)
        Pd    = (m.Pd0[b,ph]*m.alpha_bph[b,ph,t]*(1.0 + aP_v*(m.V[b,ph,t]-1.0))
                 if aP_v != 0.0 else m.Pd0[b,ph]*m.alpha_bph[b,ph,t])
        # Perda no núcleo G·V² por aproximação afim sucessiva em V̄:
        # V² ≈ 2·V̄·V−V̄²; torna-se exata quando a SCA converge V̄→V.
        vb = m.V_bar_bus[b,ph,t]
        Pcore = (_core_at_bph.get((b,ph), 0.0) *
                 (2.0*vb*m.V[b,ph,t] - vb**2))
        return in_P - out_P + Ppv + svr_i - svr_o == Pd + Pcore
    m.c_bal_P = pyo.Constraint(m.BPH, m.T, rule=bal_P_rule)

    # ── R3: Balanço de potência reativa — BFM EXATO ───────────────────────────
    # Q_entrada_líquida = Q_ij - x_ij·l_sq_ij  (perdas reativas no ramo)
    def bal_Q_rule(m, b, ph, t):
        if b == slack: return pyo.Constraint.Skip
        ps = pred(b, ph); ss = succ(b, ph)
        svr_in  = [p for p,d in svr.items() if d['lv_bus']==b and p==ph]
        svr_out = [p for p,d in svr.items() if d['mv_bus']==b and p==ph]
        if not ps and not ss and not svr_in and not svr_out and (b,ph) not in pvph_list:
            return pyo.Constraint.Skip
        # BFM: entrada líquida reativa = Q_ij - x_ij·l_sq_ij
        in_Q  = sum(m.Q[i,j,p,t] - m.x[i,j,p]*m.l_sq[i,j,p,t] for (i,j,p) in ps)
        out_Q = sum(m.Q[i,j,p,t] for (i,j,p) in ss)
        svr_i = sum(m.Q_svr[p,t] for p in svr_in)
        svr_o = sum(m.Q_svr[p,t] for p in svr_out)
        Qpv   = m.Qpv[b,ph,t] if (b,ph) in pvph_list else 0.0
        aQ_v  = _aQ.get((b,ph), 0.0)
        Qd    = (m.Qd0[b,ph]*m.alpha_bph[b,ph,t]*(1.0 + aQ_v*(m.V[b,ph,t]-1.0))
                 if aQ_v != 0.0 else m.Qd0[b,ph]*m.alpha_bph[b,ph,t])
        return in_Q - out_Q + Qpv + svr_i - svr_o == Qd
    m.c_bal_Q = pyo.Constraint(m.BPH, m.T, rule=bal_Q_rule)

    # ── R4: Queda de tensão BFM exata em V², linearizada por SCA ──────────────
    # V_j² = V_i² - 2(rP+xQ) + (r²+x²)l. Em cada QCP substitui-se
    # V² por 2·V̄·V−V̄²; no ponto fixo V̄=V e a igualdade original é recuperada.
    def vdrop_rule(m, i, j, ph, t):
        if (i,j,ph) in svr_branches: return pyo.Constraint.Skip
        r_d = branches[(i,j,ph)]['r_pu']
        x_d = branches[(i,j,ph)]['x_pu']
        z2  = r_d**2 + x_d**2
        vbi = m.V_bar_bus[i,ph,t]
        vbj = m.V_bar_bus[j,ph,t]
        vi2_aff = 2.0*vbi*m.V[i,ph,t] - vbi**2
        vj2_aff = 2.0*vbj*m.V[j,ph,t] - vbj**2
        return (vj2_aff == vi2_aff
                - 2.0*r_d*m.P[i,j,ph,t]
                - 2.0*x_d*m.Q[i,j,ph,t]
                + z2*m.l_sq[i,j,ph,t])
    m.c_vdrop = pyo.Constraint(m.CPH, m.T, rule=vdrop_rule)

    # ── R5: Restrição cônica SOCP — cone de Lorentz ───────────────────────────
    # l_sq_ij · V_i² ≥ P_ij² + Q_ij²
    # ── R5: Restrição cônica BFM-SOCP — V² parametrizado pelo warm-start ────
    # Formulação original (bilinear, bloqueada pelo LP writer do Pyomo):
    #   l_sq[ij] · V[i]² >= P[ij]² + Q[ij]²
    #
    # Reformulação com V[i] parametrizado como constante do warm-start:
    #   l_sq[ij] · V_ws²[i] >= P[ij]² + Q[ij]²
    # onde V_ws²[i] = V_ws[i,ph,t]² é um PARÂMETRO (não variável).
    # O LP writer aceita porque:
    #   l_sq · CONSTANTE  → linear em l_sq
    #   P[ij]² + Q[ij]²   → quadrático em P e Q individualmente (sem produto cruzado)
    #   → QP separável: aceito pelo LP writer do Pyomo
    #
    # Validade: exata quando V_ws ≈ V_ótimo. Para planejamento estático com
    # bom warm-start (erro tipicamente < 3%), a solução é praticamente exata.
    # Referência: Jabr (2006) — iteração de ponto fixo para BFM.
    #
    # Adicionar parâmetro V_sq_ws: V_ws[i,ph,t]² por ramo
    # (V upstream do ramo — barra i do ramo (i,j,ph))
    m.V_sq_ws = pyo.Param(
        m.CPH, m.T,
        initialize={(i,j,ph,t): max(V_ws.get((i,ph,t), 0.97)**2, 0.64)
                    for (i,j,ph) in cph_list for t in hours},
        within=pyo.NonNegativeReals,
        mutable=True)
    def socp_rule(m, i, j, ph, t):
        if (i,j,ph) in svr_branches: return pyo.Constraint.Skip
        # l_sq · V_ws²[upstream] >= P² + Q²
        # V_sq_ws é parâmetro → restrição QP separável (LP writer OK)
        return (m.l_sq[i,j,ph,t] * m.V_sq_ws[i,j,ph,t]
                >= m.P[i,j,ph,t]**2 + m.Q[i,j,ph,t]**2)
    m.c_socp = pyo.Constraint(m.CPH, m.T, rule=socp_rule)
    print(f"  SOCP-SCA: {len(cph_list)*len(hours):,} restrições cônicas/QCP "
          f"(V̄ parametrizado e refinado iterativamente)")

    # ── R6: Modelo do SVR — linearização A1 ──────────────────────────────────
    # V_lv = tap_min·V_mv + step·tap_pos  (linear em V_mv e tap_pos)
    def svr_rule(m, ph, t):
        mv = svr[ph]['mv_bus']; lv = svr[ph]['lv_bus']
        if (lv,ph) not in connected_bph: return pyo.Constraint.Skip
        return m.V[lv,ph,t] == tap_min*m.V[mv,ph,t] + step_*m.tap_pos[ph,t]
    m.c_svr = pyo.Constraint(m.SVRPH, m.T, rule=svr_rule)

    # ── R7: Curtailment PV ────────────────────────────────────────────────────
    m.c_pcurt_ub = pyo.Constraint(
        m.PVPH, m.T,
        rule=lambda m,b,ph,t: m.Pcurt[b,ph,t] <= m.Pavail[b,ph,t])
    m.c_pcurt_night = pyo.Constraint(
        m.PVPH, m.T,
        rule=lambda m,b,ph,t: (m.Pcurt[b,ph,t] == 0.0)
        if irr[t] < 1e-6 else pyo.Constraint.Skip)

    # ── R8: capacidade aparente do inversor ──────────────────────────────────
    # A capacidade reativa é determinada por S_nom, não por um limite de FP
    # aplicado ao despacho ativo. Assim, curtailment libera capacidade de Q em
    # vez de reduzi-la artificialmente.
    m.c_pv_capability = pyo.Constraint(
        m.PVPH, m.T,
        rule=lambda m,b,ph,t:
        (m.Pavail[b,ph,t] - m.Pcurt[b,ph,t])**2 + m.Qpv[b,ph,t]**2
        <= m.Snom[b,ph]**2)

    # O alvo ótimo Qpv*(t) deve pertencer ao mesmo espaço operacional usado
    # depois na classificação A/B/C. Estas duas desigualdades lineares impõem
    # |FP|>=0,85 ainda dentro do OPF, evitando ajustar políticas a um Q* inviável.
    _k_fp085 = math.sqrt(1.0-FP_MIN_VV**2)/FP_MIN_VV
    def _discovery_fp_pos_rule(m, b, ph, t):
        if DISCOVERY_RELAX_FP_FLOOR:
            return pyo.Constraint.Skip
        return (m.Qpv[b,ph,t] <=
                _k_fp085*(m.Pavail[b,ph,t]-m.Pcurt[b,ph,t]))

    def _discovery_fp_neg_rule(m, b, ph, t):
        if DISCOVERY_RELAX_FP_FLOOR:
            return pyo.Constraint.Skip
        return (-m.Qpv[b,ph,t] <=
                _k_fp085*(m.Pavail[b,ph,t]-m.Pcurt[b,ph,t]))

    m.c_qpv_fp085_pos = pyo.Constraint(
        m.PVPH, m.T, rule=_discovery_fp_pos_rule)
    m.c_qpv_fp085_neg = pyo.Constraint(
        m.PVPH, m.T, rule=_discovery_fp_neg_rule)

    # ── R9: Qpv = 0 fora da janela VV ────────────────────────────────────────
    m.c_qpv_nvv = pyo.Constraint(
        m.PVPH, m.T_NVV,
        rule=lambda m,b,ph,t: m.Qpv[b,ph,t] == 0.0)

    # Sem a função comercial de VAR noturno, não é aceitável usar toda a
    # potência aparente do inversor quando a geração ativa é quase nula.
    m.c_qpv_low_solar = pyo.Constraint(
        m.PVPH, m.T,
        rule=lambda m,b,ph,t: m.Qpv[b,ph,t] == 0.0
        if (not ENABLE_NIGHT_VAR and PV_EFF_FACTOR[t] < QPV_MIN_SOLAR_FRAC)
        else pyo.Constraint.Skip)

    # ── R10: Suavização ΔQpv ─────────────────────────────────────────────────
    t_vv_list = sorted([t for t in hours if T_VV_START <= t <= T_VV_END])
    _w_ramp   = {t: (W_RAMP_PICO if IRR_PROFILE[t] > IRR_PICO else W_RAMP_DIA)
                 for t in t_vv_list}
    def ramp_rule(m, b, ph, t):
        prev = m.Qpv[b,ph,t-1] if t-1 in hours else 0.0
        return m.ramp_pos[b,ph,t] - m.ramp_neg[b,ph,t] == m.Qpv[b,ph,t] - prev
    m.c_ramp = pyo.Constraint(
        [(b,ph,t) for (b,ph) in pvph_list for t in t_vv_list],
        rule=ramp_rule)

    # Uso de reativo em norma L1: q_abs >= |Qpv|.
    m.c_q_abs_pos = pyo.Constraint(
        m.PVPH, m.T,
        rule=lambda m,b,ph,t: m.q_abs[b,ph,t] >= m.Qpv[b,ph,t])
    m.c_q_abs_neg = pyo.Constraint(
        m.PVPH, m.T,
        rule=lambda m,b,ph,t: m.q_abs[b,ph,t] >= -m.Qpv[b,ph,t])

    # Um inversor trifásico é um único equipamento: suas fases recebem o
    # mesmo despacho de Q e o mesmo curtailment.
    _pv3_buses = sorted({b for (b,ph) in pvph_list
                         if pv[(b,ph)].get('is_3ph', False)})
    _pv3_pairs = []
    for _b3 in _pv3_buses:
        _phases3 = sorted(p for (bb,p) in pvph_list if bb == _b3)
        if _phases3:
            _pv3_pairs.extend((_b3, ph, _phases3[0]) for ph in _phases3[1:])
    m.c_q_3ph = pyo.Constraint(
        [(b,ph,ph_ref,t) for (b,ph,ph_ref) in _pv3_pairs for t in hours],
        rule=lambda m,b,ph,ph_ref,t: m.Qpv[b,ph,t] == m.Qpv[b,ph_ref,t]
        if ENABLE_3PH_INVERTER_COUPLING else pyo.Constraint.Skip)
    m.c_pcurt_3ph = pyo.Constraint(
        [(b,ph,ph_ref,t) for (b,ph,ph_ref) in _pv3_pairs for t in hours],
        rule=lambda m,b,ph,ph_ref,t: m.Pcurt[b,ph,t] == m.Pcurt[b,ph_ref,t]
        if ENABLE_3PH_INVERTER_COUPLING else pyo.Constraint.Skip)

    # ── R11: Movimento de tap e |tap(t)-tap(t-1)| ─────────────────────────────
    # CORREÇÃO CRÍTICA: a referência inicial é a POSIÇÃO INICIAL REAL do tap
    # (svr[ph]['tap_init']), não zero. Usar 0 contaria uma operação fantasma
    # no início para qualquer tap_init > tap_min.
    # Posição inteira inicial por fase (calculada uma vez para uso nas regras)
    _tap_init_pos = {
        ph: int(round((svr[ph]['tap_init'] - tap_min) / step_))
        for ph in svr_phs
    }
    def _tap_prev(m, ph, t):
        # Posição de tap no período anterior; em t=0 retorna a posição inicial
        if t == hours[0]:
            return _tap_init_pos[ph]
        return m.tap_pos[ph, hours[hours.index(t)-1]]
    # Um comando desloca no máximo UM degrau. Como tap_pos é inteiro no MISOCP,
    # u_tap=1 equivale a um movimento mecânico; na relaxação contínua ele fornece
    # a envoltória convexa do mesmo vínculo.
    m.c_tap_up = pyo.Constraint(
        m.SVRPH, m.T,
        rule=lambda m,ph,t:
        m.tap_pos[ph,t] - _tap_prev(m, ph, t)
        <= m.u_tap[ph,t])
    m.c_tap_dn = pyo.Constraint(
        m.SVRPH, m.T,
        rule=lambda m,ph,t:
        _tap_prev(m, ph, t) - m.tap_pos[ph,t]
        <= m.u_tap[ph,t])

    # Epígrafo exato do número de degraus: tap_step_abs=|Δtap| no ótimo.
    m.c_tap_abs_up = pyo.Constraint(
        m.SVRPH, m.T,
        rule=lambda m,ph,t:
        m.tap_step_abs[ph,t] >= m.tap_pos[ph,t] - _tap_prev(m, ph, t))
    m.c_tap_abs_dn = pyo.Constraint(
        m.SVRPH, m.T,
        rule=lambda m,ph,t:
        m.tap_step_abs[ph,t] >= _tap_prev(m, ph, t) - m.tap_pos[ph,t])
    m.c_tap_event_link = pyo.Constraint(
        m.SVRPH, m.T,
        rule=lambda m,ph,t: m.u_tap[ph,t] <= m.tap_step_abs[ph,t])

    # ── R12: Limite de degraus mecânicos percorridos ──────────────────────────
    m.c_tap_max = pyo.Constraint(
        m.SVRPH,
        rule=lambda m,ph:
        sum(m.tap_step_abs[ph,t] for t in hours) <= N_TAP_MAX)

    # ── R13: Delay do SVR ─────────────────────────────────────────────────────
    m.c_tap_delay = pyo.Constraint(
        m.SVRPH, m.T,
        rule=lambda m,ph,t: m.u_tap[ph,t] == 0
        if hours.index(t) < d_delay else pyo.Constraint.Skip)

    # O grid não representa exatamente 15 min. ceil(15/10)=2 torna o modelo
    # conservador: não há movimentos em amostras consecutivas, e o primeiro
    # movimento só pode ocorrer após duas amostras completas.
    def _tap_spacing_rule(m, ph, t):
        idx = hours.index(t)
        lo = max(0, idx - d_delay + 1)
        if idx == 0:
            return pyo.Constraint.Skip
        return sum(m.u_tap[ph, hours[k]] for k in range(lo, idx + 1)) <= 1
    m.c_tap_spacing = pyo.Constraint(m.SVRPH, m.T, rule=_tap_spacing_rule)

    # Fechamento hard somente para perfis realmente cíclicos. Neste caso real,
    # carga(t=0) != carga(t=T); forçar tap(T)=tap(0) pode violar tensão ou o teto
    # de operações. O custo terminal abaixo contabiliza o retorno futuro sem
    # impor que ele ocorra artificialmente no último intervalo.
    m.c_tap_terminal = pyo.Constraint(
        m.SVRPH,
        rule=lambda m,ph: m.tap_pos[ph,hours[-1]] == _tap_init_pos[ph]
        if ENFORCE_CYCLIC_TAP else pyo.Constraint.Skip)
    m.c_tap_terminal_abs_up = pyo.Constraint(
        m.SVRPH,
        rule=lambda m,ph:
        m.tap_terminal_abs[ph] >= m.tap_pos[ph,hours[-1]] - _tap_init_pos[ph])
    m.c_tap_terminal_abs_dn = pyo.Constraint(
        m.SVRPH,
        rule=lambda m,ph:
        m.tap_terminal_abs[ph] >= _tap_init_pos[ph] - m.tap_pos[ph,hours[-1]])

    # ── R14: Linearização |Vi - Vj| para VUF ─────────────────────────────────
    m.c_unbal_pos = pyo.Constraint(
        [(b,ph1,ph2,t) for b in all_trif for (ph1,ph2) in unbal_pairs for t in hours],
        rule=lambda m,b,ph1,ph2,t: m.unbal[b,ph1,ph2,t] >= m.V[b,ph1,t]-m.V[b,ph2,t])
    m.c_unbal_neg = pyo.Constraint(
        [(b,ph1,ph2,t) for b in all_trif for (ph1,ph2) in unbal_pairs for t in hours],
        rule=lambda m,b,ph1,ph2,t: m.unbal[b,ph1,ph2,t] >= m.V[b,ph2,t]-m.V[b,ph1,t])

    # ── R15: FD% — PRODIST Módulo 8 (SOFT via F.O., não hard constraint) ────
    # A restrição hard de FD% pode tornar o problema infeasível quando o
    # desequilíbrio inerente da rede (cargas delta por fase, topologia assimétrica)
    # excede o limite regulatório antes de qualquer controle.
    # Solução: manter como termo primário normalizado Σ|Vi-Vj| na F.O.
    # A conformidade FD95% é verificada e reportada no pós-processamento.
    # Para ativar como hard: descomentar o bloco abaixo.
    #
    # def _vuf_lim(b):
    #     lv = buses.get(b, {}).get('level', 'mv')
    #     fd = FD_BT_MAX if lv == 'lv' else FD_MT_MAX
    #     return fd / (K_FD_VUF * 100) * math.sqrt(3)
    # m.c_vuf_hard = pyo.Constraint(
    #     [(b,ph1,ph2,t) for b in all_trif for (ph1,ph2) in unbal_pairs for t in hours],
    #     rule=lambda m,b,ph1,ph2,t: m.unbal[b,ph1,ph2,t] <= _vuf_lim(b))

    # ── R16: Desvio |V - V_ref| linearizado ─────────────────────────────────
    # V_ref = 0,95 p.u. para barras MT com carga modelo 1 (P-const)
    # V_ref = 1,00 p.u. para todas as demais barras.
    m.c_dvp = pyo.Constraint(
        _dv_idx,
        rule=lambda m,b,ph,t: m.dv_pos[b,ph,t] >= m.V[b,ph,t] - m.Vref[b,ph])
    m.c_dvn = pyo.Constraint(
        _dv_idx,
        rule=lambda m,b,ph,t: m.dv_neg[b,ph,t] >= m.Vref[b,ph] - m.V[b,ph,t])

    # R17: margem preventiva de tensão como meta soft. Para RG60, o limite
    # inferior já é 0,95 p.u.; nas demais barras a meta fica 0,005 p.u. para
    # dentro da faixa adequada do PRODIST.
    def _v_margin_targets(b):
        if b in svr_lv:
            return 0.95, 1.05 - V_SECURITY_MARGIN
        if buses.get(b, {}).get('level') == 'mv':
            return 0.93 + V_SECURITY_MARGIN, 1.05 - V_SECURITY_MARGIN
        return 0.92 + V_SECURITY_MARGIN, 1.05 - V_SECURITY_MARGIN

    m.c_v_margin_low = pyo.Constraint(
        _dv_idx,
        rule=lambda m,b,ph,t:
        m.v_margin_low[b,ph,t] >= _v_margin_targets(b)[0] - m.V[b,ph,t])
    m.c_v_margin_high = pyo.Constraint(
        _dv_idx,
        rule=lambda m,b,ph,t:
        m.v_margin_high[b,ph,t] >= m.V[b,ph,t] - _v_margin_targets(b)[1])

    # ── FUNÇÃO OBJETIVO V3.7.6 ──────────────────────────────────────────────
    # Quatro metas primárias, todas adimensionais após normalização:
    #   0,55·N_tap/N_tap,ref + 0,20·Σ|V-Vref|/D_ref
    # + 0,10·Σ|Vφ-Vψ|/U_ref + 0,15·E_core/E_core,ref.
    # O peso de taps é maior que a soma dos demais. A garantia de que nenhum
    # termo piora é feita adicionalmente na seleção física ex-post por
    # dominância; pesos sozinhos não oferecem essa garantia matemática.
    _dt_h = DT_MIN / 60.0
    _joule_loss = sum(m.r[i,j,ph] * m.l_sq[i,j,ph,t]
                       for (i,j,ph) in cph_list for t in hours)
    _core_loss = sum(
        g * (2.0*m.V_bar_bus[b,ph,t]*m.V[b,ph,t]
             - m.V_bar_bus[b,ph,t]**2)
        for (b,ph), g in _core_at_bph.items() for t in hours)

    # Escalas fixas calculadas antes da otimização. Elas não dependem das
    # variáveis e são copiadas para todos os replays físicos do modelo.
    _eps_norm = 1.0e-9
    _tap_norm = max(float(N_TAP_MAX*len(svr_phs)), 1.0)
    _dv_norm = max(sum(
        abs(V_ws.get((b,ph,t), _vref[(b,ph)])-_vref[(b,ph)])
        for (b,ph,t) in _dv_idx), _eps_norm)
    _unbal_norm = max(sum(
        abs(V_ws.get((b,ph1,t), 1.0)-V_ws.get((b,ph2,t), 1.0))
        for b in all_trif for ph1,ph2 in unbal_pairs for t in hours),
        _eps_norm)
    _core_norm = max(_dt_h*sum(
        g*V_ws.get((b,ph,t), 1.0)**2
        for (b,ph),g in _core_at_bph.items() for t in hours), _eps_norm)
    _joule_norm = max(_dt_h*sum(
        branches[(i,j,ph)]['r_pu']*I2_ws.get((i,j,ph,t), 0.0)
        for (i,j,ph) in cph_list for t in hours), _eps_norm)
    _margin_norm = max(V_SECURITY_MARGIN*len(_dv_idx), _eps_norm)
    _q_use_norm = max(sum(
        pv[(b,ph)]['S_nom_pu'] for (b,ph) in pvph_list for _t in hours),
        _eps_norm)
    _q_ramp_norm = max(sum(
        _w_ramp[t]*pv[(b,ph)]['S_nom_pu']
        for (b,ph) in pvph_list for t in t_vv_list), _eps_norm)
    _curt_norm = max(sum(
        pv[(b,ph)]['P_rated_pu']*PV_EFF_FACTOR[t]
        for (b,ph) in pvph_list for t in hours), _eps_norm)
    for _name, _value in {
            'tap': _tap_norm, 'voltage_deviation': _dv_norm,
            'unbalance': _unbal_norm, 'core': _core_norm,
            'joule': _joule_norm, 'voltage_margin': _margin_norm,
            'q_use': _q_use_norm, 'q_ramp': _q_ramp_norm,
            'curtailment': _curt_norm}.items():
        setattr(m, f'obj_norm_{_name}', pyo.Param(initialize=float(_value)))

    _tap_term = (
        sum(m.tap_step_abs[ph,t] for ph in svr_phs for t in hours) +
        (sum(m.tap_terminal_abs[ph] for ph in svr_phs)
         if PENALIZE_TERMINAL_TAP else 0.0))
    _dv_term = sum(m.dv_pos[b,ph,t] + m.dv_neg[b,ph,t]
                   for (b,ph,t) in _dv_idx)
    _margin_term = sum(
        m.v_margin_low[b,ph,t] + m.v_margin_high[b,ph,t]
        for (b,ph,t) in _dv_idx)
    _unbal_term = sum(m.unbal[b,ph1,ph2,t]
                      for b in all_trif for ph1,ph2 in unbal_pairs
                      for t in hours)
    _q_use_term = sum(m.q_abs[b,ph,t]
                      for (b,ph) in pvph_list for t in hours)
    _q_ramp_term = sum(
        _w_ramp[t]*(m.ramp_pos[b,ph,t]+m.ramp_neg[b,ph,t])
        for (b,ph) in pvph_list for t in t_vv_list)
    _curtailment_term = sum(m.Pcurt[b,ph,t]
                            for (b,ph) in pvph_list for t in hours)
    m.obj = pyo.Objective(
        expr=(
            OBJ_WEIGHT_TAP*_tap_term/m.obj_norm_tap
            + OBJ_WEIGHT_VOLTAGE_DEVIATION*_dv_term/m.obj_norm_voltage_deviation
            + OBJ_WEIGHT_UNBALANCE*_unbal_term/m.obj_norm_unbalance
            + OBJ_WEIGHT_CORE*(_dt_h*_core_loss)/m.obj_norm_core
            + OBJ_WEIGHT_VOLTAGE_MARGIN*_margin_term/m.obj_norm_voltage_margin
            + OBJ_WEIGHT_JOULE_SAFEGUARD*(_dt_h*_joule_loss)/m.obj_norm_joule
            + OBJ_WEIGHT_Q_USE_SAFEGUARD*_q_use_term/m.obj_norm_q_use
            + OBJ_WEIGHT_Q_RAMP_SAFEGUARD*_q_ramp_term/m.obj_norm_q_ramp
            + OBJ_WEIGHT_CURTAILMENT*_curtailment_term/m.obj_norm_curtailment
        ),
        sense=pyo.minimize
    )
    n_vars   = sum(1 for v in m.component_data_objects(pyo.Var, active=True))
    n_constr = sum(1 for c in m.component_data_objects(pyo.Constraint, active=True))
    n_bin    = sum(1 for v in m.component_data_objects(pyo.Var, active=True)
                   if v.is_binary())
    print(f"  Variáveis: {n_vars:,}  ({n_bin} binárias)")
    print(f"  Restrições: {n_constr:,}")
    print(f"  (inclui {len(cph_list)*len(hours):,} cônicas SOCP)")

    return m, {
        'bph': bph_list, 'cph': cph_list, 'pvph': pvph_list,
        'svr_phs': svr_phs, 'hours': hours, 'irr': irr, 'load': load,
        't_vv': (T_VV_START, T_VV_END), 'trif_buses': all_trif,
        'tap_step': step_, 'tap_min': tap_min, 'n_pos': n_pos,
        'tap_delay_periods': d_delay, 'n_tap_max': N_TAP_MAX,
    }, I2_ws


# ══════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO
# ══════════════════════════════════════════════════════════════════════════════


# =============================================================================
# GRÁFICOS — BFM-SOCP (adaptado do LinDistFlow v4)
# No BFM-SOCP: m.V[b,ph,t] em p.u. — mesma interface do LinDistFlow.
# =============================================================================

def try_v(m, b, ph, t):
    """Extrai tensão em p.u. do modelo BFM-SOCP."""
    try:
        return pyo.value(m.V[b, ph, t])
    except Exception:
        return 1.0


# =============================================================================
# HELPERS CENTRALIZADOS PARA CÁLCULO DE FD% (PRODIST Módulo 8)
# =============================================================================
# Limitação estrutural reconhecida:
# O modelo BFM-SOCP fornece apenas MAGNITUDES de tensão por fase (sem ângulos
# explícitos). As tensões de linha são reconstruídas assumindo defasagem rígida
# de 120° entre fases. Isto significa que o FD% computado captura apenas o
# componente de desequilíbrio de AMPLITUDE; o componente angular não é
# representado. Esta limitação deve ser declarada explicitamente no paper/tese.
# =============================================================================

def _calc_vll_from_vph(V1, V2, V3):
    """Calcula tensões de linha (V_ab, V_bc, V_ca) a partir das tensões de fase
    assumindo defasagem ideal de 120°. cos(120°) = -1/2.
    V_ab² = V1² + V2² - 2·V1·V2·cos(120°) = V1² + V2² + V1·V2."""
    return (math.sqrt(max(V1**2 + V2**2 + V1*V2, 0.0)),
            math.sqrt(max(V2**2 + V3**2 + V2*V3, 0.0)),
            math.sqrt(max(V3**2 + V1**2 + V3*V1, 0.0)))


def _calc_fd_prodist(Vab, Vbc, Vca):
    """Calcula FD% pela fórmula alternativa do PRODIST Módulo 8
    (Submódulo 8.1, Seção 4.2 — Desequilíbrio de Tensão):

        beta = (Vab^4 + Vbc^4 + Vca^4) / (Vab^2 + Vbc^2 + Vca^2)^2
        FD%  = 100 * sqrt[(1 - sqrt(3 - 6·beta)) / (1 + sqrt(3 - 6·beta))]

    Esta fórmula é equivalente ao método das componentes simétricas
    (V-/V+) em regime senoidal puro, e dispensa a decomposição em
    sequência positiva/negativa.
    """
    S2 = Vab**2 + Vbc**2 + Vca**2
    if S2 < 1e-12:
        return 0.0
    beta = (Vab**4 + Vbc**4 + Vca**4) / S2**2
    arg = max(0.0, min(1.0, 3.0 - 6.0*beta))
    if abs(1.0 - arg) < 1e-14:
        return 0.0
    sqarg = math.sqrt(arg)
    return 100.0 * math.sqrt((1.0 - sqarg) / (1.0 + sqarg))


In [ ]:
#@title Internal model implementation 4/12 { display-mode: "form" }
def calc_fd_percent(m, data, sets_info):
    """Calcula FD% (PRODIST Módulo 8) a partir de m.V — retorna estatísticas ricas.

    Retorna dicionário com:
      fd_per_bus    : {bus: [FD%(t) para t in hours]}
      fd95_per_bus  : {bus: percentil 95 (proxy do FD95% regulatório)}
      fd_max_per_bus: {bus: pico instantâneo}
      t_peak_per_bus: {bus: t do pico (índice de período)}
      fd_all        : np.array com TODAS amostras (bus × tempo) flat
      fd95_system   : percentil 95 sistêmico (proxy)
      fd_max_system : pico instantâneo do sistema
      t_peak_system : período do pico sistêmico
      b_peak_system : barra do pico sistêmico
      pct_time_viol : % de amostras (bus,t) excedendo limite PRODIST
      n_viol        : nº absoluto de amostras (bus,t) em violação
      n_total       : nº total de amostras
      critical_buses: barras cuja FD95% supera o limite PRODIST
    """
    import numpy as np_fd
    buses  = data['buses']
    hours  = sets_info['hours']
    bph    = sets_info['bph']
    bph_set = set(bph)
    all_trif = sorted({b for (b, ph) in bph
                        if all((b, p) in bph_set for p in [1,2,3])})

    fd_per_bus, fd95_per_bus, fd_max_per_bus = {}, {}, {}
    t_peak_per_bus = {}
    critical_buses = []
    n_viol = 0
    n_total = 0
    fd_all_flat = []
    fd_max_system = 0.0
    t_peak_system = hours[0]
    b_peak_system = all_trif[0] if all_trif else None

    for b in all_trif:
        lv = buses.get(b, {}).get('level', 'mv')
        lim = FD_BT_MAX if lv == 'lv' else FD_MT_MAX
        fd_s = []
        t_pk_b = hours[0]
        fd_pk_b = 0.0
        for t in hours:
            V1 = try_v(m, b, 1, t)
            V2 = try_v(m, b, 2, t)
            V3 = try_v(m, b, 3, t)
            fd = _calc_fd_prodist(*_calc_vll_from_vph(V1, V2, V3))
            fd_s.append(fd)
            fd_all_flat.append(fd)
            n_total += 1
            if fd > lim:
                n_viol += 1
            if fd > fd_pk_b:
                fd_pk_b = fd
                t_pk_b = t
            if fd > fd_max_system:
                fd_max_system = fd
                t_peak_system = t
                b_peak_system = b
        fd_per_bus[b] = fd_s
        fd95_b = float(np_fd.percentile(fd_s, 95))
        fd95_per_bus[b] = fd95_b
        fd_max_per_bus[b] = fd_pk_b
        t_peak_per_bus[b] = t_pk_b
        if fd95_b > lim:
            critical_buses.append((b, fd95_b, lim))

    fd_all_arr = np_fd.array(fd_all_flat) if fd_all_flat else np_fd.array([0.0])
    fd95_system = float(np_fd.percentile(fd_all_arr, 95))
    pct_time_viol = (100.0 * n_viol / n_total) if n_total else 0.0

    return {
        'fd_per_bus':     fd_per_bus,
        'fd95_per_bus':   fd95_per_bus,
        'fd_max_per_bus': fd_max_per_bus,
        't_peak_per_bus': t_peak_per_bus,
        'fd_all':         fd_all_arr,
        'fd95_system':    fd95_system,
        'fd_max_system':  fd_max_system,
        't_peak_system':  t_peak_system,
        'b_peak_system':  b_peak_system,
        'pct_time_viol':  pct_time_viol,
        'n_viol':         n_viol,
        'n_total':        n_total,
        'critical_buses': critical_buses,
    }


def summarize_voltage_quality(m, data, sets_info):
    """Resume tensão e indicadores PRODIST usados na comparação ex-post."""
    out = {}
    for level, adequate_lo, precarious_lo in (
            ('mv', 0.93, 0.90), ('lv', 0.92, 0.87)):
        values = [try_v(m, b, ph, t)
                  for (b, ph) in sets_info['bph']
                  if data['buses'].get(b, {}).get('level', 'mv') == level
                  for t in sets_info['hours']]
        if not values:
            out[level] = {'v_min': float('nan'), 'v_max': float('nan'),
                          'v_mean': float('nan'), 'drp': 0.0, 'drc': 0.0,
                          'n': 0}
            continue
        n = len(values)
        n_prec = sum((precarious_lo <= v < adequate_lo) or
                     (1.05 < v <= 1.06) for v in values)
        n_crit = sum(v < precarious_lo or v > 1.06 for v in values)
        out[level] = {
            'v_min': min(values), 'v_max': max(values),
            'v_mean': sum(values) / n,
            'drp': 100.0 * n_prec / n,
            'drc': 100.0 * n_crit / n,
            'n': n,
        }
    out['fd'] = calc_fd_percent(m, data, sets_info)
    return out



def assess_representative_day_quality(quality, validation_bound_hits=0,
                                      baseline_quality=None,
                                      period_count=144):
    """Separa triagem diária de conformidade regulatória formal.

    Em um dia representativo, DRP/DRC e FD95 são métricas de screening. Quando
    uma baseline S1 é fornecida, a aceitação usa uma regra de não piora: cada
    indicador do candidato deve permanecer abaixo do maior valor entre o limite
    de screening e o valor observado em S1 (mais tolerância numérica). Assim,
    uma baseline que já exceda o limiar diário não transforma automaticamente
    todos os candidatos em "não conformes". A conformidade regulatória formal
    não é declarada neste horizonte; ela requer a avaliação semanal de 1008
    amostras no procedimento dedicado.
    """
    if quality is None:
        return {
            'daily_screening_ok': False,
            'daily_screening_absolute_ok': False,
            'daily_screening_nonworsening_ok': False,
            'daily_screening_mode': 'unavailable',
            'formal_regulatory_assessed': False,
            'formal_regulatory_ok': None,
            'formal_regulatory_reason': 'quality_unavailable',
            'period_count': int(period_count),
            'required_formal_periods': FORMAL_REGULATORY_WINDOW_PERIODS,
            'limits': {},
        }

    mv = quality['mv']; lv = quality['lv']; fd = quality['fd']
    absolute_ok = bool(
        mv['drp'] <= DAILY_SCREEN_DRP_MAX_PCT and
        mv['drc'] <= DAILY_SCREEN_DRC_MAX_PCT and
        lv['drp'] <= DAILY_SCREEN_DRP_MAX_PCT and
        lv['drc'] <= DAILY_SCREEN_DRC_MAX_PCT and
        fd['fd95_system'] <= FD_MT_MAX and
        int(validation_bound_hits) == 0)

    limits = {
        'mv_drp_pct': DAILY_SCREEN_DRP_MAX_PCT,
        'mv_drc_pct': DAILY_SCREEN_DRC_MAX_PCT,
        'lv_drp_pct': DAILY_SCREEN_DRP_MAX_PCT,
        'lv_drc_pct': DAILY_SCREEN_DRC_MAX_PCT,
        'fd95_system_pct': FD_MT_MAX,
    }
    mode = 'absolute_daily_screen'
    if baseline_quality is not None:
        bmv = baseline_quality['mv']; blv = baseline_quality['lv']
        bfd = baseline_quality['fd']
        limits = {
            'mv_drp_pct': max(DAILY_SCREEN_DRP_MAX_PCT,
                              float(bmv['drp'])) + DAILY_SCREEN_METRIC_TOL_PCT,
            'mv_drc_pct': max(DAILY_SCREEN_DRC_MAX_PCT,
                              float(bmv['drc'])) + DAILY_SCREEN_METRIC_TOL_PCT,
            'lv_drp_pct': max(DAILY_SCREEN_DRP_MAX_PCT,
                              float(blv['drp'])) + DAILY_SCREEN_METRIC_TOL_PCT,
            'lv_drc_pct': max(DAILY_SCREEN_DRC_MAX_PCT,
                              float(blv['drc'])) + DAILY_SCREEN_METRIC_TOL_PCT,
            'fd95_system_pct': max(FD_MT_MAX,
                                   float(bfd['fd95_system'])) +
                               DAILY_SCREEN_METRIC_TOL_PCT,
        }
        mode = 'nonworsening_vs_S1_daily_screen'

    nonworsening_ok = bool(
        mv['drp'] <= limits['mv_drp_pct'] and
        mv['drc'] <= limits['mv_drc_pct'] and
        lv['drp'] <= limits['lv_drp_pct'] and
        lv['drc'] <= limits['lv_drc_pct'] and
        fd['fd95_system'] <= limits['fd95_system_pct'] and
        int(validation_bound_hits) == 0)

    return {
        'daily_screening_ok': nonworsening_ok,
        'daily_screening_absolute_ok': absolute_ok,
        'daily_screening_nonworsening_ok': nonworsening_ok,
        'daily_screening_mode': mode,
        'formal_regulatory_assessed': False,
        'formal_regulatory_ok': None,
        'formal_regulatory_reason': (
            'representative_day_only__formal_assessment_requires_1008_samples'),
        'period_count': int(period_count),
        'required_formal_periods': FORMAL_REGULATORY_WINDOW_PERIODS,
        'limits': limits,
    }


def _vv_setting_eval(v, setting):
    """Avalia Volt-VAR; Q mantém a unidade declarada em ``Q_max``."""
    v1, v2 = setting['V1'], setting['V2']
    v3, v4 = setting['V3'], setting['V4']
    qmax = setting['Q_max']
    if v <= v1:
        return qmax
    if v <= v2:
        return qmax * (v2 - v) / max(v2 - v1, 1e-9)
    if v <= v3:
        return 0.0
    if v <= v4:
        return -qmax * (v - v3) / max(v4 - v3, 1e-9)
    return -qmax


def _vv_setting_eval_pu(v_pu, setting):
    """Avalia Volt-VAR com entrada e saída estritamente em p.u.

    O replay elétrico prefere ``Q_max_pu``. Para objetos legados sem unidade
    declarada, ``Q_max`` é interpretado como kVAr e convertido uma única vez.
    """
    if 'Q_max_pu' in setting:
        qmax_pu = float(setting['Q_max_pu'])
    elif setting.get('electrical_units') == 'per_unit':
        qmax_pu = float(setting.get('Q_max', 0.0))
    else:
        qmax_pu = float(setting.get('Q_max', 0.0))/SBASE
    setting_pu = {
        'V1': float(setting['V1']), 'V2': float(setting['V2']),
        'V3': float(setting['V3']), 'V4': float(setting['V4']),
        'Q_max': qmax_pu,
    }
    return _vv_setting_eval(float(v_pu), setting_pu)


def _curves_for_static_profile(inferred_B, data, profile):
    """Materializa um perfil Volt-VAR único e estático para todos os períodos.

    ``inferred_B`` é usado apenas para preservar a identidade dos inversores.
    Os parâmetros selecionados vêm de ``OPERATIONAL_VV_PROFILES`` e não têm
    índice temporal, impedindo reprogramação período a período.
    """
    curves = []
    for original in inferred_B:
        curve = dict(original)
        bp = (curve['bus'], curve['ph'])
        snom_pu = data['pv'][bp]['S_nom_pu']
        if (USE_INDIVIDUAL_FITTED_VV_CURVES and
                all(key in curve for key in
                    ('V1', 'V2', 'V3', 'V4', 'Q_max'))):
            qmax_pu = min(
                max(float(curve.get(
                    'Q_max_pu', curve.get('Q_max', 0.0))), 0.0),
                QPV_MAX_FRAC_S*snom_pu)
            curve.update({
                'Q_max': qmax_pu,
                'Q_max_pu': qmax_pu,
                'electrical_units': 'per_unit',
                'profile_name': 'opf_individual_fit',
                'source': 'OPF_Qstar_individual_static_fit',
            })
            curves.append(curve)
            continue
        curve.update({
            'V1': float(profile['V1']), 'V2': float(profile['V2']),
            'V3': float(profile['V3']), 'V4': float(profile['V4']),
            'Q_max': min(float(profile['q_frac']), QPV_MAX_FRAC_S) * snom_pu,
            'Q_max_pu': min(float(profile['q_frac']), QPV_MAX_FRAC_S) * snom_pu,
            'electrical_units': 'per_unit',
            'profile_name': profile['name'],
            'source': ('IEEE_1547_2018_Category_B_default'
                       if profile['name'] == 'ieee_catb_default'
                       else 'operational_static_profile'),
        })
        curves.append(curve)
    return curves


def _aggregate_fitted_vv_profile(inferred_B, data):
    """Agrega em uma curva comum os ajustes livres obtidos dos pontos Q*.

    A mediana evita que uma fase isolada determine os breakpoints. O resultado
    continua sendo um único conjunto V1..V4/Qmax sem índice temporal.
    """
    rows = [row for row in inferred_B
            if all(f'fitted_{key}' in row
                   for key in ('V1', 'V2', 'V3', 'V4', 'q_frac'))]
    if not rows:
        return None
    profile = {'name': 'opf_aggregate_fit'}
    for key in ('V1', 'V2', 'V3', 'V4', 'q_frac'):
        profile[key] = float(np.median(
            [float(row[f'fitted_{key}']) for row in rows]))
    profile['q_frac'] = min(max(profile['q_frac'], 0.01), QPV_MAX_FRAC_S)
    # Proteção numérica para medianas coincidentes.
    profile['V2'] = max(profile['V2'], profile['V1']+0.005)
    profile['V3'] = max(profile['V3'], profile['V2'])
    profile['V4'] = max(profile['V4'], profile['V3']+0.005)
    if profile['V4'] > 1.10 or profile['V1'] < 0.88:
        return None
    return profile


def _limit_q_command_slew(q_cmd, data, pvph, hours):
    """Limita ΔQ do comando local sem alterar parâmetros da curva Volt-VAR.

    O desligamento quando não há irradiância é imediato por segurança. Na
    energização, a potência reativa cresce no máximo pelo limite configurado.
    Inversores trifásicos usam um único comando e o menor limite das fases.
    """
    if not ENABLE_Q_COMMAND_SLEW_LIMIT:
        return dict(q_cmd)
    out = dict(q_cmd)
    ordered_hours = list(hours)
    phases_by_bus = {}
    singles = []
    for b, ph in pvph:
        if data['pv'][(b,ph)].get('is_3ph', False):
            phases_by_bus.setdefault(b, []).append(ph)
        else:
            singles.append((b,ph))

    def _limit_device(members):
        step_pu = min(
            Q_COMMAND_MAX_STEP_FRAC_S * data['pv'][bp]['S_nom_pu']
            for bp in members)
        previous = 0.0
        for t in ordered_hours:
            solar_ok = (ENABLE_NIGHT_VAR or
                        PV_EFF_FACTOR[t] >= QPV_MIN_SOLAR_FRAC)
            if not solar_ok:
                applied = 0.0
            else:
                target = sum(q_cmd[(b,ph,t)] for b,ph in members)/len(members)
                applied = max(previous-step_pu, min(previous+step_pu, target))
            for b, ph in members:
                out[(b,ph,t)] = applied
            previous = applied

    for b, phases in phases_by_bus.items():
        _limit_device([(b,ph) for ph in sorted(phases)])
    for bp in singles:
        _limit_device([bp])
    return out


def _summarize_inverter_operability(q_settings, data, pvph, hours):
    """Audita rampa de Q e confirma ausência de mudanças de parâmetros em t."""
    max_step_pu = 0.0
    max_step_kvar = 0.0
    max_step_frac_s = 0.0
    max_step_excess_frac_s = 0.0
    violations = 0
    ordered_hours = list(hours)
    for b, ph in pvph:
        snom_pu = max(data['pv'][(b,ph)]['S_nom_pu'], 1.0e-12)
        for prev_t, t in zip(ordered_hours[:-1], ordered_hours[1:]):
            # Desligamento por falta de irradiância é uma exceção operacional.
            if (not ENABLE_NIGHT_VAR and
                    PV_EFF_FACTOR[t] < QPV_MIN_SOLAR_FRAC):
                continue
            delta_pu = abs(q_settings[(b,ph,t)]-q_settings[(b,ph,prev_t)])
            max_step_pu = max(max_step_pu, delta_pu)
            frac_s = delta_pu/snom_pu
            max_step_kvar = max(max_step_kvar, delta_pu*SBASE)
            max_step_frac_s = max(max_step_frac_s, frac_s)
            excess = max(frac_s-Q_COMMAND_MAX_STEP_FRAC_S, 0.0)
            max_step_excess_frac_s = max(max_step_excess_frac_s, excess)
            if (ENABLE_Q_COMMAND_SLEW_LIMIT and
                    excess > Q_COMMAND_SLEW_TOL):
                violations += 1
    return {
        'parameters_static': bool(INVERTER_PARAMETERS_STATIC_OVER_HORIZON),
        'parameter_changes': 0,
        'max_q_step_pu': max_step_pu,
        'max_q_step_kvar': max_step_kvar,
        'max_q_step_frac_s': max_step_frac_s,
        'max_q_step_excess_frac_s': max_step_excess_frac_s,
        'q_slew_tolerance_frac_s': Q_COMMAND_SLEW_TOL,
        'q_slew_violations': violations,
        'ok': bool(INVERTER_PARAMETERS_STATIC_OVER_HORIZON and violations == 0),
    }


def _static_non_vv_raw_q(setting, p_pu, snom_pu):
    """Retorna o Q bruto de um controle estático não-Volt-VAR.

    ``fixed_pf`` implementa Q=kP; ``constant_q`` implementa Q=qfrac*Snom.
    O envelope do inversor é aplicado pelo chamador, da mesma forma em todos os
    replays físicos. Os parâmetros da configuração não dependem de ``t``.
    """
    mode = setting.get('mode', 'fixed_pf')
    if mode == 'constant_q':
        return float(setting.get('q_const_frac_s', 0.0))*float(snom_pu)
    if mode == 'fixed_pf':
        return float(setting.get('k_qp', 0.0))*float(p_pu)
    raise ValueError(f"Modo estático não-Volt-VAR desconhecido: {mode!r}")


def _operational_qcap_pu(p_pu, snom_pu):
    """Limite de Q que garante simultaneamente capacidade e |FP|>=0,85."""
    p_pu = max(float(p_pu), 0.0)
    snom_pu = max(float(snom_pu), 0.0)
    k_fp085 = math.sqrt(1.0-FP_MIN_VV**2)/FP_MIN_VV
    return min(
        math.sqrt(max(snom_pu**2-p_pu**2, 0.0)),
        QPV_MAX_FRAC_S*snom_pu,
        k_fp085*p_pu)


def _enforce_operational_fp_floor(q_cmd, model, data, pvph, hours):
    """Projeta comandos finais para que nenhuma amostra opere abaixo de FP=0,85."""
    out = dict(q_cmd)
    for b, ph in pvph:
        snom_pu = data['pv'][(b,ph)]['S_nom_pu']
        for t in hours:
            p_pu = max(
                pyo.value(model.Pavail[b,ph,t]) -
                pyo.value(model.Pcurt[b,ph,t]), 0.0)
            qcap = _operational_qcap_pu(p_pu, snom_pu)
            out[(b,ph,t)] = max(
                -qcap, min(qcap, float(out[(b,ph,t)])))
    return out


def _fit_static_non_vv_action(P_arr, Q_arr, snom_pu):
    """Seleciona FP fixo ou Q constante, ambos estáticos nas 24 h.

    A biblioteca contém diversos fatores de potência e comandos de Q constante
    em fração de Snom. Cada direção é testada explicitamente. A seleção minimiza
    o RMSE até Q* depois do recorte pelo círculo de potência aparente e pelo
    limite de 0,44*Snom usado no OPF e no replay físico.
    """
    P_arr = np.asarray(P_arr, dtype=float)
    Q_arr = np.asarray(Q_arr, dtype=float)
    snom_pu = float(snom_pu)
    if P_arr.size == 0:
        return {
            'mode': 'fixed_pf', 'k_qp': 0.0, 'fp': 1.0,
            'q_const_frac_s': 0.0, 'direction': 'none', 'rmse': 0.0,
            'selection_score': 0.0, 'prediction': np.zeros(0),
            'clip_fraction': 0.0, 'candidate_count': 0,
        }

    qcap = np.asarray([
        _operational_qcap_pu(p, snom_pu) for p in P_arr])
    candidates = []

    for fp in FIXED_POWER_FACTOR_CANDIDATES:
        k_abs = math.sqrt(max(1.0-fp**2, 0.0))/fp
        # Grupo A: FP capacitivo fixo na biblioteca declarada (Q>0).
        for direction in (1.0,):
            raw = direction*k_abs*P_arr
            prediction = np.clip(raw, -qcap, qcap)
            rmse = float(np.sqrt(np.mean((Q_arr-prediction)**2)))
            candidates.append({
                'mode': 'fixed_pf', 'k_qp': direction*k_abs, 'fp': fp,
                'q_const_frac_s': 0.0,
                'direction': ('capacitive' if direction > 0.0 else 'inductive'),
                'rmse': rmse, 'selection_score': rmse,
                'prediction': prediction,
                'clip_fraction': float(np.mean(np.abs(raw) > qcap+1.0e-9)),
            })

    for q_frac in CONSTANT_Q_FRAC_S_CANDIDATES:
        raw = np.full(P_arr.shape, q_frac*snom_pu, dtype=float)
        prediction = np.clip(raw, -qcap, qcap)
        rmse = float(np.sqrt(np.mean((Q_arr-prediction)**2)))
        candidates.append({
            'mode': 'constant_q', 'k_qp': 0.0, 'fp': None,
            'q_const_frac_s': q_frac,
            'direction': ('capacitive' if q_frac > 0.0 else 'inductive'),
            'rmse': rmse,
            'selection_score': (
                rmse+CONSTANT_Q_SELECTION_PENALTY_FRAC_S*snom_pu),
            'prediction': prediction,
            'clip_fraction': float(np.mean(np.abs(raw) > qcap+1.0e-9)),
        })

    # Primeiro preserva a aderência ao alvo Q*. Somente entre ajustes
    # praticamente equivalentes (tolerância declarada em fração de Snom) entra
    # o desempate operacional: Volt--VAR é tratado no nível A/B/C; dentro do
    # Grupo A, prefere-se o FP mais próximo de 0.96. Isso permite demonstrar se
    # FP=0.85 é realmente necessário, em vez de escolhê-lo pela ordem da grade.
    best_score = min(row['selection_score'] for row in candidates)
    equivalent = [
        row for row in candidates
        if row['selection_score'] <= (
            best_score + FP_EQUIVALENCE_RMSE_TOL_FRAC_S*snom_pu)]
    equivalent_ids = {id(row) for row in equivalent}
    best = min(equivalent, key=lambda row: (
        row['mode'] != 'fixed_pf',
        abs(float(row.get('fp') or 0.0)-PREFERRED_FIXED_POWER_FACTOR)
        if row['mode'] == 'fixed_pf' else float('inf'),
        row['selection_score'], row['rmse']))
    result = dict(best)
    result['candidate_count'] = len(candidates)
    result['candidate_library'] = [{
        'mode': row['mode'],
        'fp': row.get('fp'),
        'q_const_frac_s': row.get('q_const_frac_s', 0.0),
        'direction': row.get('direction'),
        'rmse': row['rmse'],
        'selection_score': row['selection_score'],
        'distance_to_preferred_pf': (
            abs(float(row.get('fp'))-PREFERRED_FIXED_POWER_FACTOR)
            if row.get('fp') is not None else None),
        'equivalent_to_best_fit': id(row) in equivalent_ids,
    } for row in candidates]
    result['source'] = 'static_multimode_library'
    return result


def _fit_fixed_pf_085_action(P_arr, Q_arr, snom_pu):
    """Alias legado: seleciona o melhor ajuste da biblioteca estática A."""
    return _fit_static_non_vv_action(P_arr, Q_arr, snom_pu)


def _fit_individual_vv_action(V_arr, P_arr, Q_arr, snom_pu):
    """Ajusta uma curva Volt-VAR estatica aos pontos otimos de toda a janela."""
    V_arr = np.asarray(V_arr, dtype=float)
    P_arr = np.asarray(P_arr, dtype=float)
    Q_arr = np.asarray(Q_arr, dtype=float)
    snom_pu = float(snom_pu)
    if V_arr.size == 0:
        return _fit_ieee_catb_action(V_arr, P_arr, Q_arr, snom_pu)
    fitted = _fit_vv_setting_grid(
        V_arr, Q_arr, QPV_MAX_FRAC_S*snom_pu)
    qmax_pu = min(max(float(fitted['Q_max']), 0.0),
                  QPV_MAX_FRAC_S*snom_pu)
    curve = {
        'V1': float(fitted['V1']), 'V2': float(fitted['V2']),
        'V3': float(fitted['V3']), 'V4': float(fitted['V4']),
        'Q_max': qmax_pu, 'Q_max_pu': qmax_pu,
        'electrical_units': 'per_unit',
        'source': 'OPF_Qstar_10min_hourly_median_static_fit',
        'fit_aggregation': VV_FIT_AGGREGATION,
        'fit_point_count': int(V_arr.size),
        'profile_name': 'opf_individual_fit',
    }
    raw = np.asarray([
        _vv_setting_eval_pu(float(v), curve) for v in V_arr])
    qcap = np.asarray([
        _operational_qcap_pu(p, snom_pu) for p in P_arr])
    prediction = np.clip(raw, -qcap, qcap)
    curve.update({
        'rmse': float(np.sqrt(np.mean((Q_arr-prediction)**2))),
        'prediction': prediction,
        'clip_fraction': float(np.mean(np.abs(raw) > qcap+1.0e-9)),
        'fitted_V1': float(fitted['V1']),
        'fitted_V2': float(fitted['V2']),
        'fitted_V3': float(fitted['V3']),
        'fitted_V4': float(fitted['V4']),
        'fitted_q_frac': qmax_pu/max(snom_pu, 1.0e-12),
    })
    return curve


def _fit_ieee_catb_action(V_arr, P_arr, Q_arr, snom_pu):
    """Avalia o Grupo B usando somente a curva IEEE Cat.B padrão."""
    V_arr = np.asarray(V_arr, dtype=float)
    P_arr = np.asarray(P_arr, dtype=float)
    Q_arr = np.asarray(Q_arr, dtype=float)
    qmax = IEEE1547_Q_FRAC*float(snom_pu)
    curve = {
        'V1': IEEE1547_V1, 'V2': IEEE1547_V2,
        'V3': IEEE1547_V3, 'V4': IEEE1547_V4,
        'Q_max': qmax,
        'Q_max_pu': qmax,
        'electrical_units': 'per_unit',
        'source': 'IEEE_1547_2018_Category_B_default',
    }
    if V_arr.size == 0:
        curve.update({'rmse': 0.0, 'prediction': np.zeros(0),
                      'clip_fraction': 0.0})
        return curve
    raw = np.asarray([_vv_setting_eval(float(v), curve) for v in V_arr])
    qcap = np.asarray([
        _operational_qcap_pu(p, snom_pu) for p in P_arr])
    prediction = np.clip(raw, -qcap, qcap)
    curve.update({
        'rmse': float(np.sqrt(np.mean((Q_arr-prediction)**2))),
        'prediction': prediction,
        'clip_fraction': float(np.mean(np.abs(raw) > qcap+1.0e-9)),
    })
    return curve


def _pv_electrical_distance_metrics(data, pvph):
    """Distância e sensibilidade elétrica da fonte até cada inversor-fase.

    O caminho é obtido no próprio grafo de ramos usado pelo BFM. A distância
    acumula ``|Z|`` e a sensibilidade reativa acumula ``|X|``, ambas em p.u.
    Não se usa número de barras nem distância geográfica. Em rede radial o
    caminho é único; Dijkstra torna a rotina robusta a eventuais ramos extras.
    """
    from heapq import heappop, heappush
    from itertools import count

    adjacency = {}
    for (i, j, ph), branch in data['branches'].items():
        r_pu = abs(float(branch.get('r_pu', 0.0)))
        x_pu = abs(float(branch.get('x_pu', 0.0)))
        z_pu = math.hypot(r_pu, x_pu)
        left, right = (i, ph), (j, ph)
        adjacency.setdefault(left, []).append(
            (right, z_pu, r_pu, x_pu))
        adjacency.setdefault(right, []).append(
            (left, z_pu, r_pu, x_pu))
    # Os SVRs pertencem ao grafo elétrico do BFM, mas são armazenados
    # separadamente de ``data['branches']``.
    for ph, regulator in data.get('svr', {}).items():
        r_pu = abs(float(regulator.get('r_pu', 0.0)))
        x_pu = abs(float(regulator.get('x_pu', 0.0)))
        z_pu = math.hypot(r_pu, x_pu)
        left = (regulator['mv_bus'], ph)
        right = (regulator['lv_bus'], ph)
        adjacency.setdefault(left, []).append(
            (right, z_pu, r_pu, x_pu))
        adjacency.setdefault(right, []).append(
            (left, z_pu, r_pu, x_pu))

    paths = {}
    phases = sorted({ph for _, ph in pvph})
    for ph in phases:
        source = (data['slack'], ph)
        best_z = {source: 0.0}
        components = {source: (0.0, 0.0, 0)}
        serial = count()
        heap = [(0.0, next(serial), source)]
        while heap:
            z_path, _, node = heappop(heap)
            if z_path > best_z.get(node, float('inf'))+1.0e-15:
                continue
            r_path, x_path, hops = components[node]
            for neighbor, z_edge, r_edge, x_edge in adjacency.get(node, ()):
                candidate_z = z_path+z_edge
                if candidate_z+1.0e-15 >= best_z.get(
                        neighbor, float('inf')):
                    continue
                best_z[neighbor] = candidate_z
                components[neighbor] = (
                    r_path+r_edge, x_path+x_edge, hops+1)
                heappush(heap, (candidate_z, next(serial), neighbor))
        for bp in pvph:
            if bp[1] != ph:
                continue
            if bp not in best_z:
                paths[bp] = {
                    'connected_to_slack': False,
                    'path_z_pu': float('nan'),
                    'path_r_pu': float('nan'),
                    'path_x_pu': float('nan'),
                    'path_hops': 0,
                }
                continue
            r_path, x_path, hops = components[bp]
            paths[bp] = {
                'connected_to_slack': True,
                'path_z_pu': best_z[bp],
                'path_r_pu': r_path,
                'path_x_pu': x_path,
                'path_hops': hops,
            }

    finite = [
        row for row in paths.values()
        if row['connected_to_slack'] and
        math.isfinite(row['path_z_pu']) and
        math.isfinite(row['path_x_pu'])
    ]
    max_z = max((row['path_z_pu'] for row in finite), default=0.0)
    max_x = max((row['path_x_pu'] for row in finite), default=0.0)
    for row in paths.values():
        if not row['connected_to_slack']:
            row['path_z_normalized'] = 0.0
            row['reactive_sensitivity_normalized'] = 0.0
            row['electrical_distance_normalized'] = 0.0
            continue
        z_norm = row['path_z_pu']/max(max_z, 1.0e-12)
        x_norm = row['path_x_pu']/max(max_x, 1.0e-12)
        row['path_z_normalized'] = z_norm
        row['reactive_sensitivity_normalized'] = x_norm
        row['electrical_distance_normalized'] = 0.5*(z_norm+x_norm)
    return paths


def _distance_reactive_support_audit(q_settings, data, pvph, hours, groups):
    """Verifica, sem impor monotonicidade, distância versus |Q| aplicado."""
    devices = {}
    group_of = {
        bp: name for name, members in groups.items() for bp in members}
    for bp in pvph:
        key = ((bp[0], '3ph') if data['pv'][bp].get('is_3ph', False)
               else bp)
        devices.setdefault(key, []).append(bp)
    distance_metrics = data.get('pv_electrical_distance')
    if not distance_metrics:
        distance_metrics = _pv_electrical_distance_metrics(data, pvph)

    rows = []
    for device, members in devices.items():
        distance = sum(
            distance_metrics[bp]['electrical_distance_normalized']
            for bp in members)/max(len(members), 1)
        sensitivity = sum(
            distance_metrics[bp]['reactive_sensitivity_normalized']
            for bp in members)/max(len(members), 1)
        q_frac_samples = [
            abs(float(q_settings[(b, ph, t)])) /
            max(data['pv'][(b, ph)]['S_nom_pu'], 1.0e-12)
            for b, ph in members for t in hours]
        q_rms = math.sqrt(
            sum(value*value for value in q_frac_samples) /
            max(len(q_frac_samples), 1))
        q_mean = sum(q_frac_samples)/max(len(q_frac_samples), 1)
        rows.append({
            'device': device,
            'members': tuple(sorted(members)),
            'group': group_of.get(members[0], ''),
            'electrical_distance_normalized': distance,
            'reactive_sensitivity_normalized': sensitivity,
            'q_rms_frac_s': q_rms,
            'q_mean_abs_frac_s': q_mean,
            'q_peak_frac_s': max(q_frac_samples, default=0.0),
        })

    def _pearson(x_values, y_values):
        if len(x_values) < 2:
            return float('nan')
        x_mean = sum(x_values)/len(x_values)
        y_mean = sum(y_values)/len(y_values)
        numerator = sum(
            (x-x_mean)*(y-y_mean) for x, y in zip(x_values, y_values))
        denominator = math.sqrt(
            sum((x-x_mean)**2 for x in x_values) *
            sum((y-y_mean)**2 for y in y_values))
        return numerator/denominator if denominator > 1.0e-15 else float('nan')

    def _ranks(values):
        order = sorted(range(len(values)), key=lambda idx: values[idx])
        ranks = [0.0]*len(values)
        start = 0
        while start < len(order):
            end = start+1
            while (end < len(order) and
                   abs(values[order[end]]-values[order[start]]) <= 1.0e-15):
                end += 1
            average_rank = 0.5*(start+end-1)+1.0
            for position in range(start, end):
                ranks[order[position]] = average_rank
            start = end
        return ranks

    x_values = [row['reactive_sensitivity_normalized'] for row in rows]
    y_values = [row['q_rms_frac_s'] for row in rows]
    rows.sort(key=lambda row: (
        -row['electrical_distance_normalized'], str(row['device'])))
    return {
        'device_count': len(rows),
        'pearson_sensitivity_vs_q_rms': _pearson(x_values, y_values),
        'spearman_sensitivity_vs_q_rms':
            _pearson(_ranks(x_values), _ranks(y_values)),
        'rows': rows,
    }


def _rl_discrete_state(context):
    """Discretiza as métricas operacionais em um estado auditável."""
    losses = context['action_loss']
    best_idx = min(range(len(RL_ACTIONS)), key=lambda i: losses[RL_ACTIONS[i]])

    def _bin(value, cuts):
        return int(sum(float(value) > cut for cut in cuts))

    return (
        best_idx,
        _bin(min(losses.values()), (0.08, 0.15, 0.35, 0.75)),
        _bin(context['dead_fraction'], (0.50, 0.85)),
        int(context['bidirectional_q']),
        _bin(context['voltage_support_need'], (0.05, 0.25, 0.50)),
        _bin(context['static_clip_fraction'], (0.01, 0.10)),
        _bin(context.get('electrical_distance_normalized', 0.0),
             RL_DISTANCE_STATE_CUTS),
        _bin(context.get('reactive_support_priority', 0.0),
             (0.05, 0.20, 0.50)),
    )


def _rl_action_rewards(context):
    """Retorno: aderência ao Q* mais simplicidade e risco operacional."""
    loss = {a: min(float(context['action_loss'][a]), RL_NRMSE_REWARD_CAP)
            for a in RL_ACTIONS}
    return {
        'A': (-loss['A']-RL_COMPLEXITY_PENALTY['A']
              -RL_STATIC_BIDIRECTIONAL_PENALTY*float(context['bidirectional_q'])
              -RL_STATIC_CLIPPING_PENALTY*context['static_clip_fraction']),
        'B': -loss['B']-RL_COMPLEXITY_PENALTY['B'],
        'C': (-loss['C']-RL_COMPLEXITY_PENALTY['C']
              -RL_UNITY_SUPPORT_PENALTY*context['voltage_support_need']
              -RL_DISTANCE_SUPPORT_PENALTY*
              context.get('reactive_support_priority', 0.0)),
    }


def _train_contextual_bandit_classifier(device_contexts):
    """Q-learning: A=estático não-VV, B=Volt-VAR, C=FP1,00.

    A ação é estática e não altera um estado futuro da classificação; por isso
    gamma=0. O efeito na rede é validado posteriormente pelo ponto fixo BFM.
    """
    if not device_contexts:
        return {}, {}, {'episodes': 0, 'mean_reward': 0.0,
                        'last_1000_mean_reward': 0.0, 'state_count': 0,
                        'coverage_adjustments': []}

    rng = np.random.default_rng(RL_RANDOM_SEED)
    actions = tuple(RL_ACTIONS)
    q_table, visits = {}, {}
    keys = sorted(device_contexts, key=lambda x: str(x))
    for key in keys:
        ctx = device_contexts[key]
        ctx['state'] = _rl_discrete_state(ctx)
        ctx['rewards'] = _rl_action_rewards(ctx)
        if ctx['state'] not in q_table:
            q_table[ctx['state']] = np.zeros(len(actions), dtype=float)
            visits[ctx['state']] = np.zeros(len(actions), dtype=int)

    reward_trace = []
    n_ep = max(int(RL_TRAINING_EPISODES), len(keys)*len(actions))
    for episode in range(n_ep):
        key = keys[int(rng.integers(0, len(keys)))]
        ctx = device_contexts[key]
        state = ctx['state']
        frac = episode/max(n_ep-1, 1)
        epsilon = RL_EPSILON_START+frac*(RL_EPSILON_END-RL_EPSILON_START)
        if rng.random() < epsilon:
            ai = int(rng.integers(0, len(actions)))
        else:
            qv = q_table[state]
            best = np.flatnonzero(np.isclose(qv, np.max(qv), atol=1.0e-12))
            ai = int(best[int(rng.integers(0, len(best)))])
        action = actions[ai]
        reward = float(ctx['rewards'][action])
        visits[state][ai] += 1
        # Atualização de bandit = Q-learning com gamma=0.
        q_table[state][ai] += RL_LEARNING_RATE*(reward-q_table[state][ai])
        reward_trace.append(reward)

    policy = {}
    blend = min(max(float(RL_POLICY_LOCAL_REWARD_BLEND), 0.0), 1.0)
    for key in keys:
        ctx = device_contexts[key]
        learned = q_table[ctx['state']]
        local = np.asarray([ctx['rewards'][a] for a in actions], dtype=float)
        values = (1.0-blend)*learned+blend*local
        order = np.argsort(-values)
        ai = int(order[0])
        if ctx.get('no_useful_generation', False):
            ai = actions.index('C')
            order = np.asarray([ai]+[i for i in range(len(actions)) if i != ai])
        margin = float(values[order[0]]-values[order[1]])
        policy[key] = {
            'action': actions[ai], 'state': ctx['state'],
            'q_values': {a: float(values[i]) for i, a in enumerate(actions)},
            'learned_q': {a: float(learned[i]) for i, a in enumerate(actions)},
            'rewards': dict(ctx['rewards']), 'confidence_margin': margin,
            'coverage_adjusted': False,
        }

    coverage_adjustments = []
    if RL_ENABLE_SAFE_MODE_COVERAGE:
        for wanted in actions:
            counts = {a: sum(p['action'] == a for p in policy.values())
                      for a in actions}
            if counts[wanted] > 0:
                continue
            options = []
            for key, p in policy.items():
                current = p['action']
                if device_contexts[key].get('no_useful_generation', False):
                    continue
                if current != wanted and counts[current] <= 1:
                    continue
                regret = max(p['q_values'].values())-p['q_values'][wanted]
                options.append((float(regret), str(key), key))
            if options:
                regret, _, key = min(options)
                if regret <= RL_MAX_COVERAGE_REGRET:
                    old = policy[key]['action']
                    policy[key]['action'] = wanted
                    policy[key]['coverage_adjusted'] = True
                    coverage_adjustments.append({
                        'device': key, 'from': old, 'to': wanted,
                        'regret': regret})

    q_export = {
        str(state): {a: float(q_table[state][i])
                     for i, a in enumerate(actions)}
        for state in q_table}
    summary = {
        'episodes': n_ep,
        'mean_reward': float(np.mean(reward_trace)) if reward_trace else 0.0,
        'last_1000_mean_reward': (
            float(np.mean(reward_trace[-1000:])) if reward_trace else 0.0),
        'state_count': len(q_table),
        'coverage_adjustments': coverage_adjustments,
    }
    return policy, q_export, summary


def _direct_reward_classifier(device_contexts):
    """Ablation determinística ``argmax_a R_g(a)``.

    Usa exatamente os mesmos contextos, recompensas e regra de desempate do
    classificador por bandit, mas não executa episódios de aprendizagem. O
    resultado é mantido separado para que o replay elétrico de ambos possa ser
    comparado com as mesmas condições iniciais.
    """
    actions = tuple(RL_ACTIONS)
    policy = {}
    started = time.perf_counter()
    for key in sorted(device_contexts, key=lambda x: str(x)):
        ctx = device_contexts[key]
        state = ctx.get('state', _rl_discrete_state(ctx))
        rewards = ctx.get('rewards', _rl_action_rewards(ctx))
        values = np.asarray([float(rewards[a]) for a in actions], dtype=float)
        # Empates são resolvidos de forma determinística pela ordem declarada de
        # RL_ACTIONS, que também é exportada no arquivo de parâmetros congelados.
        ai = int(np.flatnonzero(np.isclose(
            values, np.max(values), atol=1.0e-12, rtol=0.0))[0])
        if ctx.get('no_useful_generation', False):
            ai = actions.index('C')
        order = np.argsort(-values, kind='stable')
        second = int(order[1]) if len(order) > 1 else ai
        policy[key] = {
            'action': actions[ai],
            'state': state,
            'q_values': {a: float(values[i]) for i, a in enumerate(actions)},
            'learned_q': None,
            'rewards': dict(rewards),
            'confidence_margin': float(values[ai]-values[second]),
            'coverage_adjusted': False,
            'classifier': 'direct_argmax_reward',
        }
    summary = {
        'classifier': 'direct_argmax_reward',
        'episodes': 0,
        'state_count': len({str(p['state']) for p in policy.values()}),
        'elapsed_seconds': time.perf_counter()-started,
        'action_order_for_ties': list(actions),
    }
    return policy, summary


def compare_classifier_assignments(rl_policy, direct_policy):
    """Tabela auditável da ablação RL versus maximização direta."""
    rows = []
    for key in sorted(set(rl_policy) | set(direct_policy), key=str):
        rl = rl_policy.get(key, {})
        direct = direct_policy.get(key, {})
        rows.append({
            'device': str(key),
            'rl_action': rl.get('action'),
            'direct_action': direct.get('action'),
            'same_assignment': rl.get('action') == direct.get('action'),
            'reward_A': direct.get('rewards', rl.get('rewards', {})).get('A'),
            'reward_B': direct.get('rewards', rl.get('rewards', {})).get('B'),
            'reward_C': direct.get('rewards', rl.get('rewards', {})).get('C'),
        })
    return rows


def _fit_fixed_pf_setting(P_arr, Q_arr):
    """Ajusta Q=kP sob |k|≤tan(arccos(FP_MIN_VV))."""
    den = float(np.dot(P_arr, P_arr))
    k_lim = math.sqrt(1.0 - FP_MIN_VV**2) / FP_MIN_VV
    k_raw = float(np.dot(P_arr, Q_arr) / den) if den > 1.0e-9 else 0.0
    k = max(-k_lim, min(k_lim, k_raw))
    pred = k * P_arr
    rmse = float(np.sqrt(np.mean((Q_arr - pred)**2)))
    fp = 1.0 / math.sqrt(1.0 + k*k)
    return {'k_qp': k, 'fp': fp, 'rmse': rmse, 'prediction': pred}


def _aggregate_qstar_points_by_hour(period_arr, V_arr, Q_arr, P_arr):
    """Resume os pontos ótimos de 10 min em até 24 pontos horários.

    A mediana dos seis períodos é usada para reduzir a influência de um único
    ponto numérico extremo. Os pontos de 10 min continuam preservados para
    auditoria e a política ajustada volta a ser testada nos 144 períodos.
    """
    period_arr = np.asarray(period_arr, dtype=int)
    V_arr = np.asarray(V_arr, dtype=float)
    Q_arr = np.asarray(Q_arr, dtype=float)
    P_arr = np.asarray(P_arr, dtype=float)
    rows = []
    for hour in range(24):
        mask = (period_arr // VV_PERIODS_PER_HOUR) == hour
        count = int(np.count_nonzero(mask))
        if count < VV_MIN_POINTS_PER_HOUR:
            continue
        v_hour = V_arr[mask]
        q_hour = Q_arr[mask]
        p_hour = P_arr[mask]
        rows.append({
            'hour': hour,
            'n_points': count,
            'V': float(np.median(v_hour)),
            'Q': float(np.median(q_hour)),
            'P': float(np.median(p_hour)),
            'V_min': float(np.min(v_hour)),
            'V_max': float(np.max(v_hour)),
            'Q_min': float(np.min(q_hour)),
            'Q_max': float(np.max(q_hour)),
        })
    return rows


In [ ]:
#@title Internal model implementation 5/12 { display-mode: "form" }
def _fit_vv_setting_grid(V_arr, Q_arr, q_cap, sample_weight=None):
    """Busca uma curva Volt-VAR implementável que minimize o RMSE observado.

    A busca inclui curvas cuja região observada permanece saturada, evitando
    traduzir equivocadamente Q quase constante como FP fixo.
    """
    weight = (np.ones_like(Q_arr, dtype=float) if sample_weight is None
              else np.asarray(sample_weight, dtype=float))
    weight = weight/max(float(np.sum(weight)), 1.0e-12)
    v_lo = max(0.90, float(np.min(V_arr)) - 0.08)
    v_hi = min(1.07, float(np.max(V_arr)) + 0.08)
    v2_grid = np.linspace(v_lo, v_hi, 25)
    deadbands = (0.010, 0.020, 0.040)
    ramps = (0.020, 0.040, 0.060)
    best = None
    for v2 in v2_grid:
        for db in deadbands:
            v3 = v2 + db
            for r_cap in ramps:
                v1 = v2 - r_cap
                if v1 < 0.88:
                    continue
                for r_ind in ramps:
                    v4 = v3 + r_ind
                    if v4 > 1.10:
                        continue
                    shape = np.array([
                        _vv_setting_eval(float(v), {
                            'V1': v1, 'V2': v2, 'V3': v3, 'V4': v4,
                            'Q_max': 1.0}) for v in V_arr])
                    den = float(np.dot(weight*shape, shape))
                    qmax = (float(np.dot(weight*shape, Q_arr) / den)
                             if den > 1.0e-12 else 0.0)
                    qmax = max(0.0, min(float(q_cap), qmax))
                    pred = qmax * shape
                    rmse = float(np.sqrt(np.sum(
                        weight*(Q_arr-pred)**2)))
                    if best is None or rmse < best['rmse']:
                        best = {
                            'V1': float(v1), 'V2': float(v2),
                            'V3': float(v3), 'V4': float(v4),
                            'Q_max': qmax, 'rmse': rmse,
                            'prediction': pred,
                        }
    if best is None:
        best = {'V1': 0.92, 'V2': 0.98, 'V3': 1.02, 'V4': 1.08,
                'Q_max': 0.0,
                'rmse': float(np.sqrt(np.sum(weight*Q_arr**2))),
                'prediction': np.zeros_like(Q_arr)}
    return best


def activate_exact_fixed_tap_svr(m, data, hours, tap_schedule):
    """Ativa a relação elétrica exata do SVR quando o tap é conhecido.

    A aproximação afim continua pertencendo somente ao OPF de identificação.
    No replay, ``tap_schedule`` é discreto e conhecido, portanto

        V_out = (tap_min + tap_step*n) V_in

    é uma igualdade linear. O parâmetro mutável permite atualizar a trajetória
    entre iterações de malha fechada sem reconstruir o restante do modelo.
    """
    svr = data['svr']
    phs = sorted(svr)
    expected = {(ph, t) for ph in phs for t in hours}
    missing = expected-set(tap_schedule)
    if missing:
        raise ValueError(
            f'tap_schedule incompleto para SVR exato: {len(missing)} chaves ausentes')

    if hasattr(m, 'c_svr'):
        m.c_svr.deactivate()

    tau_values = {
        (ph, t): (float(svr[ph]['tap_min']) +
                  int(round(tap_schedule[(ph, t)]))*
                  float(svr[ph]['tap_step']))
        for ph in phs for t in hours}

    if not hasattr(m, 'tau_svr_exact'):
        m.tau_svr_exact = pyo.Param(
            m.SVRPH, m.T, initialize=tau_values, mutable=True,
            within=pyo.PositiveReals)

        def _exact_rule(model, ph, t):
            mv = svr[ph]['mv_bus']
            lv = svr[ph]['lv_bus']
            if (lv, ph) not in set(model.BPH):
                return pyo.Constraint.Skip
            return (model.V[lv, ph, t] ==
                    model.tau_svr_exact[ph, t]*model.V[mv, ph, t])

        m.c_svr_exact = pyo.Constraint(m.SVRPH, m.T, rule=_exact_rule)
    else:
        for key, value in tau_values.items():
            m.tau_svr_exact[key] = value
        m.c_svr_exact.activate()

    for (ph, t), pos in tap_schedule.items():
        m.tap_pos[ph, t].fix(int(round(pos)))
    m._svr_equation_mode = 'exact_fixed_tap'
    return tau_values


def max_exact_svr_residual_pu(m, data, hours, tap_schedule):
    """Maior resíduo absoluto da igualdade exata do SVR em p.u."""
    residuals = []
    for ph, sd in data['svr'].items():
        for t in hours:
            tau = (float(sd['tap_min']) +
                   int(round(tap_schedule[(ph, t)]))*float(sd['tap_step']))
            residuals.append(abs(
                try_v(m, sd['lv_bus'], ph, t) -
                tau*try_v(m, sd['mv_bus'], ph, t)))
    return max(residuals, default=0.0)


def _normalise_relay_state(data, initial_relay_state=None):
    """Cria/copia o estado causal do relé, incluindo a fronteira diária."""
    state = {}
    initial_relay_state = initial_relay_state or {}
    for ph, sd in data['svr'].items():
        default_pos = int(round(
            (sd['tap_init']-sd['tap_min'])/sd['tap_step']))
        raw = initial_relay_state.get(ph, initial_relay_state.get(str(ph), {}))
        state[ph] = {
            'tap_position': int(raw.get('tap_position', default_pos)),
            'pending_direction': int(raw.get('pending_direction', 0)),
            'delay_counter': int(raw.get('delay_counter', 0)),
            'queued_delta': int(raw.get('queued_delta', 0)),
            'last_relay_voltage_v': raw.get('last_relay_voltage_v'),
        }
    return state


def _svr_schedule_from_solution(
        m, data, hours, initial_relay_state=None, return_final_state=False):
    """Reconstrói causalmente a trajetória autônoma do SVR.

    A solução ``m`` corresponde à trajetória da iteração anterior. Durante a
    reconstrução, porém, o relé pode alterar sua posição corrente. A tensão
    usada na leitura LDC é então atualizada com a posição que está sendo
    efetivamente reconstruída, evitando que o relé continue lendo a tensão
    associada ao tap antigo. Assim, em cada período, o tap resulta da leitura
    de tensão/corrente produzida pela política dos inversores e pela posição
    corrente do próprio regulador.
    """
    svr = data['svr']
    phs = sorted(svr)
    relay_state = _normalise_relay_state(data, initial_relay_state)
    pos = {ph: relay_state[ph]['tap_position'] for ph in phs}
    count = {ph: relay_state[ph]['delay_counter'] for ph in phs}
    pending = {ph: relay_state[ph]['pending_direction'] for ph in phs}
    queued = {ph: relay_state[ph]['queued_delta'] for ph in phs}
    ops = {ph: 0 for ph in phs}
    traj, relay = {}, {}
    for t in hours:
        for ph in phs:
            sd = svr[ph]
            if queued[ph]:
                pos[ph] = max(0, min(
                    sd['n_tap']-1, pos[ph]+queued[ph]))
                ops[ph] += abs(queued[ph])
                queued[ph] = 0
            traj[(ph,t)] = pos[ph]
            # ``m`` foi resolvido com a trajetória da iteração anterior.
            # Para a lógica causal do relé, a tensão secundária deve refletir
            # a posição ``pos[ph]`` que está sendo reconstruída neste instante.
            v_mv = try_v(m, sd['mv_bus'], ph, t)
            tau = sd['tap_min'] + sd['tap_step']*pos[ph]
            v_lv = tau*v_mv
            p_svr = pyo.value(m.P_svr[ph,t])
            q_svr = pyo.value(m.Q_svr[ph,t])
            i_line = complex(p_svr, -q_svr) * (SBASE / VMT) / max(v_lv, 0.5)
            i_comp = i_line / max(sd.get('ctprim_A', 700.0), 1.0e-9)
            vreg = v_lv * sd.get('vbase_relay_V', 120.0)
            z_ldc = complex(sd.get('R_ldc_V', 3.0), sd.get('X_ldc_V', 9.0))
            v_relay = abs(complex(vreg, 0.0) - z_ldc*i_comp)
            relay[(ph,t)] = v_relay
            err = v_relay - sd.get('vreg_V', VREG_SVR_V)
            band = sd.get('band_V', BAND_SVR_V)
            direction = 0
            if err > band/2 and pos[ph] > 0:
                direction = -1
            elif err < -band/2 and pos[ph] < sd['n_tap']-1:
                direction = 1

            if direction == 0:
                count[ph] = 0
                pending[ph] = 0
            else:
                if pending[ph] == direction:
                    count[ph] += 1
                else:
                    pending[ph] = direction
                    count[ph] = 1
                if count[ph] >= sd.get('delay_periods', DELAY_PER):
                    if t == hours[-1]:
                        # A decisão tomada no último intervalo entra em serviço
                        # no primeiro período do dia seguinte.
                        queued[ph] = direction
                    else:
                        pos[ph] += direction
                        pos[ph] = max(0, min(sd['n_tap']-1, pos[ph]))
                        ops[ph] += 1
                    count[ph] = 0
                    pending[ph] = 0
            relay_state[ph] = {
                'tap_position': pos[ph],
                'pending_direction': pending[ph],
                'delay_counter': count[ph],
                'queued_delta': queued[ph],
                'last_relay_voltage_v': v_relay,
            }
    if return_final_state:
        return traj, ops, relay, relay_state
    return traj, ops, relay


def recalculate_original_objective_pu(
        m, data, sets_info, initial_relay_state=None,
        initial_q_command=None, include_terminal_return=None):
    """Recalcula a F.O. original diretamente do estado físico, em p.u.

    As variáveis epígrafo do OPF podem ficar não mínimas quando a função
    objetivo é desativada no fluxo ex-post. Por isso os valores absolutos são
    reconstruídos de V, Qpv, Pcurt e tap, em vez de ler ``dv_pos``, ``q_abs``
    ou ``tap_step_abs``. Nenhuma grandeza elétrica é convertida para kW/kVAr.
    """
    hours = list(sets_info['hours'])
    bph = list(sets_info['bph'])
    pvph = list(sets_info['pvph'])
    cph = list(sets_info['cph'])
    slack = data['slack']
    svr_lv = {d['lv_bus'] for d in data['svr'].values()}
    hour_set = set(hours)
    relay_state = _normalise_relay_state(data, initial_relay_state)
    initial_q_command = dict(initial_q_command or {})
    if include_terminal_return is None:
        # O retorno ao tap inicial é uma convenção de horizonte isolado. Em
        # uma cadeia cronológica ele criaria uma operação artificial à meia-noite.
        include_terminal_return = bool(
            PENALIZE_TERMINAL_TAP and initial_relay_state is None)

    voltage_deviation_pu = sum(
        abs(try_v(m, b, ph, t)-pyo.value(m.Vref[b,ph]))
        for b, ph in bph if b != slack for t in hours)

    voltage_margin_slack_pu = 0.0
    for b, ph in bph:
        if b == slack:
            continue
        if b in svr_lv:
            lo, hi = 0.95, 1.05-V_SECURITY_MARGIN
        elif data['buses'].get(b, {}).get('level', 'mv') == 'mv':
            lo, hi = 0.93+V_SECURITY_MARGIN, 1.05-V_SECURITY_MARGIN
        else:
            lo, hi = 0.92+V_SECURITY_MARGIN, 1.05-V_SECURITY_MARGIN
        for t in hours:
            v_pu = try_v(m, b, ph, t)
            voltage_margin_slack_pu += max(lo-v_pu, 0.0)+max(v_pu-hi, 0.0)

    tap_steps = 0.0
    tap_terminal_steps = 0.0
    for ph in sets_info['svr_phs']:
        sd = data['svr'][ph]
        previous = int(relay_state[ph]['tap_position'])
        initial_position = previous
        for t in hours:
            current = int(round(pyo.value(m.tap_pos[ph,t])))
            tap_steps += abs(current-previous)
            previous = current
        tap_terminal_steps += abs(previous-initial_position)

    bph_set = set(bph)
    unbalance_pu = 0.0
    for b in sets_info.get('trif_buses', []):
        for ph1, ph2 in ((1,2), (2,3), (3,1)):
            if (b,ph1) not in bph_set or (b,ph2) not in bph_set:
                continue
            for t in hours:
                unbalance_pu += abs(
                    try_v(m,b,ph1,t)-try_v(m,b,ph2,t))

    q_use_pu = sum(abs(pyo.value(m.Qpv[b,ph,t]))
                   for b, ph in pvph for t in hours)
    q_ramp_weighted_pu = 0.0
    for b, ph in pvph:
        for t in hours:
            if not (T_VV_START <= t <= T_VV_END):
                continue
            previous_q = (pyo.value(m.Qpv[b,ph,t-1])
                          if t-1 in hour_set else
                          float(initial_q_command.get((b, ph), 0.0)))
            ramp_weight = (W_RAMP_PICO
                           if IRR_PROFILE[t] > IRR_PICO else W_RAMP_DIA)
            q_ramp_weighted_pu += ramp_weight*abs(
                pyo.value(m.Qpv[b,ph,t])-previous_q)

    dt_h = DT_MIN/60.0
    joule_loss_pu_h = dt_h*sum(
        data['branches'][(i,j,ph)]['r_pu']*pyo.value(m.l_sq[i,j,ph,t])
        for i,j,ph in cph for t in hours)
    core_loss_pu_h = dt_h*sum(
        bd.get('G_core_pu', 0.0)*try_v(m,j,ph,t)**2
        for (i,j,ph), bd in data['branches'].items() for t in hours)
    curtailment_pu_period_sum = sum(
        pyo.value(m.Pcurt[b,ph,t]) for b,ph in pvph for t in hours)

    def _norm(name):
        return max(pyo.value(getattr(m, f'obj_norm_{name}')), 1.0e-12)

    weighted = {
        'tap_operations': OBJ_WEIGHT_TAP*(
            tap_steps+(tap_terminal_steps if include_terminal_return else 0.0)
        )/_norm('tap'),
        'voltage_deviation': (
            OBJ_WEIGHT_VOLTAGE_DEVIATION*voltage_deviation_pu /
            _norm('voltage_deviation')),
        'unbalance': (
            OBJ_WEIGHT_UNBALANCE*unbalance_pu/_norm('unbalance')),
        'core_loss': OBJ_WEIGHT_CORE*core_loss_pu_h/_norm('core'),
        # Salvaguardas secundárias, também adimensionais.
        'voltage_margin': (
            OBJ_WEIGHT_VOLTAGE_MARGIN*voltage_margin_slack_pu /
            _norm('voltage_margin')),
        'joule_loss_safeguard': (
            OBJ_WEIGHT_JOULE_SAFEGUARD*joule_loss_pu_h/_norm('joule')),
        'q_use_safeguard': (
            OBJ_WEIGHT_Q_USE_SAFEGUARD*q_use_pu/_norm('q_use')),
        # W_RAMP_DIA/PICO já foi aplicado amostra a amostra acima.
        'q_ramp_safeguard': (
            OBJ_WEIGHT_Q_RAMP_SAFEGUARD*q_ramp_weighted_pu /
            _norm('q_ramp')),
        'curtailment_safeguard': (
            OBJ_WEIGHT_CURTAILMENT*curtailment_pu_period_sum /
            _norm('curtailment')),
    }
    return {
        'electrical_unit_system': 'per_unit',
        'sbase_kva_per_phase_reporting_only': SBASE,
        'vref_mv_allowed_pu': (MV_VREF_CONST_P_PU, MV_VREF_OTHER_PU),
        'raw': {
            'voltage_deviation_pu_sum': voltage_deviation_pu,
            'voltage_margin_slack_pu_sum': voltage_margin_slack_pu,
            'tap_steps': tap_steps,
            'tap_terminal_steps': tap_terminal_steps,
            'unbalance_pu_sum': unbalance_pu,
            'q_use_pu_sum': q_use_pu,
            'q_ramp_weighted_pu_sum': q_ramp_weighted_pu,
            'joule_loss_pu_h': joule_loss_pu_h,
            'core_loss_pu_h': core_loss_pu_h,
            'curtailment_pu_period_sum': curtailment_pu_period_sum,
        },
        'weighted': weighted,
        'normalization': {
            name: _norm(name) for name in (
                'tap', 'voltage_deviation', 'unbalance', 'core', 'joule',
                'voltage_margin', 'q_use', 'q_ramp', 'curtailment')},
        'total': sum(weighted.values()),
    }


def validate_hybrid_fixed_pq_pu(base_model, data, sets_info, groups,
                                 p_injection_pu, q_injection_pu,
                                 tap_schedule, opt,
                                 initial_relay_state=None):
    """Certifica o estado final com P/Q e taps fixos e tensões livres.

    A versão anterior fixava também V nos terminais do Grupo B. Isso pode
    sobre-determinar numericamente o QCP (P, Q e V simultaneamente prescritos)
    e levou a retornos CPLEX ``unknown`` mesmo quando o ponto fechado anterior
    era fisicamente exato. Aqui o replay físico mantém todas as tensões
    não-slack livres, como em um fluxo de potência convencional. Para o Grupo B,
    compara-se a tensão recalculada com a tensão do ponto fixo que originou o
    comando Volt-VAR; pequena deriva confirma a consistência V-Q sem impor V.
    """
    hours = list(sets_info['hours'])
    pvph = list(sets_info['pvph'])
    cph = list(sets_info['cph'])
    slack = data['slack']
    group_of = {bp: g for g, members in groups.items() for bp in members}
    m = base_model.clone()

    for objective in list(m.component_objects(pyo.Objective, active=True)):
        objective.deactivate()

    # A certificação não reaplica a função multiobjetivo nem restrições de
    # programação de taps; os controles já foram determinados externamente.
    for cname in ('c_v_margin_low', 'c_v_margin_high',
                  'c_tap_up', 'c_tap_dn', 'c_tap_abs_up', 'c_tap_abs_dn',
                  'c_tap_event_link', 'c_tap_max', 'c_tap_delay',
                  'c_tap_spacing', 'c_tap_terminal',
                  'c_tap_terminal_abs_up', 'c_tap_terminal_abs_dn'):
        comp = getattr(m, cname, None)
        if comp is not None:
            comp.deactivate()

    if hasattr(m, 'obj_hybrid_physical'):
        m.del_component(m.obj_hybrid_physical)
    m.obj_hybrid_physical = pyo.Objective(
        expr=sum(m.l_sq[i,j,ph,t] for i,j,ph in cph for t in hours))

    # Ponto de equilíbrio produzido pela malha fechada A/B/C + LDC.
    voltage_target_pu = {
        (b,ph,t): try_v(base_model,b,ph,t)
        for b,ph in sets_info['bph'] for t in hours}

    # Todas as tensões físicas (exceto slack) ficam livres. O alvo anterior é
    # usado apenas como warm start/SCA e como auditoria de consistência do B.
    for b, ph in sets_info['bph']:
        for t in hours:
            if b != slack:
                if m.V[b,ph,t].fixed:
                    m.V[b,ph,t].unfix()
                m.V[b,ph,t].setlb(VALIDATION_V_MIN)
                m.V[b,ph,t].setub(VALIDATION_V_MAX)
            v0 = max(voltage_target_pu[(b,ph,t)], 0.80)
            m.V[b,ph,t].set_value(v0)
            m.V_bar_bus[b,ph,t] = v0
    for i,j,ph in cph:
        for t in hours:
            vi0 = max(voltage_target_pu[(i,ph,t)], 0.80)
            m.V_sq_ws[i,j,ph,t] = max(vi0*vi0, 0.64)

    # Congela exatamente o despacho ativo/reativo efetivamente aplicado.
    for b, ph in pvph:
        for t in hours:
            pav = pyo.value(m.Pavail[b,ph,t])
            ptarget = float(p_injection_pu[(b,ph,t)])
            pcurt = min(max(pav-ptarget, 0.0), max(pav, 0.0))
            m.Pcurt[b,ph,t].fix(pcurt)
            m.Qpv[b,ph,t].fix(float(q_injection_pu[(b,ph,t)]))

    for ph in sets_info['svr_phs']:
        for t in hours:
            m.tap_pos[ph,t].unfix()
            m.tap_pos[ph,t].fix(int(tap_schedule[(ph,t)]))
            m.u_tap[ph,t].fix(0.0)
    activate_exact_fixed_tap_svr(m, data, hours, tap_schedule)

    (_, termination, _, violation, sca_converged, iterations,
     max_dv_pu) = _solve_sca_voltage_consistent(
         opt, m, cph, hours, label='hybrid-freeV-fixedPQ', tee_first=False)
    usable = termination in _USABLE_TERMINATIONS

    max_fixed_p_error_pu = (max(
        (abs((pyo.value(m.Pavail[b,ph,t])-pyo.value(m.Pcurt[b,ph,t]))-
             p_injection_pu[(b,ph,t)])
         for b,ph in pvph for t in hours), default=0.0) if usable
        else float('inf'))
    max_fixed_q_error_pu = (max(
        (abs(pyo.value(m.Qpv[b,ph,t])-q_injection_pu[(b,ph,t)])
         for b,ph in pvph for t in hours), default=0.0) if usable
        else float('inf'))

    # Consistência do ponto V-Q dos inversores Volt-VAR sem fixar V.
    b_voltage_keys = [
        (b,ph,t) for b,ph in pvph if group_of.get((b,ph), 'C') == 'B'
        for t in hours]
    max_b_voltage_drift_pu = (max(
        (abs(try_v(m,b,ph,t)-voltage_target_pu[(b,ph,t)])
         for b,ph,t in b_voltage_keys), default=0.0) if usable
        else float('inf'))

    mv_reference_rows = []
    loads_mt = data.get('loads_mt', {})
    for b, ph in sets_info['bph']:
        if data['buses'].get(b, {}).get('level', 'mv') != 'mv':
            continue
        expected = (MV_VREF_CONST_P_PU
                    if loads_mt.get((b,ph), {}).get('model', None) == 1
                    else MV_VREF_OTHER_PU)
        actual = pyo.value(m.Vref[b,ph])
        mv_reference_rows.append({
            'bus': b, 'phase': ph, 'expected_pu': expected,
            'actual_pu': actual, 'ok': abs(actual-expected) <= 1.0e-12})
    mv_reference_ok = all(row['ok'] for row in mv_reference_rows)

    exactness = (summarize_bfm_exactness(m, data, sets_info)
                 if usable and sca_converged else None)
    svr_exact_residual_pu = (
        max_exact_svr_residual_pu(m, data, hours, tap_schedule)
        if usable and sca_converged else float('inf'))
    objective_pu = (recalculate_original_objective_pu(
                        m, data, sets_info,
                        initial_relay_state=initial_relay_state,
                        include_terminal_return=(initial_relay_state is None))
                    if usable and sca_converged else None)
    fixed_injections_ok = (
        max_fixed_p_error_pu <= HYBRID_FIXED_INJECTION_TOL_PU and
        max_fixed_q_error_pu <= HYBRID_FIXED_INJECTION_TOL_PU)
    b_equilibrium_ok = (
        max_b_voltage_drift_pu <= HYBRID_B_EQUILIBRIUM_V_DRIFT_TOL_PU)
    ok = bool(
        usable and sca_converged and fixed_injections_ok and b_equilibrium_ok
        and mv_reference_ok and exactness is not None
        and exactness['exactness_ok']
        and svr_exact_residual_pu <= SVR_EXACT_RESIDUAL_TOL)

    return {
        'model': m, 'ok': ok, 'termination': termination,
        'solver_violation': violation, 'sca_converged': sca_converged,
        'iterations': iterations, 'max_sca_dv_pu': max_dv_pu,
        'electrical_unit_system': 'per_unit',
        'p_fixed': True, 'q_fixed': True,
        'voltage_fixed_groups': (),
        'voltage_free_groups': ('A','B','C'),
        'fixed_voltage_count': 0,
        'free_voltage_count': len(pvph)*len(hours),
        # Mantido por compatibilidade de relatório; agora significa deriva do
        # equilíbrio V do Grupo B, e não erro de uma variável V fixada.
        'max_fixed_voltage_error_pu': max_b_voltage_drift_pu,
        'max_b_equilibrium_voltage_drift_pu': max_b_voltage_drift_pu,
        'b_equilibrium_ok': b_equilibrium_ok,
        'max_fixed_p_error_pu': max_fixed_p_error_pu,
        'max_fixed_q_error_pu': max_fixed_q_error_pu,
        'mv_reference_ok': mv_reference_ok,
        'mv_reference_rows': mv_reference_rows,
        'voltage_target_pu': voltage_target_pu,
        'p_injection_pu': dict(p_injection_pu),
        'q_injection_pu': dict(q_injection_pu),
        'objective_pu': objective_pu,
        'exactness': exactness,
        'svr_equation_mode': 'exact_fixed_tap',
        'max_svr_exact_residual_pu': svr_exact_residual_pu,
    }


In [ ]:
#@title Internal model implementation 6/12 { display-mode: "form" }
def validate_recommended_settings(opf_model, data, sets_info, groups,
                                  inferred_B, fixed_pf_A, solver_executable,
                                  source_relaxation_exact=False,
                                  autonomous_seed_schedule=None,
                                  autonomous_seed_model=None,
                                  run_qstar_diagnostic=True,
                                  baseline_quality_for_screening=None,
                                  initial_relay_state=None):
    """Valida os settings A/B/C com o SVR operando por LDC autônomo.

    O OPF fornece apenas o envelope ``Q*`` e a referência de identificação.
    No modo ``autonomous_ldc``, a avaliação final não impõe a trajetória de
    taps do OPF: todos os cenários partem da mesma trajetória física S1 usada
    apenas como *seed* numérico, e a trajetória final é reconstruída pelo relé
    a partir das tensões e correntes resultantes da atuação dos inversores.
    """
    from pyomo.environ import SolverFactory

    hours = list(sets_info['hours'])
    pvph = list(sets_info['pvph'])
    cph = list(sets_info['cph'])
    group_of = {bp: g for g, members in groups.items() for bp in members}
    curve_of = {(d['bus'], d['ph']): d for d in inferred_B}
    fixed_of = dict(fixed_pf_A)

    missing_b = [bp for bp in groups.get('B', []) if bp not in curve_of]
    if missing_b:
        raise ValueError(f"Grupo B sem curva individual: {missing_b}")

    m = opf_model.clone()
    # Na validação os settings substituem o despacho livre de Q.
    if hasattr(m, 'c_qpv_nvv'):
        m.c_qpv_nvv.deactivate()

    # A validação ex-post é um fluxo de potência, não um novo OPF regulatório.
    # Logo, a tensão precisa poder sair da faixa adequada para que DRP/DRC
    # meçam uma eventual não conformidade. Manter [0,92/0,93; 1,05] aqui
    # converteria uma configuração ruim em "infeasible" e esconderia o valor
    # físico que se deseja auditar.
    slack = data['slack']
    for (b,ph) in sets_info['bph']:
        if b == slack:
            continue
        for t in hours:
            m.V[b,ph,t].setlb(VALIDATION_V_MIN)
            m.V[b,ph,t].setub(VALIDATION_V_MAX)
    # A margem preventiva pertence à otimização. Na validação, a distância às
    # fronteiras é apenas reportada, não imposta nem penalizada.
    for cname in ('c_v_margin_low', 'c_v_margin_high'):
        comp = getattr(m, cname, None)
        if comp is not None:
            comp.deactivate()

    # A trajetória de tap é fornecida externamente: pelo próprio OPF no ensaio
    # principal ou pelo relé LDC no cenário diagnóstico. Desativam-se apenas as
    # restrições de programação; a equação elétrica do SVR permanece ativa.
    for cname in ('c_tap_up', 'c_tap_dn', 'c_tap_abs_up', 'c_tap_abs_dn',
                  'c_tap_event_link', 'c_tap_max', 'c_tap_delay',
                  'c_tap_spacing', 'c_tap_terminal',
                  'c_tap_terminal_abs_up', 'c_tap_terminal_abs_dn'):
        comp = getattr(m, cname, None)
        if comp is not None:
            comp.deactivate()
    for ph in sets_info['svr_phs']:
        for t in hours:
            m.tap_pos[ph,t].unfix()
            m.u_tap[ph,t].fix(0.0)
    q_opf = {(b,ph,t): pyo.value(opf_model.Qpv[b,ph,t])
             for (b,ph) in pvph for t in hours}
    pcurt_opf = {(b,ph,t): pyo.value(opf_model.Pcurt[b,ph,t])
                  for (b,ph) in pvph for t in hours}
    if FORCE_ZERO_CURTAILMENT_IN_OPERATIONAL_REPLAY:
        pcurt_opf = {key: 0.0 for key in pcurt_opf}
    for (b,ph) in pvph:
        for t in hours:
            m.Pcurt[b,ph,t].fix(
                0.0 if FORCE_ZERO_CURTAILMENT_IN_OPERATIONAL_REPLAY
                else pcurt_opf[(b,ph,t)])

    # Um fluxo ex-post não deve conservar a função multiobjetivo do OPF: ela
    # pode preferir l artificialmente alto para aproximar Vref. Com os controles
    # fixos, minimizar estritamente Σl aperta a relaxação e recupera o estado
    # físico radial sempre que a relaxação for exata para esse despacho.
    if hasattr(m, 'obj'):
        m.obj.deactivate()
    m.obj_physical_pf = pyo.Objective(
        expr=sum(m.l_sq[i,j,ph,t] for (i,j,ph) in cph for t in hours))

    opt = pyo.SolverFactory('cplex', executable=solver_executable)
    opt.options['qpmethod'] = 6
    opt.options['timelimit'] = 1200
    opt.options['threads'] = 4
    opt.options['barrier_convergetol'] = 1e-6

    tap_opf_schedule = {
        (ph,t): int(round(pyo.value(opf_model.tap_pos[ph,t])))
        for ph in sets_info['svr_phs'] for t in hours}

    # Q* e a trajetória de taps do OPF são apenas referência de identificação.
    # O replay de Q* é executado uma única vez no candidato completo; S2 e os
    # rollouts esparsos podem suprimi-lo para não repetir um diagnóstico caro
    # que não participa da decisão física do LDC.
    opf_pf = opf_model
    opf_pf_termination = 'skipped_redundant_qstar_replay'
    opf_pf_violation = 0.0
    opf_pf_converged = True
    opf_pf_iterations = 0
    opf_pf_dv = float('nan')
    if run_qstar_diagnostic:
        opf_pf = m.clone()
        for (b,ph) in pvph:
            for t in hours:
                opf_pf.Qpv[b,ph,t].fix(q_opf[(b,ph,t)])
        for (ph,t), pos in tap_opf_schedule.items():
            opf_pf.tap_pos[ph,t].fix(pos)
        activate_exact_fixed_tap_svr(
            opf_pf, data, hours, tap_opf_schedule)
        (opf_pf_res, opf_pf_termination, _, opf_pf_violation,
         opf_pf_converged, opf_pf_iterations, opf_pf_dv) = (
            _solve_sca_voltage_consistent(
                opt, opf_pf, cph, hours, label='replay-Q*-diagnostico',
                tee_first=False))
        if opf_pf_termination not in _USABLE_TERMINATIONS:
            raise RuntimeError(
                f"reaplicação física diagnóstica de Q* sem solução: "
                f"{opf_pf_termination}")

    if SETTINGS_TAP_MODE == 'autonomous_ldc':
        if autonomous_seed_schedule is None:
            raise ValueError(
                'S3 autônomo requer autonomous_seed_schedule da baseline S1.')
        expected = {(ph,t) for ph in sets_info['svr_phs'] for t in hours}
        if set(autonomous_seed_schedule) != expected:
            raise ValueError(
                'autonomous_seed_schedule incompleto para S3 autônomo.')
        tap_schedule = {k: int(v) for k, v in autonomous_seed_schedule.items()}
        tap_prev = dict(tap_schedule)
        q_prev = {(b,ph,t): 0.0 for (b,ph) in pvph for t in hours}

        # Todos os cenários partem do mesmo estado físico S1, não do estado
        # relaxado/otimizado do OPF. O seed é apenas numérico: na iteração
        # seguinte o LDC reconstrói toda a trajetória a partir de tap_init.
        for (ph,t), pos in tap_schedule.items():
            m.tap_pos[ph,t].fix(pos)
        activate_exact_fixed_tap_svr(m, data, hours, tap_schedule)
        for (b,ph) in pvph:
            for t in hours:
                m.Qpv[b,ph,t].fix(0.0)
        seed_source_model = (
            autonomous_seed_model if autonomous_seed_model is not None
            else (opf_pf if run_qstar_diagnostic else opf_model))
        for (b,ph) in sets_info['bph']:
            for t in hours:
                v0 = max(try_v(seed_source_model, b, ph, t), 0.80)
                m.V_bar_bus[b,ph,t] = v0
        for (i,j,ph) in cph:
            for t in hours:
                vi0 = max(try_v(seed_source_model, i, ph, t), 0.80)
                m.V_sq_ws[i,j,ph,t] = max(vi0*vi0, 0.64)
        (_, seed_term, _, _, seed_ok, _, _) = _solve_sca_voltage_consistent(
            opt, m, cph, hours, label='seed-S1-para-S3-autonomo',
            tee_first=False)
        if seed_term not in _USABLE_TERMINATIONS or not seed_ok:
            raise RuntimeError(
                'Não foi possível fechar o seed físico S1 antes de S3: '
                f'{seed_term}.')
        v_prev = {(b,ph,t): try_v(m, b, ph, t)
                  for (b,ph) in sets_info['bph'] for t in hours}
    else:
        q_prev = dict(q_opf)
        tap_prev = dict(tap_opf_schedule)
        tap_schedule = dict(tap_opf_schedule)
        v_prev = {(b,ph,t): try_v(opf_pf, b, ph, t)
                  for (b,ph) in sets_info['bph'] for t in hours}
    tap_ops = {}
    _initial_state_for_ops = _normalise_relay_state(
        data, initial_relay_state)
    for ph in sets_info['svr_phs']:
        init_pos = int(_initial_state_for_ops[ph]['tap_position'])
        prev_pos = init_pos
        tap_ops[ph] = 0
        for t in hours:
            cur_pos = tap_schedule[(ph,t)]
            tap_ops[ph] += abs(cur_pos - prev_pos)
            prev_pos = cur_pos
    relay_values = {}
    final_relay_state = _normalise_relay_state(data, initial_relay_state)
    converged = False
    termination = 'notSolved'
    last_dv = float('inf')
    last_dq = float('inf')

    # Barras trifásicas: um único equipamento recebe o mesmo Q por fase.
    phases_by_bus = {}
    for b, ph in pvph:
        if data['pv'][(b,ph)].get('is_3ph', False):
            phases_by_bus.setdefault(b, []).append(ph)

    for iteration in range(1, SETTINGS_MAX_ITER + 1):
        if SETTINGS_TAP_MODE == 'autonomous_ldc':
            (tap_schedule, tap_ops, relay_values,
             final_relay_state) = _svr_schedule_from_solution(
                m, data, hours,
                initial_relay_state=initial_relay_state,
                return_final_state=True)
        elif SETTINGS_TAP_MODE != 'opf_schedule':
            raise ValueError(
                f"SETTINGS_TAP_MODE inválido: {SETTINGS_TAP_MODE!r}")
        tap_delta = sum(tap_schedule[k] != tap_prev[k] for k in tap_schedule)
        for (ph,t), pos in tap_schedule.items():
            m.tap_pos[ph,t].fix(pos)
        activate_exact_fixed_tap_svr(m, data, hours, tap_schedule)

        q_cmd = {}
        for (b,ph) in pvph:
            g = group_of.get((b,ph), 'C')
            snom_pu = data['pv'][(b,ph)]['S_nom_pu']
            for t in hours:
                p_pu = max(pyo.value(m.Pavail[b,ph,t])
                           - pcurt_opf[(b,ph,t)], 0.0)
                qcap_pu = _operational_qcap_pu(p_pu, snom_pu)
                solar_ok = (ENABLE_NIGHT_VAR or
                            PV_EFF_FACTOR[t] >= QPV_MIN_SOLAR_FRAC)
                if not solar_ok:
                    raw_q_pu = 0.0
                elif g == 'A':
                    raw_q_pu = _static_non_vv_raw_q(
                        fixed_of[(b,ph)], p_pu, snom_pu)
                elif g == 'B':
                    raw_q_pu = _vv_setting_eval_pu(
                        v_prev[(b,ph,t)], curve_of[(b,ph)])
                else:
                    raw_q_pu = 0.0
                q_cmd[(b,ph,t)] = max(
                    -qcap_pu, min(qcap_pu, raw_q_pu))

        # Acoplamento por dispositivo: média dos comandos das três fases.
        if ENABLE_3PH_INVERTER_COUPLING:
            for b, phases in phases_by_bus.items():
                for t in hours:
                    q_common = sum(q_cmd[(b,ph,t)] for ph in phases) / len(phases)
                    for ph in phases:
                        q_cmd[(b,ph,t)] = q_common

        # Parâmetros V1..V4/Qmax são estáticos; somente o comando local Q(V)
        # evolui. O limitador transforma essa evolução em trajetória executável.
        q_cmd = _limit_q_command_slew(q_cmd, data, pvph, hours)

        q_applied = {
            k: (0.0 if (not ENABLE_NIGHT_VAR and
                        PV_EFF_FACTOR[k[2]] < QPV_MIN_SOLAR_FRAC)
                else SETTINGS_DAMPING*q_cmd[k]
                     + (1.0 - SETTINGS_DAMPING)*q_prev[k])
            for k in q_cmd}
        # A mistura de uma trajetória limitada com o Q* inicial pode voltar a
        # exceder ΔQ no começo do ponto fixo. A projeção final garante que toda
        # iterada aplicada — não apenas o alvo q_cmd — respeite a rampa física.
        q_applied = _limit_q_command_slew(
            q_applied, data, pvph, hours)
        q_applied = _enforce_operational_fp_floor(
            q_applied, m, data, pvph, hours)
        for (b,ph) in pvph:
            for t in hours:
                m.Qpv[b,ph,t].fix(q_applied[(b,ph,t)])

        # Atualiza todas as linearizações V² com a tensão anterior.
        for (b,ph) in sets_info['bph']:
            for t in hours:
                m.V_bar_bus[b,ph,t] = max(v_prev[(b,ph,t)], 0.80)
        for (i,j,ph) in cph:
            for t in hours:
                m.V_sq_ws[i,j,ph,t] = max(v_prev[(i,ph,t)]**2, 0.64)

        res, termination, _, iter_violation = _solve_with_status_recovery(
            opt, m, tee=False, label=f'closed-loop-{iteration:02d}')
        if termination not in ('optimal', 'locallyOptimal', 'feasible'):
            raise RuntimeError(
                f"iteração {iteration} sem solução utilizável: {termination}")

        v_new = {(b,ph,t): try_v(m, b, ph, t)
                 for (b,ph) in sets_info['bph'] for t in hours}
        last_dv = max(abs(v_new[k] - v_prev[k]) for k in v_new)
        last_dq = max(abs(q_applied[k] - q_prev[k]) for k in q_applied)
        print(f"      iter={iteration:02d}  max|ΔV|={last_dv:.3e} p.u.  "
              f"max|ΔQ|={last_dq:.3e} p.u./fase  "
              f"Δtap={tap_delta}")
        v_prev = v_new
        q_prev = q_applied
        tap_prev = dict(tap_schedule)
        if (last_dv <= SETTINGS_V_TOL and last_dq <= SETTINGS_Q_TOL
                and tap_delta == 0):
            converged = True
            break

    # Validação híbrida solicitada: as injeções aplicadas são congeladas em
    # p.u.; somente as tensões dos inversores Volt-VAR são fixadas no ponto de
    # equilíbrio. Nos grupos de fator de potência fixo, V continua sendo uma
    # consequência livre das equações da rede.
    p_injection_pu = {
        (b,ph,t): pyo.value(m.Pavail[b,ph,t])-pcurt_opf[(b,ph,t)]
        for b,ph in pvph for t in hours}
    if ENABLE_HYBRID_POST_CLASSIFICATION_VALIDATION:
        hybrid_validation = validate_hybrid_fixed_pq_pu(
            m, data, sets_info, groups, p_injection_pu, q_prev,
            tap_schedule, opt,
            initial_relay_state=initial_relay_state)
    else:
        hybrid_validation = {
            'model': m, 'ok': True, 'termination': 'disabled',
            'electrical_unit_system': 'per_unit',
            'p_fixed': True, 'q_fixed': True,
            'voltage_fixed_groups': (), 'voltage_free_groups': ('A','B','C'),
            'fixed_voltage_count': 0,
            'free_voltage_count': len(pvph)*len(hours),
            'max_fixed_voltage_error_pu': 0.0,
            'max_fixed_p_error_pu': 0.0,
            'max_fixed_q_error_pu': 0.0,
            'mv_reference_ok': True, 'mv_reference_rows': [],
            'voltage_target_pu': {},
            'p_injection_pu': p_injection_pu,
            'q_injection_pu': dict(q_prev),
            'objective_pu': recalculate_original_objective_pu(
                m, data, sets_info),
            'exactness': summarize_bfm_exactness(m, data, sets_info),
        }
    objective_opf_replay_pu = recalculate_original_objective_pu(
        opf_pf, data, sets_info)

    settings_evaluation_model = (
        hybrid_validation['model']
        if hybrid_validation['termination'] in _USABLE_TERMINATIONS else m)

    relaxed_opf_quality = summarize_voltage_quality(opf_model, data, sets_info)
    opf_quality = summarize_voltage_quality(opf_pf, data, sets_info)
    settings_quality = summarize_voltage_quality(
        settings_evaluation_model, data, sets_info)
    opf_bfm_exactness = summarize_bfm_exactness(opf_pf, data, sets_info)
    settings_bfm_exactness = summarize_bfm_exactness(
        settings_evaluation_model, data, sets_info)
    q_errors_pu = [q_prev[k]-q_opf[k] for k in q_prev]
    q_rmse_pu = math.sqrt(
        sum(e*e for e in q_errors_pu)/max(len(q_errors_pu), 1))
    q_ref_pu = math.sqrt(
        sum(q_opf[k]**2 for k in q_opf)/max(len(q_opf), 1))
    inverter_operability = _summarize_inverter_operability(
        q_prev, data, pvph, hours)
    validation_bound_hits = sum(
        1 for (b,ph) in sets_info['bph'] if b != slack for t in hours
        if (try_v(settings_evaluation_model,b,ph,t) <= VALIDATION_V_MIN + 1.0e-5 or
            try_v(settings_evaluation_model,b,ph,t) >= VALIDATION_V_MAX - 1.0e-5)
    )
    quality_screen = assess_representative_day_quality(
        settings_quality,
        validation_bound_hits=validation_bound_hits,
        baseline_quality=baseline_quality_for_screening,
        period_count=len(hours))
    daily_screening_ok = quality_screen['daily_screening_ok']
    # Compatibilidade: ``regulatory_ok`` deixa de representar um resultado
    # formal nesta campanha diária. Mantém-se explicitamente indefinido.
    regulatory_ok = None
    tap_schedule_matches_opf = all(
        tap_schedule[k] == tap_opf_schedule[k] for k in tap_opf_schedule)
    physical_model_ok = bool(
        settings_bfm_exactness['exactness_ok'] and
        (not run_qstar_diagnostic or opf_bfm_exactness['exactness_ok']))
    operationally_applicable = bool(
        converged and (not run_qstar_diagnostic or opf_pf_converged) and
        daily_screening_ok and physical_model_ok and hybrid_validation['ok'] and
        PV_TEMP_DATA_OK and inverter_operability['ok'] and
        (SETTINGS_TAP_MODE != 'opf_schedule' or tap_schedule_matches_opf))

    # Métricas do S3 final são calculadas somente no estado físico produzido
    # pela política A/B/C + LDC. A trajetória e Q* do OPF não entram aqui.
    settings_objective_pu = recalculate_original_objective_pu(
        settings_evaluation_model, data, sets_info,
        initial_relay_state=initial_relay_state,
        include_terminal_return=(initial_relay_state is None))
    settings_core_loss_kwh = calc_core_loss_energy_kwh(
        settings_evaluation_model, data, sets_info)
    settings_joule_loss_kwh = calc_joule_loss_energy_kwh(
        settings_evaluation_model, data, sets_info)
    settings_tap_audit = _audit_programmed_tap_schedule(
        tap_schedule, data, hours,
        initial_relay_state=initial_relay_state,
        enforce_daily_budget=(initial_relay_state is None))
    settings_svr_exact_residual_pu = max_exact_svr_residual_pu(
        settings_evaluation_model, data, hours, tap_schedule)

    # O Estágio 2 só pode certificar otimalidade contínua condicionada à
    # trajetória discreta de taps quando a própria relaxação que gerou Q* é
    # exata. O replay físico prova factibilidade dos controles, mas não converte
    # uma relaxação folgada em solução ótima do problema não convexo.
    conditional_optimality_certified = bool(
        SETTINGS_TAP_MODE == 'opf_schedule' and source_relaxation_exact and
        opf_pf_converged and opf_bfm_exactness['exactness_ok'])
    # A trajetória discreta é obtida por relaxação + arredondamento no Estágio 1;
    # portanto não existe certificado global para a combinação taps+reativos.
    global_optimality_certified = False
    return {
        'model': settings_evaluation_model,
        'converged': converged, 'iterations': iteration,
        'termination': termination, 'max_dv': last_dv,
        'max_dq_pu': last_dq,
        'max_dq_kvar': last_dq*SBASE,  # somente compatibilidade de relatório
        'relaxed_opf_quality': relaxed_opf_quality,
        'opf_quality': opf_quality, 'settings_quality': settings_quality,
        'opf_replay_model': opf_pf,
        'opf_replay_converged': opf_pf_converged,
        'opf_replay_termination': opf_pf_termination,
        'opf_replay_iterations': opf_pf_iterations,
        'opf_replay_max_dv': opf_pf_dv,
        'qstar_diagnostic_executed': bool(run_qstar_diagnostic),
        'q_rmse_pu': q_rmse_pu,
        'q_rmse_kvar': q_rmse_pu*SBASE,  # somente compatibilidade de relatório
        'q_nrmse_pct': 100.0*q_rmse_pu/max(q_ref_pu, 1e-9),
        'inverter_operability': inverter_operability,
        'vv_profile_name': (
            'opf_individual_fit' if USE_INDIVIDUAL_FITTED_VV_CURVES
            else 'ieee_catb_default'),
        'q_settings': q_prev, 'tap_schedule': tap_schedule,
        'tap_ops': tap_ops, 'relay_values': relay_values,
        'initial_relay_state': _normalise_relay_state(
            data, initial_relay_state),
        'final_relay_state': final_relay_state,
        'svr_equation_mode': 'exact_fixed_tap',
        'max_svr_exact_residual_pu': settings_svr_exact_residual_pu,
        'tap_audit': settings_tap_audit,
        'objective_metrics_pu': settings_objective_pu,
        'voltage_deviation_pu_sum':
            settings_objective_pu['raw']['voltage_deviation_pu_sum'],
        'unbalance_pu_sum':
            settings_objective_pu['raw']['unbalance_pu_sum'],
        'core_loss_kwh': settings_core_loss_kwh,
        'joule_loss_kwh': settings_joule_loss_kwh,
        'total_loss_kwh': settings_core_loss_kwh + settings_joule_loss_kwh,
        'validation_bound_hits': validation_bound_hits,
        'tap_mode': SETTINGS_TAP_MODE,
        'tap_schedule_matches_opf': tap_schedule_matches_opf,
        'opf_bfm_exactness': opf_bfm_exactness,
        'settings_bfm_exactness': settings_bfm_exactness,
        'physical_model_ok': physical_model_ok,
        'input_data_ok': PV_TEMP_DATA_OK,
        'daily_quality_screening': quality_screen,
        'daily_screening_ok': daily_screening_ok,
        'daily_screening_absolute_ok': quality_screen['daily_screening_absolute_ok'],
        'formal_regulatory_assessed': quality_screen['formal_regulatory_assessed'],
        'formal_regulatory_ok': quality_screen['formal_regulatory_ok'],
        'regulatory_ok': regulatory_ok,
        'representative_day_candidate_acceptable': operationally_applicable,
        'pending_weekly_regulatory_assessment': True,
        'operationally_applicable': operationally_applicable,
        'conditional_optimality_certified': conditional_optimality_certified,
        'global_optimality_certified': global_optimality_certified,
        'hybrid_validation': hybrid_validation,
        'hybrid_validation_ok': hybrid_validation['ok'],
        'objective_opf_replay_pu': objective_opf_replay_pu,
        'objective_settings_hybrid_pu': hybrid_validation['objective_pu'],
        'electrical_unit_system': 'per_unit',
    }



def _autonomous_sparse_device_members(data, pvph):
    """Agrupa terminais por equipamento físico para a busca esparsa."""
    devices = {}
    for bp in pvph:
        key = ((bp[0], '3ph') if data['pv'][bp].get('is_3ph', False) else bp)
        devices.setdefault(key, []).append(bp)
    return {key: tuple(sorted(members, key=lambda x: (str(x[0]), x[1])))
            for key, members in devices.items()}


def _autonomous_sparse_ranking(full_result, groups, data, pvph, hours):
    """Ordena equipamentos ativos por suporte aplicado × sensibilidade elétrica.

    O ranking serve somente para definir a ordem dos rollouts. A classe final
    continua sendo decidida pelo replay físico completo com LDC autônomo.
    """
    group_of = {bp: g for g, members in groups.items() for bp in members}
    devices = _autonomous_sparse_device_members(data, pvph)
    distance = data.get('pv_electrical_distance') or \
        _pv_electrical_distance_metrics(data, pvph)
    rows = []
    q_settings = full_result.get('q_settings', {})
    for key, members in devices.items():
        active_group = next((group_of.get(bp, 'C') for bp in members
                             if group_of.get(bp, 'C') in ('A', 'B')), 'C')
        if active_group == 'C':
            continue
        q_sq = 0.0
        n = 0
        snom = 0.0
        sensitivity = 0.0
        dist = 0.0
        for bp in members:
            snom += float(data['pv'][bp]['S_nom_pu'])
            drow = distance.get(bp, {})
            sensitivity = max(sensitivity, float(
                drow.get('reactive_sensitivity_normalized', 0.0)))
            dist = max(dist, float(
                drow.get('electrical_distance_normalized', 0.0)))
            for t in hours:
                q = float(q_settings.get((bp[0], bp[1], t), 0.0))
                q_sq += q*q
                n += 1
        q_rms = math.sqrt(q_sq/max(n, 1))
        q_rms_frac_s = q_rms/max(snom/max(len(members), 1), 1.0e-12)
        priority = q_rms_frac_s*(1.0+sensitivity)
        rows.append({
            'device': key, 'members': members, 'group': active_group,
            'q_rms_frac_s': q_rms_frac_s,
            'reactive_sensitivity_normalized': sensitivity,
            'electrical_distance_normalized': dist,
            'priority': priority,
        })
    rows.sort(key=lambda r: (-r['priority'], -r['q_rms_frac_s'],
                             str(r['device'])))
    return rows


def _autonomous_sparse_variant(groups, inferred_B, fixed_pf_A,
                               active_devices, data, pvph):
    """Mantém A/B apenas nos equipamentos selecionados; demais tornam-se C."""
    device_members = _autonomous_sparse_device_members(data, pvph)
    active_members = {
        bp for key in active_devices for bp in device_members.get(key, ())}
    group_of = {bp: g for g, members in groups.items() for bp in members}
    variant = {'A': [], 'B': [], 'C': []}
    for bp in pvph:
        g = group_of.get(bp, 'C') if bp in active_members else 'C'
        variant[g].append(bp)
    curve_lookup = {(row['bus'], row['ph']): row for row in inferred_B}
    curves = [dict(curve_lookup[bp]) for bp in variant['B']
              if bp in curve_lookup]
    fixed = {bp: fixed_pf_A[bp] for bp in variant['A']
             if bp in fixed_pf_A}
    return variant, curves, fixed


def _autonomous_candidate_acceptance(result, baseline):
    """Aplica a mesma lógica física/operacional usada na seleção do paper."""
    if not baseline.get('valid', False):
        return False, ['baseline_fisica_nao_certificada'], None
    taps = int(sum(result.get('tap_ops', {}).values()))
    dv = float(result.get('voltage_deviation_pu_sum', float('inf')))
    ub = float(result.get('unbalance_pu_sum', float('inf')))
    core = float(result.get('core_loss_kwh', float('inf')))
    total = float(result.get('total_loss_kwh', float('inf')))
    base_taps = int(baseline['tap_ops_total'])
    base_dv = float(baseline['voltage_deviation_pu_sum'])
    base_ub = float(baseline['unbalance_pu_sum'])
    base_core = float(baseline['core_loss_kwh'])
    base_total = float(baseline['total_loss_kwh'])
    operational = bool(result.get('operationally_applicable', False))
    score = (
        OP_WEIGHT_TAP*taps/max(base_taps, 1) +
        OP_WEIGHT_VOLTAGE_DEVIATION*dv/max(base_dv, 1.0e-9) +
        OP_WEIGHT_UNBALANCE*ub/max(base_ub, 1.0e-9) +
        OP_WEIGHT_CORE*core/max(base_core, 1.0e-9))
    audit = _audit_four_objective_dominance(
        taps, dv, ub, core, total,
        base_taps, base_dv, base_ub, base_core, base_total,
        operationally_admissible=operational,
        relative_score_improvement=1.0-score)
    return bool(audit['ok']), list(audit['reasons']), score


def search_autonomous_sparse_policy_v3(full_result, opf_model, data,
                                        sets_info, groups, inferred_B,
                                        fixed_pf_A, solver_executable,
                                        baseline_physical_result):
    """Level III: replays físicos progressivamente esparsos com LDC autônomo.

    Q* e a classificação A/B/C definem apenas modos e ranking. Para cada
    prefixo, os equipamentos não selecionados são colocados em FP=1 e o fluxo
    completo é resolvido novamente; o próprio LDC reconstrói todos os taps.
    Entre os prefixos que passam as guardas, seleciona primeiro o de menor
    número de operações de tap, depois o menor escore operacional e, em empate,
    o menor conjunto ativo. Se nenhum prefixo for aceito, retorna explicitamente
    a política de não ação.
    """
    pvph = list(sets_info['pvph'])
    hours = list(sets_info['hours'])
    baseline_schedule = baseline_physical_result.get('tap_schedule')
    baseline_model = baseline_physical_result.get('model')
    print("\n  [Level III] BUSCA ESPARSA AUTÔNOMA SOLICITADA")
    print(f"  sparse_enabled={ENABLE_AUTONOMOUS_SPARSE_POLICY_SEARCH} | "
          f"baseline_valid={bool(baseline_physical_result.get('valid', False))} | "
          f"baseline_schedule={bool(baseline_schedule)}")
    if not baseline_physical_result.get('valid', False) or not baseline_schedule:
        reason = ('baseline_fisica_nao_certificada' if not
                  baseline_physical_result.get('valid', False)
                  else 'baseline_sem_trajetoria_ldc')
        full_result['autonomous_sparse_search_executed'] = False
        full_result['autonomous_sparse_search_reason'] = reason
        full_result['no_action_applied'] = True
        try:
            import pandas as _pd_sparse_skip
            _pd_sparse_skip.DataFrame([{
                'executed': False,
                'reason': reason,
                'baseline_valid': bool(baseline_physical_result.get('valid', False)),
                'baseline_schedule_available': bool(baseline_schedule),
            }]).to_csv('/content/paper_autonomous_sparse_search.csv', index=False)
        except Exception:
            pass
        print(f"  [Level III] busca esparsa NÃO executada: {reason}")
        return full_result, groups, inferred_B, fixed_pf_A

    ranking = _autonomous_sparse_ranking(
        full_result, groups, data, pvph, hours)
    n_active = len(ranking)
    if not ranking:
        full_result['autonomous_sparse_search_executed'] = True
        full_result['autonomous_sparse_candidate_count'] = 0
        full_result['no_action_applied'] = True
        return full_result, groups, inferred_B, fixed_pf_A

    counts = sorted({
        max(1, min(n_active, int(math.ceil(frac*n_active))))
        for frac in AUTONOMOUS_SPARSE_PREFIX_FRACTIONS})
    if n_active not in counts:
        counts.append(n_active)
    counts = sorted(set(counts))

    # V4.7: o prefixo de 100% dos equipamentos A/B é sempre auditado. Na
    # V4.6, quando ``len(counts) == max_replays`` e o último elemento já era
    # ``n_active``, o fatiamento ``counts[:max_replays-1]`` removia justamente
    # esse caso completo. Reservamos explicitamente a última vaga para ele.
    max_replays = max(1, AUTONOMOUS_SPARSE_MAX_REPLAYS)
    if len(counts) > max_replays:
        if max_replays == 1:
            counts = [n_active]
        else:
            partial_counts = [c for c in counts if c < n_active]
            counts = partial_counts[:max_replays-1] + [n_active]
    elif n_active not in counts:
        # Salvaguarda defensiva: não deve ocorrer, mas mantém a auditoria
        # completa mesmo se a construção dos prefixos for alterada no futuro.
        counts = (counts[:max_replays-1] + [n_active]
                  if max_replays > 1 else [n_active])
    counts = sorted(set(counts))

    print("\n  [Level III] BUSCA ESPARSA — LDC AUTÔNOMO")
    print(f"  Equipamentos A/B provisórios={n_active}; prefixos testados={counts}")
    print("  Critério: redução de taps vs S1 + guardas físicas + screening diário comparativo "
          "+ tolerâncias de desvio e perdas.")

    catalog = []
    accepted = None
    accepted_payload = None
    accepted_key = None
    for count in counts:
        active_keys = [row['device'] for row in ranking[:count]]
        vg, vc, vf = _autonomous_sparse_variant(
            groups, inferred_B, fixed_pf_A, active_keys, data, pvph)
        print(f"    prefixo {count}/{n_active}: A={len(vg['A'])} "
              f"B={len(vg['B'])} C={len(vg['C'])}")
        try:
            candidate = validate_recommended_settings(
                opf_model, data, sets_info, vg, vc, vf, solver_executable,
                source_relaxation_exact=False,
                autonomous_seed_schedule=baseline_schedule,
                autonomous_seed_model=baseline_model,
                run_qstar_diagnostic=False,
                baseline_quality_for_screening=baseline_physical_result.get('quality'))
            ok, reasons, score = _autonomous_candidate_acceptance(
                candidate, baseline_physical_result)
            taps = int(sum(candidate.get('tap_ops', {}).values()))
            quality = candidate.get('settings_quality') or {}
            mv_quality = quality.get('mv') or {}
            lv_quality = quality.get('lv') or {}
            fd_quality = quality.get('fd') or {}
            bfm = candidate.get('settings_bfm_exactness') or {}
            qscreen = candidate.get('daily_quality_screening') or {}
            qlimits = qscreen.get('limits') or {}
            physical_valid = bool(
                candidate.get('converged', False) and
                candidate.get('physical_model_ok', False) and
                candidate.get('hybrid_validation_ok', False))
            operational = bool(
                candidate.get('operationally_applicable', False))
            objective_value = float(
                (candidate.get('objective_metrics_pu') or {}).get(
                    'total', score))
            pf_distance = preferred_pf_distance(vf)
            catalog.append({
                'active_physical_devices': count,
                'A_phase_terminals': len(vg['A']),
                'B_phase_terminals': len(vg['B']),
                'C_phase_terminals': len(vg['C']),
                'tap_ops_total': taps,
                # ``valid`` é mantido por compatibilidade com os CSVs da V4.6;
                # os campos abaixo separam explicitamente validade física,
                # conformidade regulatória e aplicabilidade operacional.
                'valid': operational,
                'physical_valid': physical_valid,
                'daily_screening_ok': bool(candidate.get('daily_screening_ok', False)),
                'daily_screening_absolute_ok': bool(candidate.get('daily_screening_absolute_ok', False)),
                'daily_screening_mode': qscreen.get('daily_screening_mode'),
                'daily_limit_mv_drp_pct': qlimits.get('mv_drp_pct'),
                'daily_limit_mv_drc_pct': qlimits.get('mv_drc_pct'),
                'daily_limit_lv_drp_pct': qlimits.get('lv_drp_pct'),
                'daily_limit_lv_drc_pct': qlimits.get('lv_drc_pct'),
                'daily_limit_fd95_system_pct': qlimits.get('fd95_system_pct'),
                'formal_regulatory_assessed': bool(candidate.get('formal_regulatory_assessed', False)),
                'formal_regulatory_ok': candidate.get('formal_regulatory_ok'),
                'regulatory_ok': candidate.get('regulatory_ok'),
                'representative_day_candidate_acceptable': operational,
                'pending_weekly_regulatory_assessment': True,
                'operationally_applicable': operational,
                'hybrid_validation_ok': bool(
                    candidate.get('hybrid_validation_ok', False)),
                'validation_bound_hits': candidate.get(
                    'validation_bound_hits'),
                'accepted': ok,
                'mv_vmin_pu': mv_quality.get('v_min'),
                'mv_vmax_pu': mv_quality.get('v_max'),
                'mv_drp_pct': mv_quality.get('drp'),
                'mv_drc_pct': mv_quality.get('drc'),
                'lv_vmin_pu': lv_quality.get('v_min'),
                'lv_vmax_pu': lv_quality.get('v_max'),
                'lv_drp_pct': lv_quality.get('drp'),
                'lv_drc_pct': lv_quality.get('drc'),
                'fd95_system_pct': fd_quality.get('fd95_system'),
                'bfm_exact': bool(bfm.get('exactness_ok', False)),
                'bfm_aggregate_gap_pct': bfm.get('aggregate_gap_pct'),
                'bfm_max_physical_violation_pu': bfm.get(
                    'max_physical_violation_pu'),
                'bfm_max_vdrop_residual_pu2': bfm.get(
                    'max_vdrop_residual_pu2'),
                'voltage_deviation_pu_sum': candidate.get(
                    'voltage_deviation_pu_sum'),
                'unbalance_pu_sum': candidate.get('unbalance_pu_sum'),
                'core_loss_kwh': candidate.get('core_loss_kwh'),
                'joule_loss_kwh': candidate.get('joule_loss_kwh'),
                'total_loss_kwh': candidate.get('total_loss_kwh'),
                'operational_score': score,
                'objective_value': objective_value,
                'distance_to_preferred_pf': pf_distance,
                'rejection_reasons': ';'.join(reasons),
            })
            print(f"      taps={taps}; screening_diario="
                  f"{'OK' if candidate.get('daily_screening_ok') else 'FALHA'}; "
                  f"operacional={'SIM' if candidate.get('operationally_applicable') else 'NÃO'}; "
                  f"aceito={'SIM' if ok else 'NÃO'}" +
                  (f"; motivos={';'.join(reasons)}" if reasons else ''))
            if ok:
                # Candidatos não admissíveis já foram removidos por ``ok``.
                # A ordenação restante é declaradamente lexicográfica:
                # taps -> objetivo -> FP mais próximo de 0,96 -> esparsidade.
                candidate_key = lexicographic_candidate_key({
                    'guards_ok': True,
                    'tap_ops_total': taps,
                    'objective_value': objective_value,
                    'distance_to_preferred_pf': pf_distance,
                    'candidate_id': f'prefix_{count:04d}',
                }, objective_equivalence_tol=1.0e-6) + (count,)
                if accepted_key is None or candidate_key < accepted_key:
                    if accepted is not None:
                        accepted.pop('model', None)
                        accepted.pop('opf_replay_model', None)
                        old_hyb = accepted.get('hybrid_validation')
                        if isinstance(old_hyb, dict):
                            old_hyb.pop('model', None)
                    accepted = candidate
                    accepted_payload = (vg, vc, vf, active_keys, count)
                    accepted_key = candidate_key
                    print(f"      -> melhor candidato esparso até aqui: "
                          f"taps={taps}, score={score:.6f}, ativos={count}")
                    gc.collect()
                    continue
            # Libera o clone pesado quando ele não é o melhor candidato salvo.
            candidate.pop('model', None)
            candidate.pop('opf_replay_model', None)
            hyb = candidate.get('hybrid_validation')
            if isinstance(hyb, dict):
                hyb.pop('model', None)
            gc.collect()
        except Exception as exc:
            catalog.append({
                'active_physical_devices': count,
                'accepted': False,
                'valid': False,
                'tap_ops_total': None,
                'rejection_reasons': f'exception:{type(exc).__name__}:{exc}',
            })
            print(f"      falha no replay: {type(exc).__name__}: {exc}")

    try:
        import pandas as _pd_sparse
        _pd_sparse.DataFrame(catalog).to_csv(
            '/content/paper_autonomous_sparse_search.csv', index=False)
        _pd_sparse.DataFrame(ranking).drop(columns=['members']).to_csv(
            '/content/paper_autonomous_sparse_ranking.csv', index=False)
    except Exception as exc:
        print(f"  [CSV] aviso busca esparsa: {exc}")

    if accepted is not None:
        vg, vc, vf, active_keys, count = accepted_payload
        accepted['autonomous_sparse_search_executed'] = True
        accepted['autonomous_sparse_candidate_count'] = len(catalog)
        accepted['autonomous_sparse_selected_count'] = count
        accepted['autonomous_sparse_total_active_provisional'] = n_active
        accepted['autonomous_sparse_active_devices'] = [str(k) for k in active_keys]
        accepted['autonomous_sparse_ranking'] = ranking
        accepted['selection_status'] = 'autonomous_sparse_prefix_accepted'
        accepted['no_action_applied'] = False
        accepted['selected_groups'] = vg
        accepted['selected_inferred_B'] = vc
        accepted['selected_fixed_pf_A'] = vf
        return accepted, vg, vc, vf

    # Nenhum prefixo foi aceito: materializa a decisão de não ação como um
    # replay autônomo completo, em vez de publicar uma política não conforme.
    print("  Nenhum prefixo A/B atendeu às guardas físicas, ao screening diário comparativo e às metas; aplicando não ação (FP=1).")
    unity_groups = {'A': [], 'B': [], 'C': list(pvph)}
    no_action = validate_recommended_settings(
        opf_model, data, sets_info, unity_groups, [], {}, solver_executable,
        source_relaxation_exact=False,
        autonomous_seed_schedule=baseline_schedule,
        autonomous_seed_model=baseline_model,
        run_qstar_diagnostic=False,
        baseline_quality_for_screening=baseline_physical_result.get('quality'))
    no_action['autonomous_sparse_search_executed'] = True
    no_action['autonomous_sparse_candidate_count'] = len(catalog)
    no_action['autonomous_sparse_selected_count'] = 0
    no_action['autonomous_sparse_total_active_provisional'] = n_active
    no_action['autonomous_sparse_active_devices'] = []
    no_action['autonomous_sparse_ranking'] = ranking
    no_action['selection_status'] = 'no_action_after_autonomous_sparse_search'
    no_action['no_action_applied'] = True
    no_action['selected_groups'] = unity_groups
    no_action['selected_inferred_B'] = []
    no_action['selected_fixed_pf_A'] = {}
    return no_action, unity_groups, [], {}


def calc_core_loss_energy_kwh(m, data, sets_info):
    """Energia de perdas no núcleo coerente com o balanço Pcore=G·V_j²."""
    return sum(
        bd.get('G_core_pu', 0.0) * try_v(m, j, ph, t)**2
        for (i,j,ph), bd in data['branches'].items()
        for t in sets_info['hours']) * SBASE * (DT_MIN/60.0)


In [ ]:
#@title Internal model implementation 7/12 { display-mode: "form" }
def calc_joule_loss_energy_kwh(m, data, sets_info):
    """Energia Joule física usando R·(P²+Q²)/V² em cada ramo."""
    return sum(
        bd.get('r_pu', 0.0) *
        (pyo.value(m.P[i,j,ph,t])**2 + pyo.value(m.Q[i,j,ph,t])**2) /
        max(try_v(m, i, ph, t)**2, 0.64)
        for (i,j,ph), bd in data['branches'].items()
        for t in sets_info['hours']) * SBASE * (DT_MIN/60.0)


def evaluate_physical_ldc_baseline(seed_model, data, sets_info,
                                   tap_schedule, voltage_seed, opt,
                                   initial_relay_state=None):
    """Certifica S1 por ponto fixo entre BFM físico e LDC autônomo.

    Corrige duas fragilidades da versão anterior:
    1) todas as restrições auxiliares de programação de tap do OPF são
       desativadas antes de fixar uma trajetória LDC; caso contrário,
       ``u_tap=0`` e vínculos de evento/delay podem tornar artificialmente
       inviável uma trajetória que contém operações reais;
    2) a trajetória recebida do sweep pré-OPF é apenas seed. Após cada fluxo
       físico, o LDC é reconstruído a partir das tensões/correntes resolvidas e
       o processo é repetido até a trajetória de taps permanecer inalterada.
    """
    m = seed_model.clone()
    hours = list(sets_info['hours'])
    cph = list(sets_info['cph'])

    for objective in m.component_objects(pyo.Objective, active=True):
        objective.deactivate()

    # A baseline é operação física com taps determinados externamente pelo LDC,
    # não um novo problema de scheduling. Desativar TODO o aparato temporal do
    # OPF é essencial para não criar inviabilidade espúria.
    for cname in ('c_v_margin_low', 'c_v_margin_high',
                  'c_tap_up', 'c_tap_dn', 'c_tap_abs_up', 'c_tap_abs_dn',
                  'c_tap_event_link', 'c_tap_max', 'c_tap_delay',
                  'c_tap_spacing', 'c_tap_terminal',
                  'c_tap_terminal_abs_up', 'c_tap_terminal_abs_dn'):
        component = getattr(m, cname, None)
        if component is not None:
            component.deactivate()

    for b, ph in sets_info['pvph']:
        for t in hours:
            m.Pcurt[b,ph,t].fix(0.0)
            m.Qpv[b,ph,t].fix(0.0)

    for b, ph in sets_info['bph']:
        for t in hours:
            if b != data['slack']:
                if m.V[b,ph,t].fixed:
                    m.V[b,ph,t].unfix()
                m.V[b,ph,t].setlb(VALIDATION_V_MIN)
                m.V[b,ph,t].setub(VALIDATION_V_MAX)
            v0 = max(float(voltage_seed.get((b,ph,t), 1.0)), 0.80)
            m.V[b,ph,t].set_value(v0)
            m.V_bar_bus[b,ph,t] = v0
    for i,j,ph in cph:
        for t in hours:
            vi0 = max(float(voltage_seed.get((i,ph,t), 1.0)), 0.80)
            m.V_sq_ws[i,j,ph,t] = max(vi0*vi0, 0.64)

    if hasattr(m, 'obj_baseline_physical'):
        m.del_component(m.obj_baseline_physical)
    m.obj_baseline_physical = pyo.Objective(
        expr=sum(m.l_sq[i,j,ph,t] for (i,j,ph) in cph for t in hours))

    current_schedule = {k: int(v) for k, v in tap_schedule.items()}
    last = {
        'termination': 'notSolved', 'raw_termination': 'notSolved',
        'violation': float('inf'), 'sca_ok': False, 'sca_iterations': 0,
        'sca_dv': float('inf'), 'exactness': None}
    relay_values = {}
    ldc_iterations = 0

    for ldc_iterations in range(1, BASELINE_LDC_CLOSED_LOOP_MAX_ITER + 1):
        for ph in sets_info['svr_phs']:
            for t in hours:
                m.tap_pos[ph,t].unfix()
                m.tap_pos[ph,t].fix(int(current_schedule[(ph,t)]))
                m.u_tap[ph,t].fix(0.0)
        activate_exact_fixed_tap_svr(m, data, hours, current_schedule)

        (result, termination, raw_termination, violation, sca_ok,
         sca_iterations, sca_dv) = _solve_sca_voltage_consistent(
            opt, m, cph, hours,
            label=f'baseline-LDC-fisica-CL{ldc_iterations:02d}',
            tee_first=False)
        last.update({
            'termination': termination, 'raw_termination': raw_termination,
            'violation': violation, 'sca_ok': sca_ok,
            'sca_iterations': sca_iterations, 'sca_dv': sca_dv})

        if termination not in _USABLE_TERMINATIONS or not sca_ok:
            break

        exactness = summarize_bfm_exactness(m, data, sets_info)
        svr_exact_residual_pu = max_exact_svr_residual_pu(
            m, data, hours, current_schedule)
        last['exactness'] = exactness

        new_schedule, _, relay_values = _svr_schedule_from_solution(
            m, data, hours, initial_relay_state=initial_relay_state)
        tap_delta = sum(
            int(new_schedule[k]) != int(current_schedule[k])
            for k in current_schedule)
        print(f"  baseline LDC closed-loop iter={ldc_iterations}: "
              f"Δtap={tap_delta} | BFM="
              f"{'OK' if exactness.get('exactness_ok', False) else 'FALHA'}")

        if tap_delta == 0 and exactness.get('exactness_ok', False):
            current_schedule = {k: int(v) for k, v in new_schedule.items()}
            break

        current_schedule = {k: int(v) for k, v in new_schedule.items()}
        # Usa o próprio estado físico recém-resolvido como warm start da próxima
        # iteração LDC, sem herdar V da relaxação OPF.
        for b, ph in sets_info['bph']:
            for t in hours:
                v0 = max(try_v(m,b,ph,t), 0.80)
                m.V_bar_bus[b,ph,t] = v0
        for i,j,ph in cph:
            for t in hours:
                vi0 = max(try_v(m,i,ph,t), 0.80)
                m.V_sq_ws[i,j,ph,t] = max(vi0*vi0, 0.64)
    else:
        # loop esgotou sem break
        pass

    # A baseline só é certificada se fluxo/SCA/BFM e a trajetória LDC forem
    # simultaneamente consistentes.
    if last['termination'] in _USABLE_TERMINATIONS:
        (stable_schedule, ops, relay_values_final,
         final_relay_state) = _svr_schedule_from_solution(
            m, data, hours, initial_relay_state=initial_relay_state,
            return_final_state=True)
    else:
        stable_schedule, ops, relay_values_final = {}, {}, {}
        final_relay_state = _normalise_relay_state(
            data, initial_relay_state)
    schedule_stable = bool(stable_schedule) and all(
        int(stable_schedule[k]) == int(current_schedule[k])
        for k in current_schedule)
    exactness = last.get('exactness')
    svr_exact_residual_pu = (
        max_exact_svr_residual_pu(m, data, hours, current_schedule)
        if last['termination'] in _USABLE_TERMINATIONS else float('inf'))
    physical_ok = bool(
        last['termination'] in _USABLE_TERMINATIONS and last['sca_ok'] and
        schedule_stable and exactness is not None and
        exactness.get('exactness_ok', False) and
        svr_exact_residual_pu <= SVR_EXACT_RESIDUAL_TOL)

    if physical_ok:
        objective = recalculate_original_objective_pu(
            m, data, sets_info, initial_relay_state=initial_relay_state,
            include_terminal_return=(initial_relay_state is None))
        quality = summarize_voltage_quality(m, data, sets_info)
        core_kwh = calc_core_loss_energy_kwh(m, data, sets_info)
        joule_kwh = calc_joule_loss_energy_kwh(m, data, sets_info)
        raw = objective['raw']
        audit = _audit_programmed_tap_schedule(
            current_schedule, data, hours,
            initial_relay_state=initial_relay_state,
            enforce_daily_budget=(initial_relay_state is None))
        quality_screen = assess_representative_day_quality(
            quality, validation_bound_hits=0,
            baseline_quality=quality, period_count=len(hours))
        return {
            'name': 'baseline_ldc_fisica', 'valid': True, 'model': m,
            'termination': last['termination'],
            'raw_termination': last['raw_termination'],
            'solver_violation': last['violation'],
            'sca_converged': last['sca_ok'],
            'sca_iterations': last['sca_iterations'],
            'sca_max_dv_pu': last['sca_dv'],
            'ldc_closed_loop_converged': schedule_stable,
            'ldc_closed_loop_iterations': ldc_iterations,
            'exactness': exactness, 'quality': quality,
            'daily_quality_screening': quality_screen,
            'daily_screening_ok': True,
            'daily_screening_absolute_ok': quality_screen['daily_screening_absolute_ok'],
            'formal_regulatory_assessed': False,
            'formal_regulatory_ok': None,
            'regulatory_ok': None,
            'objective_metrics_pu': objective,
            'tap_schedule': dict(current_schedule), 'tap_audit': audit,
            'relay_values': relay_values_final,
            'initial_relay_state': _normalise_relay_state(
                data, initial_relay_state),
            'final_relay_state': final_relay_state,
            'svr_equation_mode': 'exact_fixed_tap',
            'max_svr_exact_residual_pu': svr_exact_residual_pu,
            'tap_ops_total': sum(audit['ops'].values()),
            'voltage_deviation_pu_sum': raw['voltage_deviation_pu_sum'],
            'unbalance_pu_sum': raw['unbalance_pu_sum'],
            'core_loss_kwh': core_kwh, 'joule_loss_kwh': joule_kwh,
            'total_loss_kwh': core_kwh+joule_kwh,
        }

    return {
        'name': 'baseline_ldc_fisica', 'valid': False, 'model': m,
        'termination': last['termination'],
        'raw_termination': last['raw_termination'],
        'solver_violation': last['violation'],
        'sca_converged': last['sca_ok'],
        'sca_iterations': last['sca_iterations'],
        'sca_max_dv_pu': last['sca_dv'],
        'ldc_closed_loop_converged': schedule_stable,
        'ldc_closed_loop_iterations': ldc_iterations,
        'tap_schedule': dict(current_schedule),
        'initial_relay_state': _normalise_relay_state(
            data, initial_relay_state),
        'final_relay_state': final_relay_state,
        'exactness': exactness,
        'svr_equation_mode': 'exact_fixed_tap',
        'max_svr_exact_residual_pu': svr_exact_residual_pu,
    }


def replay_qstar_for_curve_interpretation(opf_model, data, sets_info,
                                          solver_executable):
    """Fecha fisicamente o despacho Q* antes de formar os pares (V,Q*).

    Q*, Pcurt e taps são copiados do OPF exploratório. Somente tensões, fluxos e
    correntes são recalculados minimizando corrente, de modo que o ajuste não
    use tensões provenientes de uma relaxação BFM com folga relevante.
    """
    m = opf_model.clone()
    hours = list(sets_info['hours'])
    cph = list(sets_info['cph'])
    for objective in m.component_objects(pyo.Objective, active=True):
        objective.deactivate()
    for cname in ('c_v_margin_low', 'c_v_margin_high',
                  'c_tap_up', 'c_tap_dn', 'c_tap_abs_up', 'c_tap_abs_dn',
                  'c_tap_event_link', 'c_tap_max', 'c_tap_delay',
                  'c_tap_spacing', 'c_tap_terminal',
                  'c_tap_terminal_abs_up', 'c_tap_terminal_abs_dn'):
        component = getattr(m, cname, None)
        if component is not None:
            component.deactivate()
    for ph in sets_info['svr_phs']:
        for t in hours:
            m.tap_pos[ph,t].fix(int(round(pyo.value(opf_model.tap_pos[ph,t]))))
            m.u_tap[ph,t].fix(0.0)
    qstar_tap_schedule = {
        (ph, t): int(round(pyo.value(opf_model.tap_pos[ph,t])))
        for ph in sets_info['svr_phs'] for t in hours}
    activate_exact_fixed_tap_svr(m, data, hours, qstar_tap_schedule)
    for b, ph in sets_info['pvph']:
        for t in hours:
            m.Pcurt[b,ph,t].fix(pyo.value(opf_model.Pcurt[b,ph,t]))
            m.Qpv[b,ph,t].fix(pyo.value(opf_model.Qpv[b,ph,t]))
    for b, ph in sets_info['bph']:
        if b == data['slack']:
            continue
        for t in hours:
            m.V[b,ph,t].unfix()
            m.V[b,ph,t].setlb(VALIDATION_V_MIN)
            m.V[b,ph,t].setub(VALIDATION_V_MAX)
    m.obj_qstar_interpretation_physical = pyo.Objective(
        expr=sum(m.l_sq[i,j,ph,t] for i,j,ph in cph for t in hours))
    opt = pyo.SolverFactory('cplex', executable=solver_executable)
    opt.options['qpmethod'] = 6
    opt.options['timelimit'] = 1200
    opt.options['threads'] = 4
    opt.options['barrier_convergetol'] = 1e-6
    (_, termination, raw_termination, violation, sca_ok,
     iterations, max_dv) = _solve_sca_voltage_consistent(
        opt, m, cph, hours, label='replay-Qstar-interpretacao',
        tee_first=False)
    exactness = (summarize_bfm_exactness(m, data, sets_info)
                 if termination in _USABLE_TERMINATIONS and sca_ok else None)
    valid = bool(exactness and exactness.get('exactness_ok', False))
    svr_exact_residual_pu = (
        max_exact_svr_residual_pu(m, data, hours, qstar_tap_schedule)
        if termination in _USABLE_TERMINATIONS else float('inf'))
    valid = bool(valid and svr_exact_residual_pu <= SVR_EXACT_RESIDUAL_TOL)
    return {
        'model': m, 'valid': valid, 'termination': termination,
        'raw_termination': raw_termination, 'solver_violation': violation,
        'sca_converged': sca_ok, 'iterations': iterations,
        'max_sca_dv_pu': max_dv, 'exactness': exactness,
        'svr_equation_mode': 'exact_fixed_tap',
        'max_svr_exact_residual_pu': svr_exact_residual_pu,
    }


def _tap_schedule_shift_events(base_schedule, data, hours, phases,
                               shift_periods):
    """Desloca eventos para mais tarde preservando ordem, degrau e quantidade.

    É uma heurística operacional auditável: tenta reduzir o tempo em tensão
    elevada sem criar operações adicionais. A auditoria posterior continua
    responsável por rejeitar delay, limites ou saltos inválidos.
    """
    target_phases = set(phases)
    out = dict(base_schedule)
    ordered_hours = list(hours)
    for ph, sd in data['svr'].items():
        if ph not in target_phases:
            continue
        init = int(round((sd['tap_init']-sd['tap_min'])/sd['tap_step']))
        previous = init
        events = []
        for idx, t in enumerate(ordered_hours):
            current = int(round(base_schedule[(ph,t)]))
            delta = current-previous
            if delta:
                # O cronograma base já possui degrau unitário; ainda assim,
                # preservamos explicitamente cada passo para auditoria robusta.
                direction = 1 if delta > 0 else -1
                for _ in range(abs(delta)):
                    events.append((idx, direction))
            previous = current
        shifted = [(min(idx+shift_periods, len(ordered_hours)-1), direction)
                   for idx, direction in events]
        by_idx = {}
        for idx, direction in shifted:
            by_idx.setdefault(idx, []).append(direction)
        position = init
        for idx, t in enumerate(ordered_hours):
            for direction in by_idx.get(idx, ()):
                position += direction
            out[(ph,t)] = max(0, min(sd['n_tap']-1, position))
    return out


def _tap_schedule_toward_initial(base_schedule, data, hours, reduction,
                                 phases=None):
    """Remove afastamento apenas nas fases indicadas, sem criar saltos."""
    out = {}
    target_phases = set(data['svr'] if phases is None else phases)
    for ph, sd in data['svr'].items():
        init = int(round((sd['tap_init']-sd['tap_min'])/sd['tap_step']))
        for t in hours:
            pos = int(round(base_schedule[(ph,t)]))
            if ph not in target_phases:
                out[(ph,t)] = pos
                continue
            delta = pos-init
            if delta > 0:
                pos = init + max(delta-reduction, 0)
            elif delta < 0:
                pos = init - max(-delta-reduction, 0)
            out[(ph,t)] = max(0, min(sd['n_tap']-1, pos))
    return out


def _tap_schedule_window_reduction(base_schedule, data, hours, phases,
                                   start_idx, end_idx, reduction=1):
    """Reduz taps apenas em [start_idx,end_idx), restaurando-os ao final."""
    target_phases = set(phases)
    out = dict(base_schedule)
    for ph, sd in data['svr'].items():
        if ph not in target_phases:
            continue
        init = int(round((sd['tap_init']-sd['tap_min'])/sd['tap_step']))
        for idx, t in enumerate(hours):
            if not (start_idx <= idx < end_idx):
                continue
            pos = int(round(base_schedule[(ph,t)]))
            delta = pos-init
            if delta > 0:
                pos = init + max(delta-reduction, 0)
            elif delta < 0:
                pos = init - max(-delta-reduction, 0)
            out[(ph,t)] = max(0, min(sd['n_tap']-1, pos))
    return out


def _generate_ldc_near_schedules(base_schedule, data, hours,
                                 max_schedules=LDC_NEAR_MAX_SCHEDULES):
    """Gera vizinhos mecânicos suprimindo exatamente um evento do LDC.

    A lista de comandos restantes é reaplicada desde a posição inicial. Isso
    preserva degrau unitário e espaçamento; a auditoria elimina qualquer caso
    que alcance limite de tap. A seleção prioriza remover comandos ``raise`` e
    reduzir a exposição acumulada a tap², proxy coerente com perdas G·V².
    """
    ordered_hours = list(hours)
    base_ops = sum(_audit_programmed_tap_schedule(
        base_schedule, data, ordered_hours)['ops'].values())

    def _tap_v2_exposure(schedule):
        return sum(
            (data['svr'][ph]['tap_min'] +
             int(round(schedule[(ph,t)]))*data['svr'][ph]['tap_step'])**2
            for ph in data['svr'] for t in ordered_hours)

    base_exposure = _tap_v2_exposure(base_schedule)
    generated = []
    signatures = set()
    for ph, sd in sorted(data['svr'].items()):
        init = int(round((sd['tap_init']-sd['tap_min'])/sd['tap_step']))
        previous = init
        events = []
        for idx, t in enumerate(ordered_hours):
            current = int(round(base_schedule[(ph,t)]))
            delta = current-previous
            if delta:
                direction = 1 if delta > 0 else -1
                for _ in range(abs(delta)):
                    events.append((idx, direction))
            previous = current
        for event_ordinal, (removed_idx, removed_direction) in enumerate(events):
            remaining = [event for k, event in enumerate(events)
                         if k != event_ordinal]
            by_idx = {}
            for idx, direction in remaining:
                by_idx.setdefault(idx, []).append(direction)
            schedule = dict(base_schedule)
            position = init
            for idx, t in enumerate(ordered_hours):
                for direction in by_idx.get(idx, ()):
                    position += direction
                schedule[(ph,t)] = position
            audit = _audit_programmed_tap_schedule(schedule, data, ordered_hours)
            total_ops = sum(audit['ops'].values())
            signature = tuple(schedule[k] for k in sorted(schedule))
            if (not audit['ok'] or total_ops != base_ops-1 or
                    signature in signatures):
                continue
            signatures.add(signature)
            distance = sum(abs(schedule[k]-base_schedule[k]) for k in schedule)
            exposure = _tap_v2_exposure(schedule)
            generated.append({
                'name': (f'ldc_menos_evento_ph{ph}_n{event_ordinal+1}_'
                         f't{ordered_hours[removed_idx]}'),
                'schedule': schedule, 'tap_audit': audit,
                'phase': ph, 'event_ordinal': event_ordinal+1,
                'event_period': ordered_hours[removed_idx],
                'event_direction': removed_direction,
                'distance_to_ldc': distance,
                'tap_v2_exposure': exposure,
                'tap_v2_exposure_reduction': base_exposure-exposure,
            })

    generated.sort(key=lambda row: (
        -row['tap_v2_exposure_reduction'], row['distance_to_ldc'],
        row['event_period'], row['phase']))
    selected = []
    beneficial = [row for row in generated
                  if row['event_direction'] > 0 and
                  row['tap_v2_exposure_reduction'] > 0.0]
    # Por fase, preserva uma intervenção suave (menor redução positiva) e
    # uma intensa (maior redução). Isso evita testar apenas remoções precoces
    # que derrubariam demais a tensão, sem voltar a incluir comandos lower.
    gentle_rows = []
    intense_rows = []
    for ph in sorted(data['svr']):
        phase_rows = [item for item in beneficial if item['phase'] == ph]
        if phase_rows:
            gentle = min(
                phase_rows, key=lambda item: item['tap_v2_exposure_reduction'])
            intense = max(
                phase_rows, key=lambda item: item['tap_v2_exposure_reduction'])
            gentle_rows.append(gentle)
            intense_rows.append(intense)
    for row in gentle_rows + intense_rows:
        if row not in selected:
            selected.append(row)
    for row in beneficial + generated:
        if row not in selected:
            selected.append(row)
        if len(selected) >= max_schedules:
            break
    return selected[:max_schedules]


def _screen_vv_profiles(profile_candidates, reference,
                        max_profiles=VV_SCREEN_MAX_PROFILES):
    """Preserva curvas diversas com bom desvio, perdas e escore primário."""
    valid = [c for c in profile_candidates
             if c.get('admissible', False) and
             math.isfinite(c.get('total_loss_kwh', float('inf')))]
    if not valid:
        return ('ieee_catb_default',)
    ref_dv = max(reference['voltage_deviation_pu_sum'], 1.0e-9)
    ref_unb = max(reference['unbalance_pu_sum'], 1.0e-9)
    ref_core = max(reference['core_loss_kwh'], 1.0e-9)
    ref_total = max(reference['total_loss_kwh'], 1.0e-9)
    for c in valid:
        c['vv_screen_score'] = (
            0.45*c['voltage_deviation_pu_sum']/ref_dv +
            0.15*c['unbalance_pu_sum']/ref_unb +
            0.15*c['core_loss_kwh']/ref_core +
            0.25*c['total_loss_kwh']/ref_total)
    selected = []
    orderings = (
        lambda c: c['voltage_deviation_pu_sum'],
        lambda c: c['total_loss_kwh'],
        lambda c: c['core_loss_kwh'],
        lambda c: c['vv_screen_score'],
    )
    for key in orderings:
        row = min(valid, key=key)
        name = row['vv_profile_name']
        if name not in selected:
            selected.append(name)
        if len(selected) >= max_profiles:
            break
    for row in sorted(valid, key=lambda c: c['vv_screen_score']):
        if row['vv_profile_name'] not in selected:
            selected.append(row['vv_profile_name'])
        if len(selected) >= max_profiles:
            break
    return tuple(selected)


def _surrogate_screen_vv_profiles(profiles, inferred_B, data, sets_info,
                                  opf_model, base_model,
                                  max_profiles=VV_SCREEN_MAX_PROFILES):
    """Pré-seleciona curvas sem criar clones nem chamar o solver.

    A triagem compara (i) aproximação ao Q* do OPF, (ii) um indicador local
    de correção de tensão e (iii) uso RMS da capacidade reativa. O indicador
    de tensão é apenas uma sensibilidade de ordenação; a admissibilidade é
    decidida depois pelo replay BFM físico completo.
    """
    hours = list(sets_info['hours'])
    members = sorted({(row['bus'], row['ph']) for row in inferred_B})
    if not members:
        return ('ieee_catb_default',), []

    rows = []
    for profile in profiles:
        curves = _curves_for_static_profile(inferred_B, data, profile)
        curve_of = {(row['bus'], row['ph']): row for row in curves}
        q_cmd = {}
        for b, ph in members:
            snom_pu = data['pv'][(b,ph)]['S_nom_pu']
            for t in hours:
                p_pu = max(pyo.value(base_model.Pavail[b,ph,t])-
                           pyo.value(base_model.Pcurt[b,ph,t]), 0.0)
                qcap_pu = _operational_qcap_pu(p_pu, snom_pu)
                solar_ok = (ENABLE_NIGHT_VAR or
                            PV_EFF_FACTOR[t] >= QPV_MIN_SOLAR_FRAC)
                raw_q = (_vv_setting_eval_pu(
                    try_v(base_model,b,ph,t), curve_of[(b,ph)])
                    if solar_ok else 0.0)
                q_cmd[(b,ph,t)] = max(-qcap_pu, min(qcap_pu, raw_q))

        # Replica o acoplamento físico e a rampa usados no replay caro.
        if ENABLE_3PH_INVERTER_COUPLING:
            phases_by_bus = {}
            for b, ph in members:
                if data['pv'][(b,ph)].get('is_3ph', False):
                    phases_by_bus.setdefault(b, []).append(ph)
            for b, phases in phases_by_bus.items():
                for t in hours:
                    common = sum(q_cmd[(b,ph,t)] for ph in phases)/len(phases)
                    for ph in phases:
                        q_cmd[(b,ph,t)] = common
        q_cmd = _limit_q_command_slew(q_cmd, data, members, hours)

        q_error_sq = 0.0
        q_ref_sq = 0.0
        q_fraction_sq = 0.0
        voltage_error = 0.0
        samples = 0
        for b, ph in members:
            snom_pu = max(data['pv'][(b,ph)]['S_nom_pu'], 1.0e-12)
            for t in hours:
                q_value = q_cmd[(b,ph,t)]
                q_star = pyo.value(opf_model.Qpv[b,ph,t])
                q_error_sq += (q_value-q_star)**2
                q_ref_sq += q_star*q_star
                q_fraction = q_value/snom_pu
                q_fraction_sq += q_fraction*q_fraction
                v_base = try_v(base_model,b,ph,t)
                v_proxy = (v_base +
                           VV_SURROGATE_VOLT_GAIN_PU_PER_QFRAC*q_fraction)
                voltage_error += abs(v_proxy-1.0)
                samples += 1
        q_nrmse = math.sqrt(q_error_sq/max(q_ref_sq, 1.0e-18))
        rows.append({
            'profile_name': profile['name'],
            'voltage_proxy_mae_pu': voltage_error/max(samples, 1),
            'q_tracking_nrmse': q_nrmse,
            'q_rms_frac_s': math.sqrt(q_fraction_sq/max(samples, 1)),
        })

    # Normalização min-max: nenhuma das três grandezas domina por unidade.
    for key in ('voltage_proxy_mae_pu', 'q_tracking_nrmse', 'q_rms_frac_s'):
        values = [row[key] for row in rows]
        lower, upper = min(values), max(values)
        span = max(upper-lower, 1.0e-12)
        for row in rows:
            row[f'{key}_normalized'] = (row[key]-lower)/span
    for row in rows:
        row['surrogate_score'] = (
            VV_SURROGATE_WEIGHT_VOLTAGE*
            row['voltage_proxy_mae_pu_normalized'] +
            VV_SURROGATE_WEIGHT_Q_TRACKING*
            row['q_tracking_nrmse_normalized'] +
            VV_SURROGATE_WEIGHT_Q_USAGE*
            row['q_rms_frac_s_normalized'])

    # A V3.7.5 escolhia a menor utilização de Q como diversidade e, no caso
    # real, promoveu uma curva praticamente inativa. Aqui uma curva só ocupa
    # vaga se atuar ou se ainda acompanhar Q* de forma material.
    for row in rows:
        row['inactive_for_screen'] = bool(
            row['q_rms_frac_s'] < VV_SURROGATE_MIN_ACTIVE_Q_RMS_FRAC_S and
            row['q_tracking_nrmse'] >= VV_SURROGATE_INACTIVE_TRACKING_NRMSE)
        row['surrogate_pareto'] = False
    eligible = [row for row in rows if not row['inactive_for_screen']]
    if not eligible:
        eligible = list(rows)
    for row in eligible:
        dominated = any(
            (other['voltage_proxy_mae_pu'] <= row['voltage_proxy_mae_pu'] and
             other['q_tracking_nrmse'] <= row['q_tracking_nrmse'] and
             other['q_rms_frac_s'] <= row['q_rms_frac_s'] and
             (other['voltage_proxy_mae_pu'] < row['voltage_proxy_mae_pu'] or
              other['q_tracking_nrmse'] < row['q_tracking_nrmse'] or
              other['q_rms_frac_s'] < row['q_rms_frac_s']))
            for other in eligible if other is not row)
        row['surrogate_pareto'] = not dominated
    pareto = [row for row in eligible if row['surrogate_pareto']]
    pool = pareto or eligible
    selected = [row['profile_name'] for row in sorted(
        pool, key=lambda item: (
            item['surrogate_score'], item['q_tracking_nrmse']))[:max_profiles]]
    if len(selected) < max_profiles:
        for row in sorted(eligible, key=lambda item: item['surrogate_score']):
            if row['profile_name'] not in selected:
                selected.append(row['profile_name'])
            if len(selected) >= max_profiles:
                break
    for row in rows:
        row['selected_for_physical_replay'] = row['profile_name'] in selected
    return tuple(selected), rows


def _group_b_seed_rows(groups, inferred_B):
    """Preserva ajustes conhecidos e cria identidades para novos membros B."""
    lookup = {(row['bus'], row['ph']): row for row in inferred_B}
    return [dict(lookup.get(bp, {'bus': bp[0], 'ph': bp[1]}))
            for bp in groups['B']]


def _vv_surrogate_seed_rows(groups, inferred_B):
    """Fornece membros para a triagem de curvas mesmo sem Grupo B inicial.

    As identidades A/C são somente uma população provisória da heurística
    barata. Elas não mudam a classificação nem entram no fluxo até que um
    rollout físico explícito A/C→B seja executado e aprovado.
    """
    if inferred_B:
        return [dict(row) for row in inferred_B], False
    seeds = [
        {'bus': bp[0], 'ph': bp[1]}
        for group_name in ('A', 'C')
        for bp in groups.get(group_name, [])
    ]
    seeds.sort(key=lambda row: (str(row['bus']), row['ph']))
    return seeds, True


def _move_device_a_to_b(groups, members):
    """Move um equipamento físico completo de A ou C para B."""
    member_set = set(members)
    variant = {name: list(values) for name, values in groups.items()}
    variant['A'] = [bp for bp in variant['A'] if bp not in member_set]
    variant['C'] = [bp for bp in variant['C'] if bp not in member_set]
    for bp in members:
        if bp not in variant['B']:
            variant['B'].append(bp)
    for name in variant:
        variant[name] = sorted(variant[name], key=lambda bp: (str(bp[0]), bp[1]))
    return variant


def _rank_a_to_b_rollout_devices(groups, fixed_pf_A, profile, opf_model,
                                  base_model, data, hours,
                                  max_devices=GLOBAL_POLICY_MAX_DEVICE_ROLLOUTS):
    """Escolhe poucos equipamentos A/C para testar migração física a B.

    O ranking é apenas uma triagem: combina a variação de Q que a troca →B
    produziria, a excursão de tensão local, a aproximação a Q* e a sensibilidade
    ``Σ|X|`` do caminho. Distância sem necessidade de Q não cria impacto. A
    recompensa decisória é calculada depois em replay físico global completo.
    """
    devices = {}
    source_group = {}
    for source in ('A', 'C'):
        for bp in groups.get(source, []):
            key = ((bp[0], '3ph') if data['pv'][bp].get('is_3ph', False)
                   else bp)
            devices.setdefault(key, []).append(bp)
            source_group[key] = source
    distance_metrics = data.get('pv_electrical_distance')
    if not distance_metrics:
        distance_metrics = _pv_electrical_distance_metrics(
            data, [bp for values in groups.values() for bp in values])
    ranked = []
    for key, members in devices.items():
        seed_rows = [{'bus': b, 'ph': ph} for b, ph in members]
        curve_of = {
            (row['bus'], row['ph']): row
            for row in _curves_for_static_profile(seed_rows, data, profile)}
        impact = 0.0
        tracking_gain = 0.0
        q_ref_sq = 0.0
        for b, ph in members:
            snom_pu = data['pv'][(b,ph)]['S_nom_pu']
            static_setting = fixed_pf_A.get((b,ph))
            for t in hours:
                p_pu = max(pyo.value(base_model.Pavail[b,ph,t])-
                           pyo.value(base_model.Pcurt[b,ph,t]), 0.0)
                qcap_pu = _operational_qcap_pu(p_pu, snom_pu)
                solar_ok = (ENABLE_NIGHT_VAR or
                            PV_EFF_FACTOR[t] >= QPV_MIN_SOLAR_FRAC)
                q_a_raw = (_static_non_vv_raw_q(
                    static_setting, p_pu, snom_pu)
                    if static_setting is not None else 0.0)
                q_a = (max(-qcap_pu, min(qcap_pu, q_a_raw))
                       if solar_ok else 0.0)
                q_b = (_vv_setting_eval_pu(
                    try_v(base_model,b,ph,t), curve_of[(b,ph)])
                    if solar_ok else 0.0)
                q_b = max(-qcap_pu, min(qcap_pu, q_b))
                q_star = pyo.value(opf_model.Qpv[b,ph,t])
                voltage_weight = 1.0+5.0*abs(try_v(base_model,b,ph,t)-1.0)
                impact += abs(q_b-q_a)*voltage_weight
                tracking_gain += (q_a-q_star)**2-(q_b-q_star)**2
                q_ref_sq += q_star*q_star
        normalized_gain = tracking_gain/max(q_ref_sq, 1.0e-12)
        distance_norm = sum(
            distance_metrics[bp]['electrical_distance_normalized']
            for bp in members)/max(len(members), 1)
        reactive_sensitivity = sum(
            distance_metrics[bp]['reactive_sensitivity_normalized']
            for bp in members)/max(len(members), 1)
        electrical_base_proxy = (
            impact*(1.0+max(-0.5, normalized_gain)))
        distance_multiplier = (
            1.0+GLOBAL_POLICY_DISTANCE_RANK_WEIGHT*reactive_sensitivity)
        ranked.append({
            'device': key, 'members': tuple(sorted(members)),
            'source_group': source_group[key],
            'impact_proxy': impact,
            'q_tracking_gain_proxy': normalized_gain,
            'electrical_distance_normalized': distance_norm,
            'reactive_sensitivity_normalized': reactive_sensitivity,
            'distance_rank_multiplier': distance_multiplier,
            'selection_proxy_without_distance': electrical_base_proxy,
            'selection_proxy': electrical_base_proxy*distance_multiplier,
        })
    ranked.sort(key=lambda row: (
        -row['selection_proxy'], -row['impact_proxy'], str(row['device'])))
    for position, row in enumerate(ranked, 1):
        row['proxy_rank'] = position
        row['selected_for_global_rollout'] = position <= max_devices
    return ranked[:max_devices], ranked


def _global_rollout_plans(ranked_devices, profile_name):
    """Cria rollouts limitados: melhores individuais e conjunto cumulativo."""
    selected = [
        row for row in ranked_devices
        if row.get('selected_for_global_rollout', True)
    ]
    plans = []
    for row in selected[:GLOBAL_POLICY_INDIVIDUALS_PER_PROFILE]:
        plans.append({
            'kind': 'individual',
            'profile_name': profile_name,
            'devices': (row['device'],),
            'members': tuple(row['members']),
            'source_groups': (row['source_group'],),
            'selection_proxy': row['selection_proxy'],
        })

    cumulative = selected[:GLOBAL_POLICY_CUMULATIVE_DEVICE_COUNT]
    if len(cumulative) > 1:
        members = tuple(sorted(
            {bp for row in cumulative for bp in row['members']},
            key=lambda bp: (str(bp[0]), bp[1])))
        plans.append({
            'kind': f'cumulative_top{len(cumulative)}',
            'profile_name': profile_name,
            'devices': tuple(row['device'] for row in cumulative),
            'members': members,
            'source_groups': tuple(row['source_group'] for row in cumulative),
            'selection_proxy': sum(
                row['selection_proxy'] for row in cumulative),
        })
    return plans


def _rollout_plan_label(plan):
    """Rótulo curto, determinístico e seguro para nomes de candidatos/CSV."""
    if plan['kind'] == 'individual':
        raw = str(plan['devices'][0])
    else:
        raw = plan['kind']
    return (raw.replace(' ', '').replace("'", '')
            .replace('(', '').replace(')', '').replace(',', '_'))


def _global_physical_policy_reward(candidate, reference):
    """Recompensa de rollout com as quatro metas físicas e salvaguarda Joule."""
    if (not candidate.get('admissible', False) or
            not math.isfinite(candidate.get('total_loss_kwh', float('inf')))):
        return -float('inf')
    tap_ops = sum(candidate['tap_audit']['ops'].values())
    score = (
        OP_WEIGHT_TAP*tap_ops/max(reference['tap_ops_total'], 1) +
        OP_WEIGHT_VOLTAGE_DEVIATION*
        candidate['voltage_deviation_pu_sum']/max(
            reference['voltage_deviation_pu_sum'], 1.0e-9) +
        OP_WEIGHT_UNBALANCE*candidate['unbalance_pu_sum']/max(
            reference['unbalance_pu_sum'], 1.0e-9) +
        OP_WEIGHT_CORE*candidate['core_loss_kwh']/max(
            reference['core_loss_kwh'], 1.0e-9))
    total_increase = max(
        candidate['total_loss_kwh']/max(reference['total_loss_kwh'], 1.0e-9)-1.0,
        0.0)
    return -(score+GLOBAL_POLICY_TOTAL_LOSS_PENALTY*total_increase)


def _project_continuous_tap_schedule(continuous_schedule, data, hours,
                                     rounding='round', bias=0,
                                     n_tap_max=N_TAP_MAX):
    """Projeta taps contínuos em uma trajetória inteira causal e auditável.

    ``rounding`` controla apenas o alvo inteiro (`floor`, `round` ou `ceil`).
    O tap efetivamente aplicado ainda percorre no máximo um degrau, respeita o
    primeiro delay, o espaçamento conservador de dois períodos e o orçamento
    por fase. ``bias`` cria vizinhos discretos sem alterar esses mecanismos.
    """
    if rounding not in ('floor', 'round', 'ceil'):
        raise ValueError(f'Estratégia de arredondamento inválida: {rounding}')
    ordered_hours = list(hours)
    schedule = {}
    for ph, sd in sorted(data['svr'].items()):
        init = int(round((sd['tap_init']-sd['tap_min'])/sd['tap_step']))
        init = max(0, min(sd['n_tap']-1, init))
        targets = []
        for t in ordered_hours:
            raw = max(0.0, min(float(sd['n_tap']-1),
                               float(continuous_schedule[(ph,t)])))
            if rounding == 'floor':
                target = math.floor(raw)
            elif rounding == 'ceil':
                target = math.ceil(raw)
            else:
                target = int(round(raw))
            targets.append(max(0, min(sd['n_tap']-1, target+bias)))

        delay = sd.get('delay_periods', DELAY_PER)
        current = init
        steps = 0
        last_move_idx = -delay
        for idx, (t, target) in enumerate(zip(ordered_hours, targets)):
            direction = 1 if target > current else -1 if target < current else 0
            if (idx >= delay and direction and idx-last_move_idx >= delay and
                    steps < n_tap_max):
                current = max(0, min(sd['n_tap']-1, current+direction))
                steps += 1
                last_move_idx = idx
            schedule[(ph,t)] = current
    return schedule


def _project_joint_tap_schedule(continuous_schedule, data, hours,
                                total_budget,
                                max_phase_separation=
                                JOINT_TAP_MAX_PHASE_SEPARATION):
    """Projeta conjuntamente as três fases sob um único orçamento de desgaste.

    Em cada instante, no máximo uma operação por fase é permitida. As operações
    elegíveis são ordenadas pela redução do erro absoluto até a trajetória-alvo,
    descontando uma penalidade quando o movimento aumenta a separação entre
    fases. O delay e o limite individual continuam sendo auditados normalmente.
    """
    ordered_hours = list(hours)
    phases = tuple(sorted(data['svr']))
    current = {}
    last_move = {}
    phase_ops = {ph: 0 for ph in phases}
    for ph in phases:
        sd = data['svr'][ph]
        current[ph] = max(
            0, min(sd['n_tap']-1, int(round(
                (sd['tap_init']-sd['tap_min'])/sd['tap_step']))))
        last_move[ph] = -sd.get('delay_periods', DELAY_PER)

    schedule = {}
    total_ops = 0
    for idx, t in enumerate(ordered_hours):
        proposals = []
        for ph in phases:
            sd = data['svr'][ph]
            target = max(
                0.0, min(float(sd['n_tap']-1),
                         float(continuous_schedule[(ph,t)])))
            direction = (1 if target > current[ph]+0.25 else
                         -1 if target < current[ph]-0.25 else 0)
            delay = sd.get('delay_periods', DELAY_PER)
            if (not direction or idx < delay or
                    idx-last_move[ph] < delay or
                    phase_ops[ph] >= N_TAP_MAX):
                continue
            proposed = current[ph]+direction
            before_error = abs(target-current[ph])
            after_error = abs(target-proposed)
            other_positions = [
                current[other] for other in phases if other != ph]
            separation_after = (
                max([proposed]+other_positions) -
                min([proposed]+other_positions))
            # Movimentos que excedem a separação recomendada ainda podem ser
            # necessários, mas perdem prioridade frente a opções coordenadas.
            separation_penalty = max(
                0, separation_after-max_phase_separation)
            benefit = before_error-after_error-2.0*separation_penalty
            proposals.append((benefit, -separation_after, ph, proposed))

        # Seleção global: o orçamento pertence ao regulador, não a cada fase.
        for benefit, _, ph, proposed in sorted(proposals, reverse=True):
            if total_ops >= total_budget or benefit <= 0.0:
                break
            current[ph] = proposed
            last_move[ph] = idx
            phase_ops[ph] += 1
            total_ops += 1
        for ph in phases:
            schedule[(ph,t)] = current[ph]
    return schedule


In [ ]:
#@title Internal model implementation 8/12 { display-mode: "form" }
def _generate_joint_budget_schedules(opf_schedule, ldc_schedule, data, hours):
    """Gera trajetórias coordenadas para a fronteira taps×qualidade.

    Os alvos são interpolações temporais completas entre OPF e LDC. Para cada
    orçamento conservam-se somente trajetórias mecanicamente distintas e mais
    próximas do LDC, evitando multiplicar replays BFM equivalentes.
    """
    candidates = []
    seen = set()
    baseline_ops = sum(
        _audit_programmed_tap_schedule(
            ldc_schedule, data, hours)['ops'].values())
    budgets = sorted({
        int(k) for k in JOINT_TAP_BUDGETS
        if 0 < int(k) < baseline_ops})
    for budget in budgets:
        rows = []
        for alpha in JOINT_TAP_BLEND_FACTORS:
            target = {
                key: ((1.0-alpha)*float(opf_schedule[key]) +
                      alpha*float(ldc_schedule[key]))
                for key in opf_schedule}
            schedule = _project_joint_tap_schedule(
                target, data, hours, budget)
            audit = _audit_programmed_tap_schedule(schedule, data, hours)
            if not audit['ok']:
                continue
            signature = tuple(schedule[k] for k in sorted(schedule))
            if signature in seen:
                continue
            seen.add(signature)
            rows.append({
                'name': f'joint_budget_{budget}_blend_{alpha:.2f}',
                'schedule': schedule,
                'budget': budget,
                'blend': alpha,
                'ops': sum(audit['ops'].values()),
                'distance_to_ldc':
                    _tap_schedule_distance_to_continuous(
                        schedule, ldc_schedule),
            })
        rows.sort(key=lambda row: (
            abs(row['budget']-row['ops']), row['distance_to_ldc']))
        candidates.extend(rows[:JOINT_TAP_MAX_CANDIDATES_PER_BUDGET])
    return candidates


def _tap_schedule_distance_to_continuous(schedule, continuous_schedule):
    """Distância L1 usada apenas para ordenar vizinhos igualmente mecânicos."""
    return sum(abs(float(schedule[k])-float(continuous_schedule[k]))
               for k in schedule)


def _audit_programmed_tap_schedule(
        schedule, data, hours, initial_relay_state=None,
        enforce_daily_budget=True):
    """Confere limites, degrau unitário, espaçamento do delay e total diário."""
    ok = True
    reasons = []
    ops = {}
    relay_state = _normalise_relay_state(data, initial_relay_state)
    for ph, sd in data['svr'].items():
        init = int(relay_state[ph]['tap_position'])
        prev = init
        last_change_idx = -10**9
        ops[ph] = 0
        for idx, t in enumerate(hours):
            pos = int(round(schedule[(ph,t)]))
            if not (0 <= pos < sd['n_tap']):
                ok = False; reasons.append(f'ph{ph}:tap_fora_limite@t{t}')
            step = abs(pos-prev)
            if step > 1:
                ok = False; reasons.append(f'ph{ph}:salto_{step}@t{t}')
            if step:
                if idx-last_change_idx < sd.get('delay_periods', DELAY_PER):
                    ok = False; reasons.append(f'ph{ph}:delay@t{t}')
                last_change_idx = idx
                ops[ph] += step
            prev = pos
        if enforce_daily_budget and ops[ph] > N_TAP_MAX:
            ok = False; reasons.append(f'ph{ph}:ops={ops[ph]}>{N_TAP_MAX}')
    return {'ok': ok, 'ops': ops, 'reasons': reasons}


def _strict_voltage_candidate_ok(quality):
    """Filtro lexicográfico: nenhuma amostra fora da faixa adequada."""
    tol = 2.0e-5
    return (
        quality['mv']['v_min'] >= 0.93-tol and
        quality['mv']['v_max'] <= 1.05+tol and
        quality['lv']['v_min'] >= 0.92-tol and
        quality['lv']['v_max'] <= 1.05+tol and
        quality['fd']['fd95_system'] <= FD_MT_MAX+tol)


def _evaluate_core_tap_candidate(seed_model, data, sets_info, groups,
                                 inferred_B, fixed_pf_A, schedule, opt,
                                 label, vv_profile_name='ieee_catb_default',
                                 max_iterations=SEARCH_CANDIDATE_MAX_ITER,
                                 v_tolerance=SEARCH_CANDIDATE_V_TOL,
                                 q_tolerance=SEARCH_CANDIDATE_Q_TOL):
    """Resolve o ponto fixo Q(V) estático para uma trajetória de tap fornecida."""
    # Força a coleta do clone anterior antes de reservar o próximo modelo.
    gc.collect()
    m = seed_model.clone()
    hours = list(sets_info['hours'])
    pvph = list(sets_info['pvph'])
    cph = list(sets_info['cph'])
    group_of = {bp: g for g, members in groups.items() for bp in members}
    curve_of = {(d['bus'],d['ph']): d for d in inferred_B}
    fixed_of = dict(fixed_pf_A)
    # O seed pode vir da validação híbrida, na qual V do Grupo B estava fixa
    # apenas para auditoria. Cada novo candidato deve voltar a resolver o ponto
    # fixo Q(V), portanto todas as tensões não slack são liberadas aqui.
    for b, ph in sets_info['bph']:
        if b == data['slack']:
            continue
        for t in hours:
            if m.V[b,ph,t].fixed:
                m.V[b,ph,t].unfix()
            m.V[b,ph,t].setlb(VALIDATION_V_MIN)
            m.V[b,ph,t].setub(VALIDATION_V_MAX)
    for (ph,t), pos in schedule.items():
        m.tap_pos[ph,t].fix(int(pos))
    activate_exact_fixed_tap_svr(m, data, hours, schedule)
    # Replays operacionais S2/S3 não herdam curtailment do OPF exploratório.
    # A comparação deve refletir apenas a política reativa e a resposta do LDC.
    for b, ph in pvph:
        for t in hours:
            m.Pcurt[b,ph,t].fix(0.0)

    v_prev = {(b,ph,t): try_v(m,b,ph,t)
              for (b,ph) in sets_info['bph'] for t in hours}
    q_prev = {(b,ph,t): pyo.value(m.Qpv[b,ph,t])
              for (b,ph) in pvph for t in hours}
    phases_by_bus = {}
    for b, ph in pvph:
        if data['pv'][(b,ph)].get('is_3ph', False):
            phases_by_bus.setdefault(b, []).append(ph)

    termination = 'notSolved'
    converged = False
    last_dv = float('inf')
    last_dq = float('inf')
    for iteration in range(1, max_iterations+1):
        q_cmd = {}
        for b, ph in pvph:
            g = group_of.get((b,ph), 'B' if IEEE1547_VV_ALL_PV else 'C')
            snom_pu = data['pv'][(b,ph)]['S_nom_pu']
            for t in hours:
                p_pu = max(pyo.value(m.Pavail[b,ph,t])-
                           pyo.value(m.Pcurt[b,ph,t]), 0.0)
                qcap_pu = _operational_qcap_pu(p_pu, snom_pu)
                solar_ok = (ENABLE_NIGHT_VAR or
                            PV_EFF_FACTOR[t] >= QPV_MIN_SOLAR_FRAC)
                if not solar_ok:
                    raw_q_pu = 0.0
                elif g == 'A':
                    raw_q_pu = _static_non_vv_raw_q(
                        fixed_of[(b,ph)], p_pu, snom_pu)
                elif g == 'B':
                    raw_q_pu = _vv_setting_eval_pu(
                        v_prev[(b,ph,t)], curve_of[(b,ph)])
                else:
                    raw_q_pu = 0.0
                q_cmd[(b,ph,t)] = max(
                    -qcap_pu, min(qcap_pu, raw_q_pu))

        if ENABLE_3PH_INVERTER_COUPLING:
            for b, phases in phases_by_bus.items():
                for t in hours:
                    q_common = sum(q_cmd[(b,ph,t)] for ph in phases)/len(phases)
                    for ph in phases:
                        q_cmd[(b,ph,t)] = q_common

        q_cmd = _limit_q_command_slew(q_cmd, data, pvph, hours)

        q_applied = {
            k: (0.0 if (not ENABLE_NIGHT_VAR and
                        PV_EFF_FACTOR[k[2]] < QPV_MIN_SOLAR_FRAC)
                else SETTINGS_DAMPING*q_cmd[k] +
                     (1.0-SETTINGS_DAMPING)*q_prev[k])
            for k in q_cmd}
        # Mesma projeção final usada na validação principal. Sem ela, o
        # amortecimento podia criar pequenas violações artificiais de rampa.
        q_applied = _limit_q_command_slew(
            q_applied, data, pvph, hours)
        q_applied = _enforce_operational_fp_floor(
            q_applied, m, data, pvph, hours)
        for b, ph in pvph:
            for t in hours:
                m.Qpv[b,ph,t].fix(q_applied[(b,ph,t)])
        for b, ph in sets_info['bph']:
            for t in hours:
                m.V_bar_bus[b,ph,t] = max(v_prev[(b,ph,t)], 0.80)
        for i,j,ph in cph:
            for t in hours:
                m.V_sq_ws[i,j,ph,t] = max(v_prev[(i,ph,t)]**2, 0.64)

        _, termination, _, _ = _solve_with_status_recovery(
            opt, m, tee=False, label=f'{label}-{iteration:02d}')
        if termination not in _USABLE_TERMINATIONS:
            print(f"      [{label}] it={iteration:02d} | "
                  f"solver={termination}", flush=True)
            break
        v_new = {(b,ph,t): try_v(m,b,ph,t)
                 for (b,ph) in sets_info['bph'] for t in hours}
        last_dv = max(abs(v_new[k]-v_prev[k]) for k in v_new)
        last_dq = max(abs(q_applied[k]-q_prev[k]) for k in q_applied)
        print(f"      [{label}] it={iteration:02d} | solver={termination} | "
              f"ΔV={last_dv:.3e} | ΔQ={last_dq:.3e} p.u.",
              flush=True)
        v_prev, q_prev = v_new, q_applied
        if last_dv <= v_tolerance and last_dq <= q_tolerance:
            converged = True
            break

    audit = _audit_programmed_tap_schedule(schedule, data, hours)
    if termination in _USABLE_TERMINATIONS:
        quality = summarize_voltage_quality(m, data, sets_info)
        exactness = summarize_bfm_exactness(m, data, sets_info)
        bound_hits = sum(
            1 for (b,ph) in sets_info['bph'] if b != data['slack'] for t in hours
            if (try_v(m,b,ph,t) <= VALIDATION_V_MIN+1.0e-5 or
                try_v(m,b,ph,t) >= VALIDATION_V_MAX-1.0e-5))
        strict_ok = _strict_voltage_candidate_ok(quality)
        quality_screen = assess_representative_day_quality(
            quality, validation_bound_hits=bound_hits,
            baseline_quality=None, period_count=len(hours))
        daily_screening_ok = quality_screen['daily_screening_ok']
        regulatory_ok = None
        core_kwh = calc_core_loss_energy_kwh(m, data, sets_info)
        joule_kwh = calc_joule_loss_energy_kwh(m, data, sets_info)
        objective_metrics = recalculate_original_objective_pu(
            m, data, sets_info)
        inverter_operability = _summarize_inverter_operability(
            q_prev, data, pvph, hours)
    else:
        quality = None; exactness = None; bound_hits = -1
        svr_exact_residual_pu = float('inf')
        strict_ok = False
        quality_screen = assess_representative_day_quality(
            None, validation_bound_hits=-1,
            baseline_quality=None, period_count=len(hours))
        daily_screening_ok = False
        regulatory_ok = None
        core_kwh = float('inf'); joule_kwh = float('inf')
        objective_metrics = None
        inverter_operability = {
            'parameters_static': True, 'parameter_changes': 0,
            'max_q_step_pu': float('inf'),
            'max_q_step_kvar': float('inf'),
            'max_q_step_frac_s': float('inf'),
            'q_slew_violations': -1, 'ok': False}
    admissible = bool(
        converged and audit['ok'] and exactness is not None and
        exactness['exactness_ok'] and
        svr_exact_residual_pu <= SVR_EXACT_RESIDUAL_TOL and
        daily_screening_ok and inverter_operability['ok'] and
        (strict_ok or not CORE_CANDIDATE_REQUIRE_ALL_ADEQUATE))
    rejection_reasons = []
    if not converged:
        rejection_reasons.append('ponto_fixo_nao_convergiu')
    if not audit['ok']:
        rejection_reasons.extend(audit['reasons'] or ['tap_mecanico_invalido'])
    if exactness is None or not exactness['exactness_ok']:
        rejection_reasons.append('fechamento_bfm')
    if not daily_screening_ok:
        rejection_reasons.append('screening_diario')
    if CORE_CANDIDATE_REQUIRE_ALL_ADEQUATE and not strict_ok:
        rejection_reasons.append('tensao_fora_faixa_adequada')
    if not inverter_operability['ok']:
        rejection_reasons.append('rampa_q')
    return {
        'name': label, 'model': m, 'schedule': dict(schedule),
        'tap_audit': audit, 'converged': converged,
        'iterations': iteration, 'termination': termination,
        'max_dv': last_dv, 'max_dq_pu': last_dq,
        'max_dq_kvar': last_dq*SBASE,  # relatório legado
        'quality': quality, 'exactness': exactness,
        'svr_equation_mode': 'exact_fixed_tap',
        'max_svr_exact_residual_pu': svr_exact_residual_pu,
        'validation_bound_hits': bound_hits,
        'strict_voltage_ok': strict_ok, 'daily_screening_ok': daily_screening_ok,
        'daily_screening_absolute_ok': quality_screen['daily_screening_absolute_ok'],
        'formal_regulatory_assessed': False, 'formal_regulatory_ok': None,
        'regulatory_ok': regulatory_ok,
        'core_loss_kwh': core_kwh, 'joule_loss_kwh': joule_kwh,
        'total_loss_kwh': core_kwh+joule_kwh,
        'objective_metrics_pu': objective_metrics,
        'voltage_deviation_pu_sum': (
            objective_metrics['raw']['voltage_deviation_pu_sum']
            if objective_metrics else float('inf')),
        'unbalance_pu_sum': (
            objective_metrics['raw']['unbalance_pu_sum']
            if objective_metrics else float('inf')),
        'inverter_operability': inverter_operability,
        'vv_profile_name': vv_profile_name,
        'admissible': admissible,
        'rejection_reasons': rejection_reasons,
        'q_settings': q_prev,
    }


def _audit_four_objective_dominance(
        tap_ops, voltage_deviation, unbalance, core_loss, total_loss,
        ref_tap_ops, ref_voltage_deviation, ref_unbalance, ref_core_loss,
        ref_total_loss, operationally_admissible=True,
        relative_score_improvement=None):
    """Aceita redução de taps sob guardas de tensão, perdas e factibilidade."""
    tap_saving = ref_tap_ops-tap_ops
    voltage_saving = ref_voltage_deviation-voltage_deviation
    unbalance_saving = ref_unbalance-unbalance
    core_saving = ref_core_loss-core_loss
    total_increase_frac = (
        (total_loss-ref_total_loss)/max(ref_total_loss, 1.0e-9)
        if math.isfinite(total_loss) else float('inf'))
    voltage_saving_frac = voltage_saving/max(abs(ref_voltage_deviation), 1.0e-9)
    unbalance_saving_frac = unbalance_saving/max(abs(ref_unbalance), 1.0e-9)
    reasons = []
    if not operationally_admissible:
        reasons.append('nao_admissivel_operacionalmente')
    if tap_saving < OP_MIN_TAP_SAVING_STEPS:
        reasons.append('sem_reducao_material_taps')
    voltage_increase_frac = (
        (voltage_deviation-ref_voltage_deviation) /
        max(abs(ref_voltage_deviation), 1.0e-9))
    if voltage_increase_frac > PARETO_MAX_VOLTAGE_DEVIATION_INCREASE_FRAC:
        reasons.append('desvio_tensao_acima_tolerancia')
    if total_increase_frac > PARETO_MAX_TOTAL_LOSS_INCREASE_FRAC:
        reasons.append('piora_perdas_totais')
    return {
        'ok': not reasons, 'reasons': reasons,
        'tap_saving_steps': tap_saving,
        'voltage_deviation_saving_pu': voltage_saving,
        'voltage_deviation_saving_frac': voltage_saving_frac,
        'unbalance_saving_pu': unbalance_saving,
        'unbalance_saving_frac': unbalance_saving_frac,
        'core_saving_kwh': core_saving,
        'total_loss_increase_frac': total_increase_frac,
        'voltage_deviation_increase_frac': voltage_increase_frac,
    }


def _rank_operational_candidates(candidates, dominance_reference=None):
    """Aplica dominância nas quatro metas e prioridade lexicográfica de taps.

    Um candidato não pode comprar a melhora de um termo com a piora de outro.
    Se a referência já tiver desvio ou desequilíbrio numericamente zero, exige-se
    não piora dentro da tolerância absoluta. Sem candidato conjunto, aplica-se
    a regra de não ação.
    """
    reference = candidates[0]  # fallback operacional/tap-base
    external_reference_ok = bool(
        dominance_reference is not None and
        dominance_reference.get('valid', False))
    comparison = dominance_reference if external_reference_ok else reference
    base_joule = comparison['joule_loss_kwh']
    base_core = comparison['core_loss_kwh']
    base_total = comparison['total_loss_kwh']
    base_ops = (comparison['tap_ops_total'] if external_reference_ok else
                sum(reference['tap_audit']['ops'].values()))
    base_dv = comparison['voltage_deviation_pu_sum']
    base_unbalance = comparison['unbalance_pu_sum']
    for c in candidates:
        ops_total = sum(c['tap_audit']['ops'].values())
        c['tap_ops_total'] = ops_total
        c['tap_budget_ok'] = bool(
            not OP_REQUIRE_TAP_OPS_NO_GREATER_THAN_BASE or ops_total <= base_ops)
        c['operationally_admissible'] = bool(
            c['admissible'] and c['tap_budget_ok'] and
            c['inverter_operability']['ok'])
        if math.isfinite(c['total_loss_kwh']):
            c['operational_score'] = (
                OP_WEIGHT_TAP*ops_total/max(base_ops, 1) +
                OP_WEIGHT_VOLTAGE_DEVIATION*
                c['voltage_deviation_pu_sum']/max(base_dv, 1.0e-9) +
                OP_WEIGHT_UNBALANCE*
                c['unbalance_pu_sum']/max(base_unbalance, 1.0e-9) +
                OP_WEIGHT_CORE*c['core_loss_kwh']/max(base_core, 1.0e-9))
        else:
            c['operational_score'] = float('inf')
        c['pareto'] = False
        c['tap_saving_steps'] = base_ops-ops_total
        c['voltage_deviation_saving_pu'] = (
            base_dv-c['voltage_deviation_pu_sum'])
        c['voltage_deviation_saving_frac'] = (
            c['voltage_deviation_saving_pu']/max(abs(base_dv), 1.0e-9))
        c['unbalance_saving_pu'] = base_unbalance-c['unbalance_pu_sum']
        c['unbalance_saving_frac'] = (
            c['unbalance_saving_pu']/max(abs(base_unbalance), 1.0e-9))
        c['core_saving_kwh'] = base_core-c['core_loss_kwh']
        c['total_loss_increase_frac'] = (
            (c['total_loss_kwh']-base_total)/max(base_total, 1.0e-9)
            if math.isfinite(c['total_loss_kwh']) else float('inf'))

    feasible = [c for c in candidates if c['operationally_admissible']]
    for c in feasible:
        dominated = any(
            (d['voltage_deviation_pu_sum'] <=
             c['voltage_deviation_pu_sum']+OP_DOMINANCE_ABS_TOL_PU and
             d['unbalance_pu_sum'] <=
             c['unbalance_pu_sum']+OP_DOMINANCE_ABS_TOL_PU and
             d['core_loss_kwh'] <= c['core_loss_kwh']+1.0e-9 and
             d['tap_ops_total'] <= c['tap_ops_total'] and
             (d['voltage_deviation_pu_sum'] <
              c['voltage_deviation_pu_sum']-OP_DOMINANCE_ABS_TOL_PU or
              d['unbalance_pu_sum'] <
              c['unbalance_pu_sum']-OP_DOMINANCE_ABS_TOL_PU or
              d['core_loss_kwh'] < c['core_loss_kwh']-1.0e-9 or
              d['tap_ops_total'] < c['tap_ops_total']))
            for d in feasible if d is not c)
        c['pareto'] = not dominated
    pareto = [c for c in feasible if c['pareto']]
    # Cada parcela da referência física normalizada vale seu peso; logo o
    # escore de referência é exatamente 1. No fallback tap-base, usa-se o valor
    # calculado da própria primeira linha.
    reference_score = 1.0 if external_reference_ok else reference['operational_score']
    for c in candidates:
        c['relative_score_improvement'] = (
            (reference_score-c['operational_score'])/max(abs(reference_score), 1.0e-9)
            if math.isfinite(c['operational_score']) else -float('inf'))
        audit = _audit_four_objective_dominance(
            c['tap_ops_total'], c['voltage_deviation_pu_sum'],
            c['unbalance_pu_sum'], c['core_loss_kwh'], c['total_loss_kwh'],
            base_ops, base_dv, base_unbalance, base_core, base_total,
            operationally_admissible=c['operationally_admissible'],
            relative_score_improvement=c['relative_score_improvement'])
        c['target_rejection_reasons'] = audit['reasons']
        c['simultaneous_target_ok'] = audit['ok']

    target = [c for c in candidates if c['simultaneous_target_ok']]
    if target:
        selected = min(
            target, key=lambda c: (
                c['tap_ops_total'], c['operational_score'],
                c['voltage_deviation_pu_sum'], c['unbalance_pu_sum'],
                c['core_loss_kwh']))
        selection_status = 'guardas_tensao_perdas_atendidas_prioridade_taps'
        no_action = False
    elif reference['operationally_admissible']:
        selected = reference
        selection_status = (
            'guardas_tensao_perdas_nao_atendidas_recomendacao_bloqueada'
            if external_reference_ok else
            'nao_acao_sem_candidato_simultaneo_material')
        no_action = True
    elif feasible:
        selected = min(feasible, key=lambda c: (
            c['operational_score'], c['total_loss_kwh']))
        selection_status = 'referencia_invalida_fallback_admissivel'
        no_action = False
    else:
        selected = reference
        selection_status = 'nenhum_candidato_admissivel'
        no_action = True
    return feasible, pareto, target, selected, selection_status, no_action


def _append_lightweight_candidate(catalog, candidate, progress_label=None):
    """Guarda apenas resultados escalares e libera o clone Pyomo pesado.

    ``model`` e ``q_settings`` são necessários somente para o candidato final.
    Retê-los para dezenas de alternativas fazia a RAM crescer linearmente.
    """
    entry = {
        key: value for key, value in candidate.items()
        if key not in ('model', 'q_settings')}
    catalog.append(entry)
    if progress_label:
        quality = entry.get('quality')
        exactness = entry.get('exactness') or {}
        bfm_closed = bool(exactness.get('exactness_ok', False))
        print(f"    [busca] concluído {progress_label}: "
              f"solver={entry.get('termination')} | "
              f"BFM_fechado={'SIM' if bfm_closed else 'NÃO'} | "
              f"operacional={'SIM' if entry.get('admissible', False) else 'NÃO'} | "
              f"taps={sum(entry.get('tap_audit',{}).get('ops',{}).values())} | "
              f"desvio={entry.get('voltage_deviation_pu_sum',float('inf')):.4f} | "
              f"VminBT={(quality['lv']['v_min'] if quality else float('nan')):.4f}")
    # Rompe ciclos e referências antes do próximo clone de 225 mil variáveis.
    candidate.pop('model', None)
    candidate.pop('q_settings', None)
    candidate.clear()
    gc.collect()
    return entry


In [ ]:
#@title Internal model implementation 9/12 { display-mode: "form" }
def search_operational_candidate(base_result, opf_model, data, sets_info,
                                 groups, inferred_B, fixed_pf_A,
                                 solver_executable,
                                 dominance_reference=None,
                                 s2_fp096_schedule=None):
    """Busca heurística, estática e auditável de uma solução operacional.

    Na V3.9.3, A/B/C permanecem estáticos no horizonte e a classificação local
    pode receber refinamento A/C→B limitado por rollouts BFM globais. A triagem
    Pareto evita curvas inativas e todo clone é liberado após suas métricas.
    Apenas o vencedor é reexecutado com tolerância rigorosa.
    """
    from pyomo.environ import SolverFactory
    hours = list(sets_info['hours'])
    pvph = list(sets_info['pvph'])
    base_schedule = dict(base_result['tap_schedule'])
    opt = pyo.SolverFactory('cplex', executable=solver_executable)
    opt.options['qpmethod'] = 6
    opt.options['timelimit'] = 1200
    opt.options['threads'] = 4
    opt.options['barrier_convergetol'] = 1e-6

    base_quality = base_result['settings_quality']
    base_audit = _audit_programmed_tap_schedule(base_schedule, data, hours)
    base_core = calc_core_loss_energy_kwh(base_result['model'], data, sets_info)
    base_joule = calc_joule_loss_energy_kwh(base_result['model'], data, sets_info)
    base_objective_metrics = recalculate_original_objective_pu(
        base_result['model'], data, sets_info)
    base_operability = base_result.get(
        'inverter_operability',
        _summarize_inverter_operability(
            base_result['q_settings'], data, pvph, hours))
    base_admissible = bool(
        base_result['converged'] and base_audit['ok'] and
        base_result['settings_bfm_exactness']['exactness_ok'] and
        base_result.get('daily_screening_ok', False) and base_operability['ok'] and
        (_strict_voltage_candidate_ok(base_quality) or
         not CORE_CANDIDATE_REQUIRE_ALL_ADEQUATE))
    base_ops = sum(base_audit['ops'].values())
    dominance_reference_required = dominance_reference is not None
    dominance_reference_ok = bool(
        dominance_reference_required and
        dominance_reference.get('valid', False))
    tap_budget_ops = (
        dominance_reference['tap_ops_total']
        if dominance_reference_ok else base_ops)

    base_runtime_candidate = {
        'name': ('tap_base__mixed_rl_ieee_b' if
                 ENABLE_RL_INVERTER_CLASSIFICATION else
                 'tap_base__ieee_catb_default'), 'reduction': 0,
        'model': base_result['model'], 'strategy': 'base',
        'window_start': None, 'window_end': None,
        'phases': tuple(data['svr']), 'schedule': base_schedule,
        'tap_audit': base_audit, 'converged': base_result['converged'],
        'iterations': base_result['iterations'],
        'termination': base_result['termination'],
        'max_dv': base_result['max_dv'],
        'max_dq_pu': base_result['max_dq_pu'],
        'max_dq_kvar': base_result['max_dq_kvar'],
        'quality': base_quality,
        'exactness': base_result['settings_bfm_exactness'],
        'validation_bound_hits': base_result['validation_bound_hits'],
        'strict_voltage_ok': _strict_voltage_candidate_ok(base_quality),
        'daily_screening_ok': base_result.get('daily_screening_ok', False),
        'formal_regulatory_assessed': False,
        'formal_regulatory_ok': None,
        'regulatory_ok': None,
        'core_loss_kwh': base_core, 'joule_loss_kwh': base_joule,
        'total_loss_kwh': base_core+base_joule,
        'objective_metrics_pu': base_objective_metrics,
        'voltage_deviation_pu_sum':
            base_objective_metrics['raw']['voltage_deviation_pu_sum'],
        'unbalance_pu_sum':
            base_objective_metrics['raw']['unbalance_pu_sum'],
        'inverter_operability': base_operability,
        'vv_profile_name': 'ieee_catb_default',
        'vv_parameters': {
            'V1': IEEE1547_V1, 'V2': IEEE1547_V2,
            'V3': IEEE1547_V3, 'V4': IEEE1547_V4,
            'q_frac': IEEE1547_Q_FRAC,
            'individual_parameters': USE_INDIVIDUAL_FITTED_VV_CURVES},
        'admissible': base_admissible,
        'groups_variant': {name: list(values)
                           for name, values in groups.items()},
        'reclassified_devices': (),
        'global_policy_refined': False,
        'rejection_reasons': ([] if base_admissible else
                              ['referencia_nao_admissivel']),
        'q_settings': base_result['q_settings'],
    }
    candidates = []
    # A referência de runtime reutiliza o modelo já existente; o catálogo
    # recebe uma cópia leve sem duplicar nem reter outro objeto Pyomo.
    candidates.append({
        key: value for key, value in base_runtime_candidate.items()
        if key not in ('model', 'q_settings')})

    # Estágio A — perfis estáticos; nenhuma configuração recebe índice t.
    # Além da família declarada, inclui-se a curva agregada que melhor ajustou
    # os pontos Q* dos equipamentos B, quando esse ajuste é numericamente válido.
    operational_profiles = [
        p for p in OPERATIONAL_VV_PROFILES
        if (not USE_INDIVIDUAL_FITTED_VV_CURVES or
            p['name'] == 'opf_individual_fit')]
    aggregate_profile = (
        None if USE_INDIVIDUAL_FITTED_VV_CURVES
        else _aggregate_fitted_vv_profile(inferred_B, data))
    if aggregate_profile is not None:
        operational_profiles.append(aggregate_profile)
    # Mantém aliases legados disponíveis para fallbacks sem recolocá-los na busca.
    profile_by_name = {p['name']: p for p in OPERATIONAL_VV_PROFILES}
    profile_by_name.update({p['name']: p for p in operational_profiles})
    profile_by_name['unity_pf_counterfactual'] = {
        'name': 'unity_pf_counterfactual',
        'V1': IEEE1547_V1, 'V2': IEEE1547_V2,
        'V3': IEEE1547_V3, 'V4': IEEE1547_V4, 'q_frac': 0.0}

    # Fronteira causal conjunta. As três fases são projetadas simultaneamente
    # sob um orçamento TOTAL de operações, evitando a assimetria criada pela
    # combinação independente de trajetórias diárias OPF/LDC por fase.
    frontier_unity_entries = []
    sparse_frontier_entries = []
    if (dominance_reference is not None and
            dominance_reference.get('valid', False) and
            dominance_reference.get('tap_schedule')):
        ldc_schedule = dominance_reference['tap_schedule']
        unity_groups = {'A': [], 'B': [], 'C': list(pvph)}
        phase_list = tuple(sorted(data['svr']))
        print("\n    FRONTEIRA TAP × REATIVO — BUSCA CONJUNTA TRIFÁSICA")
        joint_schedules = _generate_joint_budget_schedules(
            base_schedule, ldc_schedule, data, hours)
        for joint in joint_schedules:
            schedule = joint['schedule']
            audit = _audit_programmed_tap_schedule(schedule, data, hours)
            if not audit['ok']:
                continue
            ops = sum(audit['ops'].values())
            label = 'frontier_unity_' + joint['name']
            cand = _evaluate_core_tap_candidate(
                base_result['model'], data, sets_info, unity_groups, [], {},
                schedule, opt, label, 'unity_pf_counterfactual',
                max_iterations=SETTINGS_MAX_ITER,
                v_tolerance=SETTINGS_V_TOL,
                q_tolerance=SETTINGS_Q_TOL)
            cand.update({
                'name': label, 'reduction': 0,
                'strategy': 'joint_three_phase_budget_frontier_unity',
                'window_start': None, 'window_end': None,
                'phases': phase_list, 'schedule': schedule,
                'joint_tap_budget': joint['budget'],
                'joint_tap_blend': joint['blend'],
                'distance_to_ldc': joint['distance_to_ldc'],
                'vv_parameters': dict(
                    profile_by_name['unity_pf_counterfactual']),
                'groups_variant': unity_groups,
                'reclassified_devices': (),
                'global_policy_refined': False,
            })
            entry = _append_lightweight_candidate(
                candidates, cand, f"{label} ({ops} taps)")
            frontier_unity_entries.append(entry)

        ref_dv = dominance_reference['voltage_deviation_pu_sum']
        ref_loss = dominance_reference['total_loss_kwh']
        # Pré-âncoras podem ainda apresentar uma penalidade moderada de perdas:
        # é justamente nessa etapa que o reativo esparso deve tentar recuperá-la.
        # As guardas finais rigorosas continuam sendo aplicadas pelo ranking.
        unity_preanchors = [
            row for row in frontier_unity_entries
            if (row.get('admissible', False) and
                sum(row['tap_audit']['ops'].values()) in
                SPARSE_PREANCHOR_TAP_OPS and
                row['voltage_deviation_pu_sum'] <=
                (1.0+SPARSE_PREANCHOR_MAX_VOLTAGE_DEVIATION_INCREASE_FRAC)*
                ref_dv and
                row['total_loss_kwh'] <=
                (1.0+SPARSE_PREANCHOR_MAX_LOSS_INCREASE_FRAC)*ref_loss)]
        selected_preanchors = []
        for tap_ops in SPARSE_PREANCHOR_TAP_OPS:
            rows_at_budget = [
                row for row in unity_preanchors
                if sum(row['tap_audit']['ops'].values()) == tap_ops]
            rows_at_budget.sort(key=lambda row: (
                row['total_loss_kwh'],
                row['voltage_deviation_pu_sum'],
                row['unbalance_pu_sum']))
            selected_preanchors.extend(
                rows_at_budget[:SPARSE_PREANCHORS_PER_TAP_BUDGET])

        if selected_preanchors:
            print("    Pré-âncoras FP1 liberadas para recuperação por reativo:")
            for unity_anchor in selected_preanchors:
                print(
                    f"      {unity_anchor['name']} | "
                    f"taps={sum(unity_anchor['tap_audit']['ops'].values())} | "
                    f"desvio={unity_anchor['voltage_deviation_pu_sum']:.4f} | "
                    f"perdas={unity_anchor['total_loss_kwh']:.2f} kWh "
                    f"({100.0*(unity_anchor['total_loss_kwh']/ref_loss-1.0):+.2f}%)")

            # Ranking esparso por Q_rms observado × sensibilidade elétrica.
            device_members = {}
            group_of = {
                bp: g for g, members in groups.items() for bp in members}
            for bp in pvph:
                key = ((bp[0], '3ph')
                       if data['pv'][bp].get('is_3ph', False) else bp)
                device_members.setdefault(key, []).append(bp)
            ranked_devices = []
            for key, members in device_members.items():
                q_rms = math.sqrt(sum(
                    float(base_result['q_settings'][(b,ph,t)])**2
                    for b,ph in members for t in hours) /
                    max(len(members)*len(hours), 1))
                snom = max(sum(
                    data['pv'][bp]['S_nom_pu'] for bp in members), 1.0e-9)
                sensitivity = max(
                    data.get('pv_electrical_distance', {}).get(
                        bp, {}).get('reactive_sensitivity_normalized', 0.0)
                    for bp in members)
                ranked_devices.append(
                    (q_rms/snom*(1.0+sensitivity), key, tuple(members)))
            ranked_devices.sort(reverse=True, key=lambda row: row[0])
            n_devices = len(ranked_devices)
            inferred_lookup = {
                (row['bus'],row['ph']): row for row in inferred_B}
            for unity_anchor in selected_preanchors:
                anchor_schedule = unity_anchor['schedule']
                anchor_ops = sum(
                    unity_anchor['tap_audit']['ops'].values())
                anchor_blend = unity_anchor.get('joint_tap_blend', 0.0)
                for fraction in SPARSE_REACTIVE_FRACTIONS:
                    count = max(1, int(math.ceil(fraction*n_devices)))
                    active_members = {
                        bp for _,_,members in ranked_devices[:count]
                        for bp in members}
                    variant_groups = {'A': [], 'B': [], 'C': []}
                    for bp in pvph:
                        variant_groups[
                            group_of.get(bp, 'C')
                            if bp in active_members else 'C'
                        ].append(bp)
                    variant_curves = [
                        dict(inferred_lookup[bp])
                        for bp in variant_groups['B']
                        if bp in inferred_lookup]
                    label = (
                        f"sparse_reactive_taps{anchor_ops}_"
                        f"blend{anchor_blend:.2f}_{count}of{n_devices}")
                    cand = _evaluate_core_tap_candidate(
                        base_result['model'], data, sets_info, variant_groups,
                        variant_curves, fixed_pf_A, anchor_schedule, opt,
                        label, 'opf_individual_fit',
                        max_iterations=SETTINGS_MAX_ITER,
                        v_tolerance=SETTINGS_V_TOL,
                        q_tolerance=SETTINGS_Q_TOL)
                    dv_gain = (
                        unity_anchor['voltage_deviation_pu_sum'] -
                        cand['voltage_deviation_pu_sum'])
                    loss_gain = (
                        unity_anchor['total_loss_kwh'] -
                        cand['total_loss_kwh'])
                    # Uma melhora material em qualquer uma das duas métricas
                    # justifica manter o rollout. A aceitação final ainda exige
                    # simultaneamente as guardas absolutas contra a baseline.
                    material = bool(
                        dv_gain >= SPARSE_MIN_DV_IMPROVEMENT_FRAC *
                        unity_anchor['voltage_deviation_pu_sum'] or
                        loss_gain >= SPARSE_MIN_LOSS_IMPROVEMENT_FRAC *
                        unity_anchor['total_loss_kwh'])
                    cand.update({
                        'name': label, 'reduction': 0,
                        'strategy':
                            'sparse_reactive_on_fp1_loss_recovery_preanchor',
                        'window_start': None, 'window_end': None,
                        'phases': phase_list, 'schedule': anchor_schedule,
                        'joint_tap_budget':
                            unity_anchor.get('joint_tap_budget', anchor_ops),
                        'joint_tap_blend': anchor_blend,
                        'sparse_preanchor_name': unity_anchor['name'],
                        'sparse_preanchor_total_loss_kwh':
                            unity_anchor['total_loss_kwh'],
                        'vv_parameters': dict(
                            profile_by_name['opf_individual_fit']),
                        'groups_variant': variant_groups,
                        'reclassified_devices':
                            tuple(key for _,key,_ in ranked_devices[count:]),
                        'global_policy_refined': False,
                        'sparse_active_device_count': count,
                        'sparse_total_device_count': n_devices,
                        'sparse_material_gain': material,
                        'sparse_dv_gain_pu': dv_gain,
                        'sparse_total_loss_gain_kwh': loss_gain,
                    })
                    if not material:
                        cand['admissible'] = False
                        cand.setdefault('rejection_reasons', []).append(
                            'ganho_reativo_esparso_imaterial')
                    entry = _append_lightweight_candidate(
                        candidates, cand, label)
                    sparse_frontier_entries.append(entry)
                    final_guards_met = bool(
                        entry.get('admissible', False) and material and
                        entry['voltage_deviation_pu_sum'] <=
                        (1.0+PARETO_MAX_VOLTAGE_DEVIATION_INCREASE_FRAC)*
                        ref_dv and
                        entry['total_loss_kwh'] <=
                        (1.0+PARETO_MAX_TOTAL_LOSS_INCREASE_FRAC)*ref_loss)
                    if final_guards_met:
                        print(
                            f"      [{label}] guardas finais atendidas; "
                            "mantendo o menor conjunto reativo desta âncora.")
                        break
        else:
            print("    Nenhuma pré-âncora FP1 de 22/24/26 taps satisfez "
                  "a guarda de tensão e a tolerância exploratória de perdas.")

    vv_surrogate_seeds, vv_surrogate_uses_provisional_non_b = (
        _vv_surrogate_seed_rows(groups, inferred_B))
    surrogate_profile_names, vv_surrogate_screen = (
        _surrogate_screen_vv_profiles(
            operational_profiles, vv_surrogate_seeds, data, sets_info,
            opf_model, base_result['model']))
    print("\n    [busca] pré-triagem Volt-VAR sem solver:")
    if vv_surrogate_uses_provisional_non_b:
        print("      [i] Grupo B inicial vazio: a triagem usa A/C apenas como "
              "população provisória; nenhuma classe foi alterada.")
    for row in sorted(vv_surrogate_screen,
                      key=lambda item: item['surrogate_score']):
        mark = 'SIM' if row['selected_for_physical_replay'] else 'não'
        status = ('inativa' if row.get('inactive_for_screen', False) else
                  'Pareto' if row.get('surrogate_pareto', False) else
                  'dominada')
        print(f"      {row['profile_name']:<24} | "
              f"score={row['surrogate_score']:.4f} | "
              f"NRMSE(Q*)={row['q_tracking_nrmse']:.4f} | "
              f"Q_rms/S={row['q_rms_frac_s']:.4f} | {status:<8} | "
              f"replay={mark}")
    for profile in operational_profiles:
        if (profile['name'] == 'ieee_catb_default' or
                profile['name'] not in surrogate_profile_names or
                not inferred_B):
            continue
        print(f"    [busca] avaliando curva no tap-base: {profile['name']}")
        curves = _curves_for_static_profile(inferred_B, data, profile)
        cand = _evaluate_core_tap_candidate(
            base_result['model'], data, sets_info, groups, curves,
            fixed_pf_A, base_schedule, opt,
            f"tap_base__{profile['name']}", profile['name'])
        cand.update({
            'reduction': 0, 'strategy': 'static_vv_profile',
            'window_start': None, 'window_end': None,
            'phases': tuple(data['svr']),
            'vv_parameters': dict(profile),
        })
        _append_lightweight_candidate(
            candidates, cand, f"tap-base/{profile['name']}")

    screen_reference = (dominance_reference if dominance_reference_ok else {
        'voltage_deviation_pu_sum': candidates[0]['voltage_deviation_pu_sum'],
        'unbalance_pu_sum': candidates[0]['unbalance_pu_sum'],
        'core_loss_kwh': candidates[0]['core_loss_kwh'],
        'total_loss_kwh': candidates[0]['total_loss_kwh'],
    })
    # Com B existente, o segundo filtro usa os poucos replays físicos no
    # tap-base. Sem B, esses replays seriam idênticos e não informativos; nesse
    # caso as curvas pré-selecionadas seguem diretamente para rollouts A/C→B.
    profile_screen_candidates = list(candidates)
    if inferred_B:
        screened_profile_names = _screen_vv_profiles(
            profile_screen_candidates, screen_reference)
    else:
        screened_profile_names = tuple(
            surrogate_profile_names[:GLOBAL_POLICY_MAX_VV_PROFILES])

    # Estágio B — busca CONJUNTA e limitada. Perfis de suporte são cruzados com
    # redução em cada fase e nas três fases. Isso corrige a V3.3, que escolhia o
    # perfil antes de testar menos taps e podia perder a interação Q(V)×SVR.
    seen = set()
    for reduction in OPERATIONAL_GLOBAL_TAP_REDUCTIONS:
        for phases in OPERATIONAL_TAP_REDUCTION_PHASE_SETS:
            schedule = _tap_schedule_toward_initial(
                base_schedule, data, hours, reduction, phases=phases)
            signature = tuple(schedule[k] for k in sorted(schedule))
            ph_tag = ''.join(str(ph) for ph in phases)
            for profile_name in screened_profile_names:
                signature_profile = (signature, profile_name)
                if signature_profile in seen:
                    continue
                seen.add(signature_profile)
                profile = profile_by_name[profile_name]
                curves = _curves_for_static_profile(
                    inferred_B, data, profile)
                audit = _audit_programmed_tap_schedule(schedule, data, hours)
                ops_total = sum(audit['ops'].values())
                budget_ok = (
                    not OP_REQUIRE_TAP_OPS_NO_GREATER_THAN_BASE or
                    ops_total <= tap_budget_ops)
                name = (f'tap_menos_{reduction}_ph{ph_tag}__{profile_name}')
                if not audit['ok'] or not budget_ok:
                    operability = {
                        'parameters_static': True, 'parameter_changes': 0,
                        'max_q_step_pu': float('nan'),
                        'max_q_step_kvar': float('nan'),
                        'max_q_step_frac_s': float('nan'),
                        'q_slew_violations': -1, 'ok': False}
                    reasons = list(audit['reasons'])
                    if not budget_ok:
                        reasons.append('orcamento_taps')
                    rejected = {
                        'name': name, 'reduction': reduction,
                        'strategy': 'phase_tap_reduction_with_static_vv',
                        'window_start': None, 'window_end': None,
                        'phases': tuple(phases), 'model': base_result['model'],
                        'schedule': schedule, 'tap_audit': audit,
                        'converged': False, 'iterations': 0,
                        'termination': ('tapBudgetRejected' if audit['ok']
                                        else 'mechanicallyRejected'),
                        'max_dv': float('inf'), 'max_dq_pu': float('inf'),
                        'max_dq_kvar': float('inf'),
                        'quality': None, 'exactness': None,
                        'validation_bound_hits': -1,
                        'strict_voltage_ok': False, 'daily_screening_ok': False, 'regulatory_ok': None,
                        'core_loss_kwh': float('inf'),
                        'joule_loss_kwh': float('inf'),
                        'total_loss_kwh': float('inf'),
                        'objective_metrics_pu': None,
                        'voltage_deviation_pu_sum': float('inf'),
                        'unbalance_pu_sum': float('inf'),
                        'inverter_operability': operability,
                        'vv_profile_name': profile_name,
                        'vv_parameters': dict(profile),
                        'admissible': False,
                        'rejection_reasons': reasons,
                        'q_settings': base_result['q_settings'],
                    }
                    _append_lightweight_candidate(candidates, rejected, name)
                    continue
                print(f"    [busca] avaliando redução OPF: {name}")
                cand = _evaluate_core_tap_candidate(
                    base_result['model'], data, sets_info, groups, curves,
                    fixed_pf_A, schedule, opt, name, profile_name)
                cand.update({
                    'reduction': reduction,
                    'strategy': 'phase_tap_reduction_with_static_vv',
                    'window_start': None, 'window_end': None,
                    'phases': tuple(phases),
                    'vv_parameters': dict(profile),
                })
                _append_lightweight_candidate(candidates, cand, name)

    # Estágio C — reduções limitadas às janelas solares documentadas. Essas
    # alternativas podem reduzir V² nos transformadores sem manter o tap baixo
    # durante todo o dia. A mesma auditoria de degrau, delay e orçamento decide
    # se a trajetória é executável antes de chamar o solver.
    for window_name, phases, start_idx, end_idx, reduction in (
            CORE_LOSS_SELECTIVE_WINDOWS if
            (ENABLE_CORE_LOSS_CANDIDATE_SEARCH and
             ENABLE_SOLAR_WINDOW_SEARCH) else ()):
        schedule = _tap_schedule_window_reduction(
            base_schedule, data, hours, phases, start_idx, end_idx, reduction)
        signature = tuple(schedule[k] for k in sorted(schedule))
        for profile_name in screened_profile_names:
            signature_profile = (signature, profile_name)
            if signature_profile in seen:
                continue
            seen.add(signature_profile)
            profile = profile_by_name[profile_name]
            curves = _curves_for_static_profile(inferred_B, data, profile)
            audit = _audit_programmed_tap_schedule(schedule, data, hours)
            ops_total = sum(audit['ops'].values())
            budget_ok = (
                not OP_REQUIRE_TAP_OPS_NO_GREATER_THAN_BASE or
                ops_total <= tap_budget_ops)
            name = f'{window_name}__{profile_name}'
            if not audit['ok'] or not budget_ok:
                reasons = list(audit['reasons'])
                if not budget_ok:
                    reasons.append('orcamento_taps')
                rejected = {
                    'name': name, 'reduction': reduction,
                    'strategy': 'solar_window_tap_reduction_with_static_vv',
                    'window_start': start_idx, 'window_end': end_idx,
                    'phases': tuple(phases), 'model': base_result['model'],
                    'schedule': schedule, 'tap_audit': audit,
                    'converged': False, 'iterations': 0,
                    'termination': ('tapBudgetRejected' if audit['ok']
                                    else 'mechanicallyRejected'),
                    'max_dv': float('inf'), 'max_dq_pu': float('inf'),
                    'max_dq_kvar': float('inf'), 'quality': None,
                    'exactness': None, 'validation_bound_hits': -1,
                    'strict_voltage_ok': False, 'daily_screening_ok': False, 'regulatory_ok': None,
                    'core_loss_kwh': float('inf'),
                    'joule_loss_kwh': float('inf'),
                    'total_loss_kwh': float('inf'),
                    'objective_metrics_pu': None,
                    'voltage_deviation_pu_sum': float('inf'),
                    'unbalance_pu_sum': float('inf'),
                    'inverter_operability': {
                        'parameters_static': True, 'parameter_changes': 0,
                        'max_q_step_pu': float('nan'),
                        'max_q_step_kvar': float('nan'),
                        'max_q_step_frac_s': float('nan'),
                        'q_slew_violations': -1, 'ok': False},
                    'vv_profile_name': profile_name, 'admissible': False,
                    'vv_parameters': dict(profile),
                    'rejection_reasons': reasons,
                    'q_settings': base_result['q_settings'],
                }
                _append_lightweight_candidate(candidates, rejected, name)
                continue
            print(f"    [busca] avaliando janela solar: {name}")
            cand = _evaluate_core_tap_candidate(
                base_result['model'], data, sets_info, groups, curves,
                fixed_pf_A, schedule, opt, name, profile_name)
            cand.update({
                'reduction': reduction,
                'strategy': 'solar_window_tap_reduction_with_static_vv',
                'window_start': start_idx, 'window_end': end_idx,
                'phases': tuple(phases),
                'vv_parameters': dict(profile),
            })
            _append_lightweight_candidate(candidates, cand, name)

    # Estágio D — refinamento global da classificação. O bandit local sugere
    # A/B/C pela aderência a Q*. Aqui poucos equipamentos A/C completos recebem
    # rollouts físicos para B. Cada uma de até duas curvas testa o equipamento
    # mais promissor e o conjunto cumulativo dos quatro melhores. O limite de
    # quatro replays preserva o orçamento de RAM; nenhum Q(t) vira setpoint.
    global_policy_device_screen = []
    global_policy_rollouts = []
    global_policy_profiles_tested = []
    best_global_policy = None
    if (ENABLE_GLOBAL_POLICY_REFINEMENT and
            (groups.get('A') or groups.get('C')) and
            screened_profile_names):
        global_reference = (dominance_reference if dominance_reference_ok else {
            'tap_ops_total': base_ops,
            'voltage_deviation_pu_sum':
                base_runtime_candidate['voltage_deviation_pu_sum'],
            'unbalance_pu_sum': base_runtime_candidate['unbalance_pu_sum'],
            'core_loss_kwh': base_runtime_candidate['core_loss_kwh'],
            'total_loss_kwh': base_runtime_candidate['total_loss_kwh'],
        })
        base_global_reward = _global_physical_policy_reward(
            base_runtime_candidate, global_reference)
        rollout_plans = []
        for refinement_profile_name in tuple(
                screened_profile_names)[:GLOBAL_POLICY_MAX_VV_PROFILES]:
            refinement_profile = profile_by_name[refinement_profile_name]
            rollout_devices, profile_device_screen = (
                _rank_a_to_b_rollout_devices(
                    groups, fixed_pf_A, refinement_profile, opf_model,
                    base_result['model'], data, hours))
            global_policy_profiles_tested.append(refinement_profile_name)
            for row in profile_device_screen:
                enriched = dict(row)
                enriched['vv_profile_name'] = refinement_profile_name
                enriched['tested_individually'] = bool(
                    row['proxy_rank'] <=
                    GLOBAL_POLICY_INDIVIDUALS_PER_PROFILE)
                enriched['included_in_cumulative_rollout'] = bool(
                    1 < len(rollout_devices) and
                    row['proxy_rank'] <=
                    GLOBAL_POLICY_CUMULATIVE_DEVICE_COUNT)
                global_policy_device_screen.append(enriched)
            rollout_plans.extend(_global_rollout_plans(
                rollout_devices, refinement_profile_name))

        print("\n    [política global] rollouts físicos A/C→B:")
        for plan in rollout_plans:
            refinement_profile_name = plan['profile_name']
            refinement_profile = profile_by_name[refinement_profile_name]
            members = plan['members']
            source_group = ''.join(sorted(set(plan['source_groups'])))
            plan_label = _rollout_plan_label(plan)
            variant_groups = _move_device_a_to_b(groups, members)
            variant_seeds = _group_b_seed_rows(variant_groups, inferred_B)
            variant_curves = _curves_for_static_profile(
                variant_seeds, data, refinement_profile)
            name = (f'tap_base__global_{source_group}2B_{plan_label}__'
                    f'{refinement_profile_name}')
            print(f"    [política global] avaliando {name} | "
                  f"proxy={plan['selection_proxy']:.4e}")
            cand = _evaluate_core_tap_candidate(
                base_result['model'], data, sets_info, variant_groups,
                variant_curves, fixed_pf_A, base_schedule, opt,
                name, refinement_profile_name)
            reward = _global_physical_policy_reward(cand, global_reference)
            reward_gain = reward-base_global_reward
            cand.update({
                'reduction': 0,
                'strategy': 'global_physical_policy_refinement_nonB_to_B',
                'window_start': None, 'window_end': None,
                'phases': tuple(sorted({ph for _, ph in members})),
                'vv_parameters': dict(refinement_profile),
                'groups_variant': variant_groups,
                'reclassified_devices': tuple(plan['devices']),
                'reclassified_members': tuple(members),
                'reclassified_source_group': source_group,
                'global_policy_rollout_kind': plan['kind'],
                'global_policy_rollout_label': plan_label,
                'global_policy_refined': True,
                'global_policy_reward': reward,
                'global_policy_reward_gain': reward_gain,
                'global_policy_proxy': plan['selection_proxy'],
            })
            entry = _append_lightweight_candidate(candidates, cand, name)
            global_policy_rollouts.append(entry)
            print(f"    [política global] recompensa={reward:.6f} | "
                  f"ganho={reward_gain:+.6f}")
        eligible_rollouts = [
            row for row in global_policy_rollouts
            if row.get('admissible', False) and
            row.get('global_policy_reward_gain', -float('inf')) >=
            GLOBAL_POLICY_MIN_REWARD_GAIN]
        if eligible_rollouts:
            best_global_policy = max(
                eligible_rollouts,
                key=lambda row: row['global_policy_reward'])
            print("    [política global] melhor refinamento aceito para "
                  f"cruzamento com taps: {best_global_policy['name']}")
        else:
            print("    [política global] nenhuma migração A/C→B melhorou "
                  "materialmente "
                  "a recompensa no tap-base.")

    # Estágio E — intervenção mínima sobre o controle LDC. A execução
    # V3.7.3 mostrou que 20–26 taps sacrificavam o desvio de tensão. Aqui cada
    # candidato retira apenas um dos 32 comandos LDC e cruza o vizinho de 31
    # taps com as curvas vencedoras da triagem.
    ldc_near_schedules = []
    if (ENABLE_LDC_NEAR_TAP_SEARCH and dominance_reference_ok and
            dominance_reference.get('tap_schedule')):
        ldc_near_schedules = _generate_ldc_near_schedules(
            dominance_reference['tap_schedule'], data, hours)
        # Sem B inicial, curvas diferentes não alteram a rede nos vizinhos LDC.
        # Avalia-se um único perfil neutro e reservam-se curvas alternativas
        # para os cruzamentos que realmente reclassificaram equipamentos.
        ldc_base_profile_names = (
            screened_profile_names if inferred_B else
            ('ieee_catb_default',))
        for near in ldc_near_schedules:
            schedule = near['schedule']
            signature = tuple(schedule[k] for k in sorted(schedule))
            for profile_name in ldc_base_profile_names:
                signature_profile = (signature, profile_name)
                if signature_profile in seen:
                    continue
                seen.add(signature_profile)
                profile = profile_by_name[profile_name]
                curves = _curves_for_static_profile(inferred_B, data, profile)
                name = f"{near['name']}__{profile_name}"
                print(f"    [busca] avaliando vizinho LDC: {name}")
                cand = _evaluate_core_tap_candidate(
                    base_result['model'], data, sets_info, groups, curves,
                    fixed_pf_A, schedule, opt, name, profile_name)
                cand.update({
                    'reduction': 1,
                    'strategy': 'ldc_single_event_suppression_with_static_vv',
                    'window_start': near['event_period'],
                    'window_end': None, 'phases': (near['phase'],),
                    'ldc_event_direction': near['event_direction'],
                    'distance_to_ldc': near['distance_to_ldc'],
                    'tap_v2_exposure': near['tap_v2_exposure'],
                    'tap_v2_exposure_reduction':
                        near['tap_v2_exposure_reduction'],
                    'vv_parameters': dict(profile),
                })
                _append_lightweight_candidate(candidates, cand, name)

        # A melhor migração A/C→B é cruzada somente com os primeiros
        # vizinhos direcionais. Isso testa a interação controle×tap sem abrir
        # uma combinação explosiva nem aumentar a memória retida.
        if best_global_policy is not None:
            variant_groups = best_global_policy['groups_variant']
            variant_profile_name = best_global_policy['vv_profile_name']
            variant_profile = profile_by_name[variant_profile_name]
            variant_seeds = _group_b_seed_rows(
                variant_groups, inferred_B)
            variant_curves = _curves_for_static_profile(
                variant_seeds, data, variant_profile)
            source_group = best_global_policy.get(
                'reclassified_source_group', 'A')
            rollout_label = best_global_policy.get(
                'global_policy_rollout_label', 'refinamento')
            for near in ldc_near_schedules[:GLOBAL_POLICY_MAX_TAP_ROLLOUTS]:
                schedule = near['schedule']
                name = (f"{near['name']}__global_{source_group}2B_"
                        f"{rollout_label}__"
                        f"{variant_profile_name}")
                print(f"    [política global] avaliando vizinho: {name}")
                cand = _evaluate_core_tap_candidate(
                    base_result['model'], data, sets_info, variant_groups,
                    variant_curves, fixed_pf_A, schedule, opt,
                    name, variant_profile_name)
                reward = _global_physical_policy_reward(
                    cand, global_reference)
                cand.update({
                    'reduction': 1,
                    'strategy':
                        'global_policy_nonB_to_B_with_directional_ldc_neighbor',
                    'window_start': near['event_period'],
                    'window_end': None, 'phases': (near['phase'],),
                    'ldc_event_direction': near['event_direction'],
                    'distance_to_ldc': near['distance_to_ldc'],
                    'tap_v2_exposure': near['tap_v2_exposure'],
                    'tap_v2_exposure_reduction':
                        near['tap_v2_exposure_reduction'],
                    'vv_parameters': dict(variant_profile),
                    'groups_variant': variant_groups,
                    'reclassified_devices':
                        best_global_policy['reclassified_devices'],
                    'reclassified_members':
                        best_global_policy['reclassified_members'],
                    'reclassified_source_group': source_group,
                    'global_policy_rollout_kind':
                        best_global_policy.get(
                            'global_policy_rollout_kind', 'individual'),
                    'global_policy_rollout_label': rollout_label,
                    'global_policy_refined': True,
                    'global_policy_reward': reward,
                    'global_policy_reward_gain':
                        reward-best_global_policy['global_policy_reward'],
                    'global_policy_proxy':
                        best_global_policy['global_policy_proxy'],
                })
                _append_lightweight_candidate(candidates, cand, name)

    # Orçamento de desgaste, escore normalizado e fronteira de Pareto.
    (feasible, pareto, simultaneous, selected,
     selection_status, no_action) = _rank_operational_candidates(
        candidates, dominance_reference=dominance_reference)

    # O ranking opera exclusivamente sobre o catálogo leve. Recria-se agora
    # um único modelo pesado para o vencedor; nenhum dos demais clones fica
    # referenciado. A referência base reutiliza o modelo que já estava em RAM.
    selected_catalog = selected
    if selected_catalog['name'] == candidates[0]['name']:
        selected_runtime = dict(selected_catalog)
        selected_runtime.update(base_runtime_candidate)
        selected_runtime['name'] = selected_catalog['name']
        # Campos de ranking existem somente no catálogo.
        for key in (
                'operationally_admissible', 'operational_score',
                'tap_ops_total', 'tap_saving_steps',
                'voltage_deviation_saving_pu',
                'voltage_deviation_saving_frac',
                'unbalance_saving_pu', 'unbalance_saving_frac',
                'core_saving_kwh', 'total_loss_increase_frac',
                'relative_score_improvement', 'dominance_ok',
                'dominance_reasons'):
            if key in selected_catalog:
                selected_runtime[key] = selected_catalog[key]
    else:
        print(f"\n    [busca] reexecutando somente o vencedor: "
              f"{selected_catalog['name']}", flush=True)
        selected_profile = profile_by_name[selected_catalog['vv_profile_name']]
        selected_catalog_groups = selected_catalog.get(
            'groups_variant', groups)
        selected_catalog_b = _group_b_seed_rows(
            selected_catalog_groups, inferred_B)
        selected_curves = _curves_for_static_profile(
            selected_catalog_b, data, selected_profile)
        runtime_result = _evaluate_core_tap_candidate(
            base_result['model'], data, sets_info, selected_catalog_groups,
            selected_curves,
            fixed_pf_A, selected_catalog['schedule'], opt,
            f"{selected_catalog['name']}__replay_final",
            selected_catalog['vv_profile_name'],
            max_iterations=SETTINGS_MAX_ITER,
            v_tolerance=SETTINGS_V_TOL,
            q_tolerance=SETTINGS_Q_TOL)
        # Métricas físicas da reexecução prevalecem; estratégia e ranking
        # permanecem associados à entrada que venceu o catálogo.
        selected_runtime = dict(selected_catalog)
        selected_runtime.update(runtime_result)
        selected_runtime['name'] = selected_catalog['name']
        selected_runtime['operationally_admissible'] = bool(
            selected_catalog.get('operationally_admissible', False) and
            selected_runtime.get('admissible', False))
        runtime_result.clear()

    selected = selected_runtime
    if not selected.get('admissible', False):
        # Uma diferença numérica na reexecução nunca pode virar setting.
        print("    [busca] vencedor não reproduzido; aplicando não-ação.",
              flush=True)
        fallback_catalog = candidates[0]
        selected = dict(fallback_catalog)
        selected.update(base_runtime_candidate)
        selected['name'] = fallback_catalog['name']
        for key, value in fallback_catalog.items():
            if key not in ('quality', 'exactness', 'objective_metrics_pu'):
                selected[key] = value
        simultaneous = []
        no_action = True
        selection_status = (
            'falha_replay_vencedor_recomendacao_bloqueada_nao_acao')

    selected_groups = selected.get('groups_variant', groups)
    selected_inferred_B = _curves_for_static_profile(
        _group_b_seed_rows(selected_groups, inferred_B), data,
        profile_by_name[selected['vv_profile_name']])

    out = dict(base_result)
    out.update({
        'model': selected['model'], 'converged': selected['converged'],
        'iterations': selected['iterations'],
        'termination': selected['termination'], 'max_dv': selected['max_dv'],
        'max_dq_pu': selected['max_dq_pu'],
        'max_dq_kvar': selected['max_dq_kvar'],
        'settings_quality': selected['quality'],
        'settings_bfm_exactness': selected['exactness'],
        'q_settings': selected['q_settings'],
        'tap_schedule': selected['schedule'],
        'tap_ops': selected['tap_audit']['ops'],
        'validation_bound_hits': selected['validation_bound_hits'],
        'daily_screening_ok': selected.get('daily_screening_ok', False),
        'formal_regulatory_assessed': False,
        'formal_regulatory_ok': None,
        'regulatory_ok': None,
        'physical_model_ok': bool(selected['exactness'] and
                                  selected['exactness']['exactness_ok']),
        'operationally_applicable': bool(
            selected['operationally_admissible'] and PV_TEMP_DATA_OK and
            (not dominance_reference_required or
             (dominance_reference_ok and bool(simultaneous and not no_action)))),
        'tap_schedule_matches_opf': selected['schedule'] == base_schedule,
        'tap_mode': 'operational_candidate_schedule',
        'conditional_optimality_certified': False,
        'global_optimality_certified': False,
        'operational_candidates': candidates,
        'core_loss_candidates': candidates,  # compatibilidade CSV/relatórios
        'selected_candidate': selected['name'],
        'selected_tap_reduction': selected['reduction'],
        'selected_candidate_strategy': selected['strategy'],
        'selected_window_start': selected['window_start'],
        'selected_window_end': selected['window_end'],
        'selected_phases': selected['phases'],
        'selected_vv_profile': selected['vv_profile_name'],
        'selected_vv_parameters': dict(profile_by_name[selected['vv_profile_name']]),
        'vv_profile_name': selected['vv_profile_name'],
        'selected_groups': selected_groups,
        'initial_groups': {name: list(values)
                           for name, values in groups.items()},
        'selected_inferred_B': selected_inferred_B,
        'selected_reclassified_devices':
            selected.get('reclassified_devices', ()),
        'selected_global_policy_refined':
            selected.get('global_policy_refined', False),
        'inverter_operability': selected['inverter_operability'],
        'core_loss_kwh': selected['core_loss_kwh'],
        'joule_loss_kwh': selected['joule_loss_kwh'],
        'total_loss_kwh': selected['total_loss_kwh'],
        'operational_score': selected['operational_score'],
        'tap_budget_total': tap_budget_ops,
        'dominance_reference_name': (
            dominance_reference['name'] if dominance_reference_ok
            else (dominance_reference.get('name', 'baseline_fisica_invalida')
                  if dominance_reference_required else candidates[0]['name'])),
        'dominance_reference_required': dominance_reference_required,
        'dominance_reference_valid': dominance_reference_ok,
        'dominance_reference_metrics': (
            {k: dominance_reference[k] for k in (
                'tap_ops_total', 'voltage_deviation_pu_sum',
                'unbalance_pu_sum', 'core_loss_kwh', 'joule_loss_kwh',
                'total_loss_kwh')}
            if dominance_reference_ok else {}),
        'pareto_candidate_count': len(pareto),
        'simultaneous_candidate_count': len(simultaneous),
        'simultaneous_target_achieved': bool(simultaneous and not no_action),
        'selection_status': selection_status,
        'no_action_applied': no_action,
        'selected_tap_saving_steps': selected['tap_saving_steps'],
        'selected_voltage_deviation_pu_sum':
            selected['voltage_deviation_pu_sum'],
        'selected_voltage_deviation_saving_pu':
            selected['voltage_deviation_saving_pu'],
        'selected_voltage_deviation_saving_frac':
            selected['voltage_deviation_saving_frac'],
        'selected_unbalance_pu_sum': selected['unbalance_pu_sum'],
        'selected_unbalance_saving_pu': selected['unbalance_saving_pu'],
        'selected_unbalance_saving_frac': selected['unbalance_saving_frac'],
        'selected_core_saving_kwh': selected['core_saving_kwh'],
        'selected_total_loss_increase_frac': selected['total_loss_increase_frac'],
        'selected_relative_score_improvement': selected['relative_score_improvement'],
        'core_loss_search_feasible_count': len(feasible),
        'vv_screened_profiles': screened_profile_names,
        'vv_surrogate_screen': vv_surrogate_screen,
        'vv_surrogate_selected_profiles': surrogate_profile_names,
        'vv_surrogate_profile_count': len(operational_profiles),
        'vv_surrogate_seed_count': len(vv_surrogate_seeds),
        'vv_surrogate_uses_provisional_non_b':
            vv_surrogate_uses_provisional_non_b,
        'vv_profile_screen_count': len(profile_screen_candidates),
        'vv_physical_profile_replay_count': max(
            len(profile_screen_candidates)-1, 0),
        'candidate_catalog_is_lightweight': all(
            'model' not in row and 'q_settings' not in row
            for row in candidates),
        'solar_window_search_enabled': ENABLE_SOLAR_WINDOW_SEARCH,
        'ldc_near_schedule_count': len(ldc_near_schedules),
        'global_policy_device_screen': global_policy_device_screen,
        'global_policy_profiles_tested':
            tuple(global_policy_profiles_tested),
        'global_policy_rollout_count': len(global_policy_rollouts),
        'global_policy_best_candidate': (
            best_global_policy['name'] if best_global_policy else None),
    })
    q_opf = {(b,ph,t): pyo.value(opf_model.Qpv[b,ph,t])
             for (b,ph) in pvph for t in hours}
    q_errors_pu = [selected['q_settings'][k]-q_opf[k] for k in q_opf]
    q_ref_pu = math.sqrt(sum(q_opf[k]**2 for k in q_opf)/max(len(q_opf),1))
    out['q_rmse_pu'] = math.sqrt(
        sum(e*e for e in q_errors_pu)/max(len(q_errors_pu),1))
    out['q_rmse_kvar'] = out['q_rmse_pu']*SBASE  # relatório legado
    out['q_nrmse_pct'] = 100.0*out['q_rmse_pu']/max(q_ref_pu,1.0e-9)
    _pf_samples = []
    for b, ph in pvph:
        for t in hours:
            _p = max(
                pyo.value(selected['model'].Pavail[b,ph,t]) -
                pyo.value(selected['model'].Pcurt[b,ph,t]), 0.0)
            _q = abs(float(selected['q_settings'][(b,ph,t)]))
            if _p > 1.0e-9 or _q > 1.0e-9:
                _pf_samples.append(
                    _p/math.sqrt(max(_p*_p+_q*_q, 1.0e-18)))
    out['minimum_operating_power_factor'] = min(_pf_samples, default=1.0)
    out['power_factor_below_085_count'] = sum(
        pf < FP_MIN_VV-1.0e-8 for pf in _pf_samples)
    print("    Auditoria FP das políticas: "
          f"mínimo={out['minimum_operating_power_factor']:.4f} | "
          f"amostras abaixo de 0,85={out['power_factor_below_085_count']}")
    out['distance_reactive_support_audit'] = (
        _distance_reactive_support_audit(
            selected['q_settings'], data, pvph, hours, selected_groups))


    # ======================================================================
    # CENÁRIOS PRINCIPAIS DO PAPER — sequência metodológica explícita
    # S1: LDC local + FP1, sem rotina OPF.
    # S2: LDC local + FP uniforme 0,96, sem rotina OPF.
    # S3: operação coordenada taps + política A/B/C derivada de Q*.
    # ======================================================================
    unity_groups = {'A': [], 'B': [], 'C': list(pvph)}
    unity_on_selected_taps = _evaluate_core_tap_candidate(
        base_result['model'], data, sets_info, unity_groups, [], {},
        selected['schedule'], opt, 'counterfactual-selected-taps-unity-pf',
        'unity_pf_counterfactual', max_iterations=SETTINGS_MAX_ITER,
        v_tolerance=SETTINGS_V_TOL, q_tolerance=SETTINGS_Q_TOL)

    k_fp096 = math.sqrt(max(1.0 - 0.96**2, 0.0)) / 0.96
    uniform_fp096_groups = {'A': list(pvph), 'B': [], 'C': []}
    fixed_pf_096 = {
        bp: {
            'mode': 'fixed_pf', 'k_qp': k_fp096, 'fp': 0.96,
            'q_const_frac_s': 0.0, 'direction': 'capacitive',
            'nrmse': 0.0, 'candidate_count': 1,
            'source': 'uniform_fp096_ldc_s2',
        }
        for bp in pvph
    }
    _s2_schedule = (
        dict(s2_fp096_schedule)
        if s2_fp096_schedule is not None else dict(selected['schedule']))
    uniform_fp096_ldc = _evaluate_core_tap_candidate(
        base_result['model'], data, sets_info,
        uniform_fp096_groups, [], fixed_pf_096,
        _s2_schedule, opt,
        'S2-physical-replay-LDC-uniform-fp096',
        'uniform_fp096_ldc',
        max_iterations=SETTINGS_MAX_ITER,
        v_tolerance=SETTINGS_V_TOL,
        q_tolerance=SETTINGS_Q_TOL)

    # Diagnósticos de atribuição sobre os mesmos taps de S3.
    uniform_fp096_on_s3_taps = _evaluate_core_tap_candidate(
        base_result['model'], data, sets_info,
        uniform_fp096_groups, [], fixed_pf_096,
        selected['schedule'], opt,
        'diagnostic-S3-taps-uniform-fp096',
        'uniform_fp096_counterfactual',
        max_iterations=SETTINGS_MAX_ITER,
        v_tolerance=SETTINGS_V_TOL,
        q_tolerance=SETTINGS_Q_TOL)

    a_only_groups = {
        'A': list(selected_groups.get('A', [])), 'B': [],
        'C': [bp for bp in pvph if bp not in selected_groups.get('A', [])]}
    b_only_groups = {
        'A': [], 'B': list(selected_groups.get('B', [])),
        'C': [bp for bp in pvph if bp not in selected_groups.get('B', [])]}
    a_only_on_selected_taps = _evaluate_core_tap_candidate(
        base_result['model'], data, sets_info, a_only_groups, [], fixed_pf_A,
        selected['schedule'], opt, 'counterfactual-selected-taps-A-only',
        'group_a_only_counterfactual', max_iterations=SETTINGS_MAX_ITER,
        v_tolerance=SETTINGS_V_TOL, q_tolerance=SETTINGS_Q_TOL)
    b_only_curves = _group_b_seed_rows(b_only_groups, selected_inferred_B)
    b_only_on_selected_taps = _evaluate_core_tap_candidate(
        base_result['model'], data, sets_info, b_only_groups, b_only_curves, {},
        selected['schedule'], opt, 'counterfactual-selected-taps-B-only',
        'group_b_only_counterfactual', max_iterations=SETTINGS_MAX_ITER,
        v_tolerance=SETTINGS_V_TOL, q_tolerance=SETTINGS_Q_TOL)

    def _scenario_row(name, source, taps):
        return {
            'name': name,
            'valid': bool(source.get('admissible', False)),
            'converged': bool(source.get('converged', False)),
            'tap_ops_total': int(taps),
            'voltage_deviation_pu_sum':
                float(source.get('voltage_deviation_pu_sum', float('inf'))),
            'unbalance_pu_sum':
                float(source.get('unbalance_pu_sum', float('inf'))),
            'core_loss_kwh': float(source.get('core_loss_kwh', float('inf'))),
            'joule_loss_kwh': float(source.get('joule_loss_kwh', float('inf'))),
            'total_loss_kwh': float(source.get('total_loss_kwh', float('inf'))),
            'daily_screening_ok': bool(source.get('daily_screening_ok', False)),
            'formal_regulatory_assessed': False,
            'formal_regulatory_ok': None,
            'regulatory_ok': None,
            'bfm_exact': bool(
                (source.get('exactness') or
                 source.get('settings_bfm_exactness') or {}).get(
                    'exactness_ok', False)),
        }

    s1 = _scenario_row(
        'S1_LDC_local_FP1', dominance_reference,
        dominance_reference.get('tap_ops_total', 0))
    s1['valid'] = bool(dominance_reference.get('valid', False))
    s1['bfm_exact'] = bool(
        (dominance_reference.get('exactness') or {}).get('exactness_ok', False))
    s2 = _scenario_row(
        'S2_LDC_local_FP096_uniforme', uniform_fp096_ldc,
        sum(uniform_fp096_ldc['tap_audit']['ops'].values()))
    s3 = _scenario_row(
        'S3_OPF_coordenado_ABC', selected,
        sum(selected['tap_audit']['ops'].values()))

    s2c = _scenario_row(
        'D1_mesmos_taps_S3_FP096_uniforme', uniform_fp096_on_s3_taps,
        sum(uniform_fp096_on_s3_taps['tap_audit']['ops'].values()))
    s2u = _scenario_row(
        'D2_mesmos_taps_S3_FP1', unity_on_selected_taps,
        sum(unity_on_selected_taps['tap_audit']['ops'].values()))
    s2a = _scenario_row(
        'D3_mesmos_taps_S3_somente_A', a_only_on_selected_taps,
        sum(a_only_on_selected_taps['tap_audit']['ops'].values()))
    s2b = _scenario_row(
        'D4_mesmos_taps_S3_somente_B', b_only_on_selected_taps,
        sum(b_only_on_selected_taps['tap_audit']['ops'].values()))

    attribution = {
        'scenarios': (s1, s2, s3),
        'diagnostics': (s2c, s2u, s2a, s2b),
        'tap_reduction_s2_vs_s1': s1['tap_ops_total'] - s2['tap_ops_total'],
        'tap_reduction_s3_vs_s1': s1['tap_ops_total'] - s3['tap_ops_total'],
        'tap_reduction_s3_vs_s2': s2['tap_ops_total'] - s3['tap_ops_total'],
        'group_counts': {
            'A': len(selected_groups.get('A', [])),
            'B': len(selected_groups.get('B', [])),
            'C': len(selected_groups.get('C', [])),
        },
    }
    out['causal_attribution_scenarios'] = attribution

    try:
        import pandas as _pd
        _pd.DataFrame(list(attribution['scenarios'])).to_csv(
            'paper_s1_s2_s3_summary.csv', index=False)
        _pd.DataFrame(list(attribution['diagnostics'])).to_csv(
            'paper_s1_s2_s3_diagnostics.csv', index=False)
        _pd.DataFrame([attribution['group_counts']]).to_csv(
            'paper_selected_group_counts.csv', index=False)
    except Exception as _csv_exc:
        print(f"    [CSV] aviso S1/S2/S3: {_csv_exc}")

    print("\n    CENÁRIOS PRINCIPAIS — S1 -> S2 -> S3")
    print(f"    {'Cenário':<34} {'válido':>7} {'taps':>6} "
          f"{'desvio':>11} {'deseq':>11} {'Joule':>10} "
          f"{'core':>9} {'total':>10}")
    for row in attribution['scenarios']:
        print(f"    {row['name']:<34} "
              f"{('SIM' if row['valid'] else 'NÃO'):>7} "
              f"{row['tap_ops_total']:>6d} "
              f"{row['voltage_deviation_pu_sum']:>11.4f} "
              f"{row['unbalance_pu_sum']:>11.4f} "
              f"{row['joule_loss_kwh']:>10.2f} "
              f"{row['core_loss_kwh']:>9.2f} "
              f"{row['total_loss_kwh']:>10.2f}")
    print("    A/B/C final: "
          f"A={attribution['group_counts']['A']} | "
          f"B={attribution['group_counts']['B']} | "
          f"C={attribution['group_counts']['C']}")

    unity_on_selected_taps.pop('model', None)
    uniform_fp096_ldc.pop('model', None)
    uniform_fp096_on_s3_taps.pop('model', None)
    a_only_on_selected_taps.pop('model', None)
    b_only_on_selected_taps.pop('model', None)
    # Busca limitada do menor orçamento de taps, entre as trajetórias já
    # produzidas pelo algoritmo, que continua fisicamente válido em FP=1.
    # Não reivindica ótimo global; isola se uma trajetória de poucos taps
    # depende de suporte reativo.
    _schedule_pool = {}
    for _row in [selected] + list(candidates):
        _schedule = _row.get('schedule')
        if not _schedule:
            continue
        _signature = tuple(_schedule[k] for k in sorted(_schedule))
        _ops = sum(_audit_programmed_tap_schedule(
            _schedule, data, hours)['ops'].values())
        _schedule_pool.setdefault(
            _signature, (_ops, _row.get('name', 'trajetoria'), _schedule))
    _unity_screen = []
    _selected_signature = tuple(
        selected['schedule'][k] for k in sorted(selected['schedule']))
    for _signature, (_ops, _name, _schedule) in sorted(
            _schedule_pool.items(), key=lambda item: item[1][0]
            )[:UNITY_COUNTERFACTUAL_MAX_SCHEDULES]:
        if _signature == _selected_signature:
            _cf = unity_on_selected_taps
        else:
            _cf = _evaluate_core_tap_candidate(
                base_result['model'], data, sets_info, unity_groups, [], {},
                _schedule, opt, f'counterfactual-unity-{_name}',
                'unity_pf_counterfactual', max_iterations=SETTINGS_MAX_ITER,
                v_tolerance=SETTINGS_V_TOL, q_tolerance=SETTINGS_Q_TOL)
        _cf_row = _scenario_row(
            f'FP1_screen_{_name}', _cf, _ops)
        _unity_screen.append(_cf_row)
        if _cf is not unity_on_selected_taps:
            _cf.pop('model', None)
    _unity_valid = [row for row in _unity_screen if row['valid']]
    _min_unity_taps_tested = (
        min(row['tap_ops_total'] for row in _unity_valid)
        if _unity_valid else None)
    _reactive_valid_catalog = [
        row for row in candidates
        if row.get('operationally_admissible', row.get('admissible', False))]
    _min_reactive_taps_tested = min(
        [sum(selected['tap_audit']['ops'].values())] +
        [int(row.get('tap_ops_total',
                     sum(row['tap_audit']['ops'].values())))
         for row in _reactive_valid_catalog])
    attribution = {
        'scenarios': (s1, s2, s2a, s2b, s3),
        'unity_schedule_screen': tuple(_unity_screen),
        'minimum_unity_taps_among_tested_schedules':
            _min_unity_taps_tested,
        'minimum_reactive_taps_among_tested_schedules':
            _min_reactive_taps_tested,
        'reactive_tap_advantage_among_tested_schedules': (
            None if _min_unity_taps_tested is None else
            _min_unity_taps_tested-_min_reactive_taps_tested),
        'tap_reduction_total_steps':
            s1['tap_ops_total']-s3['tap_ops_total'],
        'tap_reduction_from_schedule_steps':
            s1['tap_ops_total']-s2['tap_ops_total'],
        'same_low_tap_schedule_feasible_without_reactive': s2['valid'],
        'reactive_control_enables_low_tap_schedule':
            bool(s3['valid'] and not s2['valid']),
        'reactive_incremental_voltage_deviation_pu':
            s3['voltage_deviation_pu_sum']-s2['voltage_deviation_pu_sum'],
        'reactive_incremental_total_loss_kwh':
            s3['total_loss_kwh']-s2['total_loss_kwh'],
        'group_a_incremental_voltage_deviation_pu':
            s2a['voltage_deviation_pu_sum']-s2['voltage_deviation_pu_sum'],
        'group_a_incremental_total_loss_kwh':
            s2a['total_loss_kwh']-s2['total_loss_kwh'],
        'group_b_incremental_voltage_deviation_pu':
            s2b['voltage_deviation_pu_sum']-s2['voltage_deviation_pu_sum'],
        'group_b_incremental_total_loss_kwh':
            s2b['total_loss_kwh']-s2['total_loss_kwh'],
    }
    out['causal_attribution_scenarios'] = attribution
    print("\n    CONTRAFACTUAIS DE ATRIBUIÇÃO — MESMOS DADOS E CRITÉRIOS")
    print(f"    {'Cenário':<38} {'válido':>7} {'taps':>6} "
          f"{'desvio':>11} {'deseq':>11} {'Joule':>10} "
          f"{'core':>9} {'total':>10}")
    for row in attribution['scenarios']:
        print(f"    {row['name']:<38} "
              f"{('SIM' if row['valid'] else 'NÃO'):>7} "
              f"{row['tap_ops_total']:>6d} "
              f"{row['voltage_deviation_pu_sum']:>11.4f} "
              f"{row['unbalance_pu_sum']:>11.4f} "
              f"{row['joule_loss_kwh']:>10.2f} "
              f"{row['core_loss_kwh']:>9.2f} "
              f"{row['total_loss_kwh']:>10.2f}")
    print("    Atribuição: redução total="
          f"{attribution['tap_reduction_total_steps']} degraus; "
          "redução explicada pela trajetória de taps="
          f"{attribution['tap_reduction_from_schedule_steps']} degraus; "
          "taps baixos factíveis com FP1="
          f"{'SIM' if s2['valid'] else 'NÃO'}.")
    print("    Menor orçamento entre trajetórias testadas: "
          f"FP1={_min_unity_taps_tested if _min_unity_taps_tested is not None else 'nenhum'} | "
          f"A/B/C={_min_reactive_taps_tested} | "
          "vantagem reativa="
          f"{attribution['reactive_tap_advantage_among_tested_schedules']}.")
    unity_on_selected_taps.pop('model', None)
    a_only_on_selected_taps.pop('model', None)
    b_only_on_selected_taps.pop('model', None)

    # O candidato finalmente selecionado passa novamente pela validação híbrida.
    # Assim, a F.O. reportada não fica associada ao tap-base quando outra
    # trajetória é escolhida pela busca operacional.
    selected_p_injection_pu = {
        (b,ph,t): (pyo.value(selected['model'].Pavail[b,ph,t])-
                   pyo.value(selected['model'].Pcurt[b,ph,t]))
        for b,ph in pvph for t in hours}
    selected_hybrid = validate_hybrid_fixed_pq_pu(
        selected['model'], data, sets_info, selected_groups,
        selected_p_injection_pu, selected['q_settings'],
        selected['schedule'], opt)
    out['hybrid_validation'] = selected_hybrid
    out['hybrid_validation_ok'] = selected_hybrid['ok']
    out['objective_settings_hybrid_pu'] = selected_hybrid['objective_pu']
    out['electrical_unit_system'] = 'per_unit'
    out['operationally_applicable'] = bool(
        out['operationally_applicable'] and selected_hybrid['ok'])
    if selected_hybrid['termination'] in _USABLE_TERMINATIONS:
        _discarded_candidate_model = out.get('model')
        out['model'] = selected_hybrid['model']
        selected.pop('model', None)
        if _discarded_candidate_model is not base_result['model']:
            _discarded_candidate_model = None
            gc.collect()
        out['settings_quality'] = summarize_voltage_quality(
            out['model'], data, sets_info)
        out['settings_bfm_exactness'] = selected_hybrid['exactness']
        out['physical_model_ok'] = bool(
            selected_hybrid['exactness'] and
            selected_hybrid['exactness']['exactness_ok'])
        out['core_loss_kwh'] = calc_core_loss_energy_kwh(
            out['model'], data, sets_info)
        out['joule_loss_kwh'] = calc_joule_loss_energy_kwh(
            out['model'], data, sets_info)
        out['total_loss_kwh'] = out['core_loss_kwh']+out['joule_loss_kwh']
        out['objective_settings_hybrid_pu'] = recalculate_original_objective_pu(
            out['model'], data, sets_info)
        _hybrid_raw = out['objective_settings_hybrid_pu']['raw']
        out['selected_voltage_deviation_pu_sum'] = (
            _hybrid_raw['voltage_deviation_pu_sum'])
        out['selected_unbalance_pu_sum'] = _hybrid_raw['unbalance_pu_sum']
        out['validation_bound_hits'] = sum(
            1 for b,ph in sets_info['bph'] if b != data['slack'] for t in hours
            if (try_v(out['model'],b,ph,t) <= VALIDATION_V_MIN+1.0e-5 or
                try_v(out['model'],b,ph,t) >= VALIDATION_V_MAX-1.0e-5))
        _hq = out['settings_quality']
        _legacy_screen = assess_representative_day_quality(
            _hq, validation_bound_hits=out['validation_bound_hits'],
            baseline_quality=(dominance_reference.get('quality')
                              if dominance_reference_ok else None),
            period_count=len(hours))
        out['daily_quality_screening'] = _legacy_screen
        out['daily_screening_ok'] = _legacy_screen['daily_screening_ok']
        out['daily_screening_absolute_ok'] = _legacy_screen['daily_screening_absolute_ok']
        out['formal_regulatory_assessed'] = False
        out['formal_regulatory_ok'] = None
        out['regulatory_ok'] = None
        out['operationally_applicable'] = bool(
            out['operationally_applicable'] and out['daily_screening_ok'] and
            out['physical_model_ok'])

        # A classificação A/B/C e o fechamento híbrido podem deslocar V e Q.
        # Portanto a dominância calculada durante a busca é auditada novamente
        # sobre o estado final que seria efetivamente entregue ao operador.
        if dominance_reference_ok:
            _final_operational_score = (
                OP_WEIGHT_TAP*sum(out['tap_ops'].values()) /
                max(dominance_reference['tap_ops_total'], 1) +
                OP_WEIGHT_VOLTAGE_DEVIATION*
                out['selected_voltage_deviation_pu_sum'] /
                max(dominance_reference['voltage_deviation_pu_sum'], 1.0e-9) +
                OP_WEIGHT_UNBALANCE*out['selected_unbalance_pu_sum'] /
                max(dominance_reference['unbalance_pu_sum'], 1.0e-9) +
                OP_WEIGHT_CORE*out['core_loss_kwh'] /
                max(dominance_reference['core_loss_kwh'], 1.0e-9))
            _final_relative_score_improvement = 1.0-_final_operational_score
            _final_dom = _audit_four_objective_dominance(
                sum(out['tap_ops'].values()),
                out['selected_voltage_deviation_pu_sum'],
                out['selected_unbalance_pu_sum'], out['core_loss_kwh'],
                out['total_loss_kwh'], dominance_reference['tap_ops_total'],
                dominance_reference['voltage_deviation_pu_sum'],
                dominance_reference['unbalance_pu_sum'],
                dominance_reference['core_loss_kwh'],
                dominance_reference['total_loss_kwh'],
                operationally_admissible=bool(
                    selected_hybrid['ok'] and out.get('daily_screening_ok', False) and
                    out['physical_model_ok']),
                relative_score_improvement=_final_relative_score_improvement)
            out['operational_score'] = _final_operational_score
            out['selected_relative_score_improvement'] = (
                _final_relative_score_improvement)
            out['post_hybrid_dominance_ok'] = _final_dom['ok']
            out['post_hybrid_dominance_reasons'] = _final_dom['reasons']
            out['selected_tap_saving_steps'] = _final_dom['tap_saving_steps']
            out['selected_voltage_deviation_saving_pu'] = (
                _final_dom['voltage_deviation_saving_pu'])
            out['selected_voltage_deviation_saving_frac'] = (
                _final_dom['voltage_deviation_saving_frac'])
            out['selected_unbalance_saving_pu'] = (
                _final_dom['unbalance_saving_pu'])
            out['selected_unbalance_saving_frac'] = (
                _final_dom['unbalance_saving_frac'])
            out['selected_core_saving_kwh'] = _final_dom['core_saving_kwh']
            out['selected_total_loss_increase_frac'] = (
                _final_dom['total_loss_increase_frac'])
            if not _final_dom['ok']:
                out['simultaneous_target_achieved'] = False
                out['operationally_applicable'] = False
                out['selection_status'] = (
                    'falha_dominancia_pos_validacao_hibrida_recomendacao_bloqueada')
                out['no_action_applied'] = True
        else:
            out['post_hybrid_dominance_ok'] = False
            out['post_hybrid_dominance_reasons'] = [
                'baseline_fisica_nao_certificada']
            if dominance_reference_required:
                out['simultaneous_target_achieved'] = False
                out['operationally_applicable'] = False
                out['selection_status'] = (
                    'baseline_fisica_invalida_recomendacao_bloqueada')
                out['no_action_applied'] = True
    else:
        out['post_hybrid_dominance_ok'] = False
        out['post_hybrid_dominance_reasons'] = ['validacao_hibrida_sem_solucao_util']
        out['simultaneous_target_achieved'] = False
        out['operationally_applicable'] = False
    return out


In [ ]:
#@title Internal model implementation 10/12 { display-mode: "form" }
def search_min_core_loss_candidate(*args, **kwargs):
    """Alias legado; na V3.9.3 a seleção exige dominância nas quatro metas."""
    return search_operational_candidate(*args, **kwargs)


def report_physical_transformer_thermal(m, data, sets_info, pnl_pre_total,
                                        state_label,
                                        baseline_physical=None):
    """Relata perdas de núcleo e térmica apenas para um fluxo BFM fechado.

    A temperatura é uma aproximação quasi-estacionária baseada nas elevações
    nominais da IEEE C57.91. Ela não substitui um modelo dinâmico com constantes
    de tempo do óleo e do enrolamento.
    """
    exact = summarize_bfm_exactness(m, data, sets_info)
    print("\n" + "="*68)
    print("  AVALIAÇÃO TÉRMICA FINAL — ESTADO ELÉTRICO REAPLICADO")
    print("="*68)
    print(f"  Estado: {state_label}")
    print("  Fechamento BFM: "
          f"gap={exact['aggregate_gap_pct']:.4f}% | "
          f"violação máx={exact['max_physical_violation_pu']:.3e} p.u. | "
          f"resíduo ΔV²={exact['max_vdrop_residual_pu2']:.3e} p.u.²")
    if not exact['exactness_ok']:
        print("  [!] Avaliação térmica final não emitida: estado elétrico não fechado.")
        return None

    hours = list(sets_info['hours'])
    branches = data['branches']
    trafos = data.get('trafos', {})
    use_measured_ambient = (
        PV_TEMP_INPUT_KIND == 'ambient' and TEMP_AMB_DATA_OK)
    ambient = TEMP_AMB if use_measured_ambient else [25.0]*N_PERIODS
    print("  θ_a: " + ("perfil ambiente medido" if use_measured_ambient
                       else "25°C fixos — cenário de referência"))
    print("  Método térmico: aproximação quasi-estacionária baseada na IEEE C57.91.")

    branches_by_trafo = {}
    pnl_post = {}
    for (mv, lv, ph), bd in branches.items():
        if bd.get('level') != 'trafo':
            continue
        tid = bd.get('trafo_id', f'{mv}-{lv}')
        branches_by_trafo.setdefault(tid, []).append((mv, lv, ph))
        G = bd.get('G_core_pu', 0.0)
        for t in hours:
            # O balanço elétrico posiciona G_core na barra receptora j=lv.
            pnl_post[(tid, ph, t)] = G * try_v(m, lv, ph, t)**2 * SBASE

    baseline_model = (
        baseline_physical.get('model')
        if baseline_physical and baseline_physical.get('valid', False)
        else None)
    baseline_label = 'Base_fis' if baseline_model is not None else 'Pré_aprox'
    print("\n  Perdas no núcleo G·V² — estado físico:")
    print(f"  {'Trafo':<12} {'P_NL_nom':>10} {baseline_label:>10} "
          f"{'Físico_méd':>12} {'Δ%':>9}")
    theta_hs_all = {}
    life_by_trafo = {}
    for tid, td in trafos.items():
        trafo_branches = branches_by_trafo.get(tid, [])
        phs = list(td['phases'])
        vals_post = [pnl_post.get((tid, ph, t), 0.0)
                     for ph in phs for t in hours]
        if baseline_model is not None:
            vals_pre = [
                branches[(mv,lv,ph)].get('G_core_pu', 0.0) *
                try_v(baseline_model, lv, ph, t)**2 * SBASE
                for (mv,lv,ph) in trafo_branches for t in hours]
        else:
            vals_pre = [v for td_t in pnl_pre_total.get(tid, {}).values()
                        for v in td_t.values()]
        avg_post = sum(vals_post)/max(len(vals_post), 1)
        avg_pre = sum(vals_pre)/max(len(vals_pre), 1)
        delta = 100.0*(avg_post-avg_pre)/max(avg_pre, 1.0e-9)
        print(f"  {tid:<12} {td['P_NL_nom_kw']:>10.4f} {avg_pre:>10.4f} "
              f"{avg_post:>12.4f} {delta:>8.2f}%")

        theta_hs = []
        for h in range(24):
            ts = list(range(h*6, min(h*6+6, len(hours))))
            denom = max(len(ts)*len(phs), 1)
            # No estado certificado, r·l é a perda Joule física do ramo.
            p_ll_h = sum(
                pyo.value(m.r[mv,lv,ph]) * pyo.value(m.l_sq[mv,lv,ph,t])
                * SBASE
                for (mv,lv,ph) in trafo_branches for t in ts) / denom
            p_nl_h = sum(
                pnl_post.get((tid,ph,t), 0.0)
                for ph in phs for t in ts) / denom
            p_ll_r = td['P_LL_R_kw']
            p_nl_r = td['P_NL_nom_kw']
            loss_ratio = max(
                (p_ll_h+p_nl_h)/max(p_ll_r+p_nl_r, 1.0e-9), 0.0)
            load_loss_ratio = max(p_ll_h/max(p_ll_r, 1.0e-9), 0.0)
            theta_to = td['theta_TO_R'] * loss_ratio**0.8
            theta_g = td['theta_g_R'] * load_loss_ratio**0.8
            theta_a = sum(ambient[t] for t in ts)/max(len(ts), 1)
            theta_hs.append(theta_a + theta_to + theta_g)
        theta_hs_all[tid] = theta_hs

        faa = [math.exp(15000.0*(1.0/383.0 - 1.0/(273.0+v)))
               for v in theta_hs]
        dt_h = DT_MIN/60.0 * 6
        life_pct = sum(f*dt_h for f in faa)/(20.0*8760.0)*100.0
        life_by_trafo[tid] = life_pct

    print("\n  Hot-spot e envelhecimento — estado físico:")
    print(f"  {'Trafo':<12} {'θ_hs_max':>10} {'θ_hs_méd':>10} "
          f"{'h>95°C':>9} {'h>110°C':>10} {'Vida_dia%':>12}")
    for tid, theta_hs in theta_hs_all.items():
        print(f"  {tid:<12} {max(theta_hs):>10.1f} "
              f"{sum(theta_hs)/max(len(theta_hs),1):>10.1f} "
              f"{sum(v>95.0 for v in theta_hs):>9d} "
              f"{sum(v>110.0 for v in theta_hs):>10d} "
              f"{life_by_trafo[tid]:>12.5f}")
    if life_by_trafo:
        worst_tid = max(life_by_trafo, key=life_by_trafo.get)
        mean_life = sum(life_by_trafo.values())/len(life_by_trafo)
        print("\n  Síntese térmica física:")
        print(f"    Perda de vida média/trafo: {mean_life:.5f} %/dia")
        print(f"    Pior trafo: {worst_tid} "
              f"({life_by_trafo[worst_tid]:.5f} %/dia)")
        print("    Referência nominal a 110°C: ≈0,0137 %/dia")
    return {
        'exactness': exact,
        'theta_hs': theta_hs_all,
        'life_pct_day': life_by_trafo,
        'ambient_measured': use_measured_ambient,
        'baseline_is_physical': baseline_model is not None,
        'method': 'IEEE_C57_91_quasi_stationary',
    }


def plot_results(m, data, sets_info):
    """Gráfico 1: FD% antes/depois | Gráfico 2: curva V-Q | Gráfico 3: tap SVR."""
    try:
        import matplotlib.pyplot as plt
        import matplotlib.colors as mcolors
        import numpy as np
    except ImportError:
        print("  pip install matplotlib"); return

    buses  = data['buses']
    pv     = data['pv']
    svr    = data['svr']
    hours  = sets_info['hours']
    irr    = sets_info['irr']
    load   = sets_info['load']
    bph    = sets_info['bph']
    pvph   = sets_info['pvph']
    svr_phs = sets_info['svr_phs']
    t_vv   = sets_info['t_vv']
    step_  = sets_info['tap_step']
    tap_min_ = sets_info['tap_min']
    n_pos_ = sets_info['n_pos']

    bph_set = set(bph)
    trif_buses = sorted({b for (b,ph) in bph
                          if buses.get(b,{}).get('level')=='mv'
                          and all((b,p) in bph_set for p in [1,2,3])})

    # Pré-OPF: forward sweep
    V_pre, _, _, _, _, _ = compute_warm_start(data, hours, irr, load)

    # Cálculo de FD% por amostra (bus, t) usando helpers centralizados
    # Limitação: FD% computado captura apenas desequilíbrio de amplitude
    # (modelo BFM-SOCP não fornece ângulos de tensão explícitos).
    def _fd_pre_sample(b, t):
        V1 = V_pre.get((b, 1, t), 1.0)
        V2 = V_pre.get((b, 2, t), 1.0)
        V3 = V_pre.get((b, 3, t), 1.0)
        return _calc_fd_prodist(*_calc_vll_from_vph(V1, V2, V3))
    def _fd_post_sample(b, t):
        return _calc_fd_prodist(*_calc_vll_from_vph(
            try_v(m, b, 1, t), try_v(m, b, 2, t), try_v(m, b, 3, t)))

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle("BFM-SOCP OPF — Volt-VAR + SVR | 144×10min",
                 fontsize=13, fontweight='bold')

    # ── SUB1: CDF do FD% — proxy do FD95% regulatório (PRODIST Módulo 8) ───
    # Recolhe TODAS amostras (bus, t) antes e depois do OPF.
    fd_pre_all  = np.array([_fd_pre_sample(b, t)
                            for b in trif_buses for t in hours])
    fd_post_all = np.array([_fd_post_sample(b, t)
                            for b in trif_buses for t in hours])

    if fd_pre_all.size and fd_post_all.size:
        # CDF empírica (Função de Distribuição Cumulativa)
        sorted_pre  = np.sort(fd_pre_all)
        sorted_post = np.sort(fd_post_all)
        cdf_pre  = np.arange(1, len(sorted_pre)  + 1) / len(sorted_pre)
        cdf_post = np.arange(1, len(sorted_post) + 1) / len(sorted_post)

        # Indicadores estatísticos
        fd95_pre   = float(np.percentile(fd_pre_all,  95))
        fd95_post  = float(np.percentile(fd_post_all, 95))
        fd_max_pre  = float(np.max(fd_pre_all))
        fd_max_post = float(np.max(fd_post_all))

        # Curvas CDF
        ax1.plot(sorted_pre,  cdf_pre  * 100, '-',
                 color='#E05C3A', lw=2.2, label='Before OPF')
        ax1.plot(sorted_post, cdf_post * 100, '-',
                 color='#2A7EBA', lw=2.2, label='After OPF')

        # Linha horizontal P95 (referência estatística do PRODIST)
        ax1.axhline(95, color='black', lw=1.0, ls=':', alpha=0.6)
        ax1.text(ax1.get_xlim()[1] * 0.02, 95.5, 'P95',
                 fontsize=7, color='black', alpha=0.7)

        # Linha vertical do limite PRODIST MT (2,0%)
        ax1.axvline(FD_MT_MAX, color='gray', lw=1.2, ls='--',
                    label=f'MV limit {FD_MT_MAX:.1f}% PRODIST')

        # Marcadores de FD95% antes/depois sobre as curvas
        ax1.plot(fd95_pre,  95, 'o', color='#E05C3A', ms=10,
                 mec='white', mew=1.5, zorder=5)
        ax1.plot(fd95_post, 95, 's', color='#2A7EBA', ms=10,
                 mec='white', mew=1.5, zorder=5)

        # Caixa de texto com indicadores numéricos
        viol_pre  = 100.0 * np.sum(fd_pre_all  > FD_MT_MAX) / fd_pre_all.size
        viol_post = 100.0 * np.sum(fd_post_all > FD_MT_MAX) / fd_post_all.size
        info_txt = (
            f'FD95%: {fd95_pre:.2f}% → {fd95_post:.2f}%\n'
            f'Pico:  {fd_max_pre:.2f}% → {fd_max_post:.2f}%\n'
            f'%t>lim: {viol_pre:.1f}% → {viol_post:.1f}%'
        )
        ax1.text(0.97, 0.05, info_txt, transform=ax1.transAxes,
                 fontsize=8, ha='right', va='bottom', family='monospace',
                 bbox=dict(boxstyle='round,pad=0.4',
                           facecolor='white', edgecolor='gray', alpha=0.9))

        ax1.set_xlim(0, max(fd_max_pre, FD_MT_MAX * 1.5) * 1.05)
        ax1.set_ylim(0, 102)
        ax1.set_xlabel('FD% (PRODIST Module 8)')
        ax1.set_ylabel('Empirical CDF [%]')
        ax1.set_title('Cumulative distribution of FD%\n'
                      '(daily proxy of regulatory FD95%)')
        ax1.legend(fontsize=8, loc='center right')
        ax1.grid(alpha=0.3)

    # ── SUB2: Curva Volt-VAR ───────────────────────────────────────────────
    pvph_list_g = sets_info['pvph']
    t_vv_h = [t for t in hours if t_vv[0]<=t<=t_vv[1]]
    t_peak = max(hours, key=lambda t: irr[t])
    _tan_fp = math.sqrt(1-FP_MIN_VV**2)/FP_MIN_VV
    if pvph_list_g:
        best_bph = max(pvph_list_g,
                       key=lambda bp: abs(pyo.value(m.Qpv[bp[0],bp[1],t_peak])))
        b_vv, ph_vv = best_bph
        V_op=[try_v(m,b_vv,ph_vv,t) for t in t_vv_h]
        Q_op=[pyo.value(m.Qpv[b_vv,ph_vv,t])*SBASE for t in t_vv_h]
        irr_op=[irr[t] for t in t_vv_h]
        _pv_loc = data['pv']
        Qmax_pk = _pv_loc[(b_vv,ph_vv)]['P_rated_pu']*irr[t_peak]*_tan_fp*SBASE if (b_vv,ph_vv) in _pv_loc else 50.
        v_c=np.linspace(0.88,1.12,200)
        def _vvc(v,qm):
            V1,V2,V3,V4=0.92,0.98,1.02,1.08
            if v<=V1: return qm
            if v<=V2: return qm*(V2-v)/(V2-V1)
            if v<=V3: return 0.
            if v<=V4: return -qm*(v-V3)/(V4-V3)
            return -qm
        ax2.plot(v_c,[_vvc(v,Qmax_pk) for v in v_c],'--',color='#BA7517',lw=2,label='IEEE 1547 Cat B')
        norm=mcolors.Normalize(vmin=0,vmax=max(irr_op) if irr_op else 1)
        sc=ax2.scatter(V_op,Q_op,c=irr_op,cmap='YlOrRd',norm=norm,s=60,zorder=5,label='Optimal OPF point')
        plt.colorbar(sc,ax=ax2,label='Irrad.',fraction=0.04)
        ax2.axhline(0,color='gray',lw=.7,ls=':')
        ax2.set_xlabel('Voltage [p.u.]'); ax2.set_ylabel('Qpv [kVAr/phase]')
        ax2.set_title(f'Volt-VAR curve\n({b_vv}, phase {ph_vv})'); ax2.legend(fontsize=8); ax2.grid(alpha=.3)

    # ── SUB3: Tap SVR ──────────────────────────────────────────────────────
    colors_ph={1:'#7F77DD',2:'#E0803A',3:'#1D9E75'}
    ax3.axvspan(t_vv[0]-.5,t_vv[1]+.5,alpha=0.07,color='#378ADD',label='VV window')
    for ph in svr_phs:
        tap_seq=[tap_min_+int(round(pyo.value(m.tap_pos[ph,t])))*step_ for t in hours]
        ax3.step(hours,tap_seq,where='mid',color=colors_ph[ph],lw=2.2,label=f'Phase {ph}')
        prev=tap_min_+int(round((svr[ph]['tap_init']-tap_min_)/step_))*step_
        for i,t in enumerate(hours):
            curr=tap_seq[i]
            if abs(curr-prev)>step_*0.5:
                mk='^' if curr>prev else 'v'
                ax3.scatter(t,curr,marker=mk,s=100,color=colors_ph[ph],zorder=6,edgecolors='white')
            prev=curr
    # Redefine xtk para uso aqui (variável local removida do SUB1 refatorado)
    xtk = [k for k in range(0, N_PERIODS, 12)]
    ax3.set_xticks(xtk); ax3.set_xticklabels([f"{k*DT_MIN//60:02d}h" for k in xtk],fontsize=7,rotation=45)
    ax3.set_xlabel('Period'); ax3.set_ylabel('Tap [p.u.]')
    ax3.set_title('SVR tap — 3 phases\n(▲▼=mechanical operation)'); ax3.legend(fontsize=8); ax3.grid(alpha=.3)

    plt.tight_layout(pad=2.)
    plt.savefig('opf_bfm_voltvar_tap.png',dpi=150,bbox_inches='tight')
    print("  Salvo: opf_bfm_voltvar_tap.png")
    plt.show()


def plot_voltage_mt(m, data, sets_info):
    """Heatmap e evolução temporal de tensão nas barras MT."""
    try:
        import matplotlib.pyplot as plt
        import matplotlib.colors as mcolors
        import numpy as np
    except ImportError:
        print("  pip install matplotlib"); return
    from collections import deque, defaultdict

    buses   = data['buses']
    branches= data['branches']
    svr     = data['svr']
    slack   = data['slack']
    hours   = sets_info['hours']
    irr     = sets_info['irr']
    load    = sets_info['load']
    bph     = sets_info['bph']
    svr_phs = sets_info['svr_phs']
    t_vv    = sets_info['t_vv']
    step_   = sets_info['tap_step']
    tap_min_= sets_info['tap_min']

    bph_set = set(bph)
    adj_mt  = defaultdict(list)
    for (fr,to,ph) in branches:
        if buses.get(fr,{}).get('level')=='mv' and buses.get(to,{}).get('level')=='mv':
            if to not in adj_mt[fr]: adj_mt[fr].append(to)
    for ph, d in svr.items():
        if d['lv_bus'] not in adj_mt[d['mv_bus']]: adj_mt[d['mv_bus']].append(d['lv_bus'])

    visited, order_mt = set(), []
    q = deque([slack]); visited.add(slack)
    while q:
        b = q.popleft()
        if buses.get(b, {}).get('level') == 'mv': order_mt.append(b)
        for nb in adj_mt.get(b, []):
            if nb not in visited: visited.add(nb); q.append(nb)
    mt_buses = [b for b in order_mt if any((b, ph) in bph_set for ph in [1, 2, 3])]
    if not mt_buses: print("  Sem barras MT"); return

    V_pre,_,_,_,_,_=compute_warm_start(data,hours,irr,load)
    t_peak=max(hours,key=lambda t:irr[t])
    colors_ph={1:'#7F77DD',2:'#E0803A',3:'#1D9E75'}

    fig,axes=plt.subplots(2,2,figsize=(22,12))
    fig.suptitle("MV voltages — BFM-SOCP | Analogous radial topology",
                 fontsize=13, fontweight='bold')
    ax1,ax2,ax3,ax4=axes[0,0],axes[0,1],axes[1,0],axes[1,1]

    # Sub1: perfil espacial no pico
    x_pos=list(range(len(mt_buses)))
    for ph in svr_phs:
        va=[V_pre.get((b,ph,t_peak),1.) for b in mt_buses]
        vp=[try_v(m,b,ph,t_peak) for b in mt_buses]
        ax1.plot(x_pos,va,'--',color=colors_ph[ph],lw=1.5,alpha=0.45)
        ax1.plot(x_pos,vp,'-o',color=colors_ph[ph],lw=2,ms=5,label=f'Phase {ph}')
    ax1.axhspan(V_ADEQ_MT_LO,V_ADEQ_MT_HI,alpha=0.07,color='#1D9E75',label='Adequada')
    ax1.axhline(V_ADEQ_MT_LO,color='#1D9E75',lw=.9,ls='--',alpha=.7)
    ax1.axhline(1.0,color='gray',lw=.7,ls=':',alpha=.5)
    ax1.set_xticks(x_pos); ax1.set_xticklabels(mt_buses,rotation=45,ha='right',fontsize=7)
    ax1.set_title(f'Spatial profile — peak t={t_peak}\n(dashed=pre-OPF, solid=post-OPF)')
    ax1.legend(fontsize=8); ax1.grid(alpha=.3)

    # Sub2: evolução temporal de barras selecionadas (fase 1)
    n_sel=min(6,len(mt_buses))
    sel=[mt_buses[i] for i in range(0,len(mt_buses),max(1,len(mt_buses)//n_sel))[:n_sel]]
    cmap_b=plt.cm.plasma
    for idx,b in enumerate(sel):
        vs=[try_v(m,b,1,t) for t in hours]
        ax2.plot(hours,vs,color=cmap_b(idx/max(len(sel)-1,1)),lw=1.8,label=b)
    # mudanças de tap fase 1
    for i,t in enumerate(hours[1:],1):
        p_cur=int(round(pyo.value(m.tap_pos[1,t])))
        p_prv=int(round(pyo.value(m.tap_pos[1,hours[i-1]])))
        if p_cur!=p_prv:
            ax2.axvline(t,color='#BA7517',lw=1.5,ls='--',alpha=.8)
    ax2.axhspan(V_ADEQ_MT_LO,V_ADEQ_MT_HI,alpha=0.06,color='#1D9E75')
    ax2.axhline(1.0,color='gray',lw=.7,ls=':',alpha=.5)
    xtk=[k for k in range(0,N_PERIODS,12)]
    ax2.set_xticks(xtk); ax2.set_xticklabels([f"{k*DT_MIN//60:02d}h" for k in xtk],fontsize=7)
    ax2.set_title('Temporal evolution V — MV buses\n(── = phase-1 tap change)')
    ax2.legend(fontsize=7); ax2.grid(alpha=.3)

    # Sub3: Heatmap V[barra×tempo] fase 1
    V_mat=np.array([[try_v(m,b,1,t) for t in hours] for b in mt_buses])
    vmin_h=max(0.88,V_mat.min()-.003); vmax_h=min(1.12,V_mat.max()+.003)
    vctr=max(vmin_h+1e-4,min(1.0,vmax_h-1e-4))
    norm_h=mcolors.TwoSlopeNorm(vcenter=vctr,vmin=vmin_h,vmax=vmax_h)
    im=ax3.imshow(V_mat,aspect='auto',cmap='RdYlGn',norm=norm_h,
                  extent=[-0.5,len(hours)-.5,len(mt_buses)-.5,-.5])
    for i,t in enumerate(hours[1:],1):
        if int(round(pyo.value(m.tap_pos[1,t])))!=int(round(pyo.value(m.tap_pos[1,hours[i-1]]))):
            ax3.axvline(t,color='white',lw=1.5,ls='--',alpha=.85)
    plt.colorbar(im,ax=ax3,fraction=.03,label='V [p.u.]')
    ax3.set_yticks(range(len(mt_buses))); ax3.set_yticklabels(mt_buses,fontsize=7)
    ax3.set_xticks(xtk); ax3.set_xticklabels([f"{k*DT_MIN//60:02d}h" for k in xtk],fontsize=7)
    ax3.set_title('Heatmap V[bus×time] — Phase 1')

    # Sub4: scatter tap × V[RG60]
    rg60=svr[svr_phs[0]]['lv_bus']
    norm_t=mcolors.Normalize(vmin=0,vmax=len(hours)-1)
    mk=['o','s','^']
    for ph in svr_phs:
        tv=[tap_min_+int(round(pyo.value(m.tap_pos[ph,t])))*step_ for t in hours]
        vr=[try_v(m,rg60,ph,t) for t in hours]
        ax4.scatter(tv,vr,c=list(range(len(hours))),cmap='viridis',norm=norm_t,
                    s=20,alpha=.65,marker=mk[ph-1],label=f'Phase {ph}')
    ax4.axhspan(V_ADEQ_MT_LO,V_ADEQ_MT_HI,alpha=0.07,color='#1D9E75',label='Adequada')
    plt.colorbar(plt.cm.ScalarMappable(norm=norm_t,cmap='viridis'),ax=ax4,fraction=.03,label='Period k')
    ax4.set_xlabel('Tap [p.u.]'); ax4.set_ylabel(f'V[{rg60}] [p.u.]')
    ax4.set_title(f'Tap × V[{rg60}] correlation'); ax4.legend(fontsize=8); ax4.grid(alpha=.3)

    plt.tight_layout(pad=2.5)
    plt.savefig('opf_bfm_voltage_mt.png',dpi=150,bbox_inches='tight')
    print("  Salvo: opf_bfm_voltage_mt.png")
    plt.show()


# =============================================================================
# TEMPORAL COMPARISON: BEFORE vs AFTER OPF — Voltage + Tap changes (phase 1)
# =============================================================================
# Produces a 2-panel figure side by side:
#   LEFT  : V(t) of 3-4 critical MV buses BEFORE OPF (local LDC control),
#           with vertical dashed lines marking the baseline tap-change events
#   RIGHT : V(t) of the same buses AFTER OPF, with vertical dashed lines
#           marking the OPF-optimized tap-change events
#
# Designed to be inserted in the Voltage Regulation subsection of the paper.
# Marker in paper_resultados.tex: [INSERT FIG: FIG_TAP_VOLTAGE_COMP]
# =============================================================================

def plot_voltage_temporal_comparison(m, data, sets_info,
                                     save_path='opf_bfm_voltage_temporal_comp.png'):
    """Side-by-side comparison of MV voltage evolution before and after OPF,
    with vertical markers at each tap-change event in each scenario.

    Uses 4 critical MV buses (slack, RG60 post-SVR, two downstream
    representative buses) of phase 1, to keep the figure clean."""
    try:
        import matplotlib.pyplot as plt
        import numpy as np
        from collections import deque, defaultdict
    except ImportError:
        print("  [TEMP_COMP] Install matplotlib first."); return

    buses     = data['buses']
    hours     = sets_info['hours']
    bph_set   = set(sets_info['bph'])
    branches  = data.get('branches', {})
    svr       = data.get('svr', {})
    irr       = sets_info.get('irr')
    load      = sets_info.get('load')

    # ── (1) Recompute pre-OPF voltages via warm-start (same approach used
    #       elsewhere in plot_voltage_mt and plot_voltage_profile_and_vv)
    try:
        V_pre, _, _, _, _, _ = compute_warm_start(data, hours, irr, load)
    except Exception as e:
        print(f"  [TEMP_COMP] Could not compute pre-OPF voltages: {e}")
        return

    # ── (2) Recompute pre-OPF tap trajectory via local LDC simulation
    try:
        tap_traj_pre, _, _ = simulate_svr_local_control(data, hours, irr, load)
    except Exception as e:
        print(f"  [TEMP_COMP] Could not simulate baseline tap trajectory: {e}")
        return

    # ── (3) Select 4 critical MV buses: prefer the canonical ones if present
    mv_buses_all = sorted({b for (b, ph) in bph_set
                           if buses.get(b, {}).get('level', 'mv') == 'mv'})
    preferred = ['650', 'RG60', '671', '675']
    sel = [b for b in preferred if b in mv_buses_all]
    if len(sel) < 4:
        # Fallback: take the first 4 MV buses
        sel = mv_buses_all[:4]
    if not sel:
        print("  [TEMP_COMP] No MV buses found."); return

    # ── (4) Identify tap-change events for phase 1 in each scenario
    def _tap_changes_post(ph):
        changes = []
        prev = None
        for t in hours:
            try:
                cur = int(round(pyo.value(m.tap_pos[ph, t])))
            except Exception:
                continue
            if prev is not None and cur != prev:
                changes.append(t)
            prev = cur
        return changes

    def _tap_changes_pre(ph):
        changes = []
        prev = None
        for t in hours:
            cur = tap_traj_pre.get((ph, t))
            if cur is None:
                continue
            if prev is not None and cur != prev:
                changes.append(t)
            prev = cur
        return changes

    changes_pre  = _tap_changes_pre(1)
    changes_post = _tap_changes_post(1)

    # ── (5) Build the plots
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    fig.suptitle("MV voltage evolution and tap-change events — "
                 "before vs.\\ after OPF (phase 1)",
                 fontsize=13, fontweight='bold')

    cmap_b = plt.cm.plasma
    n_sel = len(sel)

    # ── Left panel: BEFORE (baseline LDC control)
    for idx, b in enumerate(sel):
        vs = [V_pre.get((b, 1, t), 1.0) for t in hours]
        axL.plot(hours, vs, color=cmap_b(idx / max(n_sel - 1, 1)),
                 lw=1.8, label=b)
    for t_ch in changes_pre:
        axL.axvline(t_ch, color='#BA7517', lw=1.2, ls='--', alpha=0.6)
    try:
        axL.axhspan(V_ADEQ_MT_LO, V_ADEQ_MT_HI, alpha=0.06, color='#1D9E75')
    except NameError:
        axL.axhspan(0.93, 1.05, alpha=0.06, color='#1D9E75')
    axL.axhline(1.0, color='gray', lw=.7, ls=':', alpha=0.5)

    xtk = list(range(0, N_PERIODS, 12))
    axL.set_xticks(xtk)
    axL.set_xticklabels([f"{k * DT_MIN // 60:02d}h" for k in xtk], fontsize=8)
    axL.set_xlabel('Time of day', fontsize=10)
    axL.set_ylabel('Voltage [p.u.]', fontsize=10)
    axL.set_title(f'BEFORE OPF (baseline LDC)\n'
                  f'{len(changes_pre)} tap-change events (phase 1)',
                  fontsize=11)
    axL.legend(fontsize=8, loc='lower right', title='MV bus')
    axL.grid(alpha=0.3)

    # ── Right panel: AFTER (OPF-optimized)
    for idx, b in enumerate(sel):
        vs = [try_v(m, b, 1, t) for t in hours]
        axR.plot(hours, vs, color=cmap_b(idx / max(n_sel - 1, 1)),
                 lw=1.8, label=b)
    for t_ch in changes_post:
        axR.axvline(t_ch, color='#BA7517', lw=1.5, ls='--', alpha=0.85)
    try:
        axR.axhspan(V_ADEQ_MT_LO, V_ADEQ_MT_HI, alpha=0.06, color='#1D9E75')
    except NameError:
        axR.axhspan(0.93, 1.05, alpha=0.06, color='#1D9E75')
    axR.axhline(1.0, color='gray', lw=.7, ls=':', alpha=0.5)

    axR.set_xticks(xtk)
    axR.set_xticklabels([f"{k * DT_MIN // 60:02d}h" for k in xtk], fontsize=8)
    axR.set_xlabel('Time of day', fontsize=10)
    axR.set_title(f'AFTER OPF (proposed framework)\n'
                  f'{len(changes_post)} tap-change events (phase 1)',
                  fontsize=11)
    axR.legend(fontsize=8, loc='lower right', title='MV bus')
    axR.grid(alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"  [TEMP_COMP] Saved: {save_path}")
    print(f"  [TEMP_COMP] Tap changes — before: {len(changes_pre)} | "
          f"after: {len(changes_post)}")
    plt.show()
    return fig


# =============================================================================
# CONSOLIDATED VOLTAGE HEATMAP — STANDALONE FUNCTION
# =============================================================================
# Generates a 2-D heatmap V[b,phi,t] showing all bus-phase pairs of MV and LV
# along the daily horizon. Bus-phases are ordered by voltage level (MV first,
# then LV) and by proximity to the slack bus within each level.
#
# Usage: this function is self-contained and consumes only the Pyomo model,
# the data dictionary, and the sets_info dictionary already in memory after
# a solver run. It does NOT require re-solving the model.
#
#   plot_voltage_heatmap(model, data, sets_info)
#
# Output: writes 'opf_bfm_voltage_heatmap.png' to current working directory.
# =============================================================================

def plot_voltage_heatmap(m, data, sets_info, save_path=None,
                         force_phase=None):
    """Generates a voltage heatmap V[bus, time] for the most critical
    phase (or a user-specified phase), with buses ordered by topological
    distance to the slack bus.

    Selection criterion (default): the most critical phase is the one
    maximizing the spatio-temporal range max(V) - min(V) across all
    bus-time samples — i.e., the phase exhibiting the largest voltage
    variation, hence the most representative for distance-voltage
    analysis.

    Parameters
    ----------
    m : pyomo.environ.ConcreteModel
        Solved BFM-SOCP model with m.V[b, phi, t] populated.
    data : dict
        Network data dictionary (must contain 'buses' and 'branches').
    sets_info : dict
        Sets info (must contain 'hours' and 'bph').
    save_path : str, optional
        Output PNG path. Defaults to 'opf_bfm_voltage_heatmap.png'.
    force_phase : int in {1, 2, 3}, optional
        Override the automatic phase selection.
    """
    try:
        import matplotlib.pyplot as plt
        import matplotlib.colors as mcolors
        import numpy as np
        from collections import deque, defaultdict
    except ImportError:
        print("  [HEATMAP] Install matplotlib first.")
        return

    buses     = data['buses']
    hours     = sets_info['hours']
    bph_set   = set(sets_info['bph'])
    branches  = data.get('branches', {})
    if save_path is None:
        save_path = 'opf_bfm_voltage_heatmap.png'

    def _level_of(b):
        return buses.get(b, {}).get('level', 'mv')

    # ── (1) Build adjacency graph including transformers (MT→BT bridges)
    adj = defaultdict(list)
    for key in branches:
        # branches keys may be (fr, to, ph) or (fr, to)
        if isinstance(key, tuple) and len(key) >= 2:
            fr, to = key[0], key[1]
            if to not in adj[fr]:
                adj[fr].append(to)
            if fr not in adj[to]:
                adj[to].append(fr)

    # Add transformer edges: each transformer connects its MV bus to LV bus
    trafos = data.get('trafos', {}) or data.get('transformers', {})
    for tid, td in (trafos or {}).items():
        mv_b = td.get('mv_bus') or td.get('hv_bus')
        lv_b = td.get('lv_bus')
        if mv_b and lv_b:
            if lv_b not in adj[mv_b]: adj[mv_b].append(lv_b)
            if mv_b not in adj[lv_b]: adj[lv_b].append(mv_b)

    # ── (2) Identify slack bus
    slack = None
    for b, bd in buses.items():
        if bd.get('slack', False) or b == '650' or bd.get('is_slack', False):
            slack = b
            break
    if slack is None:
        # Fallback: pick first MV bus alphabetically
        mv_list = sorted([b for b in buses if _level_of(b) == 'mv'])
        slack = mv_list[0] if mv_list else next(iter(buses))

    # ── (3) BFS distance from slack
    dist = {slack: 0}
    q = deque([slack])
    while q:
        cur = q.popleft()
        for nb in adj.get(cur, []):
            if nb not in dist:
                dist[nb] = dist[cur] + 1
                q.append(nb)
    # Fallback distance for disconnected buses
    max_d = max(dist.values()) if dist else 0
    for b in buses:
        if b not in dist:
            dist[b] = max_d + 1   # put disconnected buses at the bottom

    # ── (4) Pick the most critical phase (largest spatio-temporal range)
    if force_phase in (1, 2, 3):
        crit_phase = force_phase
        sel_reason = f"forced by user (phase {force_phase})"
    else:
        ranges = {}
        for ph in [1, 2, 3]:
            vals = []
            for (b, p) in bph_set:
                if p != ph: continue
                for t in hours:
                    vals.append(try_v(m, b, ph, t))
            if vals:
                ranges[ph] = max(vals) - min(vals)
        if not ranges:
            print("  [HEATMAP] No phase data available.")
            return
        crit_phase = max(ranges, key=lambda k: ranges[k])
        sel_reason = (f"max spatio-temporal range = "
                      f"{ranges[crit_phase]*1000:.1f} mp.u. "
                      f"(phases: {{1:{ranges.get(1,0)*1000:.1f}, "
                      f"2:{ranges.get(2,0)*1000:.1f}, "
                      f"3:{ranges.get(3,0)*1000:.1f}}} mp.u.)")

    # ── (5) Order buses by (level, distance from slack)
    #       MV buses first (top), then LV; within each level, sort by distance.
    buses_with_phase = sorted({b for (b, ph) in bph_set if ph == crit_phase})
    mv_buses = sorted([b for b in buses_with_phase if _level_of(b) == 'mv'],
                      key=lambda b: (dist.get(b, 999), b))
    lv_buses = sorted([b for b in buses_with_phase if _level_of(b) == 'lv'],
                      key=lambda b: (dist.get(b, 999), b))
    ordered_buses = mv_buses + lv_buses
    n_mv = len(mv_buses)
    n_total = len(ordered_buses)

    if n_total == 0:
        print(f"  [HEATMAP] No buses with phase {crit_phase}.")
        return

    # ── (6) Build the V matrix [bus × time] for the selected phase
    V_mat = np.full((n_total, len(hours)), np.nan, dtype=float)
    for i, b in enumerate(ordered_buses):
        for j, t in enumerate(hours):
            V_mat[i, j] = try_v(m, b, crit_phase, t)

    # ── (7) Color scale centered at 1.0 p.u.
    v_lo, v_hi = 0.90, 1.06
    norm = mcolors.TwoSlopeNorm(vmin=v_lo, vcenter=1.0, vmax=v_hi)
    cmap = plt.get_cmap('RdYlGn_r')

    # ── (8) Plot
    height = max(6, n_total * 0.22)
    fig, ax = plt.subplots(figsize=(13, height))
    im = ax.imshow(V_mat, aspect='auto', cmap=cmap, norm=norm,
                   interpolation='nearest', origin='upper')

    # MV/LV separator
    if 0 < n_mv < n_total:
        ax.axhline(n_mv - 0.5, color='black', lw=1.8, ls='-')
        ax.text(-0.5, n_mv - 0.5, ' MV / LV ',
                ha='right', va='center', fontsize=9, fontweight='bold',
                bbox=dict(facecolor='white', edgecolor='black',
                          boxstyle='round,pad=0.3'))

    # X-axis: time labels every 2 h
    n_per_hour = max(1, int(60 / DT_MIN))
    xtk = list(range(0, len(hours), 2 * n_per_hour))
    ax.set_xticks(xtk)
    ax.set_xticklabels([f"{(k * DT_MIN) // 60:02d}h" for k in xtk],
                       fontsize=8, rotation=0)

    # Y-axis: bus labels with topological distance prefix
    ylabels = [f"{b}  (d={dist.get(b, '?')})" for b in ordered_buses]
    ax.set_yticks(range(n_total))
    ax.set_yticklabels(ylabels, fontsize=8)

    ax.set_xlabel('Time of day', fontsize=11)
    ax.set_ylabel('Bus  (ordered by hops from slack; d = topological distance)',
                  fontsize=10)
    ax.set_title(f'Voltage heatmap $V_b^{{{crit_phase},t}}$ — post-OPF\n'
                 f'Phase {crit_phase} (critical) | MV (top) and LV (bottom) '
                 f'× 144 periods of 10 min',
                 fontsize=12, fontweight='bold')

    # Colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Voltage [p.u.]', fontsize=10)
    ref_ticks = [0.90, 0.92, 0.93, 0.95, 1.00, 1.05, 1.06]
    cbar.set_ticks(ref_ticks)
    cbar.ax.tick_params(labelsize=8)

    # Summary box
    v_min_obs = float(np.nanmin(V_mat))
    v_max_obs = float(np.nanmax(V_mat))
    v_mean    = float(np.nanmean(V_mat))
    info_txt = (f'Phase {crit_phase} (criterion: largest range)\n'
                f'V min/max:  [{v_min_obs:.4f}, {v_max_obs:.4f}] p.u.\n'
                f'V mean:      {v_mean:.4f} p.u.\n'
                f'Buses:       {n_total}  ({n_mv} MV + {n_total-n_mv} LV)\n'
                f'Max hops:    {max(dist.get(b,0) for b in ordered_buses)}')
    ax.text(1.05, -0.05, info_txt, transform=ax.transAxes,
            fontsize=8, ha='left', va='top', family='monospace',
            bbox=dict(boxstyle='round,pad=0.4',
                      facecolor='white', edgecolor='gray', alpha=0.9))

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"  [HEATMAP] Saved: {save_path}")
    print(f"  [HEATMAP] Phase selection: {sel_reason}")
    plt.show()
    return fig


def plot_voltage_profile_and_vv(m, data, sets_info):
    """Perfil espacial MT + curva V-Q ajustada por regressão."""
    try:
        import matplotlib.pyplot as plt
        import numpy as np
    except ImportError:
        print("  pip install matplotlib"); return
    from collections import deque, defaultdict

    buses   = data['buses']
    branches= data['branches']
    pv      = data['pv']
    svr     = data['svr']
    slack   = data['slack']
    hours   = sets_info['hours']
    irr     = sets_info['irr']
    load    = sets_info['load']
    bph     = sets_info['bph']
    pvph    = sets_info['pvph']
    svr_phs = sets_info['svr_phs']
    t_vv    = sets_info['t_vv']
    step_   = sets_info['tap_step']
    tap_min_= sets_info['tap_min']

    bph_set=set(bph)
    adj_mt=defaultdict(list)
    for (fr,to,ph) in branches:
        if buses.get(fr,{}).get('level')=='mv' and buses.get(to,{}).get('level')=='mv':
            if to not in adj_mt[fr]: adj_mt[fr].append(to)
    for ph,d in svr.items():
        if d['lv_bus'] not in adj_mt[d['mv_bus']]: adj_mt[d['mv_bus']].append(d['lv_bus'])

    visited, order_mt = set(), []
    q = deque([slack]); visited.add(slack)
    while q:
        b = q.popleft()
        if buses.get(b, {}).get('level') == 'mv': order_mt.append(b)
        for nb in adj_mt.get(b, []):
            if nb not in visited: visited.add(nb); q.append(nb)
    mt_buses = [b for b in order_mt if any((b, ph) in bph_set for ph in [1, 2, 3])]

    V_pre,_,_,_,_,_=compute_warm_start(data,hours,irr,load)
    loads_mt_=data.get('loads_mt',{})
    # Mantém coerência com a lógica de V_ref da F.O.: barras com carga
    # MT modelo 1 (P-const) têm V_ref = 0,95; demais têm V_ref = 1,00.
    _m1_b={b for (b,ph),ld in loads_mt_.items() if ld.get('model',1) == 1}

    max_dev=0.; crit_bus=mt_buses[0] if mt_buses else slack; crit_ph=1; crit_t=0
    for b in mt_buses:
        TR=MV_VREF_CONST_P_PU if b in _m1_b else MV_VREF_OTHER_PU
        for ph in [1,2,3]:
            if (b,ph) not in bph_set: continue
            for t in hours:
                dev=abs(V_pre.get((b,ph,t),1.0)-TR)
                if dev>max_dev: max_dev=dev; crit_bus=b; crit_ph=ph; crit_t=t

    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6))
    fig.suptitle("BFM-SOCP — MV profile and fitted V-Q curve",fontsize=13,fontweight='bold')
    colors_ph={1:'#7F77DD',2:'#E0803A',3:'#1D9E75'}
    x_pos=list(range(len(mt_buses)))

    for ph in svr_phs:
        vp=[V_pre.get((b,ph,crit_t),1.) for b in mt_buses]
        vo=[try_v(m,b,ph,crit_t) for b in mt_buses]
        ax1.plot(x_pos,vp,'--',color=colors_ph[ph],lw=1.5,alpha=.45)
        ax1.plot(x_pos,vo,'-o',color=colors_ph[ph],lw=2,ms=5,label=f'Phase {ph}')
    ax1.axhspan(V_ADEQ_MT_LO,V_ADEQ_MT_HI,alpha=.07,color='#1D9E75',label='Adequada MT')
    ax1.axhline(V_ADEQ_MT_LO,color='#1D9E75',lw=.9,ls='--',alpha=.7)
    ax1.axhline(1.0,color='gray',lw=.7,ls=':')
    ax1.set_xticks(x_pos); ax1.set_xticklabels(mt_buses,rotation=45,ha='right',fontsize=7)
    t_hhmm=f"{crit_t*DT_MIN//60:02d}:{crit_t*DT_MIN%60:02d}"
    ax1.set_title(f'MV profile — t={crit_t} ({t_hhmm}) | critical bus: {crit_bus} ph{crit_ph}')
    ax1.legend(fontsize=8); ax1.grid(alpha=.3)

    t_vv_h=[t for t in hours if t_vv[0]<=t<=t_vv[1]]
    _tan_fp=math.sqrt(1-FP_MIN_VV**2)/FP_MIN_VV
    best_bph=(pvph[0] if pvph else None)
    if pvph:
        t_peak=max(hours,key=lambda t:irr[t])
        best_bph=max(pvph,key=lambda bp:abs(pyo.value(m.Qpv[bp[0],bp[1],t_peak])))
    if best_bph:
        b_vv,ph_vv=best_bph
        mask=[irr[t]>0.05 for t in t_vv_h]
        V_s=np.array([try_v(m,b_vv,ph_vv,t) for i,t in enumerate(t_vv_h) if mask[i]])
        Q_s=np.array([pyo.value(m.Qpv[b_vv,ph_vv,t])*SBASE for i,t in enumerate(t_vv_h) if mask[i]])
        irr_s=np.array([irr[t] for i,t in enumerate(t_vv_h) if mask[i]])
        if len(V_s)>2:
            dead_mask=np.abs(Q_s)<0.05*SBASE
            V2f=float(np.percentile(V_s[dead_mask],10)) if dead_mask.sum()>1 else 0.97
            V3f=float(np.percentile(V_s[dead_mask],90)) if dead_mask.sum()>1 else 1.03
            V1f=max(0.88,float(np.percentile(V_s,5)))
            V4f=min(1.12,float(np.percentile(V_s,95)))
            Qmx=float(np.percentile(np.abs(Q_s),95)) if len(Q_s)>0 else 10.
            v_r=np.linspace(min(0.88,V1f-.02),max(1.12,V4f+.02),300)
            def _vvc(v):
                if v<=V1f: return Qmx
                if v<=V2f: return Qmx*(V2f-v)/max(V2f-V1f,1e-6)
                if v<=V3f: return 0.
                if v<=V4f: return -Qmx*(v-V3f)/max(V4f-V3f,1e-6)
                return -Qmx
            sc=ax2.scatter(V_s,Q_s,c=irr_s,cmap='YlOrRd',s=30,alpha=.7,label='OPF points')
            plt.colorbar(sc,ax=ax2,fraction=.03,label='Irrad.')
            ax2.plot(v_r,[_vvc(v) for v in v_r],'-',color='#185FA5',lw=2.5,label='Fitted curve')
            for vbp,lbl in [(V1f,'V1'),(V2f,'V2'),(V3f,'V3'),(V4f,'V4')]:
                ax2.axvline(vbp,color='#BA7517',lw=.9,ls='--',alpha=.6)
                ax2.text(vbp,ax2.get_ylim()[0] if ax2.get_ylim()[0]<0 else -Qmx*.1,
                         lbl,ha='center',fontsize=8,color='#BA7517')
            ax2.axhline(0,color='gray',lw=.7,ls=':')
            ax2.set_title(f'Fitted V-Q curve\n({b_vv}, phase {ph_vv})')
            ax2.set_xlabel('Voltage [p.u.]'); ax2.set_ylabel('Qpv [kVAr/phase]')
            ax2.legend(fontsize=8); ax2.grid(alpha=.3)

    plt.tight_layout(pad=2.5)
    plt.savefig('opf_bfm_voltage_profile_vv.png',dpi=150,bbox_inches='tight')
    print("  Salvo: opf_bfm_voltage_profile_vv.png")
    plt.show()

# ═════════════════════════════════════════════════════════════════════════
# SIMULAÇÃO DO CONTROLE LOCAL DO REGULADOR (BASELINE PRÉ-OPF)
# ═════════════════════════════════════════════════════════════════════════
# O controle local de tap em reguladores Cooper/SEL opera com lógica de
# banda morta + delay (Line Drop Compensation simplificado), sem nenhuma
# otimização. Esta função reproduz esse comportamento para gerar a contagem
# de operações pré-OPF compatível com a simulação do OpenDSS.
#
# Lógica do controle:
#   Para cada período t:
#     1. Medir Vreg e a corrente, reduzidas pelas relações PT/CT
#     2. Calcular |Vrelay| = |Vreg - (R+jX)·Icomp| em volts do relé
#     3. Se |Vrelay| > Vreg_set + band/2: tap_pos -= 1 (após delay)
#     4. Se |Vrelay| < Vreg_set - band/2: tap_pos += 1 (após delay)
#     5. Caso contrário: manter posição
#
# Parâmetros típicos IEEE Cooper:
#   Vreg  = 122 V na escala nominal de 120 V
#   band  = 2 V (banda morta total)
#   delay = 15 min; com Δt=10 min, ceil(15/10)=2 períodos
# =========================================================================

VREG_SVR_V   = 122.0
BAND_SVR_V   = 2.0
TAP_DELAY_MIN = 15.0
DELAY_PER     = max(1, math.ceil(TAP_DELAY_MIN / DT_MIN))


In [ ]:
#@title Internal model implementation 11/12 { display-mode: "form" }
def simulate_svr_local_control(
        data, hours, irr_d, load_d, return_voltage=False,
        fixed_inverter_pf=None, pv_eff_d=None, load_mt_d=None,
        initial_relay_state=None, return_final_state=False):
    """Simula o controle local do regulador período a período.

    Para cada período, executa uma varredura direta da rede com o tap atual,
    mede V no secundário do regulador e aplica a lógica de banda morta para
    decidir a próxima posição.

    Retorna:
        tap_traj: {(ph, t): pos_inteira} — trajetória discreta do tap
        ops_per_phase: {ph: int} — número de operações por fase
        V_relay_traj: {(ph, t): |Vrelay| em volts — tensão do relé LDC
        V_traj: {(bus, ph, t): V em p.u.} — somente se return_voltage=True
    """
    from collections import deque, defaultdict
    buses    = data['buses']
    loads    = data['loads']
    pv       = data['pv']
    branches = data['branches']
    svr      = data['svr']
    slack    = data['slack']

    # Construir adjacências (mesma lógica do warm-start)
    adj = defaultdict(list)
    for (fr, to, ph) in branches:
        adj[(fr, ph)].append((to, ph, (fr, to, ph)))
    for ph, d in svr.items():
        adj[(d['mv_bus'], ph)].append((d['lv_bus'], ph, ('SVR', ph)))

    def bfs(root, rphs):
        order, visited = [], set()
        q = deque([(root, ph) for ph in rphs])
        for item in q: visited.add(item)
        while q:
            node = q.popleft(); order.append(node)
            for (to, tph, br) in adj.get(node, []):
                if (to, tph) not in visited:
                    visited.add((to, tph)); q.append((to, tph))
        return order

    order  = bfs(slack, buses[slack]['phases'])
    parent = {}
    for (b, ph) in order:
        for (to, tph, br) in adj.get((b, ph), []):
            if (to, tph) not in parent:
                parent[(to, tph)] = ((b, ph), br)

    svr_phs   = sorted(svr.keys())
    # Perfis são entradas explícitas; isso evita que uma campanha multiday use
    # acidentalmente os globais do dia-fonte em todos os dias de teste.
    _load_bt = load_d
    _load_mt = load_mt_d if load_mt_d is not None else LOAD_PROFILE_MT
    _pv_eff = pv_eff_d if pv_eff_d is not None else irr_d

    # Estado: posição atual, direção pendente, contador e ação na fronteira.
    relay_state = _normalise_relay_state(data, initial_relay_state)
    pos_cur = {ph: relay_state[ph]['tap_position'] for ph in svr_phs}
    delay_ctr = {ph: relay_state[ph]['delay_counter'] for ph in svr_phs}
    pending_dir = {ph: relay_state[ph]['pending_direction'] for ph in svr_phs}
    queued_delta = {ph: relay_state[ph]['queued_delta'] for ph in svr_phs}
    ops       = {ph: 0 for ph in svr_phs}
    tap_traj  = {}
    V_relay_traj = {}
    V_traj = {}

    for t in hours:
        for ph in svr_phs:
            if queued_delta[ph]:
                pos_cur[ph] = max(0, min(
                    svr[ph]['n_tap']-1,
                    pos_cur[ph]+queued_delta[ph]))
                ops[ph] += abs(queued_delta[ph])
                queued_delta[ph] = 0
        # Posição efetivamente aplicada durante o período t. Qualquer decisão
        # tomada ao fim deste período só entra em serviço no período seguinte.
        for ph in svr_phs:
            tap_traj[(ph, t)] = pos_cur[ph]

        # ── 1. Varredura direta com o tap atual ────────────────────────
        # Calcular potências líquidas em cada barra
        P_net = {}; Q_net = {}
        for (b, ph) in order:
            if b in _MT_LOAD_PROFILE_MAP:
                _profile_name = _MT_LOAD_PROFILE_MAP[b]
                _aw = (_load_mt[t] if _profile_name == 'industrial'
                       else _LOAD_BY_TYPE[_profile_name][t])
            else:
                lvl = buses.get(b, {}).get('level', 'mv')
                _aw = _load_mt[t] if lvl == 'mv' else _load_bt[t]
            Pd = loads.get((b, ph), {'P_pu': 0})['P_pu'] * _aw
            Qd = loads.get((b, ph), {'Q_pu': 0})['Q_pu'] * _aw
            _pv_row = pv.get((b, ph), {'P_rated_pu': 0, 'S_nom_pu': 0})
            Ppv = _pv_row.get('P_rated_pu', 0.0) * _pv_eff[t]
            Qpv = 0.0
            if fixed_inverter_pf is not None and Ppv > 0.0:
                _fp = float(fixed_inverter_pf)
                if not 0.0 < _fp <= 1.0:
                    raise ValueError('fixed_inverter_pf deve estar em (0,1].')
                _k = math.sqrt(max(1.0-_fp**2, 0.0))/_fp
                _snom = float(_pv_row.get('S_nom_pu', 0.0))
                _qcap = min(
                    math.sqrt(max(_snom**2-Ppv**2, 0.0)),
                    QPV_MAX_FRAC_S*_snom,
                    math.sqrt(1.0-FP_MIN_VV**2)/FP_MIN_VV*Ppv)
                # Convenção do modelo: Qpv>0 representa suporte capacitivo.
                Qpv = min(_k*Ppv, _qcap)
            P_net[(b, ph)] = Pd - Ppv
            Q_net[(b, ph)] = Qd - Qpv

        # Backward sweep: somar fluxos
        P_br = {}; Q_br = {}
        for (b, ph) in reversed(order):
            pf = P_net[(b, ph)]; qf = Q_net[(b, ph)]
            for (to, tph, br) in adj.get((b, ph), []):
                pf += P_br.get((to, ph), 0); qf += Q_br.get((to, ph), 0)
            P_br[(b, ph)] = pf; Q_br[(b, ph)] = qf

        # Forward sweep: calcular tensões com o tap_cur atual
        V = {}
        for (b, ph) in order:
            if b == slack:
                V[(b, ph)] = 1.0; continue
            if (b, ph) not in parent:
                V[(b, ph)] = 0.97; continue
            (up, uph), br_key = parent[(b, ph)]
            Vup = V.get((up, uph), 1.0)
            if br_key == ('SVR', ph):
                # Tap atual aplicado
                tau = (svr[ph]['tap_min'] +
                       pos_cur[ph]*svr[ph]['tap_step'])
                V[(b, ph)] = tau * Vup
            else:
                r = branches[br_key]['r_pu']
                x = branches[br_key]['x_pu']
                Pf = P_br.get((b, ph), 0); Qf = Q_br.get((b, ph), 0)
                z2 = r**2 + x**2
                l_sq = (Pf**2 + Qf**2) / max(Vup**2, 0.64)
                V[(b, ph)] = max(Vup - r*Pf - x*Qf + 0.5*z2*l_sq, 0.80)

        for (b, ph), v_pu in V.items():
            V_traj[(b, ph, t)] = v_pu

        # ── 2. Controle por relé: banda morta + delay + LDC ─────────────
        # As planilhas fornecem R e X em volts já referidos a Vreg. Logo:
        #   Vrelay = Vreg_relay - (R+jX)·(Ilinha/CTprim)
        # sem conversão para p.u. de impedância. O módulo complexo preserva
        # corretamente os sinais de P/Q, inclusive sob fluxo reverso.
        for ph in svr_phs:
            lv_bus = svr[ph]['lv_bus']
            V_lv   = V.get((lv_bus, ph), 1.0)

            P_svr = P_br.get((lv_bus, ph), 0.0)
            Q_svr = Q_br.get((lv_bus, ph), 0.0)
            V_for_I = max(V_lv, 0.5)   # evitar divisão por valores baixos
            Ibase_A = SBASE / VMT
            I_line_A = complex(P_svr, -Q_svr) * Ibase_A / V_for_I
            I_comp = I_line_A / max(svr[ph].get('ctprim_A', 700.0), 1.0e-9)

            Vreg_relay = V_lv * svr[ph].get('vbase_relay_V', 120.0)
            Z_ldc_V = complex(svr[ph].get('R_ldc_V', 3.0),
                              svr[ph].get('X_ldc_V', 9.0))
            Vrelay = complex(Vreg_relay, 0.0) - Z_ldc_V * I_comp
            V_meas = abs(Vrelay)
            V_relay_traj[(ph, t)] = V_meas

            # Referência e banda morta em volts do relé.
            V_ref_ph = svr[ph].get('vreg_V', VREG_SVR_V)
            BW_ph    = svr[ph].get('band_V', BAND_SVR_V)

            # Decisão de tap sobre |Vrelay|.
            err = V_meas - V_ref_ph
            need_change = False
            delta = 0
            if err > BW_ph/2 and pos_cur[ph] > 0:
                need_change = True; delta = -1   # V alta → descer tap
            elif err < -BW_ph/2 and pos_cur[ph] < svr[ph]['n_tap']-1:
                need_change = True; delta = +1   # V baixa → subir tap

            if need_change:
                # A direção precisa permanecer a mesma por todo o delay. Uma
                # inversão raise↔lower reinicia a temporização do relé.
                if pending_dir[ph] == delta:
                    delay_ctr[ph] += 1
                else:
                    pending_dir[ph] = delta
                    delay_ctr[ph] = 1

                delay_ph = svr[ph].get('delay_periods', DELAY_PER)
                if delay_ctr[ph] >= delay_ph:
                    if t == hours[-1]:
                        queued_delta[ph] = delta
                    else:
                        pos_cur[ph] += delta
                        pos_cur[ph] = max(
                            0, min(svr[ph]['n_tap']-1, pos_cur[ph]))
                        ops[ph] += 1
                    delay_ctr[ph] = 0
                    pending_dir[ph] = 0
            else:
                delay_ctr[ph] = 0
                pending_dir[ph] = 0

            relay_state[ph] = {
                'tap_position': pos_cur[ph],
                'pending_direction': pending_dir[ph],
                'delay_counter': delay_ctr[ph],
                'queued_delta': queued_delta[ph],
                'last_relay_voltage_v': V_meas,
            }

    if return_voltage and return_final_state:
        return tap_traj, ops, V_relay_traj, V_traj, relay_state
    if return_voltage:
        return tap_traj, ops, V_relay_traj, V_traj
    if return_final_state:
        return tap_traj, ops, V_relay_traj, relay_state
    return tap_traj, ops, V_relay_traj



# =============================================================================
# API PÚBLICA V4.9 — notebook, auditoria e experimentos comparativos
# =============================================================================

def model_size_statistics(model):
    """Retorna dimensões estruturais do modelo Pyomo sem chamar o solver."""
    variables = list(model.component_data_objects(pyo.Var, active=True))
    constraints = list(model.component_data_objects(pyo.Constraint, active=True))
    return {
        'n_variables': len(variables),
        'n_constraints': len(constraints),
        'n_binary_variables': sum(v.is_binary() for v in variables),
        'n_integer_variables': sum(v.is_integer() and not v.is_binary()
                                   for v in variables),
        'n_continuous_variables': sum(v.is_continuous() for v in variables),
        'n_active_objectives': sum(
            1 for _ in model.component_data_objects(pyo.Objective, active=True)),
    }


def solver_bound_evidence(result):
    """Extrai limites e gap relativo de um resultado Pyomo, quando disponíveis."""
    lower = upper = gap = None
    try:
        problem = result.problem[0]
        lower_raw = getattr(problem, 'lower_bound', None)
        upper_raw = getattr(problem, 'upper_bound', None)
        lower = (float(lower_raw) if lower_raw is not None else None)
        upper = (float(upper_raw) if upper_raw is not None else None)
        if (lower is not None and upper is not None and
                math.isfinite(lower) and math.isfinite(upper)):
            gap = abs(upper-lower)/max(1.0, abs(upper))
    except Exception:
        pass
    return {'lower_bound': lower, 'upper_bound': upper,
            'relative_gap': gap}


def set_active_weather_profile(
        irradiance_pu, ambient_temperature_c, profile_label='external'):
    """Atualiza explicitamente o perfil meteorológico usado por ``build_opf``.

    Esta função substitui a dependência implícita do dia selecionado durante o
    import. Ela é chamada antes de construir cada modelo de um dia de teste.
    Temperatura de célula continua sendo estimada pela equação NOCT.
    """
    global IRRAD, IRR_PROFILE, TEMP_AMB, PV_T_CELL, PV_EFF_FACTOR
    global TEMP_AMB_DATA_OK, PV_TEMP_DATA_OK, PROFILE_SOURCE
    irradiance = [float(value) for value in irradiance_pu]
    temperature = [float(value) for value in ambient_temperature_c]
    if len(irradiance) != N_PERIODS or len(temperature) != N_PERIODS:
        raise ValueError(
            f'O perfil deve conter {N_PERIODS} amostras de {DT_MIN} min.')
    if not all(math.isfinite(value) for value in irradiance+temperature):
        raise ValueError('O perfil meteorológico contém valor não finito.')
    IRRAD = irradiance
    IRR_PROFILE = IRRAD
    TEMP_AMB = temperature
    TEMP_AMB_DATA_OK = (
        min(TEMP_AMB) >= -20.0 and max(TEMP_AMB) <= 60.0)
    PV_TEMP_DATA_OK = TEMP_AMB_DATA_OK
    PV_T_CELL, PV_EFF_FACTOR = [], []
    for irradiance_value, temperature_value in zip(IRRAD, TEMP_AMB):
        cell_temperature, derate = pv_temperature_derate(
            irradiance_value, temperature_value)
        PV_T_CELL.append(cell_temperature)
        PV_EFF_FACTOR.append(irradiance_value*derate)
    PROFILE_SOURCE = str(profile_label)
    return {
        'profile_label': PROFILE_SOURCE,
        'irradiance_min_pu': min(IRRAD),
        'irradiance_max_pu': max(IRRAD),
        'ambient_temperature_min_c': min(TEMP_AMB),
        'ambient_temperature_max_c': max(TEMP_AMB),
        'cell_temperature_min_c': min(PV_T_CELL),
        'cell_temperature_max_c': max(PV_T_CELL),
        'temperature_data_ok': PV_TEMP_DATA_OK,
    }


def make_cplex_solver(executable=None, time_limit_seconds=1200, threads=4):
    """Cria o solver com as mesmas opções usadas em todas as etapas."""
    from pyomo.environ import SolverFactory
    solver = pyo.SolverFactory('cplex', executable=executable or CPLEX_EXE)
    solver.options['qpmethod'] = 6
    solver.options['timelimit'] = int(time_limit_seconds)
    solver.options['threads'] = int(threads)
    solver.options['barrier_convergetol'] = 1.0e-6
    return solver


def relax_identification_taps(model, sets_info):
    """Etapa E1: relaxa apenas a integralidade dos taps e eventos."""
    for ph in sets_info['svr_phs']:
        for t in sets_info['hours']:
            model.tap_pos[ph, t].domain = pyo.NonNegativeReals
            model.u_tap[ph, t].domain = pyo.NonNegativeReals
            model.u_tap[ph, t].setub(1.0)
    return model


def continuous_tap_schedule(model, sets_info):
    """Extrai a trajetória contínua resolvida na etapa E1."""
    return {
        (ph, t): float(pyo.value(model.tap_pos[ph, t]))
        for ph in sets_info['svr_phs'] for t in sets_info['hours']}


def fix_identification_tap_schedule(model, data, sets_info, schedule):
    """Etapa E2: fixa uma projeção discreta sem trocar a equação afim.

    A relação exata pertence aos replays finais. Esta rotina preserva a
    formulação de identificação e apenas remove as variáveis de decisão de
    tap depois da projeção mecânica.
    """
    hours = list(sets_info['hours'])
    audit = _audit_programmed_tap_schedule(schedule, data, hours)
    if not audit['ok']:
        raise ValueError('Trajetória E2 mecanicamente inválida: ' +
                         '; '.join(audit['reasons']))
    for ph in sets_info['svr_phs']:
        sd = data['svr'][ph]
        previous = int(round((sd['tap_init']-sd['tap_min'])/sd['tap_step']))
        for t in hours:
            position = int(round(schedule[(ph, t)]))
            model.tap_pos[ph, t].fix(position)
            model.u_tap[ph, t].fix(0.0)
            model.tap_step_abs[ph, t].set_value(abs(position-previous))
            previous = position
        model.tap_terminal_abs[ph].set_value(abs(
            previous-int(round(
                (sd['tap_init']-sd['tap_min'])/sd['tap_step']))))
    for name in ('c_tap_up', 'c_tap_dn'):
        component = getattr(model, name, None)
        if component is not None:
            component.deactivate()
    return audit


class RunEvidenceRecorder:
    """Coleta evidência computacional e a persiste em JSON e CSV.

    O pico de memória inclui o processo Python e, quando ``psutil`` estiver
    disponível, seus filhos (incluindo CPLEX). Ausência de uma medida é salva
    como ``None``; nunca como zero.
    """

    def __init__(self, run_id, output_dir):
        self.run_id = str(run_id)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.rows = []

    @staticmethod
    def _rss_tree_mb():
        try:
            import psutil
            process = psutil.Process(os.getpid())
            rss = process.memory_info().rss
            for child in process.children(recursive=True):
                try:
                    rss += child.memory_info().rss
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    pass
            return rss/(1024.0**2)
        except Exception:
            return None

    def record(self, *, stage, day=None, scenario=None, model=None,
               started_at=None, solver_status=None,
               termination_condition=None, solver_wall_seconds=None,
               model_build_seconds=None, sca_iterations=None,
               lower_bound=None, upper_bound=None, relative_gap=None,
               max_constraint_violation=None, exactness=None,
               max_svr_exact_residual_pu=None, candidates_planned=None,
               candidates_evaluated=None, candidates_accepted=None,
               peak_process_tree_rss_mb=None, extra=None):
        timestamp_epoch = time.time()
        monotonic_now = time.perf_counter()
        row = {
            'run_id': self.run_id,
            'stage': stage,
            'day': day,
            'scenario': scenario,
            'timestamp_utc_epoch': timestamp_epoch,
            'total_seconds': (
                monotonic_now-started_at if started_at is not None else None),
            'model_build_seconds': model_build_seconds,
            'solver_wall_seconds': solver_wall_seconds,
            'peak_process_tree_rss_mb': (
                peak_process_tree_rss_mb if peak_process_tree_rss_mb is not None
                else self._rss_tree_mb()),
            'solver_status': solver_status,
            'termination_condition': termination_condition,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'relative_gap': relative_gap,
            'max_constraint_violation': max_constraint_violation,
            'sca_iterations': sca_iterations,
            'bfm_aggregate_gap_pct': (
                exactness.get('aggregate_gap_pct') if exactness else None),
            'bfm_max_cone_violation_pu': (
                exactness.get('max_abs_gap') if exactness else None),
            'bfm_max_vdrop_residual_pu2': (
                exactness.get('max_vdrop_residual_pu2') if exactness else None),
            'max_svr_exact_residual_pu': max_svr_exact_residual_pu,
            'candidates_planned': candidates_planned,
            'candidates_evaluated': candidates_evaluated,
            'candidates_accepted': candidates_accepted,
        }
        if model is not None:
            row.update(model_size_statistics(model))
        if extra:
            row.update(dict(extra))
        self.rows.append(row)
        return row

    def save(self):
        import json
        import csv
        json_path = self.output_dir/'computational_evidence.json'
        csv_path = self.output_dir/'computational_evidence.csv'
        tmp = json_path.with_suffix('.json.tmp')
        tmp.write_text(json.dumps(
            self.rows, ensure_ascii=False, indent=2, default=str),
            encoding='utf-8')
        tmp.replace(json_path)
        fields = sorted({key for row in self.rows for key in row})
        with csv_path.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader()
            writer.writerows(self.rows)
        return {'json': str(json_path), 'csv': str(csv_path)}


class PeakMemoryMonitor:
    """Amostra o RSS do processo Python e de seus filhos durante uma etapa."""

    def __init__(self, interval_seconds=0.25):
        self.interval_seconds = float(interval_seconds)
        self.peak_mb = None
        self._stop = None
        self._thread = None

    def __enter__(self):
        import threading
        self._stop = threading.Event()

        def _sample():
            while not self._stop.is_set():
                value = RunEvidenceRecorder._rss_tree_mb()
                if value is not None:
                    self.peak_mb = (value if self.peak_mb is None else
                                    max(self.peak_mb, value))
                self._stop.wait(self.interval_seconds)

        self._thread = threading.Thread(target=_sample, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self._stop.set()
        self._thread.join(timeout=max(1.0, 2.0*self.interval_seconds))
        value = RunEvidenceRecorder._rss_tree_mb()
        if value is not None:
            self.peak_mb = (value if self.peak_mb is None else
                            max(self.peak_mb, value))
        return False


def frozen_parameter_snapshot(data=None, sets_info=None, normalizers=None,
                              kmeans_seed=None, source_commit=None):
    """Dicionário serializável com todos os parâmetros relevantes do run."""
    svr_parameters = {}
    if data is not None:
        for ph, row in data.get('svr', {}).items():
            svr_parameters[str(ph)] = {
                key: row.get(key) for key in (
                    'mv_bus', 'lv_bus', 'tap_min', 'tap_max', 'tap_step',
                    'n_tap', 'tap_init', 'delay_min', 'delay_periods',
                    'vreg_V', 'band_V', 'R_ldc_V', 'X_ldc_V',
                    'ctprim_A', 'vbase_relay_V')}
    return {
        'schema': 'acopf-frozen-parameters-v4.9',
        'version': VERSION,
        'runtime_signature': RUNTIME_SIGNATURE,
        'time_grid': {'dt_minutes': DT_MIN, 'n_periods': N_PERIODS},
        'bases': {'sbase_kva_per_phase': SBASE, 'vmt_kv_ln': VMT,
                  'vbt_kv_ln': VBT, 'zmt_ohm': ZMT, 'zbt_ohm': ZBT},
        'objective_weights': {
            'tap': OP_WEIGHT_TAP,
            'voltage_deviation': OP_WEIGHT_VOLTAGE_DEVIATION,
            'unbalance_proxy': OP_WEIGHT_UNBALANCE,
            'core_loss': OP_WEIGHT_CORE,
        },
        'objective_safeguard_weights': {
            'voltage_margin': OBJ_WEIGHT_VOLTAGE_MARGIN,
            'joule_loss': OBJ_WEIGHT_JOULE_SAFEGUARD,
            'reactive_use': OBJ_WEIGHT_Q_USE_SAFEGUARD,
            'reactive_ramp': OBJ_WEIGHT_Q_RAMP_SAFEGUARD,
            'curtailment': OBJ_WEIGHT_CURTAILMENT,
            'ramp_day': W_RAMP_DIA,
            'ramp_solar_peak': W_RAMP_PICO,
            'solar_peak_threshold_pu': IRR_PICO,
        },
        'normalizers': dict(normalizers or {}),
        'voltage_and_quality_limits': {
            'mv_adequate_pu': [V_ADEQ_MT_LO, V_ADEQ_MT_HI],
            'lv_adequate_pu': [V_ADEQ_BT_LO, V_ADEQ_BT_HI],
            'mv_precarious_lower_pu': V_PREC_MT_LO,
            'lv_precarious_lower_pu': V_PREC_BT_LO,
            'fd95_mv_percent': FD_MT_MAX,
            'fd95_lv_percent': FD_BT_MAX,
            'formal_regulatory_window_periods':
                FORMAL_REGULATORY_WINDOW_PERIODS,
            'daily_drp_screen_percent': DAILY_SCREEN_DRP_MAX_PCT,
            'daily_drc_screen_percent': DAILY_SCREEN_DRC_MAX_PCT,
            'validation_numeric_bounds_pu': [
                VALIDATION_V_MIN, VALIDATION_V_MAX],
            'security_margin_pu': V_SECURITY_MARGIN,
        },
        'pv_temperature_model': {
            'input_kind': PV_TEMP_INPUT_KIND,
            'noct_celsius': NOCT,
            'power_temperature_coefficient_per_celsius': GAMMA_PV,
            'stc_cell_temperature_celsius': 25.0,
            'irradiance_input_unit': 'per_unit_of_1000_W_per_m2',
        },
        'inverter': {
            'fp_min': FP_MIN_VV,
            'preferred_fixed_pf': PREFERRED_FIXED_POWER_FACTOR,
            'fixed_pf_candidates': list(FIXED_POWER_FACTOR_CANDIDATES),
            'q_max_fraction_snom': QPV_MAX_FRAC_S,
            'q_step_fraction_snom_per_period': Q_COMMAND_MAX_STEP_FRAC_S,
            'night_var_enabled': ENABLE_NIGHT_VAR,
            'individual_vv_curves': USE_INDIVIDUAL_FITTED_VV_CURVES,
            'ieee1547_reference_curve': {
                'V1': IEEE1547_V1, 'V2': IEEE1547_V2,
                'V3': IEEE1547_V3, 'V4': IEEE1547_V4,
                'q_fraction_snom': IEEE1547_Q_FRAC},
            'fixed_pf_equivalence_rmse_fraction_snom':
                FP_EQUIVALENCE_RMSE_TOL_FRAC_S,
        },
        'svr': svr_parameters,
        'solver': {'executable': CPLEX_EXE, 'threads': 4,
                   'barrier_convergetol': 1.0e-6},
        'tolerances': {
            'sca_max_iterations': SCA_MAX_ITER,
            'sca_voltage_pu': SCA_V_TOL,
            'sca_damping': SCA_DAMPING,
            'settings_voltage_pu': SETTINGS_V_TOL,
            'settings_q_pu': SETTINGS_Q_TOL,
            'settings_max_iterations': SETTINGS_MAX_ITER,
            'svr_exact_residual_pu': SVR_EXACT_RESIDUAL_TOL,
            'q_slew': Q_COMMAND_SLEW_TOL,
            'bfm_max_aggregate_gap_percent': BFM_MAX_AGG_GAP_PCT,
            'bfm_max_physical_violation_pu': BFM_MAX_PHYSICAL_VIOL_PU,
            'bfm_max_voltage_drop_residual_pu2': BFM_MAX_VDROP_RESID_PU2,
        },
        'classifier': {
            'actions': list(RL_ACTIONS),
            'random_seed': RL_RANDOM_SEED,
            'episodes': RL_TRAINING_EPISODES,
            'learning_rate': RL_LEARNING_RATE,
            'epsilon_start': RL_EPSILON_START,
            'epsilon_end': RL_EPSILON_END,
            'gamma': 0.0,
            'complexity_penalty': dict(RL_COMPLEXITY_PENALTY),
            'static_bidirectional_penalty':
                RL_STATIC_BIDIRECTIONAL_PENALTY,
            'static_clipping_penalty': RL_STATIC_CLIPPING_PENALTY,
            'unity_support_penalty': RL_UNITY_SUPPORT_PENALTY,
            'distance_support_penalty': RL_DISTANCE_SUPPORT_PENALTY,
            'materiality': {
                'unity_nrmse_regret': C_MAX_NRMSE_REGRET,
                'minimum_q_rms_fraction_snom':
                    C_MIN_Q_RMS_FRAC_S_FOR_REACTIVE,
                'minimum_reactive_support_priority':
                    C_MIN_REACTIVE_SUPPORT_PRIORITY,
            },
        },
        'candidate_acceptance': {
            'minimum_tap_saving_steps': OP_MIN_TAP_SAVING_STEPS,
            'minimum_relative_score_improvement':
                OP_MIN_REL_SCORE_IMPROVEMENT,
            'maximum_total_loss_increase_fraction':
                OP_MAX_TOTAL_LOSS_INCREASE_FRAC,
            'maximum_voltage_deviation_increase_fraction':
                PARETO_MAX_VOLTAGE_DEVIATION_INCREASE_FRAC,
            'lexicographic_order': [
                'all_guards', 'tap_operations', 'objective',
                'distance_to_pf_0.96', 'stable_candidate_id'],
        },
        'representative_day_upstream': {
            'kmeans_used': kmeans_seed is not None,
            'kmeans_seed': kmeans_seed,
            'source_commit': source_commit,
        },
        'equation_modes': {
            'identification_svr': 'affine_A1',
            'final_replay_svr': 'exact_fixed_tap',
        },
    }


def export_frozen_parameters(path, **kwargs):
    """Salva parâmetros por escrita atômica."""
    import json
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix+'.tmp')
    tmp.write_text(json.dumps(
        frozen_parameter_snapshot(**kwargs), ensure_ascii=False,
        indent=2, default=str), encoding='utf-8')
    tmp.replace(target)
    return str(target)


def build_s4_all_individual_vv(inv_data, device_members, data):
    """Materializa S4: curva individual para todo inversor físico, sem sparse."""
    pvph = sorted(inv_data)
    curves = []
    for _, members in sorted(device_members.items(), key=lambda item: str(item[0])):
        members = tuple(sorted(members))
        v_pool = np.concatenate([np.asarray(inv_data[bp]['V_arr']) for bp in members])
        p_pool = np.concatenate([np.asarray(inv_data[bp]['P_arr']) for bp in members])
        q_pool = np.concatenate([np.asarray(inv_data[bp]['Q_arr']) for bp in members])
        snom_common = min(data['pv'][bp]['S_nom_pu'] for bp in members)
        common = _fit_individual_vv_action(
            v_pool, p_pool, q_pool, snom_common)
        for bp in members:
            row = dict(common)
            row.update({'bus': bp[0], 'ph': bp[1],
                        'scenario': 'S4_all_individual_vv'})
            curves.append(row)
    covered = {(row['bus'], row['ph']) for row in curves}
    if covered != set(pvph):
        raise AssertionError('S4 não gerou uma curva para cada inversor-fase.')
    return {'A': [], 'B': pvph, 'C': []}, curves, {}


def build_uniform_fixed_pf_policy(data, pvph, power_factor=0.96):
    """Constrói S2 com todos os inversores em FP capacitivo uniforme."""
    power_factor = float(power_factor)
    if not FP_MIN_VV <= power_factor <= 1.0:
        raise ValueError(
            f'power_factor deve pertencer a [{FP_MIN_VV}, 1.0].')
    k_qp = math.sqrt(max(1.0-power_factor**2, 0.0))/power_factor
    settings = {
        bp: {
            'mode': 'fixed_pf', 'fp': power_factor, 'k_qp': k_qp,
            'q_const_frac_s': 0.0, 'direction': 'capacitive',
            'source': 'uniform_fixed_pf_counterfactual',
        }
        for bp in pvph}
    return {'A': list(pvph), 'B': [], 'C': []}, [], settings


def extract_qstar_inverter_features(
        opf_model, qstar_physical_model, data, sets_info):
    """Converte o despacho de identificação em dados auditáveis por inversor.

    A função é pública para que o notebook mostre explicitamente a passagem
    ``Q* -> pontos (V,Q,P) -> ajustes A/B/C``. A tensão é lida do replay
    fisicamente fechado de Q*, enquanto Q e P continuam sendo os alvos do OPF.
    """
    hours = list(sets_info['hours'])
    pvph = list(sets_info['pvph'])
    t_vv = tuple(sets_info.get('t_vv', (T_VV_START, T_VV_END)))
    distance = _pv_electrical_distance_metrics(data, pvph)
    data['pv_electrical_distance'] = distance
    inv_data = {}

    for b, ph in pvph:
        pairs = []
        for t in hours:
            if not (t_vv[0] <= t <= t_vv[1]):
                continue
            p_available = pyo.value(opf_model.Pavail[b, ph, t])
            p_dispatched = p_available-pyo.value(opf_model.Pcurt[b, ph, t])
            if p_dispatched < 1.0e-4:
                continue
            pairs.append((
                int(t), try_v(qstar_physical_model, b, ph, t),
                pyo.value(opf_model.Qpv[b, ph, t]), p_dispatched))

        snom = float(data['pv'][(b, ph)]['S_nom_pu'])
        q_max = QPV_MAX_FRAC_S*snom
        if len(pairs) < 5:
            useful_times = [t for t in hours if t_vv[0] <= t <= t_vv[1]]
            v_arr = np.asarray([
                try_v(qstar_physical_model, b, ph, t)
                for t in useful_times], dtype=float)
            p_arr = np.zeros(len(useful_times), dtype=float)
            q_arr = np.zeros(len(useful_times), dtype=float)
            fit_a = _fit_static_non_vv_action(p_arr, q_arr, snom)
            fit_b = _fit_individual_vv_action(v_arr, p_arr, q_arr, snom)
            inv_data[(b, ph)] = {
                'V_arr': v_arr, 'P_arr': p_arr, 'Q_arr': q_arr,
                'T_raw': np.asarray(useful_times, dtype=int),
                'V_raw': v_arr.copy(), 'P_raw': p_arr.copy(),
                'Q_raw': q_arr.copy(), 'hourly_rows': [],
                'Q_max_inv': q_max, 'fit_A': fit_a, 'fit_B': fit_b,
                'rmse_C': 0.0, 'nrmse': {'A': 0.0, 'B': 0.0, 'C': 0.0},
                'q_scale': max(0.05*q_max, 1.0e-9),
                'frac_dead': 1.0, 'frac_q_pos': 0.0, 'frac_q_neg': 0.0,
                'voltage_support_need': 0.0, 'q_support_need': 0.0,
                'no_useful_generation': True, **distance[(b, ph)],
            }
            continue

        t_raw = np.asarray([row[0] for row in pairs], dtype=int)
        v_raw = np.asarray([row[1] for row in pairs], dtype=float)
        q_raw = np.asarray([row[2] for row in pairs], dtype=float)
        p_raw = np.asarray([row[3] for row in pairs], dtype=float)
        hourly = _aggregate_qstar_points_by_hour(t_raw, v_raw, q_raw, p_raw)
        if len(hourly) >= 5:
            v_arr = np.asarray([row['V'] for row in hourly], dtype=float)
            q_arr = np.asarray([row['Q'] for row in hourly], dtype=float)
            p_arr = np.asarray([row['P'] for row in hourly], dtype=float)
        else:
            v_arr, q_arr, p_arr = v_raw, q_raw, p_raw

        q_limit = np.sqrt(np.maximum(snom*snom-p_arr*p_arr, 0.0))
        frac_dead = float(np.mean(np.abs(q_arr) < 0.05*q_max))
        fit_a = _fit_static_non_vv_action(p_arr, q_arr, snom)
        fit_b = _fit_individual_vv_action(v_arr, p_arr, q_arr, snom)
        rmse_c = float(np.sqrt(np.mean(q_arr*q_arr)))
        q_scale = max(float(np.percentile(np.abs(q_arr), 98)),
                      0.05*q_max, 1.0e-6)
        q_active = 0.05*q_max
        voltage_need = float(np.mean(
            (v_arr < IEEE1547_V2) | (v_arr > IEEE1547_V3)))
        q_support = min(
            float(np.sqrt(np.mean(q_arr*q_arr)))/max(q_max, 1.0e-9), 1.0)
        inv_data[(b, ph)] = {
            'V_arr': v_arr, 'P_arr': p_arr, 'Q_arr': q_arr,
            'T_raw': t_raw, 'V_raw': v_raw, 'P_raw': p_raw,
            'Q_raw': q_raw, 'hourly_rows': hourly,
            'Q_max_inv': q_max, 'fit_A': fit_a, 'fit_B': fit_b,
            'rmse_C': rmse_c,
            'nrmse': {
                'A': float(fit_a['rmse']/q_scale),
                'B': float(fit_b['rmse']/q_scale),
                'C': float(rmse_c/q_scale),
            },
            'q_scale': q_scale,
            'frac_cap': float(np.mean(q_arr >= 0.9*q_limit)),
            'frac_ind': float(np.mean(q_arr <= -0.9*q_limit)),
            'frac_dead': frac_dead,
            'frac_q_pos': float(np.mean(q_arr > q_active)),
            'frac_q_neg': float(np.mean(q_arr < -q_active)),
            'voltage_support_need': voltage_need,
            'q_support_need': q_support,
            'no_useful_generation': False, **distance[(b, ph)],
        }
    return inv_data


def build_device_contexts(inv_data, data):
    """Agrega fases de um mesmo equipamento antes da classificação."""
    device_members = {}
    for bp in sorted(inv_data):
        key = ((bp[0], '3ph')
               if data['pv'][bp].get('is_3ph', False) else bp)
        device_members.setdefault(key, []).append(bp)

    contexts = {}
    for key, members in device_members.items():
        action_loss = {
            action: float(np.mean([
                inv_data[bp]['nrmse'][action] for bp in members]))
            for action in RL_ACTIONS}
        q_pos = float(np.mean([inv_data[bp]['frac_q_pos'] for bp in members]))
        q_neg = float(np.mean([inv_data[bp]['frac_q_neg'] for bp in members]))
        voltage_need = float(np.mean([
            inv_data[bp]['voltage_support_need'] for bp in members]))
        q_need = float(np.mean([
            inv_data[bp]['q_support_need'] for bp in members]))
        distance = float(np.mean([
            inv_data[bp]['electrical_distance_normalized'] for bp in members]))
        sensitivity = float(np.mean([
            inv_data[bp]['reactive_sensitivity_normalized'] for bp in members]))
        contexts[key] = {
            'members': tuple(sorted(members)),
            'action_loss': action_loss,
            'dead_fraction': float(np.mean([
                inv_data[bp]['frac_dead'] for bp in members])),
            'bidirectional_q': bool(q_pos > 0.10 and q_neg > 0.10),
            'voltage_support_need': voltage_need,
            'q_support_need': q_need,
            'electrical_distance_normalized': distance,
            'reactive_sensitivity_normalized': sensitivity,
            'reactive_support_priority': (
                sensitivity*max(voltage_need, q_need)),
            'static_clip_fraction': float(np.mean([
                inv_data[bp]['fit_A']['clip_fraction'] for bp in members])),
            'no_useful_generation': bool(all(
                inv_data[bp]['no_useful_generation'] for bp in members)),
        }
    return device_members, contexts


def materialize_device_policy(
        raw_policy, inv_data, device_members, device_contexts, data,
        classifier_name):
    """Aplica a mesma regra de materialidade e gera os settings A/B/C.

    A política recebida não é modificada. O retorno preserva a ação bruta e
    a ação efetivamente aplicada, permitindo auditar se a regra de
    materialidade tornou RL e ``argmax`` eletricamente equivalentes.
    """
    import copy
    policy = copy.deepcopy(raw_policy)
    groups = {'A': [], 'B': [], 'C': []}
    curves = []
    fixed = {}
    rows = []
    for key in sorted(device_members, key=str):
        members = tuple(device_members[key])
        ctx = device_contexts[key]
        decision = policy[key]
        raw_action = decision['action']
        applied = raw_action
        losses = ctx['action_loss']
        c_competitive = (
            losses['C'] <= min(losses['A'], losses['B'])+C_MAX_NRMSE_REGRET)
        reactive_material = (
            ctx['q_support_need'] >= C_MIN_Q_RMS_FRAC_S_FOR_REACTIVE and
            ctx['reactive_support_priority'] >=
            C_MIN_REACTIVE_SUPPORT_PRIORITY)
        if ctx['no_useful_generation'] or c_competitive or not reactive_material:
            applied = 'C'
            reason = (
                'no_useful_generation' if ctx['no_useful_generation'] else
                'unity_within_material_regret' if c_competitive else
                'reactive_support_immaterial')
        else:
            reason = ''
        decision['raw_action'] = raw_action
        decision['action'] = applied
        decision['materiality_override'] = applied != raw_action
        decision['materiality_reason'] = reason

        if applied == 'A':
            p_pool = np.concatenate([inv_data[bp]['P_arr'] for bp in members])
            q_pool = np.concatenate([inv_data[bp]['Q_arr'] for bp in members])
            snom = min(data['pv'][bp]['S_nom_pu'] for bp in members)
            common = _fit_static_non_vv_action(p_pool, q_pool, snom)
            for bp in members:
                fixed[bp] = {
                    key_: common.get(key_) for key_ in (
                        'mode', 'k_qp', 'fp', 'q_const_frac_s', 'direction',
                        'candidate_count', 'candidate_library')}
                fixed[bp]['source'] = classifier_name
        elif applied == 'B':
            v_pool = np.concatenate([inv_data[bp]['V_arr'] for bp in members])
            p_pool = np.concatenate([inv_data[bp]['P_arr'] for bp in members])
            q_pool = np.concatenate([inv_data[bp]['Q_arr'] for bp in members])
            snom = min(data['pv'][bp]['S_nom_pu'] for bp in members)
            common = _fit_individual_vv_action(v_pool, p_pool, q_pool, snom)
            for bp in members:
                row = dict(common)
                row.update({'bus': bp[0], 'ph': bp[1],
                            'source_classifier': classifier_name})
                curves.append(row)
        groups[applied].extend(members)
        rows.append({
            'device': str(key), 'classifier': classifier_name,
            'raw_action': raw_action, 'applied_action': applied,
            'materiality_override': applied != raw_action,
            'materiality_reason': reason,
            'reward_A': decision.get('rewards', {}).get('A'),
            'reward_B': decision.get('rewards', {}).get('B'),
            'reward_C': decision.get('rewards', {}).get('C'),
            'member_count': len(members),
        })
    return {
        'policy': policy, 'groups': groups, 'inferred_B': curves,
        'fixed_pf_A': fixed, 'assignment_rows': rows,
        'active_device_count': sum(
            row['applied_action'] != 'C' for row in rows),
    }


def run_classifier_ablation(inv_data, data):
    """Executa RL e argmax direto lado a lado sobre os mesmos dados congelados."""
    device_members, contexts = build_device_contexts(inv_data, data)
    start_rl = time.perf_counter()
    rl_policy, q_table, rl_summary = _train_contextual_bandit_classifier(contexts)
    rl_seconds = time.perf_counter()-start_rl
    direct_policy, direct_summary = _direct_reward_classifier(contexts)
    rl = materialize_device_policy(
        rl_policy, inv_data, device_members, contexts, data,
        classifier_name='contextual_q_learning_gamma_0')
    direct = materialize_device_policy(
        direct_policy, inv_data, device_members, contexts, data,
        classifier_name='direct_reward_argmax')
    return {
        'device_members': device_members,
        'device_contexts': contexts,
        'rl': rl, 'direct': direct,
        'raw_assignment_comparison': compare_classifier_assignments(
            rl_policy, direct_policy),
        'rl_q_table': q_table,
        'rl_summary': {**rl_summary, 'elapsed_seconds': rl_seconds},
        'direct_summary': direct_summary,
    }


def preferred_pf_distance(fixed_settings):
    """Distância média dos equipamentos do Grupo A ao FP preferido."""
    values = []
    seen = set()
    for bp, row in (fixed_settings or {}).items():
        device = bp[0]
        if device in seen or row.get('mode') != 'fixed_pf':
            continue
        seen.add(device)
        values.append(abs(float(row['fp'])-PREFERRED_FIXED_POWER_FACTOR))
    return float(np.mean(values)) if values else 0.0


def count_active_physical_inverters(groups, data):
    """Conta equipamentos físicos ativos, sem triplicar inversores 3φ."""
    devices = set()
    for bp in tuple(groups.get('A', ()))+tuple(groups.get('B', ())):
        key = ((bp[0], '3ph')
               if data['pv'][bp].get('is_3ph', False) else bp)
        devices.add(key)
    return len(devices)


In [ ]:
#@title Internal model implementation 12/12 { display-mode: "form" }
def scenario_summary_row(name, result, groups=None, classifier_seconds=None,
                         replay_seconds=None, data=None):
    """Normaliza as saídas de S1--S4 e das duas classificações."""
    groups = groups or {'A': [], 'B': [], 'C': []}
    objective = result.get('objective_metrics_pu', {})
    quality = result.get('settings_quality') or result.get('quality') or {}
    tap_ops = result.get('tap_ops', {})
    tap_total = result.get('tap_ops_total')
    if tap_total is None:
        tap_total = sum(tap_ops.values()) if isinstance(tap_ops, dict) else None
    return {
        'scenario': name,
        'operationally_admissible': bool(
            result.get('operationally_applicable', result.get('valid', False))),
        'tap_operations': tap_total,
        'objective_total': objective.get('total'),
        'voltage_deviation_pu_sum': result.get(
            'voltage_deviation_pu_sum'),
        'unbalance_magnitude_proxy_pu_sum': result.get(
            'unbalance_pu_sum'),
        'joule_loss_kwh': result.get('joule_loss_kwh'),
        'core_loss_kwh': result.get('core_loss_kwh'),
        'total_loss_kwh': result.get('total_loss_kwh'),
        'mv_drp_percent': (quality.get('mv') or {}).get('drp'),
        'mv_drc_percent': (quality.get('mv') or {}).get('drc'),
        'lv_drp_percent': (quality.get('lv') or {}).get('drp'),
        'lv_drc_percent': (quality.get('lv') or {}).get('drc'),
        'fd95_percent': (quality.get('fd') or {}).get('fd95_system'),
        'n_inverter_phases_A': len(groups.get('A', [])),
        'n_inverter_phases_B': len(groups.get('B', [])),
        'n_inverter_phases_C': len(groups.get('C', [])),
        'n_active_physical_inverters': (
            count_active_physical_inverters(groups, data)
            if data is not None else None),
        'classifier_seconds': classifier_seconds,
        'replay_seconds': replay_seconds,
        'svr_equation_mode': result.get('svr_equation_mode'),
        'max_svr_exact_residual_pu': result.get(
            'max_svr_exact_residual_pu'),
        'solver_termination': result.get('termination'),
    }


def lexicographic_candidate_key(candidate, objective_equivalence_tol=1.0e-9):
    """Chave documentada: guardas → taps → objetivo → FP≈0.96.

    Candidatos não admissíveis devem ser filtrados antes desta função. O termo
    de tolerância é quantizado para impedir que ruído numérico escolha FP=0.85.
    """
    if not candidate.get('guards_ok', candidate.get('operationally_admissible', False)):
        raise ValueError('Candidato não admissível não pode ser ranqueado.')
    objective = float(candidate['objective_value'])
    objective_bucket = round(objective/max(objective_equivalence_tol, 1.0e-15))
    return (
        int(candidate['tap_ops_total']),
        objective_bucket,
        float(candidate.get('distance_to_preferred_pf', 0.0)),
        str(candidate.get('candidate_id', candidate.get('name', ''))),
    )


def run_stateful_multiday_validation(
        ordered_days, scenario_runners, initial_states, checkpoint_dir,
        frozen_policy_hash, stop_after_seconds=None):
    """Executa somente replays, com uma cadeia de estado independente por cenário.

    ``scenario_runners[name](day, state_in)`` deve devolver um dicionário com
    ``final_relay_state`` e as métricas. Esta rotina nunca chama OPF, Q*, RL ou
    classificação. Dias precisam estar ordenados e ser consecutivos.
    """
    import copy
    import json
    from datetime import date
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    parsed = [date.fromisoformat(str(day['date'])) for day in ordered_days]
    for previous, current in zip(parsed, parsed[1:]):
        if (current-previous).days != 1:
            raise ValueError(
                'Validação cronológica exige dias consecutivos; '
                f'encontrado salto {previous} -> {current}.')

    states = {name: state for name, state in initial_states.items()}
    started = time.monotonic()
    results = []
    for day in ordered_days:
        for scenario, runner in scenario_runners.items():
            if stop_after_seconds is not None and (
                    time.monotonic()-started) >= stop_after_seconds:
                return {'completed': False, 'results': results,
                        'states': states, 'reason': 'session_budget'}
            day_dir = checkpoint_dir/scenario/str(day['date'])
            marker = day_dir/'COMPLETED.json'
            if marker.is_file():
                saved = json.loads(marker.read_text(encoding='utf-8'))
                if saved.get('frozen_policy_hash') == frozen_policy_hash:
                    states[scenario] = saved['final_relay_state']
                    results.append(saved['result'])
                    continue
            day_dir.mkdir(parents=True, exist_ok=True)
            # O estado de entrada precisa permanecer imutável no checkpoint,
            # mesmo se um runner modificar internamente o dicionário recebido.
            state_in = copy.deepcopy(states[scenario])
            result = runner(day, copy.deepcopy(state_in))
            if 'final_relay_state' not in result:
                raise KeyError(
                    f'{scenario} não devolveu final_relay_state em {day["date"]}.')
            states[scenario] = copy.deepcopy(result['final_relay_state'])
            payload = {
                'schema': 'stateful-multiday-checkpoint-v4.9',
                'frozen_policy_hash': frozen_policy_hash,
                'scenario': scenario,
                'day': day['date'],
                'initial_relay_state': copy.deepcopy(state_in),
                'final_relay_state': copy.deepcopy(states[scenario]),
                'result': result,
            }
            tmp = marker.with_suffix('.json.tmp')
            tmp.write_text(json.dumps(
                payload, ensure_ascii=False, indent=2, default=str),
                encoding='utf-8')
            tmp.replace(marker)
            results.append(result)
    return {'completed': True, 'results': results, 'states': states,
            'reason': 'all_days_completed'}


In [ ]:
# Instala o CPLEX silenciosamente quando ainda não estiver disponível.
# O instalador deve estar em:
# /content/drive/MyDrive/Solvers/cplex_studio2212.linux_x86_64.bin

CPLEX_EXE = ensure_drive_and_cplex()
CFG.cplex_executable = CPLEX_EXE

if not Path(CPLEX_EXE).is_file():
    raise FileNotFoundError(
        f"A instalação terminou, mas o CPLEX não foi encontrado em: {CPLEX_EXE}"
    )

print("CPLEX:", CPLEX_EXE)
print("Embedded engine:", VERSION)
print("Meteorological source:", PROFILE_SOURCE)
print("Input-data commit:", SOURCE_COMMIT)

## 2. Weather-to-PV model

The CSV irradiance is already normalized by $G_{\rm STC}=1000\;\mathrm{W/m^2}$.
Cell temperature is estimated, rather than read from the supplied panel-
temperature series:

\[
T_{\rm cell,t}=T_{\rm amb,t}+G_t\,(\mathrm{NOCT}-20),
\]

\[
P_{{\rm av},g,t}=P_{{\rm STC},g}G_t
\left[1+\gamma_{\rm pv}(T_{\rm cell,t}-25)\right].
\]

Inputs are irradiance in p.u. and ambient temperature in $^\circ$C.  Outputs
are cell temperature and available active power.  The frozen-parameter file
records NOCT, $\gamma_{\rm pv}$ and the input convention.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo

data = load_data(str(BUSES_FILE), str(BRANCHES_FILE))
hours = np.arange(N_PERIODS)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].plot(hours/6, IRRAD, color="tab:orange")
axes[0].set_ylabel("Irradiance [p.u.]")
axes[1].plot(hours/6, TEMP_AMB, label="Ambient")
axes[1].plot(hours/6, PV_T_CELL, label="Estimated cell")
axes[1].set_ylabel("Temperature [°C]"); axes[1].legend()
axes[2].plot(hours/6, PV_EFF_FACTOR, color="tab:green")
axes[2].set_ylabel("Available PV factor"); axes[2].set_xlabel("Hour")
fig.tight_layout()
plt.show()


## 3. Phase-explicit MV/LV branch-flow model

For each energized branch phase $(i,j,\phi)$ and period $t$, the implemented
BFM uses active/reactive balances, voltage drop and the conic current relation:

\[
P_{ij}^{\phi t}-r_{ij}^{\phi}\ell_{ij}^{\phi t}
-\sum_{k:j\to k}P_{jk}^{\phi t}+P_{{\rm pv},j}^{\phi t}
=P_{d,j}^{\phi t}+P_{{\rm core},j}^{\phi t},
\]
\[
Q_{ij}^{\phi t}-x_{ij}^{\phi}\ell_{ij}^{\phi t}
-\sum_{k:j\to k}Q_{jk}^{\phi t}+Q_{{\rm pv},j}^{\phi t}
=Q_{d,j}^{\phi t},
\]
\[
v_j^{\phi t}=v_i^{\phi t}-2(rP+xQ)
 +(r^2+x^2)\ell_{ij}^{\phi t},
\qquad
(P_{ij}^{\phi t})^2+(Q_{ij}^{\phi t})^2
\le v_i^{\phi t}\ell_{ij}^{\phi t}.
\]

MV loads, LV loads, transformer series impedance and transformer shunt core
losses are in the same balances.  The current implementation is phase-diagonal;
the reported unbalance quantity is therefore a voltage-magnitude proxy, not a
full sequence-component VUF from a mutually coupled three-phase model.


### Successive convex approximation (SCA)

The product $\ell V^2$ is handled with a fixed voltage linearization point:

\[
\ell_{ij}^{\phi t}(\bar V_i^{\phi t})^2
\ge (P_{ij}^{\phi t})^2+(Q_{ij}^{\phi t})^2.
\]

After each QCP, $\bar V$ is updated from the solved voltage.  Convergence of
$V-\bar V$ does not by itself prove exactness.  The final state is separately
audited through the conic gap, the physical inequality violation and the exact
squared-voltage-drop residual.


## 4. SVR: identification approximation versus final exact relation

During identification, $n$ is a decision variable.  The affine approximation
used by the convex model is

\[
V_{\rm out}^{\phi t}=\tau_{\min}V_{\rm in}^{\phi t}+\Delta\tau n^{\phi t}.
\]

During every final evaluation, $n^{\phi t}$ is known and fixed.  The model
therefore deactivates the affine constraint and imposes the exact linear
equality

\[
\boxed{V_{\rm out}^{\phi t}=
(\tau_{\min}+\Delta\tau n^{\phi t})V_{\rm in}^{\phi t}}.
\]

Every final scenario must satisfy

\[
\max_{\phi,t}|V_{\rm out}^{\phi t}-\tau_{\phi t}V_{\rm in}^{\phi t}|
\le \varepsilon_{\rm SVR}.
\]


In [ ]:
import inspect
print(inspect.getsource(activate_exact_fixed_tap_svr))


## 5. Objective used for identification

The four primary metrics are normalized before weighting:

\[
J=w_N\widehat N_{\rm tap}+w_V\widehat D_V
  +w_U\widehat U_V+w_C\widehat E_{\rm core}+J_{\rm guard}.
\]

$J_{\rm guard}$ contains soft safeguards for voltage margin, Joule losses,
reactive effort, reactive-command variation and curtailment.  Numerical weights
are not hidden in this text; they are exported with the normalizers in
`frozen_parameters.json`.  Candidate acceptance is not based on the weighted
sum alone: physical closure and all operational guards are mandatory.


In [ ]:
build_started = time.perf_counter()
model, sets_info, warm_current = build_opf(data)
build_seconds = time.perf_counter() - build_started

evidence = RunEvidenceRecorder(RUN_ID, RUN_DIR)
evidence.record(
    stage="model_build", day="2019-02-15", model=model,
    model_build_seconds=build_seconds,
    extra={"svr_equation_mode": "affine_identification"},
)

normalizers = {
    name: float(pyo.value(getattr(model, f"obj_norm_{name}")))
    for name in (
        "tap", "voltage_deviation", "unbalance", "core", "joule",
        "voltage_margin", "q_use", "q_ramp", "curtailment"
    )
}
print(model_size_statistics(model))
print("Normalizers:", normalizers)


### Equation-to-code traceability

The table below binds each equation introduced above to the active Pyomo
component that implements it.  This is deliberately executed against the
constructed model, so a renamed or missing constraint fails visibly instead
of leaving the mathematical description disconnected from the code.


In [ ]:
constraint_trace = {
    "active-power balance": "c_bal_P",
    "reactive-power balance": "c_bal_Q",
    "BFM voltage drop": "c_vdrop",
    "SCA current cone": "c_socp",
    "affine SVR relation used only in identification": "c_svr",
    "PV apparent-power capability": "c_pv_capability",
    "minimum operating power factor (+Q)": "c_qpv_fp085_pos",
    "minimum operating power factor (-Q)": "c_qpv_fp085_neg",
    "reactive-command variation": "c_ramp",
    "tap-event linking": "c_tap_event_link",
    "tap delay": "c_tap_delay",
    "tap spacing": "c_tap_spacing",
    "phase-magnitude unbalance proxy (+)": "c_unbal_pos",
    "phase-magnitude unbalance proxy (-)": "c_unbal_neg",
}
trace_rows = []
for equation, component_name in constraint_trace.items():
    component = getattr(model, component_name, None)
    if component is None:
        raise AttributeError(f"Missing implementation for {equation}: {component_name}")
    trace_rows.append({
        "equation": equation,
        "pyomo_component": component_name,
        "active": bool(component.active),
        "number_of_indexed_constraints": len(component),
    })
constraint_trace_table = pd.DataFrame(trace_rows)
constraint_trace_table.to_csv(RUN_DIR / "equation_constraint_trace.csv", index=False)
display(constraint_trace_table)


## 6. Physical S1/S2 seeds

The forward/backward sweep below is used only to initialize the autonomous LDC
fixed point.  Published S1 and S2 metrics are obtained later from the full BFM
with exact fixed-tap SVR equations.


In [ ]:
(tap_seed_s1, seed_ops_s1, relay_seed_s1, voltage_seed_s1,
 seed_terminal_state_s1) = simulate_svr_local_control(
    data, sets_info["hours"], sets_info["irr"], sets_info["load"],
    return_voltage=True, return_final_state=True,
    pv_eff_d=PV_EFF_FACTOR, load_mt_d=LOAD_PROFILE_MT,
)

(tap_seed_s2, seed_ops_s2, relay_seed_s2, voltage_seed_s2,
 seed_terminal_state_s2) = simulate_svr_local_control(
    data, sets_info["hours"], sets_info["irr"], sets_info["load"],
    return_voltage=True, return_final_state=True,
    fixed_inverter_pf=0.96,
    pv_eff_d=PV_EFF_FACTOR, load_mt_d=LOAD_PROFILE_MT,
)
print("Sweep seeds only — S1/S2 taps:", sum(seed_ops_s1.values()), sum(seed_ops_s2.values()))


## 7. E1 — continuous identification problem

Only tap integrality is relaxed.  The physical network, transformer, voltage
bounds and inverter apparent-power circle remain active.  The optional relaxed
link between $Q_{\rm pv}$ and a pre-specified local policy is used solely to
identify the target $Q^\star(t)$.


In [ ]:
if not CFG.run_heavy_identification:
    raise RuntimeError("Set CFG.run_heavy_identification=True for a fresh training run.")

relax_identification_taps(model, sets_info)
solver_e1 = make_cplex_solver(
    CFG.cplex_executable, CFG.cplex_time_limit_e1_s, CFG.cplex_threads)
e1_started = time.perf_counter()
with PeakMemoryMonitor() as mem_e1:
    (res_e1, tc_e1, raw_e1, viol_e1, sca_e1_ok,
     sca_e1_iterations, sca_e1_mismatch) = _solve_sca_voltage_consistent(
        solver_e1, model, sets_info["cph"], sets_info["hours"],
        label="E1-continuous", tee_first=True)
e1_wall_seconds = time.perf_counter() - e1_started
e1_bounds = solver_bound_evidence(res_e1)

evidence.record(
    stage="E1_continuous_identification", day="2019-02-15", model=model,
    started_at=e1_started, solver_status=str(res_e1.solver.status),
    termination_condition=tc_e1, sca_iterations=sca_e1_iterations,
    max_constraint_violation=viol_e1,
    solver_wall_seconds=e1_wall_seconds,
    lower_bound=e1_bounds["lower_bound"],
    upper_bound=e1_bounds["upper_bound"],
    relative_gap=e1_bounds["relative_gap"],
    peak_process_tree_rss_mb=mem_e1.peak_mb,
    extra={"sca_voltage_mismatch_pu": sca_e1_mismatch,
           "svr_equation_mode": "affine_identification"},
)
if tc_e1 not in _USABLE_TERMINATIONS or not sca_e1_ok:
    raise RuntimeError(f"E1 did not converge: {tc_e1}, violation={viol_e1:.3e}")


## 8. Causal integer projection

The continuous tap trajectory is converted to integer positions with at most
one step per operation and the discretized 15-min relay delay.  Several
deterministic neighboring projections are retained so that a single rounding
choice does not decide feasibility.


In [ ]:
tap_continuous = continuous_tap_schedule(model, sets_info)
projection_specs = [("round", 0), ("ceil", 0), ("round", 1), ("floor", 0)]
projection_candidates = []
seen = set()
for rounding, bias in projection_specs:
    schedule = _project_continuous_tap_schedule(
        tap_continuous, data, sets_info["hours"],
        rounding=rounding, bias=bias,
        n_tap_max=sets_info.get("n_tap_max", N_TAP_MAX),
    )
    signature = tuple(schedule[key] for key in sorted(schedule))
    audit = _audit_programmed_tap_schedule(schedule, data, sets_info["hours"])
    if signature not in seen and audit["ok"]:
        seen.add(signature)
        projection_candidates.append((f"{rounding}_{bias:+d}", schedule, audit))

baseline_signature = tuple(tap_seed_s1[key] for key in sorted(tap_seed_s1))
if baseline_signature not in seen:
    projection_candidates.append((
        "autonomous_ldc_seed", dict(tap_seed_s1),
        _audit_programmed_tap_schedule(tap_seed_s1, data, sets_info["hours"]),
    ))

pd.DataFrame([
    {"candidate": name, "tap_operations": sum(audit["ops"].values()),
     "mechanically_valid": audit["ok"], "reasons": ";".join(audit["reasons"])}
    for name, _, audit in projection_candidates
])


## 9. E2 — fixed integer taps and conditioned QCP

Each projection is tested from the same converged E1 state.  The first
SCA-usable trajectory becomes the identification candidate.  The SVR relation
is still the affine identification relation here; it is replaced by the exact
fixed-ratio equality in all following physical evaluations.


In [ ]:
solver_e2 = make_cplex_solver(
    CFG.cplex_executable, CFG.cplex_time_limit_e2_s, CFG.cplex_threads)
e2_seed_model = model.clone()
e2_rows = []
identification_model = None
selected_projection = None

for name, schedule, audit in projection_candidates:
    candidate_model = e2_seed_model.clone()
    fix_identification_tap_schedule(
        candidate_model, data, sets_info, schedule)
    started = time.perf_counter()
    with PeakMemoryMonitor() as memory:
        (res, tc, raw_tc, violation, sca_ok, iterations,
         mismatch) = _solve_sca_voltage_consistent(
            solver_e2, candidate_model, sets_info["cph"], sets_info["hours"],
            label=f"E2-{name}", tee_first=(len(e2_rows) == 0))
    e2_wall_seconds = time.perf_counter() - started
    e2_bounds = solver_bound_evidence(res)
    exactness = (summarize_bfm_exactness(candidate_model, data, sets_info)
                 if tc in _USABLE_TERMINATIONS and sca_ok else None)
    usable, relaxation_exact = _classify_e2_candidate(tc, sca_ok, exactness)
    row = {
        "candidate": name, "tap_operations": sum(audit["ops"].values()),
        "termination": tc, "sca_converged": sca_ok,
        "accepted_for_physical_replay": usable,
        "relaxation_exact": relaxation_exact,
        "max_constraint_violation": violation,
    }
    e2_rows.append(row)
    evidence.record(
        stage="E2_fixed_tap_identification", day="2019-02-15",
        scenario=name, model=candidate_model, started_at=started,
        solver_status=str(res.solver.status), termination_condition=tc,
        sca_iterations=iterations, max_constraint_violation=violation,
        solver_wall_seconds=e2_wall_seconds,
        lower_bound=e2_bounds["lower_bound"],
        upper_bound=e2_bounds["upper_bound"],
        relative_gap=e2_bounds["relative_gap"],
        exactness=exactness, peak_process_tree_rss_mb=memory.peak_mb,
        extra={"sca_voltage_mismatch_pu": mismatch,
               "svr_equation_mode": "affine_identification",
               "accepted_for_physical_replay": usable},
    )
    if usable:
        identification_model = candidate_model
        selected_projection = {"name": name, "schedule": schedule, "audit": audit}
        break
    del candidate_model
    gc.collect()

pd.DataFrame(e2_rows).to_csv(RUN_DIR / "e2_candidates.csv", index=False)
if identification_model is None:
    raise RuntimeError("No E2 projection produced an SCA-usable identification point.")
print("Selected E2 projection:", selected_projection["name"])


## 10. Physical reconstruction of $Q^\star$

The E2 values of $Q^\star$, curtailment and tap position are fixed.  Voltages,
flows and currents are solved again with the exact SVR ratio and a strict
current-minimization objective.  Only these physically reconstructed voltages
are paired with $Q^\star$ to fit inverter settings.


In [ ]:
qstar_started = time.perf_counter()
with PeakMemoryMonitor() as mem_qstar:
    qstar_replay = replay_qstar_for_curve_interpretation(
        identification_model, data, sets_info, CFG.cplex_executable)
qstar_wall_seconds = time.perf_counter() - qstar_started
if not qstar_replay["valid"]:
    raise RuntimeError("The physical Q* reconstruction failed its closure checks.")
assert qstar_replay["svr_equation_mode"] == "exact_fixed_tap"
assert qstar_replay["max_svr_exact_residual_pu"] <= SVR_EXACT_RESIDUAL_TOL
evidence.record(
    stage="Qstar_physical_reconstruction", day="2019-02-15",
    model=qstar_replay["model"], started_at=qstar_started,
    termination_condition=qstar_replay["termination"],
    sca_iterations=qstar_replay["iterations"],
    max_constraint_violation=qstar_replay["solver_violation"],
    solver_wall_seconds=qstar_wall_seconds,
    exactness=qstar_replay["exactness"],
    max_svr_exact_residual_pu=qstar_replay["max_svr_exact_residual_pu"],
    peak_process_tree_rss_mb=mem_qstar.peak_mb,
)


## 11. Device-level synthesis and classifier ablation

For physical inverter $g$, the three immediate rewards are obtained from the
normalized fitting error plus declared complexity/risk penalties.  Contextual
Q-learning uses

\[
\mathcal Q_{k+1}(s,a)=\mathcal Q_k(s,a)
+\alpha\,[R_g(a)-\mathcal Q_k(s,a)], \qquad \gamma=0.
\]

The ablation removes learning and selects directly

\[
a_g^{\rm direct}=\arg\max_{a\in\{A,B,C\}}R_g(a).
\]

Both classifiers receive identical features and then pass through the same
materiality rule and electrical replay.  Raw and applied actions are exported
separately.


In [ ]:
features = extract_qstar_inverter_features(
    identification_model, qstar_replay["model"], data, sets_info)
ablation_started = time.perf_counter()
ablation = run_classifier_ablation(features, data)
ablation_seconds = time.perf_counter() - ablation_started

raw_comparison = pd.DataFrame(ablation["raw_assignment_comparison"])
applied_comparison = (
    pd.DataFrame(ablation["rl"]["assignment_rows"])
    .merge(pd.DataFrame(ablation["direct"]["assignment_rows"]),
           on="device", suffixes=("_rl", "_direct"))
)
raw_comparison.to_csv(RUN_DIR / "classifier_raw_assignments.csv", index=False)
applied_comparison.to_csv(RUN_DIR / "classifier_applied_assignments.csv", index=False)
print("RL active devices:", ablation["rl"]["active_device_count"])
print("Direct active devices:", ablation["direct"]["active_device_count"])
display(applied_comparison.head())


### S4 — all-inverter individualized Volt--VAR baseline

S4 uses the same $(V,Q^\star)$ points as S3, but bypasses classification and
sparse selection.  Every physical inverter receives its own static Volt--VAR
curve.  A three-phase inverter receives one common curve across its phases.


In [ ]:
groups_s4, curves_s4, fixed_s4 = build_s4_all_individual_vv(
    features, ablation["device_members"], data)
assert not groups_s4["A"] and not groups_s4["C"]
assert set(groups_s4["B"]) == set(sets_info["pvph"])
print("S4 inverter-phase curves:", len(curves_s4))


## 12. Closed-loop S1--S4 evaluation

The following cells solve the full physical BFM while the phase-wise LDC
reconstructs its own tap sequence.  The optimized tap sequence is not imposed.
The exact SVR equality is activated after every LDC update.


In [ ]:
solver_replay = make_cplex_solver(
    CFG.cplex_executable, CFG.cplex_time_limit_e2_s, CFG.cplex_threads)
s1_started = time.perf_counter()
with PeakMemoryMonitor() as mem_s1:
    result_s1 = evaluate_physical_ldc_baseline(
        identification_model, data, sets_info, tap_seed_s1,
        voltage_seed_s1, solver_replay)
s1_wall_seconds = time.perf_counter() - s1_started
if not result_s1["valid"]:
    raise RuntimeError("S1 physical baseline is not certified.")
assert result_s1["max_svr_exact_residual_pu"] <= SVR_EXACT_RESIDUAL_TOL
evidence.record(
    stage="closed_loop_replay", day="2019-02-15", scenario="S1",
    model=result_s1["model"], started_at=s1_started,
    termination_condition=result_s1["termination"],
    sca_iterations=result_s1["sca_iterations"],
    max_constraint_violation=result_s1["solver_violation"],
    exactness=result_s1["exactness"],
    solver_wall_seconds=s1_wall_seconds,
    max_svr_exact_residual_pu=result_s1["max_svr_exact_residual_pu"],
    peak_process_tree_rss_mb=mem_s1.peak_mb,
)
print(scenario_summary_row("S1", result_s1, data=data))


In [ ]:
def evaluate_policy(name, groups, curves, fixed_settings):
    started = time.perf_counter()
    with PeakMemoryMonitor() as memory:
        result = validate_recommended_settings(
            identification_model, data, sets_info,
            groups, curves, fixed_settings, CFG.cplex_executable,
            source_relaxation_exact=False,
            autonomous_seed_schedule=result_s1["tap_schedule"],
            autonomous_seed_model=result_s1["model"],
            run_qstar_diagnostic=False,
            baseline_quality_for_screening=result_s1["quality"],
        )
    if result.get("converged"):
        assert result["svr_equation_mode"] == "exact_fixed_tap"
        assert result["max_svr_exact_residual_pu"] <= SVR_EXACT_RESIDUAL_TOL
    elapsed = time.perf_counter() - started
    evidence.record(
        stage="closed_loop_replay", day="2019-02-15", scenario=name,
        model=result.get("model"), started_at=started,
        termination_condition=result.get("termination"),
        solver_wall_seconds=elapsed,
        sca_iterations=result.get("iterations"),
        exactness=result.get("settings_bfm_exactness"),
        max_svr_exact_residual_pu=result.get("max_svr_exact_residual_pu"),
        peak_process_tree_rss_mb=memory.peak_mb,
        extra={"active_inverter_phases": len(groups["A"])+len(groups["B"])},
    )
    return result, elapsed

groups_s2, curves_s2, fixed_s2 = build_uniform_fixed_pf_policy(
    data, sets_info["pvph"], power_factor=0.96)

policies = {
    "S2": (groups_s2, curves_s2, fixed_s2),
    "S3_RL": (ablation["rl"]["groups"],
              ablation["rl"]["inferred_B"], ablation["rl"]["fixed_pf_A"]),
    "S3_DIRECT": (ablation["direct"]["groups"],
                  ablation["direct"]["inferred_B"], ablation["direct"]["fixed_pf_A"]),
    "S4": (groups_s4, curves_s4, fixed_s4),
}

scenario_results = {"S1": result_s1}
scenario_times = {"S1": time.perf_counter() - s1_started}
for scenario, (groups, curves, fixed_settings) in policies.items():
    scenario_results[scenario], scenario_times[scenario] = evaluate_policy(
        scenario, groups, curves, fixed_settings)


### Sparse physical refinement of S3

Classification determines mode candidates; it does not certify that every
active device is needed.  Prefixes ordered by applied support and electrical
sensitivity are replayed physically.  A candidate must pass all guards before
ranking.  Among admissible candidates the order is:

1. fewer tap operations;
2. lower normalized objective;
3. when equivalent, Group-A PF closer to 0.96;
4. fewer active devices and a stable candidate identifier.

PF = 0.85 remains available.  The exported library shows whether it was
necessary or merely favored by a previous search order.


In [ ]:
if CFG.run_sparse_refinement:
    for scenario in ("S3_RL", "S3_DIRECT"):
        groups, curves, fixed_settings = policies[scenario]
        sparse_started = time.perf_counter()
        (refined, refined_groups, refined_curves,
         refined_fixed) = search_autonomous_sparse_policy_v3(
            scenario_results[scenario], identification_model, data, sets_info,
            groups, curves, fixed_settings, CFG.cplex_executable, result_s1)
        scenario_results[scenario] = refined
        policies[scenario] = (refined_groups, refined_curves, refined_fixed)
        evidence.record(
            stage="sparse_physical_refinement", day="2019-02-15",
            scenario=scenario, model=refined.get("model"),
            started_at=sparse_started,
            candidates_planned=refined.get("autonomous_sparse_candidate_count"),
            candidates_evaluated=refined.get("autonomous_sparse_candidate_count"),
            candidates_accepted=(
                1 if refined.get("autonomous_sparse_selected_count", 0) > 0
                else 0),
            termination_condition=refined.get("termination"),
            exactness=refined.get("settings_bfm_exactness"),
            max_svr_exact_residual_pu=refined.get(
                "max_svr_exact_residual_pu"),
        )

summary_rows = [scenario_summary_row("S1", result_s1, data=data)]
for scenario, result in scenario_results.items():
    if scenario == "S1":
        continue
    groups, _, _ = policies[scenario]
    classifier_time = (
        ablation["rl_summary"].get("elapsed_seconds") if scenario == "S3_RL"
        else ablation["direct_summary"].get("elapsed_seconds")
        if scenario == "S3_DIRECT" else 0.0)
    summary_rows.append(scenario_summary_row(
        scenario, result, groups=groups,
        classifier_seconds=classifier_time,
        replay_seconds=scenario_times[scenario],
        data=data,
    ))

scenario_table = pd.DataFrame(summary_rows)
scenario_table.to_csv(RUN_DIR / "s1_s2_s3_s4_summary.csv", index=False)
display(scenario_table)


In [ ]:
pf_library_rows = []
for bp, feature in features.items():
    for candidate in feature["fit_A"].get("candidate_library", []):
        pf_library_rows.append({"bus": bp[0], "phase": bp[1], **candidate})
pf_library = pd.DataFrame(pf_library_rows)
pf_library.to_csv(RUN_DIR / "group_A_candidate_library.csv", index=False)
if not pf_library.empty:
    display(pf_library[
        ["bus", "phase", "mode", "fp", "rmse",
         "distance_to_preferred_pf", "equivalent_to_best_fit"]
    ].head(20))


## 13. Freeze the selected settings and every numerical assumption

The frozen bundle contains only implementable local settings.  It contains no
$Q^\star$ time series and no optimized tap schedule.  The hash below identifies
the exact bundle used in the multiday experiment.


In [ ]:
def clean_json(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, tuple):
        return [clean_json(v) for v in value]
    if isinstance(value, list):
        return [clean_json(v) for v in value]
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()
                if k not in {"prediction", "model", "opf_replay_model"}}
    return value

def pack_policy(groups, curves, fixed):
    return {
        "groups": {g: [[bp[0], int(bp[1])] for bp in members]
                   for g, members in groups.items()},
        "curves": clean_json(curves),
        "fixed": [{"bus": bp[0], "ph": int(bp[1]), **clean_json(row)}
                  for bp, row in fixed.items()],
    }

def unpack_policy(bundle):
    groups = {g: [(row[0], int(row[1])) for row in members]
              for g, members in bundle["groups"].items()}
    curves = list(bundle["curves"])
    fixed = {(row["bus"], int(row["ph"])):
             {k: v for k, v in row.items() if k not in ("bus", "ph")}
             for row in bundle["fixed"]}
    return groups, curves, fixed

frozen_policies = {
    name: pack_policy(*policies[name])
    for name in ("S2", "S3_RL", "S3_DIRECT", "S4")
}
policy_bundle = {
    "schema": "acopf-frozen-local-settings-v4.9",
    "source_day_physical_date": "2019-02-15",
    "settings": frozen_policies,
    "training_calls": {"Qstar": 1, "classifier": 1, "sparse_selection": 1},
}
canonical = json.dumps(policy_bundle, sort_keys=True, separators=(",", ":"))
FROZEN_POLICY_HASH = hashlib.sha256(canonical.encode()).hexdigest()
policy_bundle["sha256"] = FROZEN_POLICY_HASH
(RUN_DIR / "frozen_local_settings.json").write_text(
    json.dumps(policy_bundle, indent=2, ensure_ascii=False), encoding="utf-8")

export_frozen_parameters(
    RUN_DIR / "frozen_parameters.json", data=data, sets_info=sets_info,
    normalizers=normalizers, kmeans_seed=CFG.kmeans_seed,
    source_commit=SOURCE_COMMIT,
)
print("Frozen policy SHA-256:", FROZEN_POLICY_HASH)


In [ ]:
evidence.record(
    stage="classifier_ablation", day="2019-02-15",
    started_at=ablation_started,
    candidates_planned=len(ablation["device_members"]),
    candidates_evaluated=len(ablation["device_members"]),
    candidates_accepted=ablation["rl"]["active_device_count"],
    extra={
        "rl_seconds": ablation["rl_summary"].get("elapsed_seconds"),
        "direct_seconds": ablation["direct_summary"].get("elapsed_seconds"),
        "raw_assignments_equal": bool(raw_comparison["same_assignment"].all()),
        "applied_assignments_equal": bool(
            (applied_comparison["applied_action_rl"] ==
             applied_comparison["applied_action_direct"]).all()),
    },
)
evidence_paths = evidence.save()
print(evidence_paths)


## 14. Chronological multiday validation

The following experiment uses the consecutive calendar week around the source
day.  It is not a sequence of nonconsecutive representative days.

The selected settings remain frozen.  For each scenario and each phase, the
following relay state crosses midnight:

```text
tap_position
pending_direction
delay_counter
queued_delta
last_relay_voltage_v
```

No $Q^\star$, Q-learning, direct classification or sparse search is called in
this loop.  Each scenario owns an independent state chain.  A command whose
delay expires in the final 10-min interval is queued and applied in the first
interval of the next day.

**After a Colab disconnection:** keep the same `campaign_id`, rerun the
configuration/environment/input/weather cells, and then rerun Sections 14--16.
The frozen settings are reloaded from Drive, their SHA-256 hash is checked, and
the loop starts at the first scenario/day without a valid `COMPLETED.json`.
Sections 7--13 do not need to be repeated merely to resume the weekly replay.


In [ ]:
FEBRUARY_PROFILE_FILE = PROFILES_DIR/"perfis_10min_2019_02.csv"
if not FEBRUARY_PROFILE_FILE.is_file():
    raise FileNotFoundError(f"Perfil cronológico não encontrado: {FEBRUARY_PROFILE_FILE}")
february = pd.read_csv(FEBRUARY_PROFILE_FILE, sep=None, engine="python")
february.columns = [str(c).strip() for c in february.columns]

ordered_days = []
for day_number in range(12, 19):
    day_frame = february[
        (february["month"].astype(int) == 2) &
        (february["day"].astype(int) == day_number)
    ].sort_values(["hour", "minute"])
    if len(day_frame) != N_PERIODS:
        raise ValueError(f"February {day_number}: expected 144 rows, got {len(day_frame)}")
    ordered_days.append({
        "date": f"{CFG.physical_year}-02-{day_number:02d}",
        "irradiance_pu": day_frame["irradiance_pu"].astype(float).tolist(),
        "ambient_temperature_c":
            day_frame["ambient_temperature_c"].astype(float).tolist(),
    })

print([day["date"] for day in ordered_days])


In [ ]:
MULTIDAY_DIR = RUN_DIR / "multiday"
MULTIDAY_DIR.mkdir(parents=True, exist_ok=True)
DAY_CACHE = {}

# This block also supports a fresh Colab session.  After running only the
# configuration, environment, input and weather-data cells, it reloads the
# frozen policy from Drive and resumes from the first missing checkpoint.
frozen_bundle_path = RUN_DIR / "frozen_local_settings.json"
if not frozen_bundle_path.is_file():
    raise FileNotFoundError(
        "No frozen policy was found. Run Sections 5--13 once before the "
        "multiday transfer experiment."
    )
loaded_policy_bundle = json.loads(
    frozen_bundle_path.read_text(encoding="utf-8"))
loaded_hash = loaded_policy_bundle.pop("sha256")
loaded_canonical = json.dumps(
    loaded_policy_bundle, sort_keys=True, separators=(",", ":"))
if hashlib.sha256(loaded_canonical.encode()).hexdigest() != loaded_hash:
    raise ValueError("The frozen-policy file failed its SHA-256 integrity check.")
FROZEN_POLICY_HASH = loaded_hash
frozen_policies = loaded_policy_bundle["settings"]

def clean_json(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, tuple):
        return [clean_json(v) for v in value]
    if isinstance(value, list):
        return [clean_json(v) for v in value]
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()
                if k not in {"prediction", "model", "opf_replay_model"}}
    return value

def unpack_policy(bundle):
    groups = {g: [(row[0], int(row[1])) for row in members]
              for g, members in bundle["groups"].items()}
    curves = list(bundle["curves"])
    fixed = {(row["bus"], int(row["ph"])):
             {k: v for k, v in row.items() if k not in ("bus", "ph")}
             for row in bundle["fixed"]}
    return groups, curves, fixed

def export_voltage_samples(model, day_data, day_sets, scenario):
    rows = []
    for b, ph in day_sets["bph"]:
        level = day_data["buses"].get(b, {}).get("level", "mv")
        for t in day_sets["hours"]:
            rows.append({
                "scenario": scenario, "date": CURRENT_DAY,
                "period": int(t), "bus": b, "phase": int(ph),
                "level": level, "voltage_pu": try_v(model, b, ph, t),
            })
    path = MULTIDAY_DIR / scenario / CURRENT_DAY / "voltage_samples.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)
    return str(path)

def compact_result(result, scenario, day_data, day_sets):
    model_for_samples = result.get("model")
    voltage_path = export_voltage_samples(
        model_for_samples, day_data, day_sets, scenario)
    policy_groups = ({'A': [], 'B': [], 'C': []} if scenario == 'S1'
                     else unpack_policy(frozen_policies[scenario])[0])
    compact = scenario_summary_row(
        scenario, result, groups=policy_groups, data=day_data)
    compact.update({
        "date": CURRENT_DAY,
        "final_relay_state": clean_json(result["final_relay_state"]),
        "initial_relay_state": clean_json(result["initial_relay_state"]),
        "tap_schedule": clean_json(result.get("tap_schedule", {})),
        "voltage_samples_csv": voltage_path,
    })
    return compact

def prepare_day_context(day, s1_initial_state):
    global CURRENT_DAY
    CURRENT_DAY = day["date"]
    if CURRENT_DAY in DAY_CACHE:
        return DAY_CACHE[CURRENT_DAY]
    set_active_weather_profile(
        day["irradiance_pu"], day["ambient_temperature_c"],
        profile_label=f"calendar_week:{CURRENT_DAY}")
    day_model, day_sets, _ = build_opf(data)
    (seed_schedule, _, _, voltage_seed, _) = simulate_svr_local_control(
        data, day_sets["hours"], day_sets["irr"], day_sets["load"],
        return_voltage=True, return_final_state=True,
        pv_eff_d=PV_EFF_FACTOR, load_mt_d=LOAD_PROFILE_MT,
        initial_relay_state=s1_initial_state,
    )
    day_solver = make_cplex_solver(
        CFG.cplex_executable, CFG.cplex_time_limit_e2_s, CFG.cplex_threads)
    baseline = evaluate_physical_ldc_baseline(
        day_model, data, day_sets, seed_schedule, voltage_seed, day_solver,
        initial_relay_state=s1_initial_state)
    if not baseline["valid"]:
        raise RuntimeError(f"S1 failed on {CURRENT_DAY}")
    DAY_CACHE[CURRENT_DAY] = {
        "base_model": day_model, "sets": day_sets,
        "baseline": baseline,
    }
    return DAY_CACHE[CURRENT_DAY]

def s1_runner(day, state_in):
    ctx = prepare_day_context(day, state_in)
    return compact_result(ctx["baseline"], "S1", data, ctx["sets"])

def policy_runner(scenario):
    groups, curves, fixed = unpack_policy(frozen_policies[scenario])
    def _run(day, state_in):
        global CURRENT_DAY
        CURRENT_DAY = day["date"]
        # S1 is executed first.  If it was restored from a checkpoint after a
        # Colab restart, recover its day-start state and rebuild only the
        # electrical baseline needed for comparison.
        s1_marker = MULTIDAY_DIR / "S1" / CURRENT_DAY / "COMPLETED.json"
        if CURRENT_DAY not in DAY_CACHE and s1_marker.is_file():
            saved = json.loads(s1_marker.read_text(encoding="utf-8"))
            s1_initial = saved["initial_relay_state"]
        else:
            s1_initial = (DAY_CACHE.get(CURRENT_DAY, {})
                          .get("baseline", {})
                          .get("initial_relay_state", state_in))
        ctx = prepare_day_context(day, s1_initial)
        set_active_weather_profile(
            day["irradiance_pu"], day["ambient_temperature_c"],
            profile_label=f"calendar_week:{CURRENT_DAY}")
        result = validate_recommended_settings(
            ctx["base_model"].clone(), data, ctx["sets"],
            groups, curves, fixed, CFG.cplex_executable,
            source_relaxation_exact=False,
            autonomous_seed_schedule=ctx["baseline"]["tap_schedule"],
            autonomous_seed_model=ctx["baseline"]["model"],
            run_qstar_diagnostic=False,
            baseline_quality_for_screening=ctx["baseline"]["quality"],
            initial_relay_state=state_in,
        )
        if not result.get("converged"):
            raise RuntimeError(f"{scenario} failed on {CURRENT_DAY}")
        return compact_result(result, scenario, data, ctx["sets"])
    return _run


In [ ]:
if CFG.run_multiday:
    requested = tuple(CFG.multiday_scenarios)
    if not requested or requested[0] != "S1":
        raise ValueError(
            "CFG.multiday_scenarios must start with S1 so every target day "
            "has its own physical baseline before policy evaluation."
        )
    scenario_runners = {}
    for name in requested:
        if name == "S1":
            scenario_runners[name] = s1_runner
        else:
            scenario_runners[name] = policy_runner(name)
    initial_states = {
        name: clean_json(_normalise_relay_state(data))
        for name in scenario_runners
    }
    multiday_result = run_stateful_multiday_validation(
        ordered_days=ordered_days,
        scenario_runners=scenario_runners,
        initial_states=initial_states,
        checkpoint_dir=MULTIDAY_DIR,
        frozen_policy_hash=FROZEN_POLICY_HASH,
        stop_after_seconds=CFG.colab_session_budget_s,
    )
    (RUN_DIR / "multiday_status.json").write_text(
        json.dumps(clean_json(multiday_result), indent=2, ensure_ascii=False),
        encoding="utf-8")
    print("Completed:", multiday_result["completed"], multiday_result["reason"])
    print("Restarting the cell resumes at the first missing scenario/day checkpoint.")


## 15. Weekly DRP, DRC and $\mathrm{FD}_{95}$

Formal weekly indicators are computed from the 1008 consecutive samples of
each scenario, not by averaging daily percentiles.  The same voltage-magnitude
and PRODIST expressions used in the daily screening are applied after all days
have been concatenated.


In [ ]:
def weekly_quality_from_csvs(multiday_dir, scenarios):
    rows = []
    for scenario in scenarios:
        files = sorted((multiday_dir / scenario).glob("*/voltage_samples.csv"))
        if len(files) != 7:
            continue
        frame = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
        result = {"scenario": scenario, "n_voltage_samples": len(frame)}
        for level, adequate_lo, precarious_lo in (
                ("mv", 0.93, 0.90), ("lv", 0.92, 0.87)):
            values = frame.loc[frame["level"] == level, "voltage_pu"].to_numpy()
            precarious = ((values >= precarious_lo) & (values < adequate_lo)) | \
                         ((values > 1.05) & (values <= 1.06))
            critical = (values < precarious_lo) | (values > 1.06)
            result[f"{level}_drp_percent"] = 100.0*precarious.mean()
            result[f"{level}_drc_percent"] = 100.0*critical.mean()
            result[f"{level}_vmin_pu"] = float(values.min())
            result[f"{level}_vmax_pu"] = float(values.max())

        fd_rows = []
        for (date, period, bus), block in frame.groupby(["date", "period", "bus"]):
            phases = {int(row.phase): float(row.voltage_pu)
                      for row in block.itertuples()}
            if all(ph in phases for ph in (1, 2, 3)):
                vll = _calc_vll_from_vph(phases[1], phases[2], phases[3])
                fd_rows.append({"date": date, "period": period, "bus": bus,
                                "fd_percent": _calc_fd_prodist(*vll)})
        fd = pd.DataFrame(fd_rows)
        result["fd95_system_percent"] = float(fd["fd_percent"].quantile(0.95))
        per_bus = fd.groupby("bus")["fd_percent"].quantile(0.95)
        result["fd95_max_bus_percent"] = float(per_bus.max())
        result["fd95_max_bus"] = str(per_bus.idxmax())
        rows.append(result)
    return pd.DataFrame(rows)

weekly_quality = weekly_quality_from_csvs(
    MULTIDAY_DIR, tuple(CFG.multiday_scenarios))
weekly_quality.to_csv(RUN_DIR / "weekly_voltage_quality.csv", index=False)
display(weekly_quality)


## 16. Reproducibility and integrity checks

Before a result is used in the paper, all checks below must pass:

- S1--S4 use `exact_fixed_tap` and satisfy the SVR residual tolerance;
- every accepted replay satisfies BFM closure and operational guards;
- S4 assigns a curve to every inverter phase;
- RL and direct assignments, active counts, taps, objective and runtime were
  exported even when the two methods are identical;
- every day-start relay state equals the previous day-end state within the same
  scenario;
- the multiday loop contains no OPF-identification or classifier call;
- all parameter values and computational evidence are present on disk.


In [ ]:
required_files = [
    RUN_DIR / "frozen_parameters.json",
    RUN_DIR / "frozen_local_settings.json",
    RUN_DIR / "computational_evidence.json",
    RUN_DIR / "computational_evidence.csv",
    RUN_DIR / "classifier_raw_assignments.csv",
    RUN_DIR / "classifier_applied_assignments.csv",
    RUN_DIR / "s1_s2_s3_s4_summary.csv",
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing reproducibility artifacts:\n" + "\n".join(missing))

for name, result in scenario_results.items():
    assert result.get("svr_equation_mode") == "exact_fixed_tap", name
    assert result.get("max_svr_exact_residual_pu", math.inf) <= SVR_EXACT_RESIDUAL_TOL, name

def audit_state_chain(root, scenario):
    markers = sorted((root / scenario).glob("*/COMPLETED.json"))
    for previous, current in zip(markers, markers[1:]):
        left = json.loads(previous.read_text(encoding="utf-8"))["final_relay_state"]
        right = json.loads(current.read_text(encoding="utf-8"))["initial_relay_state"]
        if left != right:
            raise AssertionError(f"State discontinuity: {previous} -> {current}")
    return len(markers)

if CFG.run_multiday:
    print({name: audit_state_chain(MULTIDAY_DIR, name)
           for name in CFG.multiday_scenarios})
print("Public-run integrity checks: OK")


## Interpretation boundary

This notebook separates four claims:

1. **identification:** the relaxed OPF produces physically auditable candidate
   targets;
2. **implementation:** the targets can be translated into static local settings;
3. **mechanism:** a spatially selective setting can change the voltage observed
   by an autonomous SVR and thereby avoid unnecessary reversals;
4. **transferability:** only the frozen, state-continuous multiday experiment can
   show whether the setting generalizes beyond the source day.

If Q-learning and direct reward maximization produce the same assignments and
electrical metrics, the paper should describe the method as **contextual
reward-based classification** and should not claim a demonstrated advantage
from reinforcement learning.  Likewise, S4 separates the value of individualized
Volt--VAR curves from the value of sparse spatial selection.
